# Fig. 8(b) 저전력 다이버시티 확인 (v35)

**재학습 없이**, v27에서 LDPC 복호 후 BER이 이미 0이었던 28 dBm보다 전력을 더 낮춥니다. 1개 수신기로는 복호가 어려워지고, 4개 수신기는 복호를 유지하는 구간이 있는지 확인합니다. 그 구간에서 4수신기 BPG+LDPC가 FSO-SC에 근접하거나 앞서는지 평가하며, 결과를 보장하지 않습니다.

- **고정:** Fig. 8(b)와 같은 16QAM, 맑음, 천정각 0°, 지향 오차 19 µrad, Málaga ρ=0.95, 배경광 복사휘도 10⁻⁴ W/(m²·sr·µm). v25 가중치, 원본 v27 BPG 비트스트림, v27 coherent 채널·v3 DSP·LDPC·MRC를 유지합니다. v34 위상 개선은 적용하지 않습니다.
- **비교:** FSO-SC 1 Rx와 BPG+LDPC 1/2/4 Rx MRC. 송신 광전력 18/20/22/24/26/28 dBm, 기존 v27 평가 영상 중 v28·v31과 동일하게 결과를 보지 않고 고른 32장. **전송 평가 768회**(28 dBm 겹침 128회 포함)와 무채널 기준 64회. 새 독립 평가 집합은 아닙니다.
- **판정:** 같은 영상에서 수신기별 복호 전후 BER, native BPG 실패율, 심볼 Es/N0, Alex LPIPS를 확인합니다. 복호 실패를 1로 표시한 LPIPS와 실제 회색 대체 영상 LPIPS를 구분합니다. 4 Rx 대 1 Rx 및 4 Rx 대 FSO-SC의 영상별 차이를 같은 영상끼리 묶어서 평가합니다.
- **한계:** 논문 값은 Fig. 8(b)에서 읽은 근삿값이며 28–36 dBm에만 표시합니다. 28 dBm 미만은 별도 저전력 확장입니다. 4 Rx가 이기더라도 논문의 동일 dBm에서의 교차를 검증한 것이 아닙니다. v27의 독립 분기 페이딩·이상적인 상대 채널 정보·분기마다 같은 개구 가정을 유지하며, 수신기 수가 다르면 난수 소비와 수신 관측까지 같지는 않습니다. 같은 원천 비트율의 비교이지 FEC·패딩을 포함한 전송량을 맞춘 비교는 아닙니다.

## Colab에서 실행

1. GPU 런타임을 선택하고 이 `.ipynb` 하나를 올려 **모두 실행**합니다. 별도 `.py` 파일은 필요 없습니다.
2. 아래 원본 파일이 Drive에 없으면 NAS에서 원래 경로로 복원하세요. 공개용 `*_aggregate.json`으로는 대체할 수 없습니다.

```text
MyDrive/FSO_data/
├── GEE_test.zip
├── stage2_error_range_to1_v25/
│   └── extension_cc8ff5b866b4c901/
│       ├── stage2_error_range_to1_summary.json
│       ├── configuration.json
│       └── resume/                          # 원본 폴더 전체
└── stage2_paper_sweep_v27/
    └── frozen_882a885033790ab2/
        ├── paper_sweep_summary.json
        └── sources/selected_native_bpg.zip
```

v27의 `cases/` 폴더, v31 결과, ImageNet·NAIP 학습 ZIP은 필요 없습니다. 코덱이 캐시되어 있지 않으면 공식 BPG 코덱을 준비하며, **원본 BPG 파일은 다시 인코딩하지 않습니다.** 로컬 여유 10 GiB와 Drive 여유 1 GiB를 확보하세요. 결과는 `FSO_data/stage2_fig8b_lowpower_v35/`에만 저장합니다. 중단되면 같은 GPU 유형과 런타임에서 위부터 다시 실행하면 확인된 기록을 재사용합니다. 같은 환경을 구하지 못하면 설정 셀의 `FIG8B_RUN_LABEL_V35`를 `run2`처럼 바꿔 새 실행으로 시작하세요. 원본이나 이전 결과는 삭제하지 않습니다.

끝나면 **`fig8b_lowpower_v35_results.zip`** 하나만 보내주세요. 수치·그래프·CSV를 묶으며 원본 영상과 가중치는 포함하지 않습니다.

[원 논문 Fig. 8(b)](https://doi.org/10.1109/TCOMM.2025.3562356) · [Geisler 외의 코히런트 4수신기 실험](https://doi.org/10.1117/12.2256581)


In [ ]:
# Colab dependencies only. No training, API credentials or checkpoint edits.
%pip -q install lpips==0.1.4 cmake==3.28.3 matplotlib
!apt-get -qq update
!apt-get -qq install build-essential yasm libpng-dev libjpeg-dev zlib1g-dev


In [ ]:
COMPACT_FINAL_INPUT_LOCK_V26={'author_original_test_set_or_receiver_claimed': False,
 'base_seed': 20261107,
 'bootstrap_samples': 2000,
 'bpg': {'bit_depth': 8,
         'build_recipe': 'libbpg098-cmake3283-v2',
         'chroma_format': '420',
         'cmake_version': '3.28.3',
         'color_space': 'ycbcr',
         'encoder': 'x265',
         'release': '0.9.8',
         'search_mode': 'binary_checked',
         'source_sha256': 'c0788e23bdf1a7d36cb4424ccb2fae4c7789ac94949563c4ad0e2569d3bf0095',
         'speed': 4},
 'chunk_size': 25,
 'clean_methods': ['fso_sc', 'bpg'],
 'clean_reconstructions': 400,
 'conditions': [{'background_light_radiance_w_m2_sr_um': 0.0001,
                 'group': 'clear16',
                 'launch_power_dbm': 30,
                 'modulation': '16QAM',
                 'pointing_error_urad': 20,
                 'scattered_power_ratio': 0.95,
                 'weather': 'clear',
                 'zenith_angle_deg': 30},
                {'background_light_radiance_w_m2_sr_um': 0.0001,
                 'group': 'clear16',
                 'launch_power_dbm': 33,
                 'modulation': '16QAM',
                 'pointing_error_urad': 20,
                 'scattered_power_ratio': 0.95,
                 'weather': 'clear',
                 'zenith_angle_deg': 30},
                {'background_light_radiance_w_m2_sr_um': 0.0001,
                 'group': 'clear16',
                 'launch_power_dbm': 36,
                 'modulation': '16QAM',
                 'pointing_error_urad': 20,
                 'scattered_power_ratio': 0.95,
                 'weather': 'clear',
                 'zenith_angle_deg': 30},
                {'background_light_radiance_w_m2_sr_um': 0.0001,
                 'group': 'snowqpsk',
                 'launch_power_dbm': 32,
                 'modulation': 'QPSK',
                 'pointing_error_urad': 15,
                 'scattered_power_ratio': 0.95,
                 'weather': 'snow',
                 'zenith_angle_deg': 0},
                {'background_light_radiance_w_m2_sr_um': 0.0001,
                 'group': 'snowqpsk',
                 'launch_power_dbm': 34,
                 'modulation': 'QPSK',
                 'pointing_error_urad': 15,
                 'scattered_power_ratio': 0.95,
                 'weather': 'snow',
                 'zenith_angle_deg': 0},
                {'background_light_radiance_w_m2_sr_um': 0.0001,
                 'group': 'snowqpsk',
                 'launch_power_dbm': 36,
                 'modulation': 'QPSK',
                 'pointing_error_urad': 15,
                 'scattered_power_ratio': 0.95,
                 'weather': 'snow',
                 'zenith_angle_deg': 0}],
 'data_manifest_json_sha256': '140d5919305f3592d144ed4a1a8f8d53dcd19e170b212416591b454c0ea4e7e2',
 'evaluation_count': 200,
 'expected_source_stage2_updates': 1375,
 'final_evaluation_selects_model': False,
 'geographic_independence_claimed': False,
 'ldpc': {'k': 1536,
          'llr_clip': 30,
          'max_iterations': 50,
          'n': 2048,
          'normalization': 0.8,
          'rate': 0.75,
          'seed': 20261028},
 'methods': ['fso_sc', 'bpg', 'bpg_ldpc'],
 'optimizer_updates': 0,
 'physical_reconstructions': 3600,
 'primary_lpips': 'alex_v0.1',
 'primary_model_selection': 'fixed v25 wide1375 selected before new evaluation by prior development3 '
                            'mean VGG; no test-dependent checkpoint switch',
 'profile': 'coherent_receiver_extension_v6',
 'rate_matching_scope': 'dataset mean source bitrate, not per-image constant rate or equal channel-use '
                        'budget',
 'rate_tolerance_bpp': 0.0001,
 'secondary_lpips': 'vgg_v0.1',
 'source_bpp_target': 0.21875,
 'source_branch': None,
 'source_branch_discovery': 'unique immediate extension_* child with exact pinned raw summary SHA256; '
                            'verify child name from common configuration',
 'source_checkpoint': {'bytes': 3250750999,
                       'configuration_file': 'configuration.json',
                       'configuration_sha256': 'bb0e546ded8dc5e6483d3f5f6d0b0a18dec6758ce1a8a535130d19e571fe590d',
                       'file': 'resume/stage2_resume_a.pth',
                       'model_state_sha256': 'c383d97f27ccbf12620a629b7666a7b63d62d8c2ca78f70d757938ee309f2f09',
                       'sha256': 'c4443bd14c0e3292e4a456a715e5de27a42855097a46244d93fc573fb656f121',
                       'updates': 1375},
 'source_code_sha256': '4a2f1764f6c00002335070dff6bd21def8d59b9557cc17775e69ae3eb3ea2324',
 'source_files_modified': False,
 'source_root_name': 'stage2_error_range_to1_v25',
 'source_runtime': {'cuda': '13.0',
                    'device': 'cuda',
                    'gpu': 'NVIDIA RTX PRO 6000 Blackwell Server Edition',
                    'torch': '2.11.0+cu130'},
 'source_summary_bytes': 5049075,
 'source_summary_file': 'stage2_error_range_to1_summary.json',
 'source_summary_file_sha256': 'd8d68f3129c91ea5e71f60c2ee64e700e8c0dab044dabe6ec5c729ac0aa3e9d1',
 'target_mean_bpg_bytes': 1792,
 'total_reconstructions': 4000,
 'training': False,
 'training_inventory_sha256': '04768543c975e0627eceb5615024145dde6fc1f4c636cfa614587a55fed3b0b3',
 'version': 'fixed_wide1375_compact_final_inputs_v26_1'}
COMPACT_FINAL_DATA_LOCK_V26={'all_file_sha256': ['0066845163f38a25828bfa1e29516b0a472661a09893d66fdf611694ad8f8284',
                     '0084f8c888def7ca8e5150edd21a7248b1feb0ba3ef9e1d0b3c988f46501f818',
                     '009bc6b1df70faa550bb399c021f4f7e9fa664eeab43d7c364d281aa28421946',
                     '00aee9401eab0787c80bed346259bd59328a10d561dcfa3ba3bb9c3c9cc3b95d',
                     '00c9a055214eedc9d0ef7dbf675b2a5d6c5dd1f936637b9052affd5f1b37ca0f',
                     '0127d7350d9710721527acfacfed0492dc9bacb0cee58108f994bee8cf03fec3',
                     '014aeb0ef7d76df016065061c6c381bcbbbd6fa8f4e287cc6fb400d6aa0cce4d',
                     '017f1ca9cf29c2ba58943ff61f089f2fbde10ca6b7327ad05283f4b2ab12f20a',
                     '02cfc775aafdb1fdd0382189d3cb813f2c890a1fb6e176d907fda86ab905a684',
                     '02f2301e2a66ba274c929367147510d2429dab0907f78acb13ced9e28ef2f1bb',
                     '030c5987e036da1be5245fbc9385db5a7cdc09f2743d5ea25478c503e0cb68f1',
                     '031288bf4dc636a13973e5523dc48f03df93d0d8ae0da94a4101cb226bf23c42',
                     '0318784a8740a6f2c4539e77cdcc938168574309e179b3a3ea56c77336802023',
                     '031dba86bd4ffa1a1d51960b39da70c5ce50b3edf183b70dda9d3218f905187b',
                     '036bca3d81f26090a944f59a858a923d5975c5c5176d5e010ab25c5a7715d3f0',
                     '0371dd8e6d255c728c45103c8adc87fbf93770559b1b2534f4fc848d1a58b9ea',
                     '03b71f226ea44462a1ff05875a802b60fe89c41a952f879979956d32b8cc57c6',
                     '03e2e9232c3afe0a759861d9ec32a74c194bd6d27997aec69db31be536f4a322',
                     '03fbbc01f1bcbdfa658955e88e044dda30d316ba3075bcac14bab0ffd6df2d61',
                     '040b69056ff6a04a13d1d6bb91c0c479efe2ac4f8fcb60efb0a83975496e750c',
                     '0428ba35ff505fa6cedc57e046a455ade950dcd487cdfee7a94743c494a68d2d',
                     '04af976817fd65aa924e2147b45d6c97c8134b18113053f41cce8c07566440e0',
                     '04dff21d7d781ff8ea2ed80f6da1a165e7797f9a398b4fd930bb935d84fae8d8',
                     '04ead216e16cdb9c6176d5c581c10dbcdf3a01a98871533e0ca504ade94860c3',
                     '04eefd04b4ea48004fa383bcbf35aa751dedaa95efab605c77674021bc794f71',
                     '058d5c353be680ab65f033a8962772325d98cd96eafb59f6e70c077576f91ed4',
                     '05b5ec1e4f49623780d4dd121309ac91579e6aa5325b9f337aea24ea143648bc',
                     '05c79664ff9424c0b3c30f9ec2ab084687d490fd6d0f4948b2e51176f923acab',
                     '05e6ba245ddd2e59a3e021823ef935ebf49f6a273dcc3f2679902f74fa31940b',
                     '05fa9cbc11ca5d438f4a7fd8f2f263f85f0f014f034dcc099a711dee58022289',
                     '0679c8b0537a146003135e865121b10b489c18b661562795074a4c46987f2edd',
                     '074a0104e1e1f72a5e989f7ffbb8331981538a168f494e2a75e9b18aee71013b',
                     '075cf56c692f03d94602f7174152164e29f6fcc3e213fe20b86394d29c044af2',
                     '07b5f1ea62e221c7dd148a47a9c62d11ac460bef909ca3a737b5c395221fb146',
                     '07cd670dd70670e173fd703dc00c298cdf81cd60cbe5bb1884e4c281a89df151',
                     '07fcee7db0fa51cb125bf301e6eba10b263abaaad08b4949d695f088b52b1e77',
                     '087a8df98839fd9ccf3cd63df3817899c260d6429424e093386cef1c4c65cb3a',
                     '09143054231e9df0f34551bc20ea40775fed5b60e64f6f9c7292ec04e772ca59',
                     '093a0130893752bf4785ac38e567c9d799a365483a376c158f3a2561fe6170c6',
                     '0999f9393820e8917812ddd25413ba91f6f152fbd1220acb379a7a5474810d24',
                     '0a91dcc15c839e62816387cb8262b74a81f9143cc5baec723099d777c9469768',
                     '0a9239f28d97f43c0ab7e9a16e5ef50731094a71f9408b194519b5a616809780',
                     '0b3376578021a541194d1f0c26c97e16b508d706e0fdc0e53b2ef7bb989aaf5f',
                     '0b34c3f737905d058400a3b299b7853c27e4cbf302bfb4f2fcaa09047cadfac8',
                     '0b7dffb63da06365e02956ee0c0520dc5ff8812e339270410d5b9364f1a9eb6b',
                     '0b9cdd6af7411c4095d9840e3ee8ed559daa6dae5beebc93c16cc51d15c63506',
                     '0ba797f0b2beb834faf4dc70cd0cf0d872817f03b8f49b14bf536e32eeacf785',
                     '0bbf9f9de7802db5ce31a6fa718abcd6ba30789ff1500b5dd01b7e9da4768321',
                     '0bdb354e03f3f76aefbf2e28dc71e5eb60e8a8af095dd57c7e2a0320cc83e6df',
                     '0cb426ae32adfdc3288ea3c4a9a10b2c123ab3d4664a32b02f052e86f1bbc4c8',
                     '0d34e9157d178298911b0240717d824589011ebc35a8909da1de3ee1a9897a94',
                     '0df6c0ecf3cb3f92cfdd8940843309292f5f8fd9d60dec27cf56b196d43827ec',
                     '0e2ddf287e7584f51552daf7d41f4d85e9cd406da782015f5031a95cd95871a0',
                     '0e408be1135a55441fb76763bd9a2aa751b847f4a0f64a9959d8a7e3700eacc7',
                     '0e5b291376a58a933f15e5ca8adbd8987ec600c138d1308bab9c147416162e19',
                     '0e8ea29d040315ac021abec592b9f033b759adb5bb98570d1d01e46539d1b62f',
                     '0e9f754a8e12b5e0f56d4fe577ec1be800f8f84bc196db18144a989daac35552',
                     '0f328a45a3cf2bfb5a4cf60e13ef3dd6324a6af2b4b15a03e68dd864136c7984',
                     '0f38a11d51b8b18807a2ff0238896414bb4334639ca0acf36695fd55ffd80b4e',
                     '0f9fd78f1a6a56b61c0a7357fc27f8ddbb4fcd69346d8459981485b4389b165f',
                     '0fa3fae44c1eb015fe9adc1c7dd2da907acce30ef080449e5b1d53b71a9d843b',
                     '0fdda50d89fde702209df478033b4da09af4650288ecb4d364ff8af666dfea37',
                     '0fe27584cc2a358f288b63532fbe7a68d2596fd9b8aecd58467f0a982e7d0cfb',
                     '105fcab86f503a97dfef2f8f9a3a2cca2641306714df060f553ada3acd96c8dd',
                     '11305e2605ce0d223099b48600bc36d20a6706314f69c859b893bfdc25d05431',
                     '11cd45d44877e9aaa9c41984f0d1651c7321b5d7049ee0d2c21cb7d3582729cb',
                     '12383bd875b6a9639e103cfbe7e31d835157db1d1dd971bedf34c70546031a42',
                     '12535b48c89b2d7a339e1fa11c4edb84ec232f5d565200a06e5dd42413a70dbd',
                     '1258d9ee5de9d744f782f81570c18b0a9e8d8c407169977d57b98d70a4d246cb',
                     '1269e5f1b5b64c8442881cf6aae6380d387eced7312ec5e89d677466967ee3d2',
                     '12bfc81c2d82674c39ca0e07a1ab4075cd53e1e45059574e0ffa7db800dd1319',
                     '134b0136fbc951cc55e75a08a9335a524835aecbed8acf91332a655fc1af8c97',
                     '136955a3177da1c55d7623487904f0d0fa75568f1f4f4cb10bb833e99d816063',
                     '136d362b7f3df7ad18a07b0778c9478e0efee3157baabcae0afea5da412fc9d6',
                     '13bc2e029468235b81cde631b0475fa3405cb3ae076088d58937d66bdf3ac3cf',
                     '13c0f19424fae5c97500c90cff9e11ea5ee76a3a8623ea0e26f315f4d8e6cbc3',
                     '13cf06d91aecfbbf8ce84553c0cee40fb0a2f8a428aa78434b5371780686a56d',
                     '13fbe49abe6eacc07a41fee4cf891a280ce0ea3b02097e8f51942e855673af0b',
                     '13fce8aedce5fab9133488366c17473423687df9ccd5682f038ca815b008c817',
                     '1403729748e4eb03c9a7df679357e3788438e5b8ab6d49e351f6d993cd4e1319',
                     '14078b172fc8858b0b638c731fca5ac0bd40a18001005529f22c0525e3af18ff',
                     '141fe37ce0253cb31db8fd9b604c4c9673f5b7de9aea16d7c00c4758aa6f35e6',
                     '1422861477e119b080e3e93293b9f45cd0ce20067e9beceb1bac42ddb083428b',
                     '142f21c6e71616efbd5dc9ea64b99c6d0f07bb9c18a871a95b216715b8e1c827',
                     '1490bfe18e8954e3094927b5f38c66768ca55e7bbaa947db2b96ad94d28071e1',
                     '14b62856e863f705d95d59c8fc30cab9e9a2a3e07be3c5ea248fe841e0088444',
                     '14d704549005045ccdef7464bad5df4f81f8e3313d92fafad26d551d1b2e02e2',
                     '151bd74973256bf122a0260db34c96e4e102f20e3bc1fd0a6c93d049c7654d97',
                     '153b6bad06d0fde37a8defebaebada3a0e10926b7fceb967b944cdd1783bb603',
                     '15747090059d549da3c39e1f08c4928adca0378b0a00559922556cd3d0e64b45',
                     '1628c2c04b97b49be78116becb52a0c881654de00604da82928aa7b4dae90013',
                     '167b3d90ebbffc9837a5cee597b72b178c524652090642a58c0286621737dbb9',
                     '16877e5a83f8b18b70b3a9e5c01ba954cbac334efad1b5f5a6c49ff69777e96c',
                     '168fa335f79d34804812bdc3379df3ca3ecdbb470530caed5dfa028250ba4e02',
                     '16a21a2f6f98374ea4c2bad76692ceaa1c493f7e3ef1e494ec1059e6b17617c3',
                     '16b6565dc760a3f5ff57369247163d9c7dbfc5b7bb9446d8c65e480caaaa9f73',
                     '176260b73073aae1dcb99f05af28555b4682c100e063d4d4c45aebc6bb209a48',
                     '17836231febd88d85f1fe707ed8aff7c801fc3059e1579d9460c0322d7340b8d',
                     '17d71d00032256f7aa022b26da0cb06267b554a692893213b315feacc39d6ac5',
                     '181bd43cadd95ab231952fc078cacfe2384fe1125f877b4370147dd2685813df',
                     '1820bdb61280be77c3e6eb491953711815de08e201c8f6b2d6a9b65ce2b2c1f6',
                     '1832519d80c865562a7f448d63b365f9289a8d63764c5c08766cf9886f9d7087',
                     '18b6af8db9f87bb9fbeedc355792c117a12f04eeacc3e51e9e239917e6e7e7ae',
                     '18cb13aebb80e32029fe77bceb4a67a124f779e24fa87231288a59a30873b731',
                     '18d63182f56c67e61fc78dc3179ad89ab7e443d6c95f431721202fe0129de547',
                     '18e95729761855bd29115435ea4aa3720418e954fab04dc1996a423317d5ba75',
                     '191224e97d30f7ea88221396753fe3b179640d179f1248addbbd5e1ecafcdd75',
                     '1932b91f7bbe4cde3fe574328ab25ee7c7c92f8a1c8889ade9a5bcce8bb4a6fa',
                     '19ac7efc4f73f3596bcc209c2aa678c292aa11d5bc6366e835e2044c8002bb78',
                     '19c4310b6e09bc4b22082dfb82e88d33c8fd53daea76c08bbf32c099c93e22e2',
                     '1a012e6a3fa504fbf634665a7c8294daef728a218d6d8e5b3a51da49498dc017',
                     '1a4b3599f15372469347e6a3ba262d663076626d90890c31417d50e36a856149',
                     '1b8930c4e710f3484d1b6bbde884a4c24f83cd8b77a694a59716e849ee506841',
                     '1b902ece7dd95344dc5b01077ecce2550ab8b0535d5a0b885ae4d2db86964f3d',
                     '1bee85ca8055cdc8125538e865caa59bcece6aa90a149758d4bdca3ae855835e',
                     '1bf9feae32fe5ce0a380a3d37b4754ce1b18b63d05d6bb931854519ea1a661b6',
                     '1caaa04d1678b772bf5bdeee0982880657cdc8741609a1dd7beb4a67d0212e55',
                     '1d36b5bb3a75f95e0c2d624ba5acf329117b2a331f7f83624b0a819966a5e605',
                     '1d65cbf7f778fc51185fd3d4a883e44d60ae3df2d0333b55e31d90664e9889fa',
                     '1e0a49b6a81176ac9096f259f96258490442029f298a8dd6943c4b09076ddc7c',
                     '1ed1c79090c93e3f3ba4503b11b19d8f4a027c9ddcb4394fc3b6859584ea80f8',
                     '1f2b852bde28f71a1e9334e223dc81d717addd893e7870a5295fc78d33df212b',
                     '1f736a38f947b4863750c3508765ee1ab1d55d48398c6104f29e862a8f62c4d0',
                     '1fe265ca1362508807b461ed6103e4b815323de60f70cd2bfa11fd4640e0d808',
                     '20aaedee6d4486a09ff247ad24d80a562f0be2a338e35bdf8f82740d1255b0cf',
                     '20f7ff1137e25b4238efe36281032f8e219e48ff2bec837c02d6941c83b2c46c',
                     '20ff7ce26d56fdbb1f57239cc7f13d9c7187a2fe76d8d41fc4040cc67b793228',
                     '210cdd5b6d4be8d597547d69766af2e6e5d5087a3a9e4bfcaaaafc116451152b',
                     '217be8d51db955e31e325acf1f7c1e4e6032cce31647353d113b931128ffd3b9',
                     '217c563dc3fddfb462f4abbb7ccd872ded097413301f0fb166c180ffaf10680c',
                     '21977530aa79b9e0e499784b1827e6b75877f1f30b3debd2419377a9fccbeefb',
                     '21a1ca36770cac71ae26295310db18513925786dbf6fdc9c8e4ed8f99dd50154',
                     '21bff1f8c093cc63fead0b7638ffc9940ae579d8362b986fe77cf6835abf8c67',
                     '21ea5e7d5d4f6a52ae5b1775eb83bde484d36cd1c1cf08fd4cd6cdc01688cd03',
                     '22790101b93cb7b9e5f4136c6ac0ad559271f5d3130e1e488e863ce1ad5ec16e',
                     '228c1a35ffdf65cb9a368379ed028cbe98e0968d4a827522c6d7ccf09f574eb0',
                     '2295f88b3afc6eb315197029301c0611399638894e36d518de076c7c95112cf2',
                     '22b050d12e7582bd0b6b6a39bb5cdfc3b5b23ce3b376b88d5871877ec6b4b5d0',
                     '233bca9c796ea4e12411240ae6efaedb6187ccb4257cd2faeb825d758f72f7a3',
                     '234b820997661167ff7a92381ef0c3a8876b900b6890a24a54b748f290a4a2d5',
                     '2364459e6ed9584b7eb6513c29c655ea7db9debd65722a5f441e86b6f9000b72',
                     '23cb62fa2e1dc3c9dfd92c1190e7cdfa51889b067045a01de9db28b379171d85',
                     '240c1357b7891c548e77f9069bc09e03799c79e085eba7d24cfceb00387475e3',
                     '241f921d4cd85481e077b18605fb2d2b1ae4d52f954c86a8a743e603af0b75ec',
                     '2420281d08d9fd14717298d71717302f35a9299601607e41b548b37011e86c9f',
                     '242f9140b8650ed6065382d89b4df19bbc281fb8388f7e76b096daa17c6efe01',
                     '24cd687a287375d73d5cf90864e7c0314a180fe81af899cce5fe86ddb7f26d64',
                     '24f79f0a5e3b82171a9702b32f698668549ec806f5df4eae2f2eec402caed843',
                     '24f9acbe34a04732264fb9105f54cc565760bbdab686faff514336de0fe0cb3c',
                     '2591258305dd07328766dcffb7232b23ae78a88a61d9a878ed913105cd21c3e8',
                     '25f4bdb171cb66d609c3b41e5af6d9fe6c4b43a20478fa8817e4204ab33634e6',
                     '260c6068d56fddfecdaa367fbf870801776df37aaa2d954a6a0ddab092769a0c',
                     '263a7ec3ee646b32ca436cae9a4baeddeeeb3cff8b98c67aac39a8b099a6f7b9',
                     '26b9c2917759aa9d90241b26b7fc1de4e64ad52b1a6307a5ba33095d054e19c2',
                     '26c32b6b61f258765c735ccaa8f3377fd4aeffd8ab6c4d9eb1ba72ac34b7b73d',
                     '26fcc503337caf416882e5de6b7762129877c4926d72927db93c6dd3b3aecd6e',
                     '2732d8b5234671375ad29100adbf7bf36ca52500697e384398b5ed24d9eabdd4',
                     '2783792192d71135c704a7d48f4335e8c3b43673df6f77673edb3561afec80e7',
                     '279cc38806a4e37f44a799845db932d4839b34e7b14b933146bcff349661d78f',
                     '27a780b7a4eb0ca137c56aa2261ae63b279279ca452565fa8cbced8a40636ee6',
                     '27b8b43eae7602fda0940a6c698b675fdadf8d201f56146dcfcf32c5e7bcde59',
                     '27eb5e3c07fcc322a18a8a5a63fd03877e16b86c0516e6fe4cd656da5e280c05',
                     '27fd2d7e1e17bef7a17b67a751f6e48dd6d58c870df93fe72935a67224a9cf8f',
                     '2840069ca725acd3e097f15bd532375ddef326880f3683f2054f118e70ef98a4',
                     '284fab2961066535c65cb962f88aa86b463048395b305e1793c4849dd5603156',
                     '2909154fe14311910e85e24c5abfe8fd9523d819071f3f6556f2d910ca1b0f90',
                     '298dcc57143531d79d9dd82730cbe360c46766a4f573dcac764e947382556a7b',
                     '29f63c30a424c41864cf024bff7badecc8a25f141f50236c7eaca44c851d148f',
                     '2a3a2a6b2a5b321c19baf4de1a0431816c9e1299e1a67aa3bf1f2fccb447a6ad',
                     '2a40eac16287efd2e163305ea3a4b517ab642c23b229cc77ffdeb6f600e4e544',
                     '2a4f80a0d5d95539103c5c67136ecde9c3f1e62fe65fdcede56ef22628e8d091',
                     '2a64cd8728cbc2950dbcd72f61e3c88025e4892157c106c5bf1ae4d43bb7335e',
                     '2b19544397470829dab6f7a402fe4196b2654d8140465715e2a7e23a1b225a37',
                     '2b85d6901691431bde7b2cf2e1de969ffccdedacd63a1b2cd1047a7ebc69e10a',
                     '2c1c7e342534fd04b325eb242db82ea8d9cd8c58e996676330905cb2a45a59e7',
                     '2c353cb2f84bdc3b46be2025a09349b66886bd7a6e21764dafabf2ef73cabb36',
                     '2cb2bf160f187ff6be6a300886bbae7d855388251fde410f3ea962f8dea672ca',
                     '2ce23fdc68c43b25b9169ab09fa124885e5e4e0dc2c90cf875032a5950b5d8e8',
                     '2d28a2897efa0b74c752b4287e2facee710509c8e840aa849620a1a2b3b37104',
                     '2d90ad29da4cc1144a890a98c668f275336330d6a7b6ec3aebe8ea0b71b0c26c',
                     '2da7cc2b46d1630701efddaafeb0d17618649382f2e2adc6ceca325d208dab98',
                     '2dadda6e349ab535222e8937b573e325b662300c7dd7ac77782ab0ecf93b8525',
                     '2e63938b2fcd265175ae05748f4e266ac8dc01bd2a38294bd42d515f3a21ac85',
                     '2e8151a6bb8aeec62a83a29bdb93303bb7265cb28ef5deb9fc2bb6f933e4801e',
                     '2ef5e210eadc3638aecb896f3c3e906c284585ff0a1d1476b94ad3333feaf314',
                     '2f94a3f3b360d0847336e6167d47d7d3f81d8d36b34c970dcd36befccead0eee',
                     '2fad80f5c24928cfd53dce72ff07573f693291059baf3d78a37b139b195b8796',
                     '2fb3107b17ebb27324d93eaea065b85dc2f4538c305e8f1045df56fb583f7909',
                     '2fe7c85b683d6b09700bfa1d1e2ab63e696e53621b1255dbffbb41789ad50a18',
                     '3030f8580b36db8eecb0fbbc889a0ea673a3acad45836dd991f98e20e9e11517',
                     '3084c85b7fa890e77560bc72e757fadae863d9d32bfa30b3cb28cd74188ebcd9',
                     '30c131076a387144a12eea1f98f9a4fca2f5af812c0e4a57d7325adff5fb0c49',
                     '31bcd149f49c97e3fdac5b661e2b6b3cec17a132623ee307c36a92b3d39228db',
                     '31d7041f2573fe087dd00a9faa0f6c372fb958bea8b15a956af218977e933f4d',
                     '31f704c359ea51a9f5f599c7f331d0656792ba4f3fd6417cd35bd970d19c608b',
                     '321eabc5858dd5f95463c798407ba16a9a26459943269cdf0228ba025f20825f',
                     '3220a60a4764025feb88510610f61f0ba228041e66a56f3dc4a332ce1b9a884b',
                     '322bb75b4f5a05b0eba7ba80b796474b81e5f47951d18342e2b9223b137a2c46',
                     '3238bd3fa558879d98a3512ba61046f2f34272ae9319d7ca12a2fca67de1aedd',
                     '32955221a7888fae9aa1ba5a7bab8d6ff03e326ae972c3594cf4e49a1988334b',
                     '32aaab923cda5b9aa647da5818096ed7c6f2c9f857bf5c04f6238428fcd9bf4d',
                     '32b668e9e325e12ef684de614ce85d30fd2b2dd3496fa6a196f3f70270a65728',
                     '32cbf015d1710168320a9866696e2746807f2793b602a3328d18583cb8ba45ec',
                     '32dc68b3f5dbaf39eaa39b34639d7e34099b8cce9adc7ff75cd16be78db66878',
                     '3327f2a690c915ced7bcca0d4f81597dc24daa4a7a2c454826c46fb70695ab39',
                     '332c392379f7a2307826bc57368d861dae2c596475f3193d861ca586cf3ad873',
                     '339b927f038ba1c46dff3a7497ba07609578b8a85ee910819aebe439770f57dd',
                     '33aa26f99805b56707861c3c359796c18fc6f1a8b434cbe8dcb6e9c02901e00e',
                     '33b4869594fcfa5d1e19313cf9d9e79493a06d4e072464a4670c62c3b401d701',
                     '33fd7f7a26c5859a578948cc837d6b02038d763558c4c5c205dac8eca16ffe3a',
                     '34e30aa6fed35f75372c32706537f0c8053da8706567a99639742d36e23b297c',
                     '3558079ab6c1f10699922b3dae791721a79dea43128312de14bcc85f46d9165b',
                     '3567e09c58476c5c793bfa9db08e16833c0efcb31a723b11dede08b2e383420d',
                     '358ad67805609e1377ee55dd68ab5389b3659855b7f06c6274f3f54f8a7c6d4e',
                     '358bc297fcb7bc33ac812029f1b6917c9e25618300ed46b7b93c68eab5ac2b34',
                     '35eff67e26cac12bc12c75cdaee75d76b3dddb27ea3b4a0eb259b300addf3633',
                     '35f5b161b03998a66c750105c9c9d01a326c076f7f41ef5ec730ba4a6e9e1404',
                     '3635ff890769e4f426eb238fc1097748d7915d0ddadc0d01fe3e124dc6b36718',
                     '363c6a14c10015b31f753994b513daa3dde497fe485b75fb90ab2695f49d873f',
                     '36559d2545c16bb7eca9ee449b893c48596d2114b7b000a1819e16d4035e826e',
                     '367126122831aa7424a9498c641885faf8461456fd6e86b1d8b915d920401d81',
                     '3684e33c3883dab27bbab61eaec02a028c332b10416c3668810ff80593273227',
                     '36930bfca54fb42fa28a0a8939bdcb6181bfda3da5295f8e6d30d8b658ee8079',
                     '36984c7cebec9af59bdd0ced39e22c29d5208c02346001388601a989adfbd098',
                     '36df2fa4a13411acf35dbfae062a15ca58c58331e52a20019fffb17c3ebbe8a7',
                     '36e5a076cf2ea05749a06cb2bcc26e4fdf39cc250f8139d40b8182d03c7148f4',
                     '37604d0b16ed45d50ad73e92f69292b29ab552895940e6344fd0c27848841bde',
                     '37a34ab5c0eec0ca861e91e4a042b51fea62c0b2dfc0f3081719cfea57be04cb',
                     '37e7e20d527ee6051d49773937c09beb8d178c20de96786ec44494927e9e5190',
                     '38285e3424177b69ec49d8db865a0c83452012dd4a00b46e9dda6f406836110c',
                     '383f1af4bb2eaa49811f0027d7ec0f89db9b340aef36616367c692b5c81d5f24',
                     '384e0cda59102c2e30428d0327ed6d9de19a1ed8a5634423fc8d1456d950f920',
                     '38ef6a56173fd38bd4129130a9632bc3767d5369a2d486900f8568e43eb8d9f4',
                     '38f327e85514eb45fef5ae3eb06117ab2e16df93ec4a8b85e475d3e04265c69b',
                     '3938bbb08b82fe0378d1f55530f94e5645fb61ee09867340aefe5dc258f0070e',
                     '3983a97cb60d76efcdbb1b037e8b03a770cba61d413d2dfc58fd1017207d8635',
                     '39e1dba8068d2af620de27421472afbe76be97fdad46683b25745fc6deb883a8',
                     '39e989e179963e8cf061abc0bd32c30579dc0d77eb7ec363356284915c5d294e',
                     '3a4cde911a3bec8da83cd0b644ee297788d5b85a0af9f2eff2a5cfcd779c333e',
                     '3a52e5716470b7ee0a695cb522e1d38858bba011de37e93a3f31bac90874f3e3',
                     '3af527a45ed23d571fb6a8e24ab8eb5699621194310f012da20118a900e7a7f7',
                     '3b2c492954627e8bc310bc32893d825da555b631c34c5acaf633ca62e8d5a401',
                     '3b8b929104cf13efab270b7e56ba0f38183857955095ec84191495eac11c740f',
                     '3b96f65812c35e52be5d10261363d221385700c1cbd9e1cae02bca2eb358b771',
                     '3c68b109aa59c24b0bee0a32a25f67e6764731d6db59e604a29401e5f67076ce',
                     '3c892fa76bb770337f86ac817941f6623c04663e4e7bace7f0b8c3ced70f5d66',
                     '3c9c672675554754ee7385dfb80303f8fcf0915e80aafe56495bd3505ecfbe5c',
                     '3cbfaa6d4563c235b0907c2375dd44f3b96d591099def6946e870d66a2a86547',
                     '3d356c05b4ae18471dbbe483d7558246322d4f1ad06616b748239e9a257f68d8',
                     '3d5cf0e29b74cbd7119cdb100fb760769be2c0201990dbe0894abec1bb58212e',
                     '3dcd17e02aae26c95a49837a391966b0b650d44d575c751e3a47bb1c73d3c23e',
                     '3dd1e77c4355d1ab77b029fbeb02d4112853e312cc78cba125ae02e54ca49f5d',
                     '3e516e19ca3370762eb87c89f0eeb9a70979b5185aa3640609a97d09cc668753',
                     '3efa128f5ede36086fcc5ba77fc91e380b6da1e0c6f90c2f2362f7740c0ce5d4',
                     '3f103a95be2a938f16ec55e5a2a5fec46f3eb34b76fd10536e5fdefbe99e7813',
                     '3fc5066e006c9978ee0fe225c223a63ac4958acdaefba6d763cd6d3a7284f3c7',
                     '3fc9e0e2a63e532e2d55563c09ca19d2797bdb207f32d65419cbe5a19ed4d050',
                     '3fca60957977d0994804c0158199755aff91032f2c9c91db55154255ba36eb8a',
                     '3fdcbc48a62c2fcff67e436a6bdf7aae0a40d277694895c7e80144ba41a8ebe4',
                     '403d86d395da009dfea68201bb656e7af6cec9c56ca6f5a145ae5722e62c8698',
                     '40d9ebc0983e831d0f8780cac97d2dd788a681bc11f56f85d429311dceee5902',
                     '413f07cfa1eb10f6413ea8d977b392d05de767bf4bda512038954b9a004a13a0',
                     '4178ffc9cb3279e8269a677822af23c47e97ae3ae88873aad0cb5978f31ad19d',
                     '417d998a8270f1ef1050f140355d995126b567a3f02dba0186b6b56725ac8005',
                     '41bb5562b8edca1028ff34db563dd1d43445aeed5b9e1d080bbe63b0c4e657e9',
                     '41bd481208205366912cd9a36cfa1cc5c0407552b88c38d846b579692f7af208',
                     '41ff58423114103524a85024eb29549a416af37b154de6966992f43847ca0564',
                     '422945d0c129ed7803b523c91cde540f976da7618f4e2bc59aec187d0989dc53',
                     '4300d25abee91559787f8d70f087ffcc87e63b55c9f7f475947eef2ca7d0ec7f',
                     '4333f1987986561eafb75a8d6c91ab2d7d142b5d5305263cba0b571eca1b2eb4',
                     '43d5a5f4f30c04de152b912b9e50128ec2ff837ec4c0b340aee15e025a1eb905',
                     '43f460a6393a2b675e86d0b308cacb5fba50add169ed477976e7b66084aa7a43',
                     '44507d44cce9d30e3a58ed0a140ce9ebedc22a44c0ab9eff825f1253a638a0f2',
                     '445ecfb409bc55ea2edbf185a3301d54bb48978c552d66da81ab5f654ab3ba26',
                     '446896b6489600ee23701379b84232fca38edc64e8c4cc90efeb579006be6662',
                     '4477fcdd39a8d7496347489d19943f5e3034a19fdb1fbe7efc4ac80f967a71f8',
                     '44a5ccab459559e6c665ce80951d0e5a679d5c67950682ec8c4519836553b0df',
                     '455a8e3207deaab6af6caa2a85c4577aaebd1b494f6e5d2fb47ce1f7471a0347',
                     '455fbff1af91c14371e3ae707558819cf97c55f50e08f1273ff17c3e8352bb55',
                     '45b78077660897b40a0f02bd1166111058fc7a0c885f090a64562b5c95af7791',
                     '45de925223f0a62875e752bb4c4bdaddb13b60c7fdc7c2846c83130bfef27eb1',
                     '45fb36f975631ab002fef0cf6cbb878ecc1ee949eba5519c8dd53f70aec6cbdf',
                     '46533741c06ed67856bbaf606dcd67357979e06c9434991d8cbc88aa5ede851d',
                     '4723537bb00acb02744e4438846c85f14005d8673c66a842d30f8cc4a3a187df',
                     '47551a2489eb09ad2da883c6adec5e4fc2f6f12dc47d376d17f6dc7cc6fef285',
                     '4755d79b5742075aafe9136853b3a45c18e2c74de1a2e037d6a292604c17b154',
                     '47731bdc412904e4d597b0a40365380d6a69c373dd3cc8924bd1a73ce357d635',
                     '47b2eb51a1f560d0be0833c6d040a97d523054c16c052d0c6d0d8500f7be5507',
                     '47d78f81622fab2b16724c81c6cf0c578e2fb2796f232bd4eb24e594373559ca',
                     '48097204f0e1e608f4050b14ee5ed523e1fbd7f42e64144f3733b27634b2928c',
                     '481d4d1b796c64e8d9447a128154774a0e68d3168d06751c942704ef18e2b9e7',
                     '48229a8b14150adae365f0cb009bab911d9771dbbbd7bf959ea7c6403e8dd78e',
                     '48644c96046d9bfb85294d1df49ac65c615515b59c6ea7af721b2f0b2a88933a',
                     '486b18e3bd5b423384fe51027e0cdb1384e827ae1045c8a5b33626ab98d4e25c',
                     '4909caa91644f73e0c459679143722235a1aef1fc58621cce427861966c6a28d',
                     '494177b028681bf52c409f064078f954db33534a72e96370c1a7bfa11429cc39',
                     '49551daa261549f2f64482ccfb65b32fbb79e139be7a7f19cdcd12bb579df5d8',
                     '499c8b5c58b4cce859361df6854f179566072c4426c8d9ed020749ffe7f9b50f',
                     '49b38edf9a960824ab73270077ed5e9d52aea300ffe454336d35daf0dd90f812',
                     '49d7aed962c2ed37ab2c75bc32a946b70f07ac1d8fe56a91c15e1d4822dadf34',
                     '49fef414fb763998c606ba88cb83cb653a769244fcf0d4bd2026076045c14ecb',
                     '4a5666f982cdc7b359ba16bb64b8a2c57bc334d5cce141179377324d4e5a9611',
                     '4a6a2e6fc197b3f61e187b6c420399f456bd99d9ec4f7e678b4098792310ca0d',
                     '4af55d03853a2c14ed992d81f6db525d502f7d22aa21905879ac1f1ef1f12d52',
                     '4b656b4f43b3de43565b38e1f5a013193ae8fbceb330505aa30f46e0c7d266aa',
                     '4b6fb1c47826d61e76839279441979db0a33fad6d071fc5d360a0a9651c0d5ac',
                     '4bb7d85a488846ac1a91d8a95193831d8ab2d01b65939d6b50e439198302fb04',
                     '4bdfe57004a8cd7dae812253b057800ef3f265e0ea92ece98514856c15bdefaf',
                     '4bf6c554fa4928d8da6dc7a784cb7e6f275ebec4de98b15ddf02fa5d0a5dfb2d',
                     '4c0dc526b6db4afeace727eba01436f121fa40d20c366e48efbf98f46f50fda0',
                     '4c722fa5afb35046c62ddb85504502d535cbed2fecdd9fbc1ee6b464a11a8d0a',
                     '4c9cdbe77c5df3682c148b2f6d82f53d8273a815a8ee173d5ca7967b88706bc1',
                     '4d135c5e1622a00434e847ceda99c2a093a2127712ad3ab51594e18616bdc798',
                     '4d8dcb6103b5cde63d643b86d5232a092c0adb5ac64d5805cecacfa4301867a9',
                     '4da39f2289e89580e18ece044b3df3ea9694b8cccf0583aa6f8d2f7384b71f10',
                     '4dfbdd4d41adb1016f919671481acc0416531d65f119bc8f85743db5ff3cc616',
                     '4e2e74c6905213cfdbc9062f1318493c03bae0f8b383a6776cfa281d6e837afd',
                     '4e6784537fbf5e264f16ddd041cecc2bb9bd1a14af2717010d0823015b6e0a6f',
                     '4e77140028023d0c72512da6344a5a434a7123659aca077267d9f361fcd0d253',
                     '4e88410983856bf3a09e91b936db2ced9d720d09259f120d6f10011917b94fbd',
                     '4fc4725e453bb817aa0189d37f12123078c7608a04462d210c0c1c815be89dbe',
                     '501a19a8f83296edd3e291d899153d4af2b8d4307219ac31a257dd6e70c9eefb',
                     '502ae35ae6960632b52a750cd3dea6f181b75dfae008428db818483125274d2c',
                     '50c36b1dc6fe948923c1653b715c4806ebc86f061b2ab48f0e95c42eec544b94',
                     '51272a4151c4e994491cdc6c2ec65d64bf3dd2d70e4c223b578531635eb11042',
                     '513a05cf9fffd22ef77fff7686a9bf0054c06ed46ea506fde62763b867415e20',
                     '5144ab1e87c86a75492c7a6495c1bfef5778980a3803ca2caf00f3d8fc410f28',
                     '51d5b11aca2a022fceeb7f40f853cca843acb0927dbcd1451525a6d048c6a52d',
                     '521642452f15e3c142bd07fbcd015e9cb141fd29f947b584fcff64f9987e07fc',
                     '5229daa45017552fd32669dcf4a96e45dff97671f5adcff6aa2e46f7d0140112',
                     '523335490b51187974cf36b6ca64c91830fb4a7672609a4f671dd667fef1617a',
                     '5250c06edc7da6c627c5621e9c08ec7c3218efc99c3383b07de989438de3e393',
                     '526845f850b649ead4487cc741e0949e96fd81ba36f4da9810e93caff4e9aceb',
                     '52faea885ae054ee7266959cd930816ebf9fffa581e34729d05dc2394a64508c',
                     '531977dcd74ea6ec96f68e5c0496bdbb882ef80187bf6922542ebedfe30406b2',
                     '531c76caf1ac8c69f1d4d07a8e17a86c20f4e45977dd676d9c267f3a1b448e74',
                     '536ea5cfa9c5e967fc3134241720847df974adf0d33ba3a7607fa577f2fe7ad3',
                     '539db8a0a48e494ad3c116ca192447326b78da107c79f18308a17f70240a2ba3',
                     '53ae5f55519bab8de2d01e6979fbe0db650be9cfc718bca921091d316cb0f941',
                     '53beecbd7f9a7eee5aef8a60144d417cf8ab350c8bf9399cc13bb61733b6cc65',
                     '53cd7c148f0c8e7aef05de9686449ccb178fd4a1f695de2cbd0784fe4cb311d5',
                     '5416f31fef1d2f4b52d001216e07fbe78ba73b88251ae698689f9a6b218f34b9',
                     '54679d0adb8a82eac4afde1ab7fdcde2308e1156254d8fa040d6e5236cfb6e3c',
                     '54792b4b4041509792612a46a3a9e3c61cf7db058cfd3ce54df9bfba958658b1',
                     '548287a4813fa0fdbe20d67578484f075ef37bd1dd6f2a8385076cf7a12e13b8',
                     '54890c5a60d81109680fe246b525c5fb10a5ea0040489d86c48312b2d2fec3b5',
                     '54bebe36f250ef417216594b1cef00bb5594aee3ea36f3df54c0f18e540a8ada',
                     '553338da2701b87521a32ba805028a46d66ee0c6c6ad4e85c50df9cf50e167b6',
                     '55a7deba4bce9f2a249ce09a0b49b4447f748be5e30b9f343913c7a1a40bce1b',
                     '55dcb7e2220bfa7fc6500252b8db0113bd2ef50b15d43a583f032d87825dd791',
                     '5647ff3c18cadc44baa80528bf6aaa790f05a1b0d67d96ac4fd99112c5c42238',
                     '56f89f771ebb863c0de8cc3016e931e76ca427b59161e2bd32e1b96806c99e4a',
                     '5715abe0581dba527e32f73cc3a6d92c67ba69448b31636135501012da89841b',
                     '575549bba4ada4a16aa82123728211408b480b8fde752bd071af9eec053eb75e',
                     '57c7f9893706b98cb1b263db5da76f6b7336380684efcfe9a167b77384cf5c6b',
                     '58008f06339e96301472f4146cec2558e25e79df4d0e0aa8511d08a9fe5d409e',
                     '583a382cc04cc2c8b0db2a06c108994f8b5d0f14162db5b9630da75f6764fbf7',
                     '589ea0e14d751b5ecefc3ed53d7685f72fcbf1caa3ad6b616129d3797702235e',
                     '58d4d35e699fe59f3d5e0d147fb2702d37c4c02a817c3d5c99d5fb54986c0504',
                     '5958f91d8bb273bbf211d7ab1511d885afee981e219b4943eae1ecfab3370b17',
                     '59f6c867d4f120303824620f8b9378b410dec603c8ac6150aec9ded24192bf43',
                     '5a2ff3c6aa24a2fa0eb0a25d0c74bb392e387d2cce1a0c4b2cf125fe2448446e',
                     '5a540eca79a0d67917f57f3e0d7207e0dd1f60c0f25d3394711a4d0ac9ede1dd',
                     '5a6a952153dd90401ec40eceda54ab9c37709676eaa21f8ad83503a7b24b1951',
                     '5a9e0fcf1ebb9a309c67c2bdf48fb2c3b62c75184aa11097788ec0f5efd1b846',
                     '5aa2d1269adab9c2c36d2787c1f357f5d8161b8ed87a0698847ccce315b09664',
                     '5ae385ed224de1f7172cd56c30ab55188becbe0a9bb7dcbaa8b0a247dd73845c',
                     '5ae768884ba8c293738e481645b30e939930a6494ed00f2df6c1b2a7140148de',
                     '5aedb3e6a7e8c0c6a06f67ed854ab11b02884ff38448ba4088ef87b4fe289339',
                     '5b44f9500cbf14d62e713a24adf813b92bb206d1eb842943aaf90a655037d7ef',
                     '5b6b164ca7c58b4ad9be13b016aa2b66a64cfb5f27ec24498a73d2927b869aa5',
                     '5bb51e4ddda8bf153b6623460c5029d355d7ccd835a9ce1b63e46f29bc27dae0',
                     '5c120f923a95c00b5a9c1f6328bbc9cda0850ab7cd9fe68c35b60bbe51ece0f3',
                     '5c223b60f1086863b605479c580f3ef29ee22bfbb7c76c22691184d173f21fc5',
                     '5c79464863cac894c5ad13d984d6cdb6834b72c8437b9de4b9a3e4072f3fea15',
                     '5c954a4f79e4aac172708c08351b58752e66b90742925f7bdcad2ef696141266',
                     '5ca636821c1d4914eddf26a49f8592a83ca6d1630f096e44ff4f2b8352aefb02',
                     '5cb541536ef41d7d011429679a0628423e27df7b4c2f1d921506333e354bb463',
                     '5cd3b15957e86d85be27e14c05c35b8b1d3a0aa9646995404117f630b18a758e',
                     '5d542cf81e85c032a23824fb06a5954b8471f77bb7ea92fdef962002e4413a37',
                     '5d83c7a55d1684799a32354258b06da03937744a04ef843d47c73fae116151ca',
                     '5debd9bd1bcf027bd3df503b3fb73a68924fbd5b3d7c55852f379c06cb61f238',
                     '5e30e31cd36bfe003c71b8d0e7675c5dc82c4f70f637963a4540c744f8466c64',
                     '5e525649f0e748cb0c6e6f62d6d338e884cfc1baef6d5df5d228b542df2d43a7',
                     '5ebce6409062d0d09c88ecd8b8873a68e3bf252d4d43f896a8bb2ff06b767a80',
                     '5f09033528e4da13c2ebefd8dafc8d749293d6a12e1021a6c2ff78aa0e54d157',
                     '5f77be7c9b614908c147be524eb802b464eefb1fd8b08bca03f286e1842d057b',
                     '5fa9c23441645312c7101423c3d670854475f2261e725a81b8372340446fbdd5',
                     '5fb1e2ff524adfae2f878fa8ec1653e6cec86112020c2c1d2dc69993c163429e',
                     '5fd5ea59567a4d929551fc89ff66d786cb34673d350f52207d7d0e0ce5ea4e2f',
                     '600043039e4262ce95e81c82a970c426f132bc413e97c0c6b73afc992e08c52c',
                     '6005ccb5612b23aba5df83fb1481e96647f15c6e34d11eb26225ee342ecca069',
                     '6012e6be5921ea56820771f7c5f69cb83ed7f76cab3ddb45e0d271b6e98ee613',
                     '602b7b7180d02c143d7d1364110a2e96113a6dfeb70d6c2300e114afa004f47a',
                     '605195b4df6d5c9b8694a313337bd3b8a2ef4759c1813a638bf967c728f7f3d7',
                     '6055a941f23c0adfabdf76e54581c320ad8f2315d69bee5521d5416ecaef7a6c',
                     '609e504207c753ddd5e1cd970d651576b96a41ede0fc6366711a76a8bfb7fb82',
                     '61357eca6b6068fb3a47597031319d933d97893076c86d3c9ab346d913e1b600',
                     '61616312b79ce0a6ebea0c1edb9d2dec7cd4727165e8f7d2529393983d401447',
                     '61622e5cf391c002c83dd24ac4d38737dbead13edc921b8618172f3b95db4ecf',
                     '618ad1b14bdfeb156d401375001b811948ffd21943d644b8b4fbac28801eb12f',
                     '6197f56fd48512c651dc9d17532a2f08afb691c1b0902e02212680d496f73683',
                     '61bf83cb9f40ecf7cd3ebd724c9d61f14b9305306a9e8914639ea703486524b1',
                     '6212fdd2df4135dff2a4f1e8952041ef98143dd4dbb0974f0fb172a21ca4adc5',
                     '625023052b7d4ec21f9449e9e2983590976dca8ad4040b42961775bdd94b7e13',
                     '627051b23a3628de3cc5a104f482849437d719a42bdebcd59ee548faef94d0ec',
                     '62a61125f006927196d35fcaab47bba5b4ba3ffa6a023edc8270a632efe35acf',
                     '630da7ce1bfde039240eb9b37b0f3546ac9d2a24eb724f33dd7cad0eacd0fab3',
                     '6318d44a2ce4e3378a20f1713cd47237c71b194f4235196ba6ea5a978402fb0d',
                     '633242f139c4c894ba04a49308ba5a94a23e0ad0b634924a5827405eb5436c8c',
                     '634ee8ab28a8b1ddfa650054230cdcc97c970f8c51c45c719c34b0605964200e',
                     '63ebb29d693bb899b4f8271559b08766c525fd555519f0d20d70df319dee630a',
                     '63f7c384b9e293cb12ce7740866bdcaae6c710930fbd81a521e7b9512ae30717',
                     '643fd6108a4f055165d18e3233cb086084f8116955fcdf8564b51638779da849',
                     '6445ae0879de1d6f17abc5b64fe0e21bdf2a5cba055267e518565c2f3b33b1b5',
                     '64585202cc972d21ab8dd0f603b76f9833f981f29b98908bc36a3bdf5dd77d51',
                     '648b75001ebf872b35bbee5d498a94ec5d35c0086b8e03981ecb45f3b1f0c257',
                     '6494be55906ce443273f21d6829cb2edbf38a73933f0b5299c1d1e0c5a43f887',
                     '65223c7f321b6ceeeefc982049cd0fe96bd41df4f93c8125c9b29f72956e9159',
                     '6546982643a9a818a288b099c5f87e0c7f16a18597b104da060239b405a0313a',
                     '657965471d6d3d059a76badf497c3d0ee3613564b9698c4326b25b7e8536f4ac',
                     '66886428bcd081bb7db43e6220dcbdd0c02cd99015c8a102cdceaaf7b8bc2236',
                     '669b967662ebb763669d3d3d1486b36b269a8bf9b8805dc21481138eb4ec5505',
                     '669f272262a785fb651d25fac88a0e461ae2669f6ea3fe3619172f9ac78b4e4a',
                     '66a658465049aa95d0105690b8d79d9eb5f556bb97aa58e4fb0aab3c264b094a',
                     '66df0f9e8befe6c405699906669c03bfc1c28d3a7e74a1fc5af1431c6ab4bdaa',
                     '671226cea6820bb36a38267e9eea5591ae1a7f1c351b57a59d8cb326af7701c5',
                     '675954a180056d28e080a78fcbf0efb49e619e17dcf56f22369051a2331470b3',
                     '6773700bf36ca39556ce968a73811531ed9fd27db54b72b959a1a33505f653b9',
                     '678ef7d21f94b80531d7e67eac5bc0c63f7387cfc8fd0f32f4aa3bf0a060e854',
                     '67bd27e0e0202fc0f7c911f71233c63c2191df086d0c5fe99cec8cea9e1ae46f',
                     '680a0673b02d0e1157d321f8597d09419f1d4ddcae44d515eb310b718621fdb5',
                     '682d8e92bc195d1b0ac55206279b85330f612a07de0cea46913ef1dcb1a0a1fd',
                     '6875b7f3b4696412011b80ccac0ae7b1afafab19e70a195bce1186d49e021cda',
                     '6878cbee5f711d1b6b67cba9f3258f126e269d62e795e6a43e7641e0683f903b',
                     '68d030b51a5c51f42c8914531a75672882ac1efe2628123eb262086f7b0445fd',
                     '68e43b527592abd4fdc0e35f901f25019f9a7562097128632a3fb85805cb4f14',
                     '68fea67ec0719386b07199fa625761fde0926e75a6b84fe1cc84455c2d03a5f9',
                     '69024f7e56c62dc4feca94ef84032e39ecd935a91ce75216a0bfb1e8b2e1ef99',
                     '694a9d3ef39ee9a9a8c5e37d3ce3721f435e8265547752b713bb4e03fd23eb65',
                     '697212be1d00f0ebd4d251fd66e9aca0b2a2e56db4db8b38de5e066deddfedc5',
                     '697c90155f4b75a5236cb78246c65d7a39a4e7fc13080dbc0d44919b99551857',
                     '69876366726703ea021b734a84489bb4163a4bfedfb5659aabc2d1717ea001db',
                     '6a68787b742f537eb20b088a5c9834bb03c28d0eb03ff671f9b8ff3f07395388',
                     '6b379aecea6514f6a76fd867ed5812830569fd228f9779b6726999cc3d3f3ecd',
                     '6b3b65b93269d46972a31f9a62d88575f7215c594fbb51c9701bfcd3f9781b28',
                     '6ba25a25b0fc8f2e0b8e61004ff7d594801a845e78ea3733024d85ee8bc9fb41',
                     '6bec79e8a2b94767be0e74dde2ce246946e2690b924dbd820379d68969230261',
                     '6bee3eb2eb0c7ffacfb67b4d18df51f272c59285c3866e9c14eaf518a7dd8040',
                     '6c6b71c394797b942e87cd7037a729547e0baa1a83bf9bed818f70200f1d78b9',
                     '6ce7d2bad658bb48845625e87136ad74422072322416f430039151fc498f2b43',
                     '6d1f0862f5e340d61854b140619859d5941cf9835788f3e71711c839ead556b4',
                     '6de599b17713fe4664ce8d3e6b861a63f8f399f87a9f69b46a187735c14f68a5',
                     '6de91cbf7e2fbd7d913674223dce53d96ff951141c497677aed9634a5905c927',
                     '6e13000b4639a55e40ee13717b9539b8802b01a18a09e68d2601a15f7e7c9f87',
                     '6e21f864e12c8895ab11f41e37cd2da110e47f550373ac99c17cf75045cbecd3',
                     '6f27f7aba151e830cf48e2f5df8031b3810729e419fa618b6b30ba3355fa8ff3',
                     '6f49f0f05d2df15e70a03ca6caf35d98655beb6d7fdcf036160958749f25abc4',
                     '6f7dd32e5457545710a9da309ec1a3813c9c2d8ad955686d57c8d314cf296bf9',
                     '6fe56567be2ca329e75234c985ee95430839817b2030c9918025dc917983f7ef',
                     '70337223dcf51900995cb0732bba08085688b72295e5e792b224f7779e834fb1',
                     '706d2aac56c2dca935746f7d5de572d200a7b1138c1525b42007b8a1539f3fb1',
                     '70e047dcf9cb8e47d91ffa0a8c05f42eb4a54de0526cfaa07178f81713ceeb69',
                     '711429f02f197c6aa9d1e405152cd99a5c2bc2942d231ddda6fdc11239643ae7',
                     '7114d2751df79fb97a951b37f7d7aa25983dbbc758d566b955075fd207dbbd59',
                     '71284f9ad2bf5619ed99d70f89839a85092eb67bb45c9e8da52eebf958ee49e3',
                     '71372c599c85aafa4e9714e33f875aef48f00da63640b1bc387c77ce30b09a9a',
                     '71a53233501116dec0b7a2306a681dcaf83343ad24f418a056fb594006cfa173',
                     '71bcdc5e7cf534d0de602e8052147b1a88ffc40b622904caca87102bcf8be2b0',
                     '71c504fc93d8b39a24c79f5ca6c1792f38dd34145cc075bd7d2a3c1bc6196ad2',
                     '72e4dcb218e48f09df4eab58412389c259a85fe6fc1b649f0b6f4d8c92faad81',
                     '73867ebc8c1becc179bcecce843fb21f6a9dfa1f5580101c0119552b736af699',
                     '748a12289c1e5c39d6ddf297aad473c94c5471d52bfcc26b925b86431e4d9f50',
                     '749a2c0ae14ffac85a74ae6ad8f518175790c4ca1d997895ffa9b863ccb62dbb',
                     '74e2ae59d0c07c953be1be0036dc681324bb7b350dd43db9ae09049e358003d9',
                     '74f2e2b84e80926b1d1cfa12820a74b4a2734f14779890559f8966c5a9d2f2a0',
                     '7521219c4dc87eb3caba5341f1a0ee2d23721e51d2fbbd72b3eb7bfb00fd4f07',
                     '7524c3ae8661a7c7d361b8149056d31acb09e9e31ec5fc2f5a4e62889552e3ef',
                     '75300d527a44073bdc241b9f87a402e5389fcaad05ea3ff5af191da24e5aea4f',
                     '75b884dd33d394d344c21ee8e7b66421b43ea8cfa0982cbb76f78685428cd9f2',
                     '75fe4b476e2d596cbf623bda69ed58bc09434d0168c67776fdf2059a52098e1e',
                     '762bee91ab3300e45c3e67c791694f5e9e40dd48676458697ecdc45d09a14569',
                     '76cd5408e575953f7162c493f1c2b30773be0d4ac843bedeb60267391603d23b',
                     '76ebba41843d5e5007eca1207ba93950938ac0d2c82163bcd6bbee0bdb55e861',
                     '77157d41ddece5b386573af5f5d82113cc4e1bd5827411144ab912eb5241ac29',
                     '77cde4cee01ec4f412069aa2ae6ca3d50154e6a95587596652f4d002d83325d7',
                     '77d458a52bc14e015e74fc2534a8c38520284435cf5663aae1e5355b99547ce6',
                     '77df4b1761723477ed9c65124f6843f97c960bfe5c80b69443724f78b5d85870',
                     '78174f9ee66a7fc9366eccf25520a7e251e173fd5ca1d744bf1f5ad6e2c0bc56',
                     '7819eb8cc1453988f305a6d84103f8a3dd366916aa8414e63b9b84a7dfa5c4a0',
                     '78fd175ad214e57673ca88fcfab5619a1f0efc40b60433e8d8eece1594bdb0e6',
                     '792506db3de2185698771bcba1ab498bc7ec04e1d6d40406167cadccdd378770',
                     '79294e97c363afb031381a69c8f86c1742c842a55ec108730efb354078f53122',
                     '7930d7d0cc620352ee89a5f549c07e06ebdd0bc199e53684333b198c8a51e188',
                     '79c1d30e3334e1762680107597bde0fa5eaa791998fd674cceffa8c20bfe5a1b',
                     '79d2ee709fe90a80392c3ea347b245853a2df6885b5a08e12695e95a5b1fa5ae',
                     '79f7031a4e43a6c097ac52e3cd3c2f2e0902c7f4992259092c855dc591712ea1',
                     '7a2391d2a2a38dbef609e990a824b07332bff012adb280dce711c019b0636fa9',
                     '7a66bf7c0560ddc5d003b0869c6b7e8ef4bb06029a3cb0c258820a2390522a41',
                     '7a9010cfa49379a1e234839532f5aad546561373dc5dbb60d0231c7f361fe837',
                     '7a935559ec33c860f6fc4eb3d7fb5f1488cefc60d5412f377562826633a7bd00',
                     '7abe3f31882f40b4e38c9c2f6468b7dc696a7687d97ed7599646b293d5b781b8',
                     '7adfb0914d1916012dab439ecf342839835f41a68eaaf1c4517afa9d7847b626',
                     '7aefcb954cfe40eb083fb2f35e0b334ef45b43c2ffffc25c825a0b6a3df1d839',
                     '7b85f86b38759a1c0c4716a2755340fc2bf6f8d60d4da5cd5a75a9a4d20d21fa',
                     '7b93f1f472b90c5000cf999220b99e9dbd8ecb9db806851f4f3a002a3c79c6fa',
                     '7bcdd0f5faf5011f4a12267f0968b33edd53a8a6b496eb9896cf41e73155c450',
                     '7c5c847563d9e772388777c232d50716a3685e50a702138f9244def7a90635eb',
                     '7d1c26659b5abc30b4461304e0d55a45938a0202fb4dda162642ff6cdbc7583b',
                     '7d21b69701d4c9da4cb0876448f089b3529af2c70e63c52aba7bf368e8b450cf',
                     '7d33bf62ab19e16fd55c54a0860216fd57f4b39944133b40d9be0eacd635cd9c',
                     '7d33d0329a2b9d2c2784176df8e36680b9ccd16896c9677e9cc96307c4e7c5b0',
                     '7dc3bb17b14ee6737329517be715a9a8cf175b5e169be2d6ab7e35f77fbcc6b1',
                     '7dcf4a149ee83236fc6467851c1a992b4107d833c99d7977324bc24ed989c927',
                     '7e0bdb6d99d8328a7ade0036ab35392de3b7753e3c87ad4656db59e23b1f76e7',
                     '7e51e3fc640024c1a77502b6b8564be1d4305b67590b4f60f0f135963787a155',
                     '7e65c5ab2140d1a28103fba10c2bc53e9da2cf7b43afb0a61e0c651992bd5e25',
                     '7e74b61f3e6334fb3c0b359f71c9cd53e1982b50cd5427a94f36940db9de8fb2',
                     '7e82b6c309cfbf1f791bd1f19623d5e6862eec2bcc44c5b44f3c2520da1ffd80',
                     '7ec5f47b35338bb532fa1fc0cf6893691bf8c712af1b55b942a9dc2225b6369a',
                     '7f804b7022765c0ec7233fda4241ce784305afcd23e365c12d23398b94012545',
                     '7f85d9d3836a7943bf98317d347614821d15c01a9030113f7de5e60439c0dc89',
                     '7fbed8b563cdee3197fd7b380bd5150452b5c3dbe674c395b4325943fa873a49',
                     '7fefa921a6f14bca5cc222a3036cdeedb95a1b7058f18752969289ec2648f4ed',
                     '801ed1c5358365b330eebb20cb4f7480fe83a96cb240544509db85b37410deaf',
                     '8020a0d1ff778a8e8ea8385188891bad0841a590bbe445d2176dc2616d44748c',
                     '809e23b1b836ad9341ae10ba31858b25035fa32b4733e7a1ec1b9dce55cfca4b',
                     '809ec82680cb5fbcbad008d83a971f58d48e4327e2642d41cf7cbf69bbbde9a6',
                     '80ac13bb6a2f846365ca4d0359de753ba0946695565c8e980cd675f0c07d89fd',
                     '812e1c5a661fd1120ac325cafcc5da65932555f2816dd45d578545ff0c8ae616',
                     '81fea856794e207094db96aad6c8201d44dd885d7c03d8388f045bc650456a3f',
                     '820006f4ed92e36119f0469ea3311e2739f34c88500eb583a9c37737c9f4fb45',
                     '82045496f034b46fe151140490787b0795564edcfa0b4b6f7027f63726901e21',
                     '824072b3ce8c6abef26bbb045d3c9abca759b0a8b1e8e8cdcaeb64a4e403ef34',
                     '82aa3ccdf29aeaa6f8b3a1255532f8f58c550f0f343615d0ba8d0cd6b762267a',
                     '82ba018e345e09ec588090eaa6d5030da344d6393ecbc0b091d7d6c01027403d',
                     '832d52d7219035a23a80c986cd41c2961c1be6aa7f83300e928d9a7435d5f893',
                     '83ce0bb096a862358a45d465ecddeebc1249cf4a97fa29072123cc94f9bcc59a',
                     '83e5a8ed400d291c17f48e8b432b8633aaa0de3c3b482808552a7928ac3c6a62',
                     '83f8436a8ad19dc5f504b38407177ce9a761b8550b2dd78f6394676221c2bca8',
                     '840db84c75cbeca1f2147634e8f410d96a758627824cf5a9ea4570135cd6d1e4',
                     '8411ef2357129651282c83c5658e1fd6ed5af85edebe985abc56f41f48eac25e',
                     '8462f8dba78b9dd70cb78df9732b695992863d00f429897644f6af1d8228de27',
                     '84f7da83fd4a19d839b38b02fe9cfc5e4e7a232e8799794b3013cf0bde1a7066',
                     '855bbb0ca119676f9c9135fa70d90547f606e5ae78faa5b0d8a463097f4b9ce6',
                     '85d25a5325a781e3e0d276e47a4cfbced0e428e63ffcda50f2654ab2be699c3d',
                     '861d81fd660255594cf9a736c87ca68c82ae34f45607f925da9c6967c658c231',
                     '867884ca840bb4cffc94cb14784a59146ec17d21445fcbd912a76dcd56a288d9',
                     '86c50af95f34be370076127e6a447c555f45ae8581ffa5ef42ce70dcec44700e',
                     '871e2cb50a8aa7427d0875e8f46f600864787978cbb5ffb9199f523fb96f8f7a',
                     '877827498290281af14ea2667c83ea0d1ba796a853aff5e101d03448c6f2240a',
                     '878c11cfb20e4561c89e6ce8d973d17d7b02857f0e3481b1cff39bd954579a11',
                     '87c9fa95545393b913efee2667d28f4fe46e23a08315602be8a4f3052dd1a729',
                     '881e5969b0cc5ae398f845580772401efe205756f3a17d63549f23ff5c6a0a6d',
                     '8865258e4cb393d1cd57ea265936df75da7fc46aa91e6ba16bb21e0d9d16a6a4',
                     '88aafa92a657bbf38e233d76a6bbbf312c43dfc68f8db36b6fab1a9516cd9c42',
                     '88b8d4589f35fffc2553ee7229badc66a0bfec0ca0bfdae2431fa50116ea798c',
                     '8952b58cabd16eaec4df0db56be1ea2c8222755e5a3fef7789d983e95ef696fd',
                     '89dfbfd337cebf268d49b75bc5513607be26f76058f709fb9bfc0fe5b419882a',
                     '89f8d84b36432628b137c6455f17c425d49bc5b8b274fedc2103b668f7be7a04',
                     '8a0543c1de72f5538c3a5e5a056e0139fa703eac01b1b27528bd9ca1879b77d0',
                     '8a090e223e0ea8b594b339816bdb6205b4c46c401be9a5b937f72ae965652299',
                     '8a8d20199886a3ef7714620594ed1f0261d99693616842b6ce4f1d71b845be10',
                     '8ba078f1f9220437c84953f3491b2fa59b7caf3bb6f4fff2861e67669ed00bc3',
                     '8bd404d2fc31b447ce91a7fe116316806b05e729f6150b04243d221c5a136588',
                     '8ce56387c6c068ddbd1694f74c3c9032a83892f615f7fa94205e70246c4317c4',
                     '8d0575e83fe369ccd672f27ad889a9dfdc4f517dfb0ece25e4c291d5724732b9',
                     '8d3be63c4da550a3913178f9b2de4f073b8391bab78aac1e5a02a8c656749814',
                     '8d47b475097e840bf895f5ecaf19c05df56d90d47b21cd858f090e1c677ff29e',
                     '8d7c37d7b3c2ae9536a79dc73c978546bdae97f5050531989354ec29f2a7dd59',
                     '8e55b99db889e80fb254f328283da5a53874c787ef4e4b6ad20876f5e65a06e4',
                     '8e5f776a6caa856cae5010234a899971f74549dad7df58cbe7d1d6afe6b803b8',
                     '8e9d5716cdb8b34a4e88bbc0bfd89a7239b26dc28a5677cf1bc892ae513b9a85',
                     '8ec11e81552d3696e99982476a898b20ef63825e77e3cf2a7c5cc0d8411a381f',
                     '8ef0899200ac11910cabe7a7db0f903fcbcae68a353d3fed04811df414f726ce',
                     '8ef55ac94c3d2e418592b37bbab5dbb9fa0e085988d949da289e8361334be06d',
                     '8f0f4324cced8cdf87b4aceff242c7ae78bdaca735f1f3ae2281eb6080caa1d9',
                     '8f363f31f3dff7f5311bc296aa677441c61a4f7635472aa841c1198027037e7d',
                     '8f3bc84cd99e1e485f567795b569421cc839176a422105d12f123c63c8d99a05',
                     '8fa574972578872d1b818a67c4dfa1fd585e66f8c605023fb0d81075abd81339',
                     '8faadf42f61d007e7fd5bf2305426a5499c94ab3ec5b9ddd08c23e8c7baff408',
                     '900deae74c7de1a6635f99bc11278ddd09d98078b7df8dd0839103d51ab0ad97',
                     '907e694caeb5cfa5a2dd1553a6e847b757d6a20a013435afad338ab45619e885',
                     '90e839ef0d50dba6337d776d6a08d6c9686912415b32cebf861e772ecf9555e4',
                     '90f28dcd3b0705cde9bf71db1eb36d1471be78d7b5c64a61c77db2c9c006b5cf',
                     '9169d847d0daea32a05a772d12b309e793ccc1c91f82fd483cd49a5a7591bc5b',
                     '923016553382e157b61c598820d60ea4e1f86839523968396be794ac69231567',
                     '927cc8e6a81091f01c534c53c04dac2ea17b398ff4b99d7726c996e21024d3fb',
                     '929fdc642069bea33341b3c2460e3ceda1622a031bb9b02a3f1570bb1a5a6c1c',
                     '92e0727f52430f0f457dcd108ca3ae6df86e38f153d3281f2f9d7c3c0d635805',
                     '948c52fac43e824a898f1b322ef815d1c85ae5d2e84001d078585ad0ce53347c',
                     '94b957fe02c8b2166040635a490c0571e47f45a2e664e67c7d9f6c8228d960c0',
                     '94c9fe03acf5799bea7c605273cdb6a0b772711f17ba369ed9ed25643539c400',
                     '94d6437634b1311bec99174fad65520de1a385543eed9feb7dd67016f18d035a',
                     '94d75d14c434f35a4cd69c76973a093a02287c2ad01ce8a01f0d6052a04e19a3',
                     '94e36539ff24293128ff394f1455a1c2cf8af129424a2b13080c5a4c569d3677',
                     '94e57d886c69e577b9eedf1858a8533477d597b6e11738542cd28d701bc6c005',
                     '9505898d4d8813394c9b6cb7c0b240b5ef0bf45436a4ee83c89743b990031805',
                     '9569d188333c72f87939e4fdb401a94af7aec89a8878209ef113a17b4b9481f2',
                     '95dcf2b60941b5910ab09a008eddc61fad56dfb03a59b3a7ab1b6f845cd0fe03',
                     '95ee20d1b5189d5b4233925435218569f908e3053c94848d5d0d651b1bdbe83b',
                     '96c1db9596316e1c7a4f0d7000782a35c749292993a6eda9c180eb0ba65ee73a',
                     '972a679c850c5ccb2ef2a5217a1692b5371693d086485a07bc0a63a84f983927',
                     '9746e86839c203f87344ae97c67204004eb99c35a04074a2b3a76e1c62600798',
                     '975d180742931dafbc745cd64bd338be1514f8dbebbeb2137401db4166a5a08c',
                     '976a330772d02cedab2f0d183f7f793f93117ab62e2f6efd7c25eee1634a4265',
                     '9792b8709c6ac19d37bcb83aaca038f7c34b171f0cc3c0ad907179d70eab5fd2',
                     '97a2b95067c1c51e4bfe50d4a19eaaa18758f3e5b92080b19eac2d429dfd4d51',
                     '97de593258d3376b43cb3f34896d98c025e4529517e9b8914a7d5bb88c74c5fc',
                     '982bbfff8cdc53dd939dd43265f4c5c8787ba2dc52b5e6ec37ad757c69684554',
                     '9847193e99e3e4a8d5df9fb8dfdbb7db903630ffbf651fa6662b986f1ba4a16f',
                     '986f5977972ec1fc0879556c5cbe75e1b871c7f5aff819996003d6c8552d53bc',
                     '98aa4599a2a4076d7aa41d7659e257d6a62bbff69cce3464b0ebae45af5322da',
                     '98f7731a4426d8c50ad3f9203f1e1d3bbf643aed9564e29139f29f9f0dcab26e',
                     '99c3b84ddf987d4ca7d4034bc82b45cdc386bb7b59b757ff8d136d71ec118509',
                     '9a1820f5de0cefd881355d8ac5e8d4578f6db2e204430132e97b539d222a8320',
                     '9a4aee6a95b54619f7c8541a6497e69133bde39d782f18aa6b119ecdd6da0ffb',
                     '9b2370aabd801d45f73bda922cb6939173181898894f95a0dd8309a94a265896',
                     '9b3bace51edbb59fcc110966337df057830f08393d2d72d26ccbea69a1eb4817',
                     '9b6c5d785c00afe6d77ca81ad052d8d55bf034dbd734093d4b72aa2d3e4d2afa',
                     '9c72c13f66fad520b50e438ff8b0d221ba935e367ff7022db24a16d9e2ce8022',
                     '9c94b4e851eb03ba13d6f5c6a9630261b7eb45bd4d817592f7d13cac529831ae',
                     '9c9d4d1735d5a581186801df1412bfc768ac65976a3cffa85127640f7d9245a4',
                     '9cbabc345948633cb689c7409dea95d1416fd82141725b327d701d2e5e2fdf4d',
                     '9d25f09c5077033f0c33abcef3b585c5c54d25c86251ffaeb36ca377a7d9350f',
                     '9d7258d88bad246c31aae1fe27311016399fed9409d188ef8ec6b5971d130e8f',
                     '9db4679cbd14bac82a2270d094c18046b9f7dabd07802b493e7c25a220aead90',
                     '9df229ed3b72c6f9ca679843b977eedc9fd4a0c7d9cba087242950483e98266f',
                     '9e13fc095ac630e20c2148357a9042d615fbd3c8d9a986a70dcf85d193f95825',
                     '9ea81f44498d04af4cd2b84af078e2ca6aa42bc2a0e6939d229ce5a96d21f49c',
                     '9ef8d947eb2796927fb9fa8926b44df99a1888cb92b40060f78ce6f532bac088',
                     '9f670bfb650dc6add106d49afb2b8fd2256c1a887649f444d7c108c170cbc216',
                     '9f74a9537358bd01beedb7ae65a5abeb9ba74c06d1cd036aa70256e3b7793029',
                     '9f879f6d8aa9e5eb62cb3af349cfe8fc15e8eb9fc6262cb8a377688c597dd242',
                     '9fa4dad55e35af304fdde434747cb718a18ceb3617257d1d936f0d09c4df5f41',
                     '9fc869fdc7abf58c463486c7316e769bd7afd60be59fd089f7241b6a8cdf090a',
                     '9fcadd64f6c0f9c3cdebf1030814a9f3d88e40971a5665f39b9fbd20be82002b',
                     'a076cb646f301211bc6b986c59a3f6506b6c42e1e6f645d8aa335b1a6c9bbafb',
                     'a08b6a901fca192459f24e64173f78247a955a404f6b54d706ff8c71a4bb0660',
                     'a090bebaa5187803f002adb0aa189ee9ce5c6c434a95b80f8b62d4eb072224fa',
                     'a11786337ff4f59ec22cb8b9292df7220c20debc689ec52ff0328b218d041939',
                     'a124ded59b97ffe042097a735b80b1a92fc568753bcb1d09226ae34a306234ed',
                     'a1f8788a02599524a177b2452e712c201246453fa44f49ca2697150860566234',
                     'a20efed7426e1a5e2ac2337fce00bcbdd8259d7b3444df7c7f14368f8579c69a',
                     'a232c63616c5d104835a81c72c440af4a185f1929da16c473bd7c4f07be9bb12',
                     'a233dad262372d2384c51036e6be069d8187616dc31d6de51ddc4387d30600d4',
                     'a236495c4bccc421c257fa520149f2041e6759494e36eb779d62c1fa19292e9a',
                     'a27e8db63340cb177a85d996eba4e9fe23764fb05fb9687133ee99910a0ea34a',
                     'a294a4e5c0be525728ed82c78d07e4433f4fbfb03f01a520154bc4025da14be6',
                     'a2d63e41d1c12c1cf46f2bd3aa5efde29710e69a512122e7c9f7c25ddebd4cd8',
                     'a31bcc6af1b7dc739114ef1d8037be760b99a6fec1c93efd3fb594bc06f99358',
                     'a33dd56b32848a246dff2bc7f46da43a29c49ef8a7a669c4bfd942a192bd4aba',
                     'a35875fa300436200c4fe9b56687c01feb06f1ce504fc6d3977013643a925f37',
                     'a399dc2f02c8d4432dc14604458e3ab591b6205a8bad07c965efc84ffb7384a6',
                     'a3bd77ef4e109265d159ac450fd96b2b2bd3630fafdb57fe57fa57a0519637bf',
                     'a3d52830978adc38964ede3026a65893f5d05d1169115ee550cd04823048c830',
                     'a3db4522f6a2f4ab697c3d108848d708113fc8ef5114ae564d5fb214dc304d5f',
                     'a409ed9ff99ba4e15aeaa32db6bfb186a72204a376899033f62a1ba270355940',
                     'a488e5780fb8bcd2676617c5fe2a78df84cbed7fe997c3ae13a81baf0b29bb65',
                     'a496df64d6db9c001644f1f3e41a1e45f22969effd772168f2feee9fd6c2bb6e',
                     'a4c2056540a85324deb93c58a008d6332ab9ce8a5a1b657ef975f2446a450518',
                     'a4faa72f0286e0f7e7ebc4d7c43cf6e553eb0414eb83ac5618d0c1b22a0037e0',
                     'a52c26a9be9cd50a57d2516e4c157704a2a80ccd21337649d4b638870c9f09bb',
                     'a539eb0160f433664ffb9ac52dc0f9ff61b21518dce0fe4d5d4b20c495a8f586',
                     'a54662435c1d76fa7e591b18593a38fb653a4f807158a3fcf72625a4e4679f7a',
                     'a55621a66ebff5ef62a96a300b1167c7331c2a9fad28396cdba67b9cf9d73455',
                     'a5e6bf45b685bb32c0b426e3b5360c1ed7645b067dc2973d0de258910cc23095',
                     'a6348e3bcd5456b8b69cd5e09b133653a211fe0634aacb3634a5f815d270e589',
                     'a6914c89e4c89af82929f0126a26d08e2297f650a8ceec97e0cf000a9fb65d4a',
                     'a696c02a38cce507ecad933f06a47a1a65c26b2c3c1c6452a0a403737f958a53',
                     'a6b40cde31c8955c49a6d3c167c6fd82b1a4b277b6033f933c3a8a2bd8fe3107',
                     'a6b632bcdc8e1b77a219c6464867b4fb3ae55d453cd0fa6244e29d9a54bb80a3',
                     'a6c119963b0d00f60e8b58e2beda4573aafba98f9660d001710a6021123d471e',
                     'a750a1bb9d6934dba40c0060946b195fce0d66209f03846afc550f334f154929',
                     'a7a207334af3649a356e6a9b2a05c05300eef762aa73627e9773d329704051f4',
                     'a7b5aab179ed65fc4b0ebe2ebb28a31a8535f93fa3e1f899dd3c6f81053b8deb',
                     'a7c50f21b0490ec573d73a2c1a92861140d8d6ea65f4edb0eed448c849687f97',
                     'a7e3f1b3772181ebc9dd1f14028659b654087640535343159281822e5644994f',
                     'a80102d0f0d2c04c19e13ff408076c8b8057795029609eeb31a1744436932565',
                     'a815408dd658c55f3e345691960afdd13ee58707815666d3fbd8f1e32f3d5cff',
                     'a899cb41a8bab3fd9f422610392f65f2f0102419294af7724cf72ea417c4e9ae',
                     'a8ee1c3079e2cd9b194cd1070856e202a5e86e4070f2190566098b8645cb7bf3',
                     'a8f21e01fe3828636c0a50e126554627ca10371815e24d99d2b167ac6f896cdf',
                     'a938cef46b1ad6a97ace5276cac89a6ebee184f77ede4f300508b9cdf5d3ff62',
                     'a976e717049353a5df7197df29909477b5f31d37d9890a43f066280588e6acdb',
                     'a990fc5b32d6958f31feeecf0357c7e79ec463f418f1a341ece481314f7d1f55',
                     'aa0b920e1fbabc76a620c87fbd5218f9de182d66e2e9a2f07fd3fdd3e85a25c4',
                     'aa5563a85b650cd318402efed4f90bdaaf1d040a6603f1af4fecd17bb1a3d2c3',
                     'aa709feaa1e3d2960c85e36455aba82e5cf383cdd997f83a81e0d79c8fc096b6',
                     'aa9fede7e598760298a43ddc1ea9aee4b4b4e6efd6c716f84196cb293abe8c09',
                     'ab6453bfbb3fff8ba6aac2da2cfcc891d733bec706d8eff493fb4879dc4d9131',
                     'ac52365e2ff8a12563648794f6cda371e1815d371f05cb65ccd3ccbf49a4b2f9',
                     'acaac93c49422cca18b7bd47f6d02f7a4086a7925bba9dad3895b205d0a0ae5c',
                     'acffc003128794b2f892293c555a34cdaa0ff31b351158e2918ac9ebedbccba6',
                     'ad9ee9207c21df2aa11f8b0ecd612f39f9216d8659949d4bfeda2f3d62c76880',
                     'adaf8b8b6dc4d12aefb4a5e30cf3515f7cff4ad89358d2d56768d7c02230b4e0',
                     'ae2015c7a2ca470b32af4321f30eb839dfc3ef015fcbc19e59092fa11c1f13a8',
                     'ae21b433cce10924ab235527fd783b53a603cf09cd3668ce414262a72696c815',
                     'ae6fc32d44c547003328f6d51bbdaebe4decd35953aeb8f4f7c87ab44d6398bc',
                     'aea678149050e99ba73153691b0dbdf0710098c8d944aefc0b72b2b32700b698',
                     'af2bbfaa1cdd9aec1d5eef54f195b70122c3039bcad509ecc78233ffd5e52f6a',
                     'af6d5f0780a7136025bba4d48b7e3fd8489e9051dc0323fe131e346eb28844d3',
                     'af7c7ca13c38ef62398372003b12e59c8e4cbb2a2bda8bad39afcf4efa0d2e20',
                     'b038b4c09b549bf2059829904902303c86f5d13f37edbdb4d5f0b27111ef9668',
                     'b038bb40e413a4e6fc00eafda15c191b31b1f2c125b59957ffd89fad51e184c9',
                     'b15d370de3c38dee39f1c718d58c1aad17538d5cfc41aec76cd57dd74762f3af',
                     'b21922a3d4de5f21a041a9756c92a69bcace3d7fdb2f12e696b708047f9e1edd',
                     'b26a3ead9dfa4626bf6afeafb5ba7e23d17f0979408f61b12023b388d132bece',
                     'b310a4a603a4b7b49c4f73fd15cf1a8500625a65fc9509547ef0cfd5b13a3678',
                     'b313fe9a426189bdbf0fdf969a5f6f2a127a0ab756c8eb4b76b4a1b944d3c29b',
                     'b368f01114bbe6ff3fd1279127e41a853ac52887b3ced034024d07d6f82f34c9',
                     'b3824000098fe2a3c9a9841314ff4980370c38b12f71f760003b66a61818d9f7',
                     'b391002be2d84e14954792a154edf59b87e7f0e3ead76d2eecfbe775f833d371',
                     'b484b13816596cd872e0d5125e0c8728cb428be2a1127446f5ae7823da26cab9',
                     'b4c61ef4d26dcee7928ed6fee2f904cdc98138de00b2673fa2eebf06cb334fc9',
                     'b4ebac3adf1ea2f7036580c7fc75985e13ca189a8d4c48aad11c0652679ebe17',
                     'b4fe52fb4349d23096a5f30437abfdf6361e281115f68bb5bdcb8eba56e4843d',
                     'b545a7301eeda10758632efc66ff54cd5cadb987cdda65878c98a4d40d40d382',
                     'b559c50b6ea318bd89bfa111e302cebfd84550633082ce995cc0e5fd3504c01e',
                     'b5c5e3230703e8fb02da5d7a7f5283b68c489585fdcfe672c5dfe12ebb3ae2aa',
                     'b5ccc17c1dc51e0266ecccf365ef2eff576cf0b4b781e48cd571996c190bae06',
                     'b5fd1f7d3fee17d1659fe5f93380db2a71514aeeef88d874add9725d39c7923e',
                     'b62b27361add7e88063dd3d41046eac188e6b29074f4e426ff4f0938a6090615',
                     'b630a2132166a89323db96d7791f8553ef92f42d09e771c252a88b0772851576',
                     'b660783e297bf955a11c1927e3ae18ecd28e348136937d220a2ae2cc8773f4fb',
                     'b69f30c601a3d0c67cb5fa3c0ecbed038797e08ec028940abbf4574ccc7e7361',
                     'b6dae4a3da1577c54fe51fd607f131fd05b273457b9499ae5441a8d2ff800f89',
                     'b73baa26e681c985f3b9798957e03bde2b4e573896b3904287a88c80a2b93b3e',
                     'b73f28d24703239c6af8bb16b80c7afba5ae6a6ef35f719799fd4a56774a9a3e',
                     'b75ef8e8cd44faf109ddc477d87262566dbfb69287947f859ae0c6344903eafa',
                     'b7694a0d8cc90a321d708c46804f1185bab74a1070aac880453dfe932e9e0ccb',
                     'b7e3823573fce41e1c8a714e8c04dafeb1875e7b31dbc509d045d9343458294b',
                     'b84981587fcd77f74bb02fce0f6bd4369b00f9582d089f272324d8eea13b94b9',
                     'b88923957715de2135cce6e9b26478c3c43940bb296d3cecdf2b95902d837ff7',
                     'b924bbb286ab652117d29e05849c1e13e9542867030de7b050ff1fa2e444c25b',
                     'b98e94478d3a1a2dc968b22b8514b183b7c73f69203366248338e98f11a5f8b1',
                     'ba26fb73f55eb60ea55bdd0c5b8b57fac3f708a6e65708411ad01c9d74ee07a2',
                     'ba31ae258a567cd94531ef9009dcaa08622bed1722bc458457659f9dbcbeb408',
                     'bac1d1dd9d14912cd89ae645f71a16f899da27a8572a9af755efeacd323bd200',
                     'bba85602ac4e5649950c167c080f4fbf06e8fe4faac7caa1a3c1aaa1f27b8dbc',
                     'bbb4d6e5741265bc87dd1898ba80c49989f97854f4c489fff088cb624166fd54',
                     'bbcf644b5adf140abfc5dba6a7b536a3af260258dbff6237bb1cb0bf8e6d7605',
                     'bc63ed066c67c28bb0d812fdfddb8eeb4989677760561b2c23398b8409141054',
                     'bc7069da33611ec30ff0a67d51895a7136093053d6ada339bc5426bb331bd3e5',
                     'bc731129b257d06d9127d9a1f96384f33c7320e1a8c4df30bad7a4da987e8d22',
                     'bc7eb8de8925187e8cec597e27caa7eccc6464027220e47cfaeea9c88c477c5c',
                     'bca6d7a387adbe4d6265334459c69cccebac5e0e1b4fc2aa8fc3b39787bd216f',
                     'bcff4ce05719705fbc30bf7af94ea1b2867134ba610f2b7b5b2cc6e97eb1b898',
                     'bd0262ecadd5cc20ef2409c09246847e571256f21bdc55186bf42979851f553e',
                     'bd2eb87777ce359dc2c616e117de7fcca2b7535b51741695864c36187bbd8931',
                     'bd3887ce77f616184eda2fd55c1363c99768a15df3ecc1fb9a1352d0140c3da0',
                     'bd756121532e8766d1f2dc4b5cf8eecf591c0b77fa34adea6ad8b0cc915aae7c',
                     'bd87844fd3a0223e45940f8f0c24481c1e819ae0b3421ecde3fe0a2bff9e5de4',
                     'be0a617f1090affc229d0c10ef0c46cd983a73f5d42f3cdbf166e95313d8b2ae',
                     'be11dc86e3b3469143ced4e0df76de5e4cec79ad02182fd3f78a66c7ca80120f',
                     'be71cabede2b75f5f4225fb6f102eb667f889d4c5501d4b268155e4b46b05b13',
                     'be7fbfce2e5921bfffee3a00c06ad7c5879cfaf0fbfb8f7a6f3ca3f5900f7b51',
                     'beb0bbe1093c7f232cbe1f1cb8da467265c671cee47f51f114697706b5f9261f',
                     'bf025830ab71f7afeb66ca9a28e27e456a5928119ea2c18d00bcc6656170cd0f',
                     'bf3a56b486e0a3b54c875a4f7071b8c72026de84fba0370eee6d73f8b6083d0b',
                     'bf4819ae63911316798b91d7663f6a48c686930e1517305b307da44a68f8da26',
                     'bf72e50e77b84cccb3ee457ba4b78f0bd2350a15a6ecd7cafb6d4c2f3681e86e',
                     'c01881c04b7f32256bfb77bcfcd6fe882b9d1150ca1d5a50d8bb65163b4c04e6',
                     'c038d74af6c548476bf889955b5fe53e678ef03ae1d3822901579d15c2a38b2f',
                     'c0f77f115e4502dd098e37f9bbeb7cc2020665e14c5ee461e2ce4b515eeec76d',
                     'c1031d72142bb2a637870b199a826740a1a219249aa1f913ec28080cf3fc7f32',
                     'c11abd196a83a787b797ab1b2c4923bc8c3cd36cdc905cb69bfe670a4fe683d6',
                     'c11e050ffc6028715b1b944742b976cca52912f4c8e9e4e9e2b2b83e7e710a03',
                     'c1687df1cafab395dc4667d979a03d93ece90a53af90e000ed55918c1bf6ea82',
                     'c21e4d7fa110e18f14f911efd7181fa2363dba1032adcad2492796d7d988995e',
                     'c24cb1b2a747e2b34fb0ee2d492898363a771dbaa738ce1dea97dff66823bb89',
                     'c36125e5ff9bd3a281dfa03c22d11538231fc373668feafdbbd63c1d451cba58',
                     'c3cfd6f1aecc0d7d66cbb2ba31ef1dfc4d36379e314052b18f1a0802d50be45c',
                     'c3df380a22f968ee266e1f9f652372b4af4d69d5aea7411472baf0322902b73a',
                     'c3eaf509b339c27bc416b1306da0a573458e24b00cb03c81174719495f8ee0a6',
                     'c42ab1a1bdcad4016ff71a6cd9911448ede8004bf79a66c8b0e4e9ea2d929ee0',
                     'c492444dc30340239f974cf8cc238b4c56e20c53c7d2e74e82a2440f5ddf629f',
                     'c4a2a1e053ad027e180ce34af277d236ed4ec5015c64cf30bf1fc9e7731ba666',
                     'c4da157f7c39aafae8c0661eb062391d442442202a3f8b740dc0e92cd6940299',
                     'c4e80e2436893859aeaa571736d0a5ba894d94ff08948128d8f0c5e7e2fa7164',
                     'c4ee2dceb58aaddd65a5b81f70497eb5b9d1b8728d210cbe3cbbb51f79f2e3e6',
                     'c5b481f52db15d2c57f790f18ac4da2280b0ed43c292b92b7e6e9a6359748d62',
                     'c64050d262fb76a726894c61f9e236eb0f210ff89afb4a466f6957ed6aba8311',
                     'c68065abf9be32ce7861f996cd150366c479e14e991aac58b4f9c77909b17c54',
                     'c6fa3622ed69437a80d1f23a0ba24d079d70cb17990dd72d39e35ca8eb714083',
                     'c7114b8b0457f5d66da3f5cfdaefc188b300cc66e270768a48d0df9fca9ddae4',
                     'c7a79fc1f02af283f8c3d41ccb94084ad4634a247694fe8a333a596ebe5ba49b',
                     'c7cdc1001c97e745aca47c5fd87bbed4f1304810c3dada34703ebb74d3fa4c2f',
                     'c7d83dcdfdb8d4d70ccd4d7d8f37e46ac5eacf3858e1d6073826ee3125c1a0c9',
                     'c7e035122c3cd95105d57dab4e8dc8a784b451f610e7e27ae868167dc57b33f5',
                     'c7e618454af33dd23340e8ec92d344dd3427cba435323ecd7e853c14022297c8',
                     'c7e7d5b4bd057a2d920a23ed6d9afb27e658179936767a73cdf72eae83798ce0',
                     'c8a7e9d6f07edeb91b951762b6a785d021fb412383de193dde325e2cbd00aa6d',
                     'c8ceaa52532b8a203d78a98f3075d7ea83ddac86d83c238d68730dc7ad2243ab',
                     'c8edcc3f7184a8c812ec255b2fdce5450699632b4afc44ffa3632eaba984bde1',
                     'c9bdb1f1f6498132c84cf3ed98fd05678472043d35834aa0d144337a8dda4f77',
                     'ca04b5d93626b9c802db94d584ea872d2d4d8292ec5f082c90818ac6fbab5c8b',
                     'ca23ec30d8348ce16a99baf9e360dba6440c67f05de44e7ab7019ef8169516a9',
                     'ca3cfb8173eb54060de39f931ac3d407825b65260c1b461424cc032efa849552',
                     'ca81fc43af7c64f7154c6ac29db881a181d61da42cf82d2617f6d70204af512d',
                     'ca86df99a10963102a44ce193d4d7a280965e46474166f1517d6cff55ad6141e',
                     'ca973d22482a4c8a7abeb6b7cdffae5ab02d2b25e7ef0ba9cb309bbe923b4059',
                     'cae555cb609253ed508be00d4aa6b6b4b0a19d765f48c4c516950737e527ecf2',
                     'cb19a5529faa2463a7bbe73270eb7d2a80853cfa4bf13b3bf1b83c0f67e6f57f',
                     'cb403476bfc828da561f22d80fa6c10956012eb16e69e0c9d1f076737a19a589',
                     'cb6dc46bb6cef40661228bddffa9866b16f8afa10e3fb9c132e720cac9b212ca',
                     'cc03caae1aa3dd14afadaff694dc1f600845a8f230f464bf4e36f81d1eb3f0d7',
                     'cc359600371abde4485f388b87616f589f5228e0cca128ad427a32c13189bb6b',
                     'cc3bd6fdcee10168ec426e8680b0f458b7fe3fac71fbefe3c6276b32afe42073',
                     'cc4da25beebcee8043a745438aca9d5cb7db726eedb373206f0c44b2ae2de1e6',
                     'cc4ddb7541bd48fde370d6c3f7497ac0058ad4aebe7f4c9254ca64b4e35496e9',
                     'cc54a9a090a2fded909e70c0526911dd47c0cb60458175dc36f6b206da4ca06c',
                     'cc86679651f5593decafab9d26150a955a7fdb402fbcf5bd91d31f0af1c9368b',
                     'cca03dda23bbe014aaa17a9e51a66c34bd481cbfd73b942d74e644936e136022',
                     'cd0e8ec20009a6ea5d0ad3c9f4fb13d8c7f63d041210414ced946a198f077761',
                     'cd1b1f48ec5c57c9ec6140c91458cdc65b8b698c4f81f2547d6856167af6962d',
                     'cd66b977fb9dad411fbf3b2830e3ad4e73f89b7c3aabc481d8c76ee85d964e2a',
                     'cd80785da155141ea207b6f66deaf78d185ed5e8f41ac6dd58055f0ee1c54ebc',
                     'cdb74e18bcb651c95e4a9fcc4e5e6f585442bc0083a25e24793a99e907358334',
                     'cdd46c341d141dc0a9156546726bdfe90581eb5ad0ac6d2f5b5b1a04ca3dbf67',
                     'cde315e713ded0863544545a4f50bcae520ee4af5aa89bd0297d654ed0f8d68b',
                     'ce77c42d9cfd2148934b6bba5adbd879deb6933e378fa39943f145527e582bd6',
                     'ceae795083b4bd31c52fc6c6915c6f71f8840b57dd285ede753d21757bd84211',
                     'cebfc6fb7a013acce43261abd4f0b07a3d74444c037e474ebb4cd7a53d4cfaff',
                     'cf0311467347d93d724f8a782c72a9907ca9af718d13075661e471540e30194b',
                     'cf3e1804573c00dc978084f5d4ca9aaa56879a280fe2631f3c853c074bdf65cf',
                     'cf513ce58259ab0ba6d6d363410cb90a78a40a33dc0086200a36fa3c4dc9324d',
                     'cfc65174175f84fbcd577a4c7c194d898f55ec6547502c3281f89fb35c4c9a95',
                     'cfddfb7feb0220159a6a026254eb4049f1eaee8563532998e27a4b41280fdded',
                     'd0143d7aa4de6887b1464913d08707509bb39d1cbf9c81a5d128619852e63d1b',
                     'd093be4212c184efbb87daead69bb0b55d0da6e84325eac9f33e65658a1d7f8b',
                     'd0eb40a24d2147cae5b08cc0bb8774994c2cce989266d686507352ed601e7f79',
                     'd233430dff625ac4747727605a31a30892ba8fdbc2a69c831177d1cba1e80eb3',
                     'd236b796c8d65f93d3df173d26ea568d7bc794d45f97a75933729dcb0163b004',
                     'd23cbd926904691315e39a5d935b7dc3b256764bae2547784b3c4d631ee11947',
                     'd27372a7c7e6ca823331bedee20131e5cb477e6ed886217bb00bbb690e6caa62',
                     'd277aea55028d6ecf6362b67a7c62d75ddeebcb6fb19c700d0ddf01f58be84a3',
                     'd2b247b474ae9ffc26634789ee1478154e3d816629e4571bf77280d09a1e1508',
                     'd2bafcbead55ac01b0678608926f787942d8c356fdae79b2a3930cff44890b70',
                     'd2ccef9b3bd71cbe5ef6ed83c740730ceba96d16cde0fd060b34c59c74b347c9',
                     'd2f6845706eb555b10713575e7b852454bc3b5620e869a4bf7a198ce56879777',
                     'd33826f37c63c459ca40b9e5c50d65ae298a46472923135a10cd5397e6a4dbab',
                     'd34727afb4ffd8d7c3f812156cfcb52b7dfb0498b7e5ffa3247a59fdaab40e44',
                     'd366fab0cd8aa3ee39534b74c5c03be31ec6b385f26bfbe7a75fe28bea870d3f',
                     'd3ff9c97574f8866edc51cd1023c49a0d5e614380650d7a04c4df5d4f99b3d72',
                     'd49a682bcaedc5bcb442b784967a4ac496dc470f6d3c2945f976314f71ab6858',
                     'd4dc0e275bfbadf5bf612b69fb6c9fc661fb338c7cc6ddd5a47d49b05ea85916',
                     'd50f39bb907d192d963ee4f7308e890ea35a55412647b6a515f491c5cf251142',
                     'd5d3911f303bad74e013b8b87950f8b3d9717d4bed508ed5bd8a21d9dbbf9b5b',
                     'd6e4b7cf5a1c0121456ea0836c6f10a86ea90df1b72f592b645d82ae3064f056',
                     'd72c616c02ed0095d22bcf3eb891edb90a1949ecd5f9691d35b6cb7ca8e194f5',
                     'd7c4e92cd959583908e7f3b211572c3caa69be2bfdf770ecad9dcc4f7b2ed71f',
                     'd7cca1a9ed20b8edf8b50e8882cf0a1ae705349957bff78211dbd08f4bf5e192',
                     'd8179d722bb94be1465ec1c6521a5f901ed7d42cc0f14790237936f603b30056',
                     'd8221b6593140465a6c000b21a9103811d97242f23349b130bc637065daa625b',
                     'd840d6a5d2a5ad8bcbe53354a243978687f89f8f8e2e32974a2ff3742aa6d7c1',
                     'd88af9872162ac4aa114a352e8437afbd24d238fb639b8e73f2abd8a85428fc4',
                     'd8e603fd5523d01ae05821e07851f469cb64142c8f3e192836e75a89f6c03dd8',
                     'd8f11414f5edbc58a8375e336d8d54693a1a72ce99b4752494d29ae0971de174',
                     'd902618e5fd70fafa264b9403794e9b8b8c94f54611422caa937bd5d1f2ce08e',
                     'd92fc51c6146d93e6046933a8a129b6f7d48c3ee002f438dd453d121b2ed597e',
                     'd9341c759c0898d91307cf9b3ae5efcb25bd5384b498e2720db8253ca7e8188f',
                     'd935f8de53e51d2c65282043779e213bbb30bc57c454eeef36dde7194dc72511',
                     'd969db429415ebe59b6d8d77857262ef3a03a1746e0df78b22d08160ff6c1e89',
                     'd974955e725729cb2fd395cce764af1f957747dca7c515c416c6c450c63bd758',
                     'd9a04f6fb43b1d111c3f2d7c607582b85df021297e74aca4c10dd2379c668d7f',
                     'd9a08d251e3a48b0a8565b3503f5a6356859b19f03a7c556f15510af762a1ea2',
                     'd9e16bba25112e60241dc6610d0ecf9a5cbad1bf36cb818059e9c0cf53a280bb',
                     'd9ef933ffd908fb9676fde4f74f2ddad1fa37cd86aefb0511ea543190a1c6ab2',
                     'da036e121151a78ab1fe4a4dca346f897200d65598f8370ac943ddb8805ec0c1',
                     'dabdb3ad5aab1e2c7606e065f9ec0b5ccd222634563149176046ec6a8ea7faa4',
                     'dadf3d07ecb656edfca3fff9ee957ef5ec1e3c14d3d2c1f823e0e09db570be5f',
                     'db13a623339e7eb113b514e526b8dd4c48ea0bf4fe5e6b1ab67e46e0ae8755dd',
                     'db6de543c42b65a7565657a3c4e77caedf2511b0c811127bf869b31411c86cad',
                     'db76760e12ebdbb1902cec332f7bdcf24dafe33fe8eba0bf7ab0a2b05006d434',
                     'dbd9516dd9f7342bbe1a2508066e59241b26149fa63d2ef2ddf6679d0f889c27',
                     'dc2d61864f7af14f4a8eebf7898483f1acc9e5b2f7e4ca2da3caaad7679e61a8',
                     'dc2dfc2cd6c6ad7a84d41976d2785c76558136266719687a319cda903c5fb043',
                     'dc73c114a23d9a276a30159b29644b60fc23d7e8b993dc60033a4b19897cf786',
                     'dc862e6a19b75191e5e06461b52f356a0f6fd7d7c11c8715271fa2109fb1689c',
                     'dcfaf7d2bb65afd16465bbffdd36a52c415e9dca45dcb8e0cbe0ba7d640ccb33',
                     'dd57973bed8793da8745cbbda0c478556fd132b0e5793b8ddc0d8676da89ca5c',
                     'dde736197952b12f8c239f70f050a129e2afc81d37fa389547f3470dd790bd2e',
                     'de35b8d3a9e2097661607d71f0cea4a81494e857a66c798d9d0fab7e7c877cc0',
                     'de61449801dc066b44fcf0853318ce58af2c8d984a0126742d097380c49ee751',
                     'de929ec280ef96561b8735c5ae9494f057cb9de0720c5f0972708c7cc1fc70cc',
                     'defd624650aff73657630419ac2b51de85a2234d6ef2f9e479f38180346220fe',
                     'df1d175b14bb0ae259ff6298354fedbb71afa25daf4e8c524795ac40c10ae23c',
                     'dfa53fa00d7f6c4b7783367e4be0a16ea069b4e89eacbccd612f261ed079817f',
                     'dfb3fdf3af1d7e368f14fea574b096c53e89d5cc1d6c477b5a63022300086430',
                     'e01644a008693a71a7f2448684091f0536062c402476d16309bc418fce52f18f',
                     'e080309ce951fef55a1352366e8afb4b420c3d6871c3ad673ce90b103f4e93ce',
                     'e12febef05ccfa43359e560f630b571263f677b2aa42038626d8ceb05b7e1e3a',
                     'e14457b5749ec675fcc92e2ee2dee0898f73fea0aa30562934ce397428435a1f',
                     'e14b2a14b3bbac0f69e14e44c4d090c53b2ef144db2c101bcc0b270c4e2c6d14',
                     'e1566d858f9a68cdaf9896631e6d6ab106ff599b5f8db532138af0b42825b7f2',
                     'e19e9e9153cf33a6769c5b4cc56f7591144dd973591136899f83cc9b5cc852db',
                     'e1e99ea08226b72a67793ca36c187695ad647d021677edc3f27166384df63c8e',
                     'e2f65ef69904f43aa56eba11f5b03451df30d446cc9e0e1132c74fafe4d9032f',
                     'e2fa79999a98f5b3cabb16ebb588174ddc4d05bf08863394efa723b46dca8486',
                     'e3164b923baab1fbca4ab356b80fb9e1a9daa944bfd78d9268b11ae8434020ce',
                     'e32f5f34729d144d028482e251ee067bfe6535a534dca4354e6bd5cad54b7c83',
                     'e345ec6f23f6de8abc8194790b24b86bf456626076e1deebb013f4bb9ea863e0',
                     'e37c886c5c2caa7041bca9e1d9608d3c04e509bf7d48e4ff4f9e73c4687e5f5c',
                     'e415eee54ef7c2e9c9a7cc535aee24aac6693b5434dd5f06b0622341561e9457',
                     'e4ef9d3b81bdc0e92111503a6c60f0f25bcb1130e707e815cfaf8b82b0fa2993',
                     'e5534e43cdc30b89af72785768e9decb0a60ef74f3408cb8f8bd2ebec44589ef',
                     'e574bc98610b37e4c4b4422efdfb3666db6f72bb1b67388ee655b163cb962ca1',
                     'e5f04d86078362171936c15e342bf08ca88df1353a48502a6f1e7acf131fbb83',
                     'e6357c6de6c2352032c9e3ca16e149efc5e7e2bcd5b5ec4a7807dd99ea7b5034',
                     'e6586e2d84d8ced4cf02469af1e39686e6a3aecfe97df9f1c373d6f4b8d3a293',
                     'e69d0876a300db1e031d0cb0a01f3bb0e63c01276176fcd5a2cdd279c07af807',
                     'e6b0566c4b4dc0e89ed0cc5d4f622cd4a54b0a9213c232b93897d011b6ee8b94',
                     'e6b79cb88cf921708e332d1b88ada87026c0b04483bcef7b9f200aa5f973a996',
                     'e6b92e3fc05a80b270909b215ecb8fe6236c4da2df3d93ce225675115c19ea84',
                     'e6e4d9466de21843659473842cf8adb8588b097e6b065e3878a77576830bcdc6',
                     'e77a6e19909d4ec2bbfce3357fd7518db1042b0d5fc947db9e5275db886b545c',
                     'e7ab25d666f074fc03fc3f77490bd3808e3fd77d7d8f081c87e5c239e4948753',
                     'e7f46b76749b6c4ee329ec71bdec6a14b0c4f2e1489d4dc417ba63cdc699eb4a',
                     'e8272dfb2f764870274fd990d606a6403450a302ee0206e224afcc5102b9a003',
                     'e84b504de234cce8d89145cad2e9450606332704ea3b1b477ef68835cfb67937',
                     'e8e1d414971b5b6d696f83ddd2603a7021c07a3557c226684628ddb312569fef',
                     'e8f57178c9dda1ad3b181f6486f55c5564a20f1e86a4d56655537522cd01f21b',
                     'e97d01c10a20c598100f1217c569803de9c62a2b64a4e7a17e05a474aaacd97d',
                     'ea1680874df1971470cb0a4155aa99d92d501c8eaf27010f2ca17c2904fd2045',
                     'ea428fb6437b8ac6679d093becbfab21602e4ae2df58a26d44448fe2abefef96',
                     'eaa9d7a487710aeb6548e16c2e0dd3efc7daa03231ffa735fca26ae91d3bd158',
                     'eab35ad7c9758306b0ba00d707c82fcb235c75d96a1cdd5dc1f82481ca9458dc',
                     'ead0074e4e8cab8deb3642e4d79a3906706375233c3404d351c2e94b9a461355',
                     'eb024b1f260cad8cc06aa53080d3c878d461a76e712237d83e234b46a072d00d',
                     'eb1ca8d5f10b62163c9ed65701d2bd1e67bff9125c8b7f0bcaf5c31f1f02ac3e',
                     'eba55db0c939c9dfc0c4b393858f5f871c1531d3189731257c22a23c0037179d',
                     'ec0c2972c0b16806d604aadc9317cbadb056f7d6c5045396629d20783c1392c5',
                     'ec1168b261b9ef82895992f313604bf97e081c14ec81a9e3f940aa22ad97b950',
                     'ec95b54ef913d7937e0137da18fe873a8fdbc145e4dfbe3b9834859600f91e40',
                     'ed55f699efb2f09689d7b8ea4075a8a610a7d0f7860a833eec9c5d060439f5db',
                     'ed66f190ab90f84295ade5cedb41acff6a251a8c5ca738c5a7b7c41522801dd7',
                     'ed96c947fa5895d0ab7d38332e4019d5aecadf0253ff0753c8047e0e25306761',
                     'edbd398e13fc51376fca8effc35b797d5dbf0cb77cf3a12eff6169bf27e48d49',
                     'edfbe86f13ceffa5042815ba102513ba8d19f4ad9e17207503abd7e831de69d6',
                     'ee4cec3bda0a56d6226781d4402b8f91886d09f18e107e3c893b6a6e499e4c69',
                     'ee7bf71097a5622c351f08abad14c11a9adec0b358c858c83302ee7d4b2fef03',
                     'eec49884333a7884063392202575e3f3342aa6a4ae4c0d6a9b93db5db0a64b9a',
                     'eed09b78d2b484d666a5fa6e084df72e2b60ee7122deb52a52fd05e6bdd2ffd1',
                     'eeeeb302b91f620621b0af659c763796161ad1af8c389cdd0cee8b79a0b34c3b',
                     'eef1bc0bb45b1979e5f69920c3c7cf6d030a4ca6accfd93c0a6d36a64ec93391',
                     'ef0cad2e9d77116b8fd0882e5eb722827358172564f4c8982a214a692d26eaae',
                     'efa84ef539854106d893867c732a9253f0dd32b89e3170363ee0c9977d7afd0b',
                     'efc76c8399762286029c3f42f33dccb6e4d61bfb25a558ed7c091572a2d63265',
                     'f004393b40719704ee099a14cdae647900ab407b220946d5273feeb301e5d08b',
                     'f0318103d62cdc10f51419aa5fd71ce6d7ebac8b283a78f09a7e11b13072be12',
                     'f06533715e8df31c80ea9db678f2da295132190f46f7c80eaf288b5fe9bbde2e',
                     'f06805289345f5e8b24c25507edf4d45316d9775aaddd18c34a0febf3277ca1a',
                     'f0b363599559a9bbf9ff2b973f655f010c1478c84d74b6fe70d388224710df09',
                     'f11f56c76490305d1dff4ad0f39fd03cbb4208ad618fee401c3b25a85ecc8929',
                     'f1527fec7f84aa65818f46672a38a39617ae498e62ec741c96f2a16d6a9d9d2f',
                     'f1c35a4bd9fb0eb3f9bb3e97d80846c5432561acb6d38688311d597f7d6c7c4b',
                     'f239b953914ed24b32b8b0986cdb15de521f70fd05bdd13964edbb30ae30bc3e',
                     'f466b59899686c51c42b17f4105047d0370cddad50a64031e09a9a548f5da029',
                     'f4810de1c010ff776db227074cf01b562d5adc6a25e68cf35ae4e9a7defad789',
                     'f51b99c78324c461e23f3c78216b5bf26e6467fe4aba1a350a5fe17494e9c479',
                     'f51d6d9efded51863601fa04eadd76433b8cc96e9328798d1045365937e53c21',
                     'f58e07602cb7a38f1c9d5ec35863924a4ebfeaaed4265c95388570f0863e744b',
                     'f58ec063bab0102aa2f7d27e3b13d34b0e31e5a60af64d31776e23e1fb29e376',
                     'f59b75a7b0b397a618686b749b51232ba37dadfa465451c66c413177a2ac29de',
                     'f61f6eaacfc4b770f5e69eaebf931af158305ca1fee7aa2454b038245c20b8fd',
                     'f6dc04342b5e3813cc18175c06c18391fef1563e15849d5c14e9e598ab194ad8',
                     'f78440d114a63869ee60b76aa3fd52069d74fcc5556dae9f7e362266a8cf029a',
                     'f8126fb31bd1ecea2b84152f6674f5b437e561102e63d6f4ceb03100434d7772',
                     'f82a254bd1e8a7f922feb52bc2ba3f8ced6ee800e5bfec15cd1e9f7ce24dd9be',
                     'f897e5a9a258dcfbe1874c2d011be02e0cdf9773fc8ace2c509ffc5b1d56b732',
                     'f8d639fb05aede8c77161e4a506de726c75b024f39c5a54ca16751c72acb16a8',
                     'f90f20464e3533fca466bf0a2a181fa929b9720814297632bb539e363e3763f1',
                     'f927560f9717e9bfc3db93530abc7e8654444162c84677bdf547d4018c0d3670',
                     'f9f9901487f4c64e36c4441e00362a9f012a945b6db5cfe2030fe800215c0bb7',
                     'fa0521049456a3323e3e3c5cd39274a4e6e3a9622f3bf4e9131f3245c58b61ed',
                     'fa6446ab014661dfd372f375987f0c5cd83701b7666a31911b7eb79b86a92e3b',
                     'fa7da05a1023ec96754bcb456a7bb4893f96cc7bb3f62c08c8c84a9d0c88881c',
                     'fae6ebd585ac5b0f558231478ad79fe19dd4c58faa94beb2cb1ebfa002dce89d',
                     'fb162a0c965408d952024767012c39958bb7fb6a0fe555ce107f7141c1f906f5',
                     'fb200baa05919dbed625ba042f147faf717b52701211b491f9204ff696994feb',
                     'fb2e3ba905df1ae445ec49e8177940a17407e04bd2059a64715740039fc92d4c',
                     'fb48aec8cacb40a3851e8157d5ee9a8110888306f73c931290b897ba8607a55f',
                     'fb4aedce8f218b1e0bca705da987cbec8072744ba1b1ffa055b0f34bb0565b68',
                     'fb4dda03341891307ff3f1f0e8f5cb73ab6c9b0a44d16e77dd4ea273ae56f641',
                     'fb52cd526ff3768381eda8772955dd139630b1c92448d3ecedd800ecf14eec90',
                     'fb66870b3d4622b4393f5523bd9ee8463da677a02a08d4c3f46169d4c509abcb',
                     'fb731d322a6128b06ed703e346e861fcf0c2d9ab01211bf0dad4967c486ae3f5',
                     'fc93a2eb65773a02a1711593d82795c65528f9212f025e48f79e6f9eddfdfa54',
                     'fcb3808d9e02251e3e77933b60473c7e286a26c0d435c52468e019e418bcd245',
                     'fcb6a5ba8ffa2df8138bac17f50d72fdb9acd7d0a3a31f0f420803f81fc69264',
                     'fcd4618ac716df616eba2510c6afd4e11c2f67237836c0c97d6ba71fec1621f1',
                     'fd91f378d07178bc4ab2ffdae4702a328f8d3e026d9d7f802109abf05df933b5',
                     'fe39d216b6307c6809c5d4acea9d4f5e40e79778ec05e78022f3c33dbcc6c4ff',
                     'fe76d9c7b46f5c29f6db376129e54ce177326809ef784ed3339be7309603bdfa',
                     'fe8a75ed93a09e5384ceb4783f6baac43949d1a9f2b543a35a7681da6d053480',
                     'ff29150152186fcee2398603ada81055195bdbb1721361d4074d2581ffcee0eb',
                     'ffcf3eee073fa2bc0daa9316c1f848e79a65ec27e8f63b29f7b7f2d1e80b7898',
                     'ffdf6d08f16d933fe13a421a52df348c6f90fc54bbe10e4a76c9e5b2cb21cc9a',
                     'ffeaf0b81a670d518edcd623dde67ed4a34d40754377a0bb6b3c56a4b382d664'],
 'count': 200,
 'data_lock_sha256': '448d5ff6f97c828213877830d73f7f629b6e0973ad109a92145c9cdd1639bce6',
 'eligible_images': 500,
 'entirely_new_external_dataset_claimed': False,
 'excluded_roles': {'prior_development_200': ['0066845163f38a25828bfa1e29516b0a472661a09893d66fdf611694ad8f8284',
                                              '0127d7350d9710721527acfacfed0492dc9bacb0cee58108f994bee8cf03fec3',
                                              '0371dd8e6d255c728c45103c8adc87fbf93770559b1b2534f4fc848d1a58b9ea',
                                              '03b71f226ea44462a1ff05875a802b60fe89c41a952f879979956d32b8cc57c6',
                                              '0428ba35ff505fa6cedc57e046a455ade950dcd487cdfee7a94743c494a68d2d',
                                              '04dff21d7d781ff8ea2ed80f6da1a165e7797f9a398b4fd930bb935d84fae8d8',
                                              '05fa9cbc11ca5d438f4a7fd8f2f263f85f0f014f034dcc099a711dee58022289',
                                              '0679c8b0537a146003135e865121b10b489c18b661562795074a4c46987f2edd',
                                              '075cf56c692f03d94602f7174152164e29f6fcc3e213fe20b86394d29c044af2',
                                              '0e2ddf287e7584f51552daf7d41f4d85e9cd406da782015f5031a95cd95871a0',
                                              '0fa3fae44c1eb015fe9adc1c7dd2da907acce30ef080449e5b1d53b71a9d843b',
                                              '1269e5f1b5b64c8442881cf6aae6380d387eced7312ec5e89d677466967ee3d2',
                                              '13cf06d91aecfbbf8ce84553c0cee40fb0a2f8a428aa78434b5371780686a56d',
                                              '141fe37ce0253cb31db8fd9b604c4c9673f5b7de9aea16d7c00c4758aa6f35e6',
                                              '1628c2c04b97b49be78116becb52a0c881654de00604da82928aa7b4dae90013',
                                              '176260b73073aae1dcb99f05af28555b4682c100e063d4d4c45aebc6bb209a48',
                                              '18d63182f56c67e61fc78dc3179ad89ab7e443d6c95f431721202fe0129de547',
                                              '18e95729761855bd29115435ea4aa3720418e954fab04dc1996a423317d5ba75',
                                              '1932b91f7bbe4cde3fe574328ab25ee7c7c92f8a1c8889ade9a5bcce8bb4a6fa',
                                              '19c4310b6e09bc4b22082dfb82e88d33c8fd53daea76c08bbf32c099c93e22e2',
                                              '1b8930c4e710f3484d1b6bbde884a4c24f83cd8b77a694a59716e849ee506841',
                                              '1bf9feae32fe5ce0a380a3d37b4754ce1b18b63d05d6bb931854519ea1a661b6',
                                              '1d36b5bb3a75f95e0c2d624ba5acf329117b2a331f7f83624b0a819966a5e605',
                                              '1ed1c79090c93e3f3ba4503b11b19d8f4a027c9ddcb4394fc3b6859584ea80f8',
                                              '20f7ff1137e25b4238efe36281032f8e219e48ff2bec837c02d6941c83b2c46c',
                                              '2295f88b3afc6eb315197029301c0611399638894e36d518de076c7c95112cf2',
                                              '241f921d4cd85481e077b18605fb2d2b1ae4d52f954c86a8a743e603af0b75ec',
                                              '24cd687a287375d73d5cf90864e7c0314a180fe81af899cce5fe86ddb7f26d64',
                                              '2591258305dd07328766dcffb7232b23ae78a88a61d9a878ed913105cd21c3e8',
                                              '26b9c2917759aa9d90241b26b7fc1de4e64ad52b1a6307a5ba33095d054e19c2',
                                              '279cc38806a4e37f44a799845db932d4839b34e7b14b933146bcff349661d78f',
                                              '27eb5e3c07fcc322a18a8a5a63fd03877e16b86c0516e6fe4cd656da5e280c05',
                                              '29f63c30a424c41864cf024bff7badecc8a25f141f50236c7eaca44c851d148f',
                                              '2a40eac16287efd2e163305ea3a4b517ab642c23b229cc77ffdeb6f600e4e544',
                                              '2e63938b2fcd265175ae05748f4e266ac8dc01bd2a38294bd42d515f3a21ac85',
                                              '2e8151a6bb8aeec62a83a29bdb93303bb7265cb28ef5deb9fc2bb6f933e4801e',
                                              '2fad80f5c24928cfd53dce72ff07573f693291059baf3d78a37b139b195b8796',
                                              '3084c85b7fa890e77560bc72e757fadae863d9d32bfa30b3cb28cd74188ebcd9',
                                              '30c131076a387144a12eea1f98f9a4fca2f5af812c0e4a57d7325adff5fb0c49',
                                              '3220a60a4764025feb88510610f61f0ba228041e66a56f3dc4a332ce1b9a884b',
                                              '32aaab923cda5b9aa647da5818096ed7c6f2c9f857bf5c04f6238428fcd9bf4d',
                                              '32b668e9e325e12ef684de614ce85d30fd2b2dd3496fa6a196f3f70270a65728',
                                              '332c392379f7a2307826bc57368d861dae2c596475f3193d861ca586cf3ad873',
                                              '339b927f038ba1c46dff3a7497ba07609578b8a85ee910819aebe439770f57dd',
                                              '3558079ab6c1f10699922b3dae791721a79dea43128312de14bcc85f46d9165b',
                                              '36559d2545c16bb7eca9ee449b893c48596d2114b7b000a1819e16d4035e826e',
                                              '367126122831aa7424a9498c641885faf8461456fd6e86b1d8b915d920401d81',
                                              '36e5a076cf2ea05749a06cb2bcc26e4fdf39cc250f8139d40b8182d03c7148f4',
                                              '37e7e20d527ee6051d49773937c09beb8d178c20de96786ec44494927e9e5190',
                                              '38285e3424177b69ec49d8db865a0c83452012dd4a00b46e9dda6f406836110c',
                                              '383f1af4bb2eaa49811f0027d7ec0f89db9b340aef36616367c692b5c81d5f24',
                                              '384e0cda59102c2e30428d0327ed6d9de19a1ed8a5634423fc8d1456d950f920',
                                              '38ef6a56173fd38bd4129130a9632bc3767d5369a2d486900f8568e43eb8d9f4',
                                              '38f327e85514eb45fef5ae3eb06117ab2e16df93ec4a8b85e475d3e04265c69b',
                                              '39e1dba8068d2af620de27421472afbe76be97fdad46683b25745fc6deb883a8',
                                              '3a4cde911a3bec8da83cd0b644ee297788d5b85a0af9f2eff2a5cfcd779c333e',
                                              '3b8b929104cf13efab270b7e56ba0f38183857955095ec84191495eac11c740f',
                                              '3cbfaa6d4563c235b0907c2375dd44f3b96d591099def6946e870d66a2a86547',
                                              '3dcd17e02aae26c95a49837a391966b0b650d44d575c751e3a47bb1c73d3c23e',
                                              '40d9ebc0983e831d0f8780cac97d2dd788a681bc11f56f85d429311dceee5902',
                                              '413f07cfa1eb10f6413ea8d977b392d05de767bf4bda512038954b9a004a13a0',
                                              '41ff58423114103524a85024eb29549a416af37b154de6966992f43847ca0564',
                                              '446896b6489600ee23701379b84232fca38edc64e8c4cc90efeb579006be6662',
                                              '4477fcdd39a8d7496347489d19943f5e3034a19fdb1fbe7efc4ac80f967a71f8',
                                              '45b78077660897b40a0f02bd1166111058fc7a0c885f090a64562b5c95af7791',
                                              '46533741c06ed67856bbaf606dcd67357979e06c9434991d8cbc88aa5ede851d',
                                              '47d78f81622fab2b16724c81c6cf0c578e2fb2796f232bd4eb24e594373559ca',
                                              '499c8b5c58b4cce859361df6854f179566072c4426c8d9ed020749ffe7f9b50f',
                                              '4c722fa5afb35046c62ddb85504502d535cbed2fecdd9fbc1ee6b464a11a8d0a',
                                              '4d8dcb6103b5cde63d643b86d5232a092c0adb5ac64d5805cecacfa4301867a9',
                                              '4da39f2289e89580e18ece044b3df3ea9694b8cccf0583aa6f8d2f7384b71f10',
                                              '4e88410983856bf3a09e91b936db2ced9d720d09259f120d6f10011917b94fbd',
                                              '51272a4151c4e994491cdc6c2ec65d64bf3dd2d70e4c223b578531635eb11042',
                                              '513a05cf9fffd22ef77fff7686a9bf0054c06ed46ea506fde62763b867415e20',
                                              '51d5b11aca2a022fceeb7f40f853cca843acb0927dbcd1451525a6d048c6a52d',
                                              '5229daa45017552fd32669dcf4a96e45dff97671f5adcff6aa2e46f7d0140112',
                                              '531c76caf1ac8c69f1d4d07a8e17a86c20f4e45977dd676d9c267f3a1b448e74',
                                              '5416f31fef1d2f4b52d001216e07fbe78ba73b88251ae698689f9a6b218f34b9',
                                              '54679d0adb8a82eac4afde1ab7fdcde2308e1156254d8fa040d6e5236cfb6e3c',
                                              '553338da2701b87521a32ba805028a46d66ee0c6c6ad4e85c50df9cf50e167b6',
                                              '5715abe0581dba527e32f73cc3a6d92c67ba69448b31636135501012da89841b',
                                              '57c7f9893706b98cb1b263db5da76f6b7336380684efcfe9a167b77384cf5c6b',
                                              '58008f06339e96301472f4146cec2558e25e79df4d0e0aa8511d08a9fe5d409e',
                                              '5958f91d8bb273bbf211d7ab1511d885afee981e219b4943eae1ecfab3370b17',
                                              '5ae768884ba8c293738e481645b30e939930a6494ed00f2df6c1b2a7140148de',
                                              '5bb51e4ddda8bf153b6623460c5029d355d7ccd835a9ce1b63e46f29bc27dae0',
                                              '5c120f923a95c00b5a9c1f6328bbc9cda0850ab7cd9fe68c35b60bbe51ece0f3',
                                              '5cd3b15957e86d85be27e14c05c35b8b1d3a0aa9646995404117f630b18a758e',
                                              '5d542cf81e85c032a23824fb06a5954b8471f77bb7ea92fdef962002e4413a37',
                                              '5debd9bd1bcf027bd3df503b3fb73a68924fbd5b3d7c55852f379c06cb61f238',
                                              '5f09033528e4da13c2ebefd8dafc8d749293d6a12e1021a6c2ff78aa0e54d157',
                                              '6055a941f23c0adfabdf76e54581c320ad8f2315d69bee5521d5416ecaef7a6c',
                                              '6212fdd2df4135dff2a4f1e8952041ef98143dd4dbb0974f0fb172a21ca4adc5',
                                              '625023052b7d4ec21f9449e9e2983590976dca8ad4040b42961775bdd94b7e13',
                                              '634ee8ab28a8b1ddfa650054230cdcc97c970f8c51c45c719c34b0605964200e',
                                              '63ebb29d693bb899b4f8271559b08766c525fd555519f0d20d70df319dee630a',
                                              '6546982643a9a818a288b099c5f87e0c7f16a18597b104da060239b405a0313a',
                                              '6875b7f3b4696412011b80ccac0ae7b1afafab19e70a195bce1186d49e021cda',
                                              '6a68787b742f537eb20b088a5c9834bb03c28d0eb03ff671f9b8ff3f07395388',
                                              '6b379aecea6514f6a76fd867ed5812830569fd228f9779b6726999cc3d3f3ecd',
                                              '6d1f0862f5e340d61854b140619859d5941cf9835788f3e71711c839ead556b4',
                                              '749a2c0ae14ffac85a74ae6ad8f518175790c4ca1d997895ffa9b863ccb62dbb',
                                              '74e2ae59d0c07c953be1be0036dc681324bb7b350dd43db9ae09049e358003d9',
                                              '75b884dd33d394d344c21ee8e7b66421b43ea8cfa0982cbb76f78685428cd9f2',
                                              '762bee91ab3300e45c3e67c791694f5e9e40dd48676458697ecdc45d09a14569',
                                              '76ebba41843d5e5007eca1207ba93950938ac0d2c82163bcd6bbee0bdb55e861',
                                              '77cde4cee01ec4f412069aa2ae6ca3d50154e6a95587596652f4d002d83325d7',
                                              '792506db3de2185698771bcba1ab498bc7ec04e1d6d40406167cadccdd378770',
                                              '79d2ee709fe90a80392c3ea347b245853a2df6885b5a08e12695e95a5b1fa5ae',
                                              '7d1c26659b5abc30b4461304e0d55a45938a0202fb4dda162642ff6cdbc7583b',
                                              '7d33d0329a2b9d2c2784176df8e36680b9ccd16896c9677e9cc96307c4e7c5b0',
                                              '7dcf4a149ee83236fc6467851c1a992b4107d833c99d7977324bc24ed989c927',
                                              '7e74b61f3e6334fb3c0b359f71c9cd53e1982b50cd5427a94f36940db9de8fb2',
                                              '801ed1c5358365b330eebb20cb4f7480fe83a96cb240544509db85b37410deaf',
                                              '82aa3ccdf29aeaa6f8b3a1255532f8f58c550f0f343615d0ba8d0cd6b762267a',
                                              '82ba018e345e09ec588090eaa6d5030da344d6393ecbc0b091d7d6c01027403d',
                                              '832d52d7219035a23a80c986cd41c2961c1be6aa7f83300e928d9a7435d5f893',
                                              '83f8436a8ad19dc5f504b38407177ce9a761b8550b2dd78f6394676221c2bca8',
                                              '867884ca840bb4cffc94cb14784a59146ec17d21445fcbd912a76dcd56a288d9',
                                              '881e5969b0cc5ae398f845580772401efe205756f3a17d63549f23ff5c6a0a6d',
                                              '8d7c37d7b3c2ae9536a79dc73c978546bdae97f5050531989354ec29f2a7dd59',
                                              '8ec11e81552d3696e99982476a898b20ef63825e77e3cf2a7c5cc0d8411a381f',
                                              '92e0727f52430f0f457dcd108ca3ae6df86e38f153d3281f2f9d7c3c0d635805',
                                              '948c52fac43e824a898f1b322ef815d1c85ae5d2e84001d078585ad0ce53347c',
                                              '94d75d14c434f35a4cd69c76973a093a02287c2ad01ce8a01f0d6052a04e19a3',
                                              '95dcf2b60941b5910ab09a008eddc61fad56dfb03a59b3a7ab1b6f845cd0fe03',
                                              '9792b8709c6ac19d37bcb83aaca038f7c34b171f0cc3c0ad907179d70eab5fd2',
                                              '98f7731a4426d8c50ad3f9203f1e1d3bbf643aed9564e29139f29f9f0dcab26e',
                                              '9a4aee6a95b54619f7c8541a6497e69133bde39d782f18aa6b119ecdd6da0ffb',
                                              '9d7258d88bad246c31aae1fe27311016399fed9409d188ef8ec6b5971d130e8f',
                                              '9df229ed3b72c6f9ca679843b977eedc9fd4a0c7d9cba087242950483e98266f',
                                              '9ef8d947eb2796927fb9fa8926b44df99a1888cb92b40060f78ce6f532bac088',
                                              '9f74a9537358bd01beedb7ae65a5abeb9ba74c06d1cd036aa70256e3b7793029',
                                              '9f879f6d8aa9e5eb62cb3af349cfe8fc15e8eb9fc6262cb8a377688c597dd242',
                                              'a27e8db63340cb177a85d996eba4e9fe23764fb05fb9687133ee99910a0ea34a',
                                              'a31bcc6af1b7dc739114ef1d8037be760b99a6fec1c93efd3fb594bc06f99358',
                                              'a35875fa300436200c4fe9b56687c01feb06f1ce504fc6d3977013643a925f37',
                                              'a399dc2f02c8d4432dc14604458e3ab591b6205a8bad07c965efc84ffb7384a6',
                                              'a3bd77ef4e109265d159ac450fd96b2b2bd3630fafdb57fe57fa57a0519637bf',
                                              'a409ed9ff99ba4e15aeaa32db6bfb186a72204a376899033f62a1ba270355940',
                                              'a55621a66ebff5ef62a96a300b1167c7331c2a9fad28396cdba67b9cf9d73455',
                                              'a6b632bcdc8e1b77a219c6464867b4fb3ae55d453cd0fa6244e29d9a54bb80a3',
                                              'a7b5aab179ed65fc4b0ebe2ebb28a31a8535f93fa3e1f899dd3c6f81053b8deb',
                                              'a7e3f1b3772181ebc9dd1f14028659b654087640535343159281822e5644994f',
                                              'a8f21e01fe3828636c0a50e126554627ca10371815e24d99d2b167ac6f896cdf',
                                              'a990fc5b32d6958f31feeecf0357c7e79ec463f418f1a341ece481314f7d1f55',
                                              'aa5563a85b650cd318402efed4f90bdaaf1d040a6603f1af4fecd17bb1a3d2c3',
                                              'aa9fede7e598760298a43ddc1ea9aee4b4b4e6efd6c716f84196cb293abe8c09',
                                              'ae2015c7a2ca470b32af4321f30eb839dfc3ef015fcbc19e59092fa11c1f13a8',
                                              'af2bbfaa1cdd9aec1d5eef54f195b70122c3039bcad509ecc78233ffd5e52f6a',
                                              'b310a4a603a4b7b49c4f73fd15cf1a8500625a65fc9509547ef0cfd5b13a3678',
                                              'b73baa26e681c985f3b9798957e03bde2b4e573896b3904287a88c80a2b93b3e',
                                              'ba31ae258a567cd94531ef9009dcaa08622bed1722bc458457659f9dbcbeb408',
                                              'bc7eb8de8925187e8cec597e27caa7eccc6464027220e47cfaeea9c88c477c5c',
                                              'bf025830ab71f7afeb66ca9a28e27e456a5928119ea2c18d00bcc6656170cd0f',
                                              'bf4819ae63911316798b91d7663f6a48c686930e1517305b307da44a68f8da26',
                                              'c0f77f115e4502dd098e37f9bbeb7cc2020665e14c5ee461e2ce4b515eeec76d',
                                              'c1031d72142bb2a637870b199a826740a1a219249aa1f913ec28080cf3fc7f32',
                                              'c1687df1cafab395dc4667d979a03d93ece90a53af90e000ed55918c1bf6ea82',
                                              'c4da157f7c39aafae8c0661eb062391d442442202a3f8b740dc0e92cd6940299',
                                              'c7114b8b0457f5d66da3f5cfdaefc188b300cc66e270768a48d0df9fca9ddae4',
                                              'c7e618454af33dd23340e8ec92d344dd3427cba435323ecd7e853c14022297c8',
                                              'ca04b5d93626b9c802db94d584ea872d2d4d8292ec5f082c90818ac6fbab5c8b',
                                              'd093be4212c184efbb87daead69bb0b55d0da6e84325eac9f33e65658a1d7f8b',
                                              'd236b796c8d65f93d3df173d26ea568d7bc794d45f97a75933729dcb0163b004',
                                              'd277aea55028d6ecf6362b67a7c62d75ddeebcb6fb19c700d0ddf01f58be84a3',
                                              'd2ccef9b3bd71cbe5ef6ed83c740730ceba96d16cde0fd060b34c59c74b347c9',
                                              'd34727afb4ffd8d7c3f812156cfcb52b7dfb0498b7e5ffa3247a59fdaab40e44',
                                              'd4dc0e275bfbadf5bf612b69fb6c9fc661fb338c7cc6ddd5a47d49b05ea85916',
                                              'd50f39bb907d192d963ee4f7308e890ea35a55412647b6a515f491c5cf251142',
                                              'd7cca1a9ed20b8edf8b50e8882cf0a1ae705349957bff78211dbd08f4bf5e192',
                                              'd969db429415ebe59b6d8d77857262ef3a03a1746e0df78b22d08160ff6c1e89',
                                              'db76760e12ebdbb1902cec332f7bdcf24dafe33fe8eba0bf7ab0a2b05006d434',
                                              'dc2d61864f7af14f4a8eebf7898483f1acc9e5b2f7e4ca2da3caaad7679e61a8',
                                              'dc73c114a23d9a276a30159b29644b60fc23d7e8b993dc60033a4b19897cf786',
                                              'de929ec280ef96561b8735c5ae9494f057cb9de0720c5f0972708c7cc1fc70cc',
                                              'e14457b5749ec675fcc92e2ee2dee0898f73fea0aa30562934ce397428435a1f',
                                              'e2f65ef69904f43aa56eba11f5b03451df30d446cc9e0e1132c74fafe4d9032f',
                                              'e3164b923baab1fbca4ab356b80fb9e1a9daa944bfd78d9268b11ae8434020ce',
                                              'e32f5f34729d144d028482e251ee067bfe6535a534dca4354e6bd5cad54b7c83',
                                              'e345ec6f23f6de8abc8194790b24b86bf456626076e1deebb013f4bb9ea863e0',
                                              'e574bc98610b37e4c4b4422efdfb3666db6f72bb1b67388ee655b163cb962ca1',
                                              'e69d0876a300db1e031d0cb0a01f3bb0e63c01276176fcd5a2cdd279c07af807',
                                              'e6b92e3fc05a80b270909b215ecb8fe6236c4da2df3d93ce225675115c19ea84',
                                              'e8e1d414971b5b6d696f83ddd2603a7021c07a3557c226684628ddb312569fef',
                                              'eb024b1f260cad8cc06aa53080d3c878d461a76e712237d83e234b46a072d00d',
                                              'eb1ca8d5f10b62163c9ed65701d2bd1e67bff9125c8b7f0bcaf5c31f1f02ac3e',
                                              'ee7bf71097a5622c351f08abad14c11a9adec0b358c858c83302ee7d4b2fef03',
                                              'eec49884333a7884063392202575e3f3342aa6a4ae4c0d6a9b93db5db0a64b9a',
                                              'f06805289345f5e8b24c25507edf4d45316d9775aaddd18c34a0febf3277ca1a',
                                              'f0b363599559a9bbf9ff2b973f655f010c1478c84d74b6fe70d388224710df09',
                                              'f1527fec7f84aa65818f46672a38a39617ae498e62ec741c96f2a16d6a9d9d2f',
                                              'f58ec063bab0102aa2f7d27e3b13d34b0e31e5a60af64d31776e23e1fb29e376',
                                              'f8126fb31bd1ecea2b84152f6674f5b437e561102e63d6f4ceb03100434d7772',
                                              'f927560f9717e9bfc3db93530abc7e8654444162c84677bdf547d4018c0d3670',
                                              'fae6ebd585ac5b0f558231478ad79fe19dd4c58faa94beb2cb1ebfa002dce89d',
                                              'fb2e3ba905df1ae445ec49e8177940a17407e04bd2059a64715740039fc92d4c',
                                              'fb4aedce8f218b1e0bca705da987cbec8072744ba1b1ffa055b0f34bb0565b68',
                                              'fb52cd526ff3768381eda8772955dd139630b1c92448d3ecedd800ecf14eec90',
                                              'fc93a2eb65773a02a1711593d82795c65528f9212f025e48f79e6f9eddfdfa54'],
                    'prior_holdout_200': ['0084f8c888def7ca8e5150edd21a7248b1feb0ba3ef9e1d0b3c988f46501f818',
                                          '02cfc775aafdb1fdd0382189d3cb813f2c890a1fb6e176d907fda86ab905a684',
                                          '031dba86bd4ffa1a1d51960b39da70c5ce50b3edf183b70dda9d3218f905187b',
                                          '05c79664ff9424c0b3c30f9ec2ab084687d490fd6d0f4948b2e51176f923acab',
                                          '05e6ba245ddd2e59a3e021823ef935ebf49f6a273dcc3f2679902f74fa31940b',
                                          '09143054231e9df0f34551bc20ea40775fed5b60e64f6f9c7292ec04e772ca59',
                                          '0999f9393820e8917812ddd25413ba91f6f152fbd1220acb379a7a5474810d24',
                                          '0a91dcc15c839e62816387cb8262b74a81f9143cc5baec723099d777c9469768',
                                          '0b7dffb63da06365e02956ee0c0520dc5ff8812e339270410d5b9364f1a9eb6b',
                                          '0df6c0ecf3cb3f92cfdd8940843309292f5f8fd9d60dec27cf56b196d43827ec',
                                          '134b0136fbc951cc55e75a08a9335a524835aecbed8acf91332a655fc1af8c97',
                                          '136d362b7f3df7ad18a07b0778c9478e0efee3157baabcae0afea5da412fc9d6',
                                          '15747090059d549da3c39e1f08c4928adca0378b0a00559922556cd3d0e64b45',
                                          '167b3d90ebbffc9837a5cee597b72b178c524652090642a58c0286621737dbb9',
                                          '191224e97d30f7ea88221396753fe3b179640d179f1248addbbd5e1ecafcdd75',
                                          '1a4b3599f15372469347e6a3ba262d663076626d90890c31417d50e36a856149',
                                          '1b902ece7dd95344dc5b01077ecce2550ab8b0535d5a0b885ae4d2db86964f3d',
                                          '1d65cbf7f778fc51185fd3d4a883e44d60ae3df2d0333b55e31d90664e9889fa',
                                          '1f2b852bde28f71a1e9334e223dc81d717addd893e7870a5295fc78d33df212b',
                                          '20aaedee6d4486a09ff247ad24d80a562f0be2a338e35bdf8f82740d1255b0cf',
                                          '20ff7ce26d56fdbb1f57239cc7f13d9c7187a2fe76d8d41fc4040cc67b793228',
                                          '21a1ca36770cac71ae26295310db18513925786dbf6fdc9c8e4ed8f99dd50154',
                                          '22790101b93cb7b9e5f4136c6ac0ad559271f5d3130e1e488e863ce1ad5ec16e',
                                          '228c1a35ffdf65cb9a368379ed028cbe98e0968d4a827522c6d7ccf09f574eb0',
                                          '22b050d12e7582bd0b6b6a39bb5cdfc3b5b23ce3b376b88d5871877ec6b4b5d0',
                                          '2420281d08d9fd14717298d71717302f35a9299601607e41b548b37011e86c9f',
                                          '242f9140b8650ed6065382d89b4df19bbc281fb8388f7e76b096daa17c6efe01',
                                          '24f79f0a5e3b82171a9702b32f698668549ec806f5df4eae2f2eec402caed843',
                                          '25f4bdb171cb66d609c3b41e5af6d9fe6c4b43a20478fa8817e4204ab33634e6',
                                          '260c6068d56fddfecdaa367fbf870801776df37aaa2d954a6a0ddab092769a0c',
                                          '263a7ec3ee646b32ca436cae9a4baeddeeeb3cff8b98c67aac39a8b099a6f7b9',
                                          '27fd2d7e1e17bef7a17b67a751f6e48dd6d58c870df93fe72935a67224a9cf8f',
                                          '2d90ad29da4cc1144a890a98c668f275336330d6a7b6ec3aebe8ea0b71b0c26c',
                                          '2ef5e210eadc3638aecb896f3c3e906c284585ff0a1d1476b94ad3333feaf314',
                                          '2fb3107b17ebb27324d93eaea065b85dc2f4538c305e8f1045df56fb583f7909',
                                          '31d7041f2573fe087dd00a9faa0f6c372fb958bea8b15a956af218977e933f4d',
                                          '31f704c359ea51a9f5f599c7f331d0656792ba4f3fd6417cd35bd970d19c608b',
                                          '321eabc5858dd5f95463c798407ba16a9a26459943269cdf0228ba025f20825f',
                                          '322bb75b4f5a05b0eba7ba80b796474b81e5f47951d18342e2b9223b137a2c46',
                                          '32955221a7888fae9aa1ba5a7bab8d6ff03e326ae972c3594cf4e49a1988334b',
                                          '34e30aa6fed35f75372c32706537f0c8053da8706567a99639742d36e23b297c',
                                          '35eff67e26cac12bc12c75cdaee75d76b3dddb27ea3b4a0eb259b300addf3633',
                                          '3684e33c3883dab27bbab61eaec02a028c332b10416c3668810ff80593273227',
                                          '36984c7cebec9af59bdd0ced39e22c29d5208c02346001388601a989adfbd098',
                                          '36df2fa4a13411acf35dbfae062a15ca58c58331e52a20019fffb17c3ebbe8a7',
                                          '3983a97cb60d76efcdbb1b037e8b03a770cba61d413d2dfc58fd1017207d8635',
                                          '3a52e5716470b7ee0a695cb522e1d38858bba011de37e93a3f31bac90874f3e3',
                                          '3b2c492954627e8bc310bc32893d825da555b631c34c5acaf633ca62e8d5a401',
                                          '3fca60957977d0994804c0158199755aff91032f2c9c91db55154255ba36eb8a',
                                          '3fdcbc48a62c2fcff67e436a6bdf7aae0a40d277694895c7e80144ba41a8ebe4',
                                          '41bb5562b8edca1028ff34db563dd1d43445aeed5b9e1d080bbe63b0c4e657e9',
                                          '445ecfb409bc55ea2edbf185a3301d54bb48978c552d66da81ab5f654ab3ba26',
                                          '455a8e3207deaab6af6caa2a85c4577aaebd1b494f6e5d2fb47ce1f7471a0347',
                                          '455fbff1af91c14371e3ae707558819cf97c55f50e08f1273ff17c3e8352bb55',
                                          '45fb36f975631ab002fef0cf6cbb878ecc1ee949eba5519c8dd53f70aec6cbdf',
                                          '47b2eb51a1f560d0be0833c6d040a97d523054c16c052d0c6d0d8500f7be5507',
                                          '481d4d1b796c64e8d9447a128154774a0e68d3168d06751c942704ef18e2b9e7',
                                          '48644c96046d9bfb85294d1df49ac65c615515b59c6ea7af721b2f0b2a88933a',
                                          '486b18e3bd5b423384fe51027e0cdb1384e827ae1045c8a5b33626ab98d4e25c',
                                          '4909caa91644f73e0c459679143722235a1aef1fc58621cce427861966c6a28d',
                                          '494177b028681bf52c409f064078f954db33534a72e96370c1a7bfa11429cc39',
                                          '49b38edf9a960824ab73270077ed5e9d52aea300ffe454336d35daf0dd90f812',
                                          '49d7aed962c2ed37ab2c75bc32a946b70f07ac1d8fe56a91c15e1d4822dadf34',
                                          '4b656b4f43b3de43565b38e1f5a013193ae8fbceb330505aa30f46e0c7d266aa',
                                          '4c0dc526b6db4afeace727eba01436f121fa40d20c366e48efbf98f46f50fda0',
                                          '4d135c5e1622a00434e847ceda99c2a093a2127712ad3ab51594e18616bdc798',
                                          '50c36b1dc6fe948923c1653b715c4806ebc86f061b2ab48f0e95c42eec544b94',
                                          '521642452f15e3c142bd07fbcd015e9cb141fd29f947b584fcff64f9987e07fc',
                                          '536ea5cfa9c5e967fc3134241720847df974adf0d33ba3a7607fa577f2fe7ad3',
                                          '54792b4b4041509792612a46a3a9e3c61cf7db058cfd3ce54df9bfba958658b1',
                                          '54bebe36f250ef417216594b1cef00bb5594aee3ea36f3df54c0f18e540a8ada',
                                          '55dcb7e2220bfa7fc6500252b8db0113bd2ef50b15d43a583f032d87825dd791',
                                          '5647ff3c18cadc44baa80528bf6aaa790f05a1b0d67d96ac4fd99112c5c42238',
                                          '583a382cc04cc2c8b0db2a06c108994f8b5d0f14162db5b9630da75f6764fbf7',
                                          '589ea0e14d751b5ecefc3ed53d7685f72fcbf1caa3ad6b616129d3797702235e',
                                          '58d4d35e699fe59f3d5e0d147fb2702d37c4c02a817c3d5c99d5fb54986c0504',
                                          '5a6a952153dd90401ec40eceda54ab9c37709676eaa21f8ad83503a7b24b1951',
                                          '5ae385ed224de1f7172cd56c30ab55188becbe0a9bb7dcbaa8b0a247dd73845c',
                                          '5d83c7a55d1684799a32354258b06da03937744a04ef843d47c73fae116151ca',
                                          '5e30e31cd36bfe003c71b8d0e7675c5dc82c4f70f637963a4540c744f8466c64',
                                          '5ebce6409062d0d09c88ecd8b8873a68e3bf252d4d43f896a8bb2ff06b767a80',
                                          '5f77be7c9b614908c147be524eb802b464eefb1fd8b08bca03f286e1842d057b',
                                          '600043039e4262ce95e81c82a970c426f132bc413e97c0c6b73afc992e08c52c',
                                          '602b7b7180d02c143d7d1364110a2e96113a6dfeb70d6c2300e114afa004f47a',
                                          '605195b4df6d5c9b8694a313337bd3b8a2ef4759c1813a638bf967c728f7f3d7',
                                          '61357eca6b6068fb3a47597031319d933d97893076c86d3c9ab346d913e1b600',
                                          '6197f56fd48512c651dc9d17532a2f08afb691c1b0902e02212680d496f73683',
                                          '630da7ce1bfde039240eb9b37b0f3546ac9d2a24eb724f33dd7cad0eacd0fab3',
                                          '648b75001ebf872b35bbee5d498a94ec5d35c0086b8e03981ecb45f3b1f0c257',
                                          '6494be55906ce443273f21d6829cb2edbf38a73933f0b5299c1d1e0c5a43f887',
                                          '66886428bcd081bb7db43e6220dcbdd0c02cd99015c8a102cdceaaf7b8bc2236',
                                          '66a658465049aa95d0105690b8d79d9eb5f556bb97aa58e4fb0aab3c264b094a',
                                          '671226cea6820bb36a38267e9eea5591ae1a7f1c351b57a59d8cb326af7701c5',
                                          '678ef7d21f94b80531d7e67eac5bc0c63f7387cfc8fd0f32f4aa3bf0a060e854',
                                          '67bd27e0e0202fc0f7c911f71233c63c2191df086d0c5fe99cec8cea9e1ae46f',
                                          '680a0673b02d0e1157d321f8597d09419f1d4ddcae44d515eb310b718621fdb5',
                                          '6878cbee5f711d1b6b67cba9f3258f126e269d62e795e6a43e7641e0683f903b',
                                          '68fea67ec0719386b07199fa625761fde0926e75a6b84fe1cc84455c2d03a5f9',
                                          '6b3b65b93269d46972a31f9a62d88575f7215c594fbb51c9701bfcd3f9781b28',
                                          '6c6b71c394797b942e87cd7037a729547e0baa1a83bf9bed818f70200f1d78b9',
                                          '6ce7d2bad658bb48845625e87136ad74422072322416f430039151fc498f2b43',
                                          '71284f9ad2bf5619ed99d70f89839a85092eb67bb45c9e8da52eebf958ee49e3',
                                          '7524c3ae8661a7c7d361b8149056d31acb09e9e31ec5fc2f5a4e62889552e3ef',
                                          '76cd5408e575953f7162c493f1c2b30773be0d4ac843bedeb60267391603d23b',
                                          '79f7031a4e43a6c097ac52e3cd3c2f2e0902c7f4992259092c855dc591712ea1',
                                          '7a2391d2a2a38dbef609e990a824b07332bff012adb280dce711c019b0636fa9',
                                          '7a935559ec33c860f6fc4eb3d7fb5f1488cefc60d5412f377562826633a7bd00',
                                          '7aefcb954cfe40eb083fb2f35e0b334ef45b43c2ffffc25c825a0b6a3df1d839',
                                          '7bcdd0f5faf5011f4a12267f0968b33edd53a8a6b496eb9896cf41e73155c450',
                                          '83e5a8ed400d291c17f48e8b432b8633aaa0de3c3b482808552a7928ac3c6a62',
                                          '855bbb0ca119676f9c9135fa70d90547f606e5ae78faa5b0d8a463097f4b9ce6',
                                          '85d25a5325a781e3e0d276e47a4cfbced0e428e63ffcda50f2654ab2be699c3d',
                                          '87c9fa95545393b913efee2667d28f4fe46e23a08315602be8a4f3052dd1a729',
                                          '8865258e4cb393d1cd57ea265936df75da7fc46aa91e6ba16bb21e0d9d16a6a4',
                                          '8a090e223e0ea8b594b339816bdb6205b4c46c401be9a5b937f72ae965652299',
                                          '8ba078f1f9220437c84953f3491b2fa59b7caf3bb6f4fff2861e67669ed00bc3',
                                          '8d47b475097e840bf895f5ecaf19c05df56d90d47b21cd858f090e1c677ff29e',
                                          '8e9d5716cdb8b34a4e88bbc0bfd89a7239b26dc28a5677cf1bc892ae513b9a85',
                                          '8fa574972578872d1b818a67c4dfa1fd585e66f8c605023fb0d81075abd81339',
                                          '900deae74c7de1a6635f99bc11278ddd09d98078b7df8dd0839103d51ab0ad97',
                                          '90f28dcd3b0705cde9bf71db1eb36d1471be78d7b5c64a61c77db2c9c006b5cf',
                                          '923016553382e157b61c598820d60ea4e1f86839523968396be794ac69231567',
                                          '96c1db9596316e1c7a4f0d7000782a35c749292993a6eda9c180eb0ba65ee73a',
                                          '9746e86839c203f87344ae97c67204004eb99c35a04074a2b3a76e1c62600798',
                                          '976a330772d02cedab2f0d183f7f793f93117ab62e2f6efd7c25eee1634a4265',
                                          '97a2b95067c1c51e4bfe50d4a19eaaa18758f3e5b92080b19eac2d429dfd4d51',
                                          '982bbfff8cdc53dd939dd43265f4c5c8787ba2dc52b5e6ec37ad757c69684554',
                                          '986f5977972ec1fc0879556c5cbe75e1b871c7f5aff819996003d6c8552d53bc',
                                          '9cbabc345948633cb689c7409dea95d1416fd82141725b327d701d2e5e2fdf4d',
                                          '9e13fc095ac630e20c2148357a9042d615fbd3c8d9a986a70dcf85d193f95825',
                                          'a08b6a901fca192459f24e64173f78247a955a404f6b54d706ff8c71a4bb0660',
                                          'a20efed7426e1a5e2ac2337fce00bcbdd8259d7b3444df7c7f14368f8579c69a',
                                          'a232c63616c5d104835a81c72c440af4a185f1929da16c473bd7c4f07be9bb12',
                                          'a3db4522f6a2f4ab697c3d108848d708113fc8ef5114ae564d5fb214dc304d5f',
                                          'a4faa72f0286e0f7e7ebc4d7c43cf6e553eb0414eb83ac5618d0c1b22a0037e0',
                                          'a52c26a9be9cd50a57d2516e4c157704a2a80ccd21337649d4b638870c9f09bb',
                                          'a54662435c1d76fa7e591b18593a38fb653a4f807158a3fcf72625a4e4679f7a',
                                          'a6914c89e4c89af82929f0126a26d08e2297f650a8ceec97e0cf000a9fb65d4a',
                                          'a7a207334af3649a356e6a9b2a05c05300eef762aa73627e9773d329704051f4',
                                          'a976e717049353a5df7197df29909477b5f31d37d9890a43f066280588e6acdb',
                                          'aa0b920e1fbabc76a620c87fbd5218f9de182d66e2e9a2f07fd3fdd3e85a25c4',
                                          'ac52365e2ff8a12563648794f6cda371e1815d371f05cb65ccd3ccbf49a4b2f9',
                                          'ae21b433cce10924ab235527fd783b53a603cf09cd3668ce414262a72696c815',
                                          'b4ebac3adf1ea2f7036580c7fc75985e13ca189a8d4c48aad11c0652679ebe17',
                                          'b4fe52fb4349d23096a5f30437abfdf6361e281115f68bb5bdcb8eba56e4843d',
                                          'b559c50b6ea318bd89bfa111e302cebfd84550633082ce995cc0e5fd3504c01e',
                                          'b5c5e3230703e8fb02da5d7a7f5283b68c489585fdcfe672c5dfe12ebb3ae2aa',
                                          'b630a2132166a89323db96d7791f8553ef92f42d09e771c252a88b0772851576',
                                          'b75ef8e8cd44faf109ddc477d87262566dbfb69287947f859ae0c6344903eafa',
                                          'b7694a0d8cc90a321d708c46804f1185bab74a1070aac880453dfe932e9e0ccb',
                                          'b924bbb286ab652117d29e05849c1e13e9542867030de7b050ff1fa2e444c25b',
                                          'b98e94478d3a1a2dc968b22b8514b183b7c73f69203366248338e98f11a5f8b1',
                                          'bbb4d6e5741265bc87dd1898ba80c49989f97854f4c489fff088cb624166fd54',
                                          'bc63ed066c67c28bb0d812fdfddb8eeb4989677760561b2c23398b8409141054',
                                          'bc7069da33611ec30ff0a67d51895a7136093053d6ada339bc5426bb331bd3e5',
                                          'bcff4ce05719705fbc30bf7af94ea1b2867134ba610f2b7b5b2cc6e97eb1b898',
                                          'bf3a56b486e0a3b54c875a4f7071b8c72026de84fba0370eee6d73f8b6083d0b',
                                          'bf72e50e77b84cccb3ee457ba4b78f0bd2350a15a6ecd7cafb6d4c2f3681e86e',
                                          'c038d74af6c548476bf889955b5fe53e678ef03ae1d3822901579d15c2a38b2f',
                                          'c24cb1b2a747e2b34fb0ee2d492898363a771dbaa738ce1dea97dff66823bb89',
                                          'c36125e5ff9bd3a281dfa03c22d11538231fc373668feafdbbd63c1d451cba58',
                                          'c6fa3622ed69437a80d1f23a0ba24d079d70cb17990dd72d39e35ca8eb714083',
                                          'c8ceaa52532b8a203d78a98f3075d7ea83ddac86d83c238d68730dc7ad2243ab',
                                          'c8edcc3f7184a8c812ec255b2fdce5450699632b4afc44ffa3632eaba984bde1',
                                          'cc359600371abde4485f388b87616f589f5228e0cca128ad427a32c13189bb6b',
                                          'cc4da25beebcee8043a745438aca9d5cb7db726eedb373206f0c44b2ae2de1e6',
                                          'cd66b977fb9dad411fbf3b2830e3ad4e73f89b7c3aabc481d8c76ee85d964e2a',
                                          'ceae795083b4bd31c52fc6c6915c6f71f8840b57dd285ede753d21757bd84211',
                                          'cf0311467347d93d724f8a782c72a9907ca9af718d13075661e471540e30194b',
                                          'cf513ce58259ab0ba6d6d363410cb90a78a40a33dc0086200a36fa3c4dc9324d',
                                          'd27372a7c7e6ca823331bedee20131e5cb477e6ed886217bb00bbb690e6caa62',
                                          'd2bafcbead55ac01b0678608926f787942d8c356fdae79b2a3930cff44890b70',
                                          'd7c4e92cd959583908e7f3b211572c3caa69be2bfdf770ecad9dcc4f7b2ed71f',
                                          'd8f11414f5edbc58a8375e336d8d54693a1a72ce99b4752494d29ae0971de174',
                                          'd9e16bba25112e60241dc6610d0ecf9a5cbad1bf36cb818059e9c0cf53a280bb',
                                          'da036e121151a78ab1fe4a4dca346f897200d65598f8370ac943ddb8805ec0c1',
                                          'dc2dfc2cd6c6ad7a84d41976d2785c76558136266719687a319cda903c5fb043',
                                          'dcfaf7d2bb65afd16465bbffdd36a52c415e9dca45dcb8e0cbe0ba7d640ccb33',
                                          'dde736197952b12f8c239f70f050a129e2afc81d37fa389547f3470dd790bd2e',
                                          'e14b2a14b3bbac0f69e14e44c4d090c53b2ef144db2c101bcc0b270c4e2c6d14',
                                          'e415eee54ef7c2e9c9a7cc535aee24aac6693b5434dd5f06b0622341561e9457',
                                          'e77a6e19909d4ec2bbfce3357fd7518db1042b0d5fc947db9e5275db886b545c',
                                          'e8272dfb2f764870274fd990d606a6403450a302ee0206e224afcc5102b9a003',
                                          'ea1680874df1971470cb0a4155aa99d92d501c8eaf27010f2ca17c2904fd2045',
                                          'ec0c2972c0b16806d604aadc9317cbadb056f7d6c5045396629d20783c1392c5',
                                          'ee4cec3bda0a56d6226781d4402b8f91886d09f18e107e3c893b6a6e499e4c69',
                                          'efa84ef539854106d893867c732a9253f0dd32b89e3170363ee0c9977d7afd0b',
                                          'f06533715e8df31c80ea9db678f2da295132190f46f7c80eaf288b5fe9bbde2e',
                                          'f1c35a4bd9fb0eb3f9bb3e97d80846c5432561acb6d38688311d597f7d6c7c4b',
                                          'f239b953914ed24b32b8b0986cdb15de521f70fd05bdd13964edbb30ae30bc3e',
                                          'f51b99c78324c461e23f3c78216b5bf26e6467fe4aba1a350a5fe17494e9c479',
                                          'f51d6d9efded51863601fa04eadd76433b8cc96e9328798d1045365937e53c21',
                                          'f59b75a7b0b397a618686b749b51232ba37dadfa465451c66c413177a2ac29de',
                                          'f78440d114a63869ee60b76aa3fd52069d74fcc5556dae9f7e362266a8cf029a',
                                          'f897e5a9a258dcfbe1874c2d011be02e0cdf9773fc8ace2c509ffc5b1d56b732',
                                          'fa0521049456a3323e3e3c5cd39274a4e6e3a9622f3bf4e9131f3245c58b61ed',
                                          'fa7da05a1023ec96754bcb456a7bb4893f96cc7bb3f62c08c8c84a9d0c88881c',
                                          'fb48aec8cacb40a3851e8157d5ee9a8110888306f73c931290b897ba8607a55f',
                                          'fd91f378d07178bc4ab2ffdae4702a328f8d3e026d9d7f802109abf05df933b5',
                                          'ffeaf0b81a670d518edcd623dde67ed4a34d40754377a0bb6b3c56a4b382d664'],
                    'wire_confirmation_100': ['2783792192d71135c704a7d48f4335e8c3b43673df6f77673edb3561afec80e7',
                                              '14b62856e863f705d95d59c8fc30cab9e9a2a3e07be3c5ea248fe841e0088444',
                                              '929fdc642069bea33341b3c2460e3ceda1622a031bb9b02a3f1570bb1a5a6c1c',
                                              '11cd45d44877e9aaa9c41984f0d1651c7321b5d7049ee0d2c21cb7d3582729cb',
                                              'a488e5780fb8bcd2676617c5fe2a78df84cbed7fe997c3ae13a81baf0b29bb65',
                                              '49551daa261549f2f64482ccfb65b32fbb79e139be7a7f19cdcd12bb579df5d8',
                                              'f82a254bd1e8a7f922feb52bc2ba3f8ced6ee800e5bfec15cd1e9f7ce24dd9be',
                                              '53ae5f55519bab8de2d01e6979fbe0db650be9cfc718bca921091d316cb0f941',
                                              '694a9d3ef39ee9a9a8c5e37d3ce3721f435e8265547752b713bb4e03fd23eb65',
                                              '657965471d6d3d059a76badf497c3d0ee3613564b9698c4326b25b7e8536f4ac',
                                              '748a12289c1e5c39d6ddf297aad473c94c5471d52bfcc26b925b86431e4d9f50',
                                              '4e6784537fbf5e264f16ddd041cecc2bb9bd1a14af2717010d0823015b6e0a6f',
                                              '575549bba4ada4a16aa82123728211408b480b8fde752bd071af9eec053eb75e',
                                              'aa709feaa1e3d2960c85e36455aba82e5cf383cdd997f83a81e0d79c8fc096b6',
                                              '697c90155f4b75a5236cb78246c65d7a39a4e7fc13080dbc0d44919b99551857',
                                              '6bee3eb2eb0c7ffacfb67b4d18df51f272c59285c3866e9c14eaf518a7dd8040',
                                              '087a8df98839fd9ccf3cd63df3817899c260d6429424e093386cef1c4c65cb3a',
                                              '5aa2d1269adab9c2c36d2787c1f357f5d8161b8ed87a0698847ccce315b09664',
                                              'ab6453bfbb3fff8ba6aac2da2cfcc891d733bec706d8eff493fb4879dc4d9131',
                                              '031288bf4dc636a13973e5523dc48f03df93d0d8ae0da94a4101cb226bf23c42',
                                              'b391002be2d84e14954792a154edf59b87e7f0e3ead76d2eecfbe775f833d371',
                                              '9505898d4d8813394c9b6cb7c0b240b5ef0bf45436a4ee83c89743b990031805',
                                              '3c68b109aa59c24b0bee0a32a25f67e6764731d6db59e604a29401e5f67076ce',
                                              'ec95b54ef913d7937e0137da18fe873a8fdbc145e4dfbe3b9834859600f91e40',
                                              '040b69056ff6a04a13d1d6bb91c0c479efe2ac4f8fcb60efb0a83975496e750c',
                                              'bac1d1dd9d14912cd89ae645f71a16f899da27a8572a9af755efeacd323bd200',
                                              'a8ee1c3079e2cd9b194cd1070856e202a5e86e4070f2190566098b8645cb7bf3',
                                              'b88923957715de2135cce6e9b26478c3c43940bb296d3cecdf2b95902d837ff7',
                                              'edfbe86f13ceffa5042815ba102513ba8d19f4ad9e17207503abd7e831de69d6',
                                              '093a0130893752bf4785ac38e567c9d799a365483a376c158f3a2561fe6170c6',
                                              '37604d0b16ed45d50ad73e92f69292b29ab552895940e6344fd0c27848841bde',
                                              '71c504fc93d8b39a24c79f5ca6c1792f38dd34145cc075bd7d2a3c1bc6196ad2',
                                              'd366fab0cd8aa3ee39534b74c5c03be31ec6b385f26bfbe7a75fe28bea870d3f',
                                              '07b5f1ea62e221c7dd148a47a9c62d11ac460bef909ca3a737b5c395221fb146',
                                              '358bc297fcb7bc33ac812029f1b6917c9e25618300ed46b7b93c68eab5ac2b34',
                                              '8462f8dba78b9dd70cb78df9732b695992863d00f429897644f6af1d8228de27',
                                              '71a53233501116dec0b7a2306a681dcaf83343ad24f418a056fb594006cfa173',
                                              'c7a79fc1f02af283f8c3d41ccb94084ad4634a247694fe8a333a596ebe5ba49b',
                                              'efc76c8399762286029c3f42f33dccb6e4d61bfb25a558ed7c091572a2d63265',
                                              'be71cabede2b75f5f4225fb6f102eb667f889d4c5501d4b268155e4b46b05b13',
                                              'fe76d9c7b46f5c29f6db376129e54ce177326809ef784ed3339be7309603bdfa',
                                              'e1566d858f9a68cdaf9896631e6d6ab106ff599b5f8db532138af0b42825b7f2',
                                              'defd624650aff73657630419ac2b51de85a2234d6ef2f9e479f38180346220fe',
                                              'beb0bbe1093c7f232cbe1f1cb8da467265c671cee47f51f114697706b5f9261f',
                                              '284fab2961066535c65cb962f88aa86b463048395b305e1793c4849dd5603156',
                                              '840db84c75cbeca1f2147634e8f410d96a758627824cf5a9ea4570135cd6d1e4',
                                              '036bca3d81f26090a944f59a858a923d5975c5c5176d5e010ab25c5a7715d3f0',
                                              'fe8a75ed93a09e5384ceb4783f6baac43949d1a9f2b543a35a7681da6d053480',
                                              'cc86679651f5593decafab9d26150a955a7fdb402fbcf5bd91d31f0af1c9368b',
                                              'b6dae4a3da1577c54fe51fd607f131fd05b273457b9499ae5441a8d2ff800f89',
                                              'cc4ddb7541bd48fde370d6c3f7497ac0058ad4aebe7f4c9254ca64b4e35496e9',
                                              'acaac93c49422cca18b7bd47f6d02f7a4086a7925bba9dad3895b205d0a0ae5c',
                                              '48097204f0e1e608f4050b14ee5ed523e1fbd7f42e64144f3733b27634b2928c',
                                              '669b967662ebb763669d3d3d1486b36b269a8bf9b8805dc21481138eb4ec5505',
                                              'e97d01c10a20c598100f1217c569803de9c62a2b64a4e7a17e05a474aaacd97d',
                                              '88aafa92a657bbf38e233d76a6bbbf312c43dfc68f8db36b6fab1a9516cd9c42',
                                              '02f2301e2a66ba274c929367147510d2429dab0907f78acb13ced9e28ef2f1bb',
                                              '49fef414fb763998c606ba88cb83cb653a769244fcf0d4bd2026076045c14ecb',
                                              '181bd43cadd95ab231952fc078cacfe2384fe1125f877b4370147dd2685813df',
                                              '83ce0bb096a862358a45d465ecddeebc1249cf4a97fa29072123cc94f9bcc59a',
                                              'e2fa79999a98f5b3cabb16ebb588174ddc4d05bf08863394efa723b46dca8486',
                                              'f004393b40719704ee099a14cdae647900ab407b220946d5273feeb301e5d08b',
                                              '0318784a8740a6f2c4539e77cdcc938168574309e179b3a3ea56c77336802023',
                                              '6ba25a25b0fc8f2e0b8e61004ff7d594801a845e78ea3733024d85ee8bc9fb41',
                                              'bba85602ac4e5649950c167c080f4fbf06e8fe4faac7caa1a3c1aaa1f27b8dbc',
                                              '2da7cc2b46d1630701efddaafeb0d17618649382f2e2adc6ceca325d208dab98',
                                              '68d030b51a5c51f42c8914531a75672882ac1efe2628123eb262086f7b0445fd',
                                              'a124ded59b97ffe042097a735b80b1a92fc568753bcb1d09226ae34a306234ed',
                                              '6e21f864e12c8895ab11f41e37cd2da110e47f550373ac99c17cf75045cbecd3',
                                              '8bd404d2fc31b447ce91a7fe116316806b05e729f6150b04243d221c5a136588',
                                              'fb200baa05919dbed625ba042f147faf717b52701211b491f9204ff696994feb',
                                              '4bf6c554fa4928d8da6dc7a784cb7e6f275ebec4de98b15ddf02fa5d0a5dfb2d',
                                              'e37c886c5c2caa7041bca9e1d9608d3c04e509bf7d48e4ff4f9e73c4687e5f5c',
                                              'aea678149050e99ba73153691b0dbdf0710098c8d944aefc0b72b2b32700b698',
                                              '5cb541536ef41d7d011429679a0628423e27df7b4c2f1d921506333e354bb463',
                                              '21977530aa79b9e0e499784b1827e6b75877f1f30b3debd2419377a9fccbeefb',
                                              '71372c599c85aafa4e9714e33f875aef48f00da63640b1bc387c77ce30b09a9a',
                                              'cf3e1804573c00dc978084f5d4ca9aaa56879a280fe2631f3c853c074bdf65cf',
                                              '7e65c5ab2140d1a28103fba10c2bc53e9da2cf7b43afb0a61e0c651992bd5e25',
                                              'c7e7d5b4bd057a2d920a23ed6d9afb27e658179936767a73cdf72eae83798ce0',
                                              'ff29150152186fcee2398603ada81055195bdbb1721361d4074d2581ffcee0eb',
                                              'cb403476bfc828da561f22d80fa6c10956012eb16e69e0c9d1f076737a19a589',
                                              'f8d639fb05aede8c77161e4a506de726c75b024f39c5a54ca16751c72acb16a8',
                                              '2d28a2897efa0b74c752b4287e2facee710509c8e840aa849620a1a2b3b37104',
                                              'eeeeb302b91f620621b0af659c763796161ad1af8c389cdd0cee8b79a0b34c3b',
                                              '63f7c384b9e293cb12ce7740866bdcaae6c710930fbd81a521e7b9512ae30717',
                                              '80ac13bb6a2f846365ca4d0359de753ba0946695565c8e980cd675f0c07d89fd',
                                              '8faadf42f61d007e7fd5bf2305426a5499c94ab3ec5b9ddd08c23e8c7baff408',
                                              '7c5c847563d9e772388777c232d50716a3685e50a702138f9244def7a90635eb',
                                              'a815408dd658c55f3e345691960afdd13ee58707815666d3fbd8f1e32f3d5cff',
                                              'cc3bd6fdcee10168ec426e8680b0f458b7fe3fac71fbefe3c6276b32afe42073',
                                              'a294a4e5c0be525728ed82c78d07e4433f4fbfb03f01a520154bc4025da14be6',
                                              'cc54a9a090a2fded909e70c0526911dd47c0cb60458175dc36f6b206da4ca06c',
                                              '2ce23fdc68c43b25b9169ab09fa124885e5e4e0dc2c90cf875032a5950b5d8e8',
                                              'cfddfb7feb0220159a6a026254eb4049f1eaee8563532998e27a4b41280fdded',
                                              '26c32b6b61f258765c735ccaa8f3377fd4aeffd8ab6c4d9eb1ba72ac34b7b73d',
                                              '711429f02f197c6aa9d1e405152cd99a5c2bc2942d231ddda6fdc11239643ae7',
                                              'a2d63e41d1c12c1cf46f2bd3aa5efde29710e69a512122e7c9f7c25ddebd4cd8',
                                              '4e77140028023d0c72512da6344a5a434a7123659aca077267d9f361fcd0d253',
                                              'b038bb40e413a4e6fc00eafda15c191b31b1f2c125b59957ffd89fad51e184c9']},
 'excluded_unique_images': 500,
 'expected_total_images': 1000,
 'geographic_or_near_duplicate_independence_established': False,
 'known_prior_membership_scope': 'Archived compact evaluation, channel/BPG/final checks, v7-v25 '
                                 'development subsets and v14 confirmation; file identities only',
 'preprocessing': 'center-square/PIL-bicubic256/reducing_gap1/RGB[-1,1]',
 'prior_membership_sources': [{'file': 'config/holdout_manifest.json',
                               'roles': ['prior_development_200', 'prior_holdout_200'],
                               'sha256': '02a3044407b6296e0e8611843d11c6bc596ed5008322155052cab52340932752'},
                              {'file': 'config/wire_label_confirmation_lock_v14.json',
                               'roles': ['wire_confirmation_100'],
                               'sha256': 'cba8fb9fd75df0a87cee28cc68701a062e2998ece30f87e38a33d5931204f1de'}],
 'selected_file_sha256': ['fb66870b3d4622b4393f5523bd9ee8463da677a02a08d4c3f46169d4c509abcb',
                          '878c11cfb20e4561c89e6ce8d973d17d7b02857f0e3481b1cff39bd954579a11',
                          'd974955e725729cb2fd395cce764af1f957747dca7c515c416c6c450c63bd758',
                          '7a66bf7c0560ddc5d003b0869c6b7e8ef4bb06029a3cb0c258820a2390522a41',
                          '94d6437634b1311bec99174fad65520de1a385543eed9feb7dd67016f18d035a',
                          'f4810de1c010ff776db227074cf01b562d5adc6a25e68cf35ae4e9a7defad789',
                          '9d25f09c5077033f0c33abcef3b585c5c54d25c86251ffaeb36ca377a7d9350f',
                          '820006f4ed92e36119f0469ea3311e2739f34c88500eb583a9c37737c9f4fb45',
                          'd5d3911f303bad74e013b8b87950f8b3d9717d4bed508ed5bd8a21d9dbbf9b5b',
                          '017f1ca9cf29c2ba58943ff61f089f2fbde10ca6b7327ad05283f4b2ab12f20a',
                          'c492444dc30340239f974cf8cc238b4c56e20c53c7d2e74e82a2440f5ddf629f',
                          'fe39d216b6307c6809c5d4acea9d4f5e40e79778ec05e78022f3c33dbcc6c4ff',
                          '0cb426ae32adfdc3288ea3c4a9a10b2c123ab3d4664a32b02f052e86f1bbc4c8',
                          '824072b3ce8c6abef26bbb045d3c9abca759b0a8b1e8e8cdcaeb64a4e403ef34',
                          'c4e80e2436893859aeaa571736d0a5ba894d94ff08948128d8f0c5e7e2fa7164',
                          '5fd5ea59567a4d929551fc89ff66d786cb34673d350f52207d7d0e0ce5ea4e2f',
                          '1422861477e119b080e3e93293b9f45cd0ce20067e9beceb1bac42ddb083428b',
                          '12383bd875b6a9639e103cfbe7e31d835157db1d1dd971bedf34c70546031a42',
                          '3fc5066e006c9978ee0fe225c223a63ac4958acdaefba6d763cd6d3a7284f3c7',
                          '8f0f4324cced8cdf87b4aceff242c7ae78bdaca735f1f3ae2281eb6080caa1d9',
                          'b26a3ead9dfa4626bf6afeafb5ba7e23d17f0979408f61b12023b388d132bece',
                          '9db4679cbd14bac82a2270d094c18046b9f7dabd07802b493e7c25a220aead90',
                          'b62b27361add7e88063dd3d41046eac188e6b29074f4e426ff4f0938a6090615',
                          'fb162a0c965408d952024767012c39958bb7fb6a0fe555ce107f7141c1f906f5',
                          '98aa4599a2a4076d7aa41d7659e257d6a62bbff69cce3464b0ebae45af5322da',
                          '2b19544397470829dab6f7a402fe4196b2654d8140465715e2a7e23a1b225a37',
                          '13fce8aedce5fab9133488366c17473423687df9ccd5682f038ca815b008c817',
                          '99c3b84ddf987d4ca7d4034bc82b45cdc386bb7b59b757ff8d136d71ec118509',
                          '927cc8e6a81091f01c534c53c04dac2ea17b398ff4b99d7726c996e21024d3fb',
                          '8ce56387c6c068ddbd1694f74c3c9032a83892f615f7fa94205e70246c4317c4',
                          '41bd481208205366912cd9a36cfa1cc5c0407552b88c38d846b579692f7af208',
                          '014aeb0ef7d76df016065061c6c381bcbbbd6fa8f4e287cc6fb400d6aa0cce4d',
                          '37a34ab5c0eec0ca861e91e4a042b51fea62c0b2dfc0f3081719cfea57be04cb',
                          'e1e99ea08226b72a67793ca36c187695ad647d021677edc3f27166384df63c8e',
                          'b484b13816596cd872e0d5125e0c8728cb428be2a1127446f5ae7823da26cab9',
                          '7f85d9d3836a7943bf98317d347614821d15c01a9030113f7de5e60439c0dc89',
                          'b5fd1f7d3fee17d1659fe5f93380db2a71514aeeef88d874add9725d39c7923e',
                          'c8a7e9d6f07edeb91b951762b6a785d021fb412383de193dde325e2cbd00aa6d',
                          '71bcdc5e7cf534d0de602e8052147b1a88ffc40b622904caca87102bcf8be2b0',
                          '2840069ca725acd3e097f15bd532375ddef326880f3683f2054f118e70ef98a4',
                          'c21e4d7fa110e18f14f911efd7181fa2363dba1032adcad2492796d7d988995e',
                          '6005ccb5612b23aba5df83fb1481e96647f15c6e34d11eb26225ee342ecca069',
                          '16877e5a83f8b18b70b3a9e5c01ba954cbac334efad1b5f5a6c49ff69777e96c',
                          'cebfc6fb7a013acce43261abd4f0b07a3d74444c037e474ebb4cd7a53d4cfaff',
                          '4c9cdbe77c5df3682c148b2f6d82f53d8273a815a8ee173d5ca7967b88706bc1',
                          '7e51e3fc640024c1a77502b6b8564be1d4305b67590b4f60f0f135963787a155',
                          'cde315e713ded0863544545a4f50bcae520ee4af5aa89bd0297d654ed0f8d68b',
                          '8d3be63c4da550a3913178f9b2de4f073b8391bab78aac1e5a02a8c656749814',
                          '9ea81f44498d04af4cd2b84af078e2ca6aa42bc2a0e6939d229ce5a96d21f49c',
                          '2909154fe14311910e85e24c5abfe8fd9523d819071f3f6556f2d910ca1b0f90',
                          '526845f850b649ead4487cc741e0949e96fd81ba36f4da9810e93caff4e9aceb',
                          'a11786337ff4f59ec22cb8b9292df7220c20debc689ec52ff0328b218d041939',
                          '21bff1f8c093cc63fead0b7638ffc9940ae579d8362b986fe77cf6835abf8c67',
                          'b038b4c09b549bf2059829904902303c86f5d13f37edbdb4d5f0b27111ef9668',
                          '0b3376578021a541194d1f0c26c97e16b508d706e0fdc0e53b2ef7bb989aaf5f',
                          'a090bebaa5187803f002adb0aa189ee9ce5c6c434a95b80f8b62d4eb072224fa',
                          'b660783e297bf955a11c1927e3ae18ecd28e348136937d220a2ae2cc8773f4fb',
                          '0e408be1135a55441fb76763bd9a2aa751b847f4a0f64a9959d8a7e3700eacc7',
                          '0f328a45a3cf2bfb5a4cf60e13ef3dd6324a6af2b4b15a03e68dd864136c7984',
                          '363c6a14c10015b31f753994b513daa3dde497fe485b75fb90ab2695f49d873f',
                          '058d5c353be680ab65f033a8962772325d98cd96eafb59f6e70c077576f91ed4',
                          'd23cbd926904691315e39a5d935b7dc3b256764bae2547784b3c4d631ee11947',
                          '2c353cb2f84bdc3b46be2025a09349b66886bd7a6e21764dafabf2ef73cabb36',
                          'c9bdb1f1f6498132c84cf3ed98fd05678472043d35834aa0d144337a8dda4f77',
                          'eaa9d7a487710aeb6548e16c2e0dd3efc7daa03231ffa735fca26ae91d3bd158',
                          'ca973d22482a4c8a7abeb6b7cdffae5ab02d2b25e7ef0ba9cb309bbe923b4059',
                          '47551a2489eb09ad2da883c6adec5e4fc2f6f12dc47d376d17f6dc7cc6fef285',
                          'b73f28d24703239c6af8bb16b80c7afba5ae6a6ef35f719799fd4a56774a9a3e',
                          'd72c616c02ed0095d22bcf3eb891edb90a1949ecd5f9691d35b6cb7ca8e194f5',
                          '217c563dc3fddfb462f4abbb7ccd872ded097413301f0fb166c180ffaf10680c',
                          '7fefa921a6f14bca5cc222a3036cdeedb95a1b7058f18752969289ec2648f4ed',
                          'ef0cad2e9d77116b8fd0882e5eb722827358172564f4c8982a214a692d26eaae',
                          'cdd46c341d141dc0a9156546726bdfe90581eb5ad0ac6d2f5b5b1a04ca3dbf67',
                          '4bdfe57004a8cd7dae812253b057800ef3f265e0ea92ece98514856c15bdefaf',
                          '609e504207c753ddd5e1cd970d651576b96a41ede0fc6366711a76a8bfb7fb82',
                          'd33826f37c63c459ca40b9e5c50d65ae298a46472923135a10cd5397e6a4dbab',
                          '4755d79b5742075aafe9136853b3a45c18e2c74de1a2e037d6a292604c17b154',
                          'd8221b6593140465a6c000b21a9103811d97242f23349b130bc637065daa625b',
                          'b545a7301eeda10758632efc66ff54cd5cadb987cdda65878c98a4d40d40d382',
                          'dbd9516dd9f7342bbe1a2508066e59241b26149fa63d2ef2ddf6679d0f889c27',
                          'c64050d262fb76a726894c61f9e236eb0f210ff89afb4a466f6957ed6aba8311',
                          '74f2e2b84e80926b1d1cfa12820a74b4a2734f14779890559f8966c5a9d2f2a0',
                          '2a3a2a6b2a5b321c19baf4de1a0431816c9e1299e1a67aa3bf1f2fccb447a6ad',
                          'eba55db0c939c9dfc0c4b393858f5f871c1531d3189731257c22a23c0037179d',
                          '17d71d00032256f7aa022b26da0cb06267b554a692893213b315feacc39d6ac5',
                          '2fe7c85b683d6b09700bfa1d1e2ab63e696e53621b1255dbffbb41789ad50a18',
                          'cc03caae1aa3dd14afadaff694dc1f600845a8f230f464bf4e36f81d1eb3f0d7',
                          '501a19a8f83296edd3e291d899153d4af2b8d4307219ac31a257dd6e70c9eefb',
                          '210cdd5b6d4be8d597547d69766af2e6e5d5087a3a9e4bfcaaaafc116451152b',
                          '3af527a45ed23d571fb6a8e24ab8eb5699621194310f012da20118a900e7a7f7',
                          'b15d370de3c38dee39f1c718d58c1aad17538d5cfc41aec76cd57dd74762f3af',
                          '0b9cdd6af7411c4095d9840e3ee8ed559daa6dae5beebc93c16cc51d15c63506',
                          '3238bd3fa558879d98a3512ba61046f2f34272ae9319d7ca12a2fca67de1aedd',
                          '697212be1d00f0ebd4d251fd66e9aca0b2a2e56db4db8b38de5e066deddfedc5',
                          '8020a0d1ff778a8e8ea8385188891bad0841a590bbe445d2176dc2616d44748c',
                          '0ba797f0b2beb834faf4dc70cd0cf0d872817f03b8f49b14bf536e32eeacf785',
                          '79294e97c363afb031381a69c8f86c1742c842a55ec108730efb354078f53122',
                          'a1f8788a02599524a177b2452e712c201246453fa44f49ca2697150860566234',
                          'be0a617f1090affc229d0c10ef0c46cd983a73f5d42f3cdbf166e95313d8b2ae',
                          'a6348e3bcd5456b8b69cd5e09b133653a211fe0634aacb3634a5f815d270e589',
                          '5e525649f0e748cb0c6e6f62d6d338e884cfc1baef6d5df5d228b542df2d43a7',
                          '9847193e99e3e4a8d5df9fb8dfdbb7db903630ffbf651fa6662b986f1ba4a16f',
                          '04eefd04b4ea48004fa383bcbf35aa751dedaa95efab605c77674021bc794f71',
                          '8d0575e83fe369ccd672f27ad889a9dfdc4f517dfb0ece25e4c291d5724732b9',
                          '43d5a5f4f30c04de152b912b9e50128ec2ff837ec4c0b340aee15e025a1eb905',
                          'cd0e8ec20009a6ea5d0ad3c9f4fb13d8c7f63d041210414ced946a198f077761',
                          '3b96f65812c35e52be5d10261363d221385700c1cbd9e1cae02bca2eb358b771',
                          '7adfb0914d1916012dab439ecf342839835f41a68eaaf1c4517afa9d7847b626',
                          '3c892fa76bb770337f86ac817941f6623c04663e4e7bace7f0b8c3ced70f5d66',
                          '3030f8580b36db8eecb0fbbc889a0ea673a3acad45836dd991f98e20e9e11517',
                          '66df0f9e8befe6c405699906669c03bfc1c28d3a7e74a1fc5af1431c6ab4bdaa',
                          'f0318103d62cdc10f51419aa5fd71ce6d7ebac8b283a78f09a7e11b13072be12',
                          'c7e035122c3cd95105d57dab4e8dc8a784b451f610e7e27ae868167dc57b33f5',
                          '422945d0c129ed7803b523c91cde540f976da7618f4e2bc59aec187d0989dc53',
                          'a539eb0160f433664ffb9ac52dc0f9ff61b21518dce0fe4d5d4b20c495a8f586',
                          'ffcf3eee073fa2bc0daa9316c1f848e79a65ec27e8f63b29f7b7f2d1e80b7898',
                          '3c9c672675554754ee7385dfb80303f8fcf0915e80aafe56495bd3505ecfbe5c',
                          '84f7da83fd4a19d839b38b02fe9cfc5e4e7a232e8799794b3013cf0bde1a7066',
                          '9fa4dad55e35af304fdde434747cb718a18ceb3617257d1d936f0d09c4df5f41',
                          'dabdb3ad5aab1e2c7606e065f9ec0b5ccd222634563149176046ec6a8ea7faa4',
                          'f9f9901487f4c64e36c4441e00362a9f012a945b6db5cfe2030fe800215c0bb7',
                          '675954a180056d28e080a78fcbf0efb49e619e17dcf56f22369051a2331470b3',
                          '5144ab1e87c86a75492c7a6495c1bfef5778980a3803ca2caf00f3d8fc410f28',
                          'ca81fc43af7c64f7154c6ac29db881a181d61da42cf82d2617f6d70204af512d',
                          'dd57973bed8793da8745cbbda0c478556fd132b0e5793b8ddc0d8676da89ca5c',
                          '39e989e179963e8cf061abc0bd32c30579dc0d77eb7ec363356284915c5d294e',
                          '5b6b164ca7c58b4ad9be13b016aa2b66a64cfb5f27ec24498a73d2927b869aa5',
                          '6f7dd32e5457545710a9da309ec1a3813c9c2d8ad955686d57c8d314cf296bf9',
                          '0b34c3f737905d058400a3b299b7853c27e4cbf302bfb4f2fcaa09047cadfac8',
                          'c3eaf509b339c27bc416b1306da0a573458e24b00cb03c81174719495f8ee0a6',
                          'e8f57178c9dda1ad3b181f6486f55c5564a20f1e86a4d56655537522cd01f21b',
                          'db13a623339e7eb113b514e526b8dd4c48ea0bf4fe5e6b1ab67e46e0ae8755dd',
                          'e84b504de234cce8d89145cad2e9450606332704ea3b1b477ef68835cfb67937',
                          'cb19a5529faa2463a7bbe73270eb7d2a80853cfa4bf13b3bf1b83c0f67e6f57f',
                          '44a5ccab459559e6c665ce80951d0e5a679d5c67950682ec8c4519836553b0df',
                          '53beecbd7f9a7eee5aef8a60144d417cf8ab350c8bf9399cc13bb61733b6cc65',
                          '64585202cc972d21ab8dd0f603b76f9833f981f29b98908bc36a3bdf5dd77d51',
                          '45de925223f0a62875e752bb4c4bdaddb13b60c7fdc7c2846c83130bfef27eb1',
                          '4a6a2e6fc197b3f61e187b6c420399f456bd99d9ec4f7e678b4098792310ca0d',
                          '1caaa04d1678b772bf5bdeee0982880657cdc8741609a1dd7beb4a67d0212e55',
                          '1bee85ca8055cdc8125538e865caa59bcece6aa90a149758d4bdca3ae855835e',
                          '6f49f0f05d2df15e70a03ca6caf35d98655beb6d7fdcf036160958749f25abc4',
                          'd2f6845706eb555b10713575e7b852454bc3b5620e869a4bf7a198ce56879777',
                          '3327f2a690c915ced7bcca0d4f81597dc24daa4a7a2c454826c46fb70695ab39',
                          '6fe56567be2ca329e75234c985ee95430839817b2030c9918025dc917983f7ef',
                          '3f103a95be2a938f16ec55e5a2a5fec46f3eb34b76fd10536e5fdefbe99e7813',
                          '105fcab86f503a97dfef2f8f9a3a2cca2641306714df060f553ada3acd96c8dd',
                          '12535b48c89b2d7a339e1fa11c4edb84ec232f5d565200a06e5dd42413a70dbd',
                          '682d8e92bc195d1b0ac55206279b85330f612a07de0cea46913ef1dcb1a0a1fd',
                          'b313fe9a426189bdbf0fdf969a5f6f2a127a0ab756c8eb4b76b4a1b944d3c29b',
                          '14d704549005045ccdef7464bad5df4f81f8e3313d92fafad26d551d1b2e02e2',
                          '72e4dcb218e48f09df4eab58412389c259a85fe6fc1b649f0b6f4d8c92faad81',
                          'e5f04d86078362171936c15e342bf08ca88df1353a48502a6f1e7acf131fbb83',
                          'bd3887ce77f616184eda2fd55c1363c99768a15df3ecc1fb9a1352d0140c3da0',
                          '11305e2605ce0d223099b48600bc36d20a6706314f69c859b893bfdc25d05431',
                          'fb4dda03341891307ff3f1f0e8f5cb73ab6c9b0a44d16e77dd4ea273ae56f641',
                          '9b6c5d785c00afe6d77ca81ad052d8d55bf034dbd734093d4b72aa2d3e4d2afa',
                          '618ad1b14bdfeb156d401375001b811948ffd21943d644b8b4fbac28801eb12f',
                          '89f8d84b36432628b137c6455f17c425d49bc5b8b274fedc2103b668f7be7a04',
                          'cca03dda23bbe014aaa17a9e51a66c34bd481cbfd73b942d74e644936e136022',
                          '5c79464863cac894c5ad13d984d6cdb6834b72c8437b9de4b9a3e4072f3fea15',
                          '8f363f31f3dff7f5311bc296aa677441c61a4f7635472aa841c1198027037e7d',
                          '7d21b69701d4c9da4cb0876448f089b3529af2c70e63c52aba7bf368e8b450cf',
                          '3938bbb08b82fe0378d1f55530f94e5645fb61ee09867340aefe5dc258f0070e',
                          '9569d188333c72f87939e4fdb401a94af7aec89a8878209ef113a17b4b9481f2',
                          '47731bdc412904e4d597b0a40365380d6a69c373dd3cc8924bd1a73ce357d635',
                          '35f5b161b03998a66c750105c9c9d01a326c076f7f41ef5ec730ba4a6e9e1404',
                          '6de599b17713fe4664ce8d3e6b861a63f8f399f87a9f69b46a187735c14f68a5',
                          '07cd670dd70670e173fd703dc00c298cdf81cd60cbe5bb1884e4c281a89df151',
                          '94b957fe02c8b2166040635a490c0571e47f45a2e664e67c7d9f6c8228d960c0',
                          'eef1bc0bb45b1979e5f69920c3c7cf6d030a4ca6accfd93c0a6d36a64ec93391',
                          '151bd74973256bf122a0260db34c96e4e102f20e3bc1fd0a6c93d049c7654d97',
                          '0e5b291376a58a933f15e5ca8adbd8987ec600c138d1308bab9c147416162e19',
                          '6f27f7aba151e830cf48e2f5df8031b3810729e419fa618b6b30ba3355fa8ff3',
                          'e6357c6de6c2352032c9e3ca16e149efc5e7e2bcd5b5ec4a7807dd99ea7b5034',
                          '04ead216e16cdb9c6176d5c581c10dbcdf3a01a98871533e0ca504ade94860c3',
                          '75fe4b476e2d596cbf623bda69ed58bc09434d0168c67776fdf2059a52098e1e',
                          'b368f01114bbe6ff3fd1279127e41a853ac52887b3ced034024d07d6f82f34c9',
                          '5b44f9500cbf14d62e713a24adf813b92bb206d1eb842943aaf90a655037d7ef',
                          'eed09b78d2b484d666a5fa6e084df72e2b60ee7122deb52a52fd05e6bdd2ffd1',
                          '27b8b43eae7602fda0940a6c698b675fdadf8d201f56146dcfcf32c5e7bcde59',
                          'd9341c759c0898d91307cf9b3ae5efcb25bd5384b498e2720db8253ca7e8188f',
                          '8a8d20199886a3ef7714620594ed1f0261d99693616842b6ce4f1d71b845be10',
                          'ca23ec30d8348ce16a99baf9e360dba6440c67f05de44e7ab7019ef8169516a9',
                          '27a780b7a4eb0ca137c56aa2261ae63b279279ca452565fa8cbced8a40636ee6',
                          '1a012e6a3fa504fbf634665a7c8294daef728a218d6d8e5b3a51da49498dc017',
                          'c42ab1a1bdcad4016ff71a6cd9911448ede8004bf79a66c8b0e4e9ea2d929ee0',
                          'c01881c04b7f32256bfb77bcfcd6fe882b9d1150ca1d5a50d8bb65163b4c04e6',
                          'a938cef46b1ad6a97ace5276cac89a6ebee184f77ede4f300508b9cdf5d3ff62',
                          '61616312b79ce0a6ebea0c1edb9d2dec7cd4727165e8f7d2529393983d401447',
                          '8ef55ac94c3d2e418592b37bbab5dbb9fa0e085988d949da289e8361334be06d',
                          'b5ccc17c1dc51e0266ecccf365ef2eff576cf0b4b781e48cd571996c190bae06',
                          '26fcc503337caf416882e5de6b7762129877c4926d72927db93c6dd3b3aecd6e',
                          'd902618e5fd70fafa264b9403794e9b8b8c94f54611422caa937bd5d1f2ce08e',
                          '4300d25abee91559787f8d70f087ffcc87e63b55c9f7f475947eef2ca7d0ec7f',
                          'ed96c947fa5895d0ab7d38332e4019d5aecadf0253ff0753c8047e0e25306761',
                          '627051b23a3628de3cc5a104f482849437d719a42bdebcd59ee548faef94d0ec',
                          '861d81fd660255594cf9a736c87ca68c82ae34f45607f925da9c6967c658c231',
                          '31bcd149f49c97e3fdac5b661e2b6b3cec17a132623ee307c36a92b3d39228db',
                          '009bc6b1df70faa550bb399c021f4f7e9fa664eeab43d7c364d281aa28421946'],
 'selection_role': 'fixed model confirmation on images outside known prior evaluation memberships',
 'selection_seed': 20261107,
 'selection_uses_image_content_or_quality_scores': False,
 'selection_version': 'exclude_known500_sha256_rank_v26_1',
 'source_data_manifest_sha256': '140d5919305f3592d144ed4a1a8f8d53dcd19e170b212416591b454c0ea4e7e2',
 'source_training_images': 11000,
 'source_training_inventory_sha256': '04768543c975e0627eceb5615024145dde6fc1f4c636cfa614587a55fed3b0b3',
 'source_zips_modified': False,
 'test_zip_file': 'GEE_test.zip',
 'test_zip_sha256': '558916f53c3a1cabf04d3bda5d061a120a0ab3c5bc597bb56956e2f0e6adf21a',
 'training_RGB_duplicate_check_this_run': False,
 'version': 'fixed_compact_final_data_v26_1'}


In [ ]:
import copy,gc,hashlib,io,json,math,os,shutil,time,zipfile
from pathlib import Path
import numpy as np
import torch
from PIL import Image
from google.colab import drive

if not torch.cuda.is_available():raise RuntimeError("Select a Colab GPU runtime")
device=torch.device("cuda")
torch.set_num_threads(min(8,os.cpu_count() or 2))
drive.mount("/content/drive")
DRIVE_DATA_DIR=Path("/content/drive/MyDrive/FSO_data")
SOURCE_V25_ROOT=DRIVE_DATA_DIR/COMPACT_FINAL_INPUT_LOCK_V26["source_root_name"]
PILOT_ROOT=DRIVE_DATA_DIR/"stage2_fig8b_lowpower_v35"
LOCAL_CACHE=Path("/content/fso_sc_fig8b_lowpower_v35")
LOCAL_CACHE.mkdir(parents=True,exist_ok=True)
if shutil.disk_usage(LOCAL_CACHE).free<10*1024**3:
    raise RuntimeError("Keep10GiB local disk free for the verified checkpoint and codec build")
if (COMPACT_FINAL_INPUT_LOCK_V26["evaluation_count"]!=200
        or COMPACT_FINAL_DATA_LOCK_V26["count"]!=200):
    raise RuntimeError("Keep the frozen primary model and compact200-image budget")
print("GPU:",torch.cuda.get_device_name(),"| Torch:",torch.__version__,"| CUDA:",torch.version.cuda)
print("Inference only; no optimizer, weight updates, source edits or new checkpoints")
print("Keep1GiB actual Drive quota free; mounted disk estimates can differ from account quota")

FIG8B_RUN_LABEL_V35 = "run1"
AUTO_DOWNLOAD_RESULTS_V35 = True


In [ ]:
import math

import torch
import torch.nn as nn
import torch.nn.functional as F

MODEL_VERSION = "fso_sc_paper_faithful_v6"
SEMANTIC_MODEL_CONFIG = {
    "group_norm_groups": 32,
    "activation": "SiLU",
    "downsample": "asymmetric_right_bottom_pad_then_stride2_conv",
    "upsample": "nearest_neighbor_then_conv",
    "loss_reduction": "mean",
    "input_preprocessing": "largest-center-square+bicubic256+RGB[-1,1]",
    "conv_initialization": "PyTorch_default",
    "codebook_initialization": "uniform(-1/K,1/K)",
    "decoder_attention_order": "VQGAN_inferred_interleaved_Res_Attn",
    "decoder_condition": "raw_Uq; post_quant_conv_only_for_decoder_feature",
}


def group_norm(channels, affine=True):
    return nn.GroupNorm(32, channels, eps=1e-6, affine=affine)


class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.norm1 = group_norm(in_channels)
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, padding=1)
        self.norm2 = group_norm(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, padding=1)
        self.shortcut = (
            nn.Conv2d(in_channels, out_channels, 1)
            if in_channels != out_channels
            else nn.Identity()
        )

    def forward(self, x):
        h = self.conv1(F.silu(self.norm1(x)))
        h = self.conv2(F.silu(self.norm2(h)))
        return self.shortcut(x) + h


class AttnBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.norm = group_norm(channels)
        self.q = nn.Conv2d(channels, channels, 1)
        self.k = nn.Conv2d(channels, channels, 1)
        self.v = nn.Conv2d(channels, channels, 1)
        self.proj_out = nn.Conv2d(channels, channels, 1)

    def forward(self, x):
        h = self.norm(x)
        q, k, v = self.q(h), self.k(h), self.v(h)
        batch, channels, height, width = q.shape
        q = q.reshape(batch, channels, height * width).permute(0, 2, 1)
        k = k.reshape(batch, channels, height * width)
        attention = torch.bmm(q, k) * (channels ** -0.5)
        attention = F.softmax(attention, dim=2)
        v = v.reshape(batch, channels, height * width)
        h = torch.bmm(v, attention.permute(0, 2, 1)).reshape(
            batch, channels, height, width
        )
        return x + self.proj_out(h)


class SpatialNorm(nn.Module):
    """Paper Eq. (25): gamma(U_q) * GN(F) + beta(U_q)."""

    def __init__(self, channels, latent_dim):
        super().__init__()
        # MoVQ replaces VQGAN's normalization with spatial conditioning while
        # retaining GroupNorm's learned affine parameters.
        self.norm = group_norm(channels, affine=True)
        self.conv_gamma = nn.Conv2d(latent_dim, channels, 1)
        self.conv_beta = nn.Conv2d(latent_dim, channels, 1)

    def forward(self, x, quantized_condition):
        if x.shape[2:] != quantized_condition.shape[2:]:
            quantized_condition = F.interpolate(
                quantized_condition, size=x.shape[2:], mode="nearest"
            )
        gamma = self.conv_gamma(quantized_condition)
        beta = self.conv_beta(quantized_condition)
        return gamma * self.norm(x) + beta


class SpatialResBlock(nn.Module):
    def __init__(self, in_channels, out_channels, latent_dim=4):
        super().__init__()
        self.norm1 = SpatialNorm(in_channels, latent_dim)
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, padding=1)
        self.norm2 = SpatialNorm(out_channels, latent_dim)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, padding=1)
        self.shortcut = (
            nn.Conv2d(in_channels, out_channels, 1)
            if in_channels != out_channels
            else nn.Identity()
        )

    def forward(self, x, quantized_condition):
        h = self.conv1(F.silu(self.norm1(x, quantized_condition)))
        h = self.conv2(F.silu(self.norm2(h, quantized_condition)))
        return self.shortcut(x) + h


class SpatialAttnBlock(nn.Module):
    def __init__(self, channels, latent_dim=4):
        super().__init__()
        self.norm = SpatialNorm(channels, latent_dim)
        self.q = nn.Conv2d(channels, channels, 1)
        self.k = nn.Conv2d(channels, channels, 1)
        self.v = nn.Conv2d(channels, channels, 1)
        self.proj_out = nn.Conv2d(channels, channels, 1)

    def forward(self, x, quantized_condition):
        h = self.norm(x, quantized_condition)
        q, k, v = self.q(h), self.k(h), self.v(h)
        batch, channels, height, width = q.shape
        q = q.reshape(batch, channels, height * width).permute(0, 2, 1)
        k = k.reshape(batch, channels, height * width)
        attention = torch.bmm(q, k) * (channels ** -0.5)
        attention = F.softmax(attention, dim=2)
        v = v.reshape(batch, channels, height * width)
        h = torch.bmm(v, attention.permute(0, 2, 1)).reshape(
            batch, channels, height, width
        )
        return x + self.proj_out(h)


class Upsample(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv = nn.Conv2d(channels, channels, 3, padding=1)

    def forward(self, x):
        return self.conv(F.interpolate(x, scale_factor=2.0, mode="nearest"))


class Downsample(nn.Module):
    """VQGAN downsample: asymmetric right/bottom pad then padding-free Conv."""

    def __init__(self, channels):
        super().__init__()
        self.conv = nn.Conv2d(channels, channels, 3, stride=2, padding=0)

    def forward(self, x):
        return self.conv(F.pad(x, (0, 1, 0, 1)))


class Encoder(nn.Module):
    def __init__(self, in_channels=3, latent_dim=4):
        super().__init__()
        self.in_conv = nn.Conv2d(in_channels, 256, 3, padding=1)
        self.res1 = nn.Sequential(ResidualBlock(256, 256), ResidualBlock(256, 256))
        self.down1 = Downsample(256)
        self.res2 = nn.Sequential(ResidualBlock(256, 512), ResidualBlock(512, 512))
        self.down2 = Downsample(512)
        self.res3 = nn.Sequential(ResidualBlock(512, 512), ResidualBlock(512, 512))
        self.down3 = Downsample(512)
        self.res4 = nn.Sequential(ResidualBlock(512, 1024), ResidualBlock(1024, 1024))
        self.attn1 = nn.Sequential(AttnBlock(1024), AttnBlock(1024))
        self.res5 = ResidualBlock(1024, 1024)
        self.attn2 = AttnBlock(1024)
        self.res6 = ResidualBlock(1024, 1024)
        self.out_norm = group_norm(1024)
        self.out_conv = nn.Conv2d(1024, latent_dim, 3, padding=1)

    def forward(self, x):
        x = self.res1(self.in_conv(x))
        x = self.res2(self.down1(x))
        x = self.res3(self.down2(x))
        x = self.res4(self.down3(x))
        x = self.attn1(x)
        x = self.attn2(self.res5(x))
        return self.out_conv(F.silu(self.out_norm(self.res6(x))))


class Decoder(nn.Module):
    """Fig. 3 block set arranged in the dimensionally valid MoVQ/VQGAN order."""

    def __init__(self, latent_dim=4):
        super().__init__()
        self.in_conv = nn.Conv2d(latent_dim, 1024, 3, padding=1)

        # VQGAN mid block: Res -> Attn -> Res.
        self.mid_res1 = SpatialResBlock(1024, 1024, latent_dim)
        self.mid_attn = SpatialAttnBlock(1024, latent_dim)
        self.mid_res2 = SpatialResBlock(1024, 1024, latent_dim)

        # Deepest level: three Res+Attn pairs, then 32 -> 64.
        self.deep_res = nn.ModuleList([
            SpatialResBlock(1024, 1024, latent_dim) for _ in range(3)
        ])
        self.deep_attn = nn.ModuleList([
            SpatialAttnBlock(1024, latent_dim) for _ in range(3)
        ])
        self.up_1024 = Upsample(1024)

        # 64x64 level: 1024 -> 512 followed by two 512 blocks, then 64 -> 128.
        self.level_512a = nn.ModuleList([
            SpatialResBlock(1024, 512, latent_dim),
            SpatialResBlock(512, 512, latent_dim),
            SpatialResBlock(512, 512, latent_dim),
        ])
        self.up_512a = Upsample(512)

        # 128x128 level: three 512 blocks, then 128 -> 256.
        self.level_512b = nn.ModuleList([
            SpatialResBlock(512, 512, latent_dim) for _ in range(3)
        ])
        self.up_512b = Upsample(512)

        # 256x256 level: 512 -> 256 followed by two 256 blocks.
        self.level_256 = nn.ModuleList([
            SpatialResBlock(512, 256, latent_dim),
            SpatialResBlock(256, 256, latent_dim),
            SpatialResBlock(256, 256, latent_dim),
        ])
        self.out_norm = SpatialNorm(256, latent_dim)
        self.out_conv = nn.Conv2d(256, 3, 3, padding=1)

    def forward(self, decoder_input, quantized_condition):
        h = self.mid_res1(self.in_conv(decoder_input), quantized_condition)
        h = self.mid_attn(h, quantized_condition)
        h = self.mid_res2(h, quantized_condition)

        for res_block, attn_block in zip(self.deep_res, self.deep_attn):
            h = attn_block(
                res_block(h, quantized_condition), quantized_condition
            )
        h = self.up_1024(h)

        for block in self.level_512a:
            h = block(h, quantized_condition)
        h = self.up_512a(h)

        for block in self.level_512b:
            h = block(h, quantized_condition)
        h = self.up_512b(h)

        for block in self.level_256:
            h = block(h, quantized_condition)
        return self.out_conv(F.silu(self.out_norm(h, quantized_condition)))


class VectorQuantizer(nn.Module):
    def __init__(
        self,
        num_embeddings=16384,
        embedding_dim=4,
        commitment_cost=1.0,
        distance_chunk_size=2048,
    ):
        super().__init__()
        self.embedding_dim = embedding_dim
        self.num_embeddings = num_embeddings
        self.commitment_cost = commitment_cost
        self.distance_chunk_size = distance_chunk_size
        self.embedding = nn.Embedding(num_embeddings, embedding_dim)
        self.embedding.weight.data.uniform_(
            -1.0 / num_embeddings, 1.0 / num_embeddings
        )

    @torch.no_grad()
    def nearest_indices(self, flat_input):
        # VQ assignment remains FP32 even when the surrounding model uses AMP.
        vectors = flat_input.detach().float()
        codebook = self.embedding.weight.detach().float()
        codebook_norm = codebook.square().sum(dim=1).unsqueeze(0)
        nearest = []
        for chunk in vectors.split(self.distance_chunk_size, dim=0):
            distances = (
                chunk.square().sum(dim=1, keepdim=True)
                + codebook_norm
                - 2.0 * chunk @ codebook.t()
            )
            nearest.append(distances.argmin(dim=1))
        return torch.cat(nearest, dim=0)

    def lookup(self, encoding_indices, spatial_shape=None):
        if spatial_shape is None:
            if encoding_indices.ndim != 3:
                raise ValueError("indices must have shape [batch, height, width]")
            batch, height, width = encoding_indices.shape
        else:
            batch, height, width = spatial_shape
        flat_indices = encoding_indices.reshape(-1).long()
        quantized = F.embedding(flat_indices, self.embedding.weight)
        return quantized.view(batch, height, width, self.embedding_dim).permute(
            0, 3, 1, 2
        ).contiguous()

    def forward(self, inputs):
        channels_last = inputs.permute(0, 2, 3, 1).contiguous()
        batch, height, width, channels = channels_last.shape
        if channels != self.embedding_dim:
            raise ValueError(f"Expected embedding_dim={self.embedding_dim}, got {channels}")

        flat_input = channels_last.view(-1, self.embedding_dim)
        flat_indices = self.nearest_indices(flat_input)
        encoding_indices = flat_indices.view(batch, height, width)
        quantized = self.lookup(encoding_indices, (batch, height, width))

        encoder_loss = F.mse_loss(quantized.detach().float(), inputs.float())
        codebook_loss = F.mse_loss(quantized.float(), inputs.detach().float())
        vq_loss = codebook_loss + self.commitment_cost * encoder_loss

        quantized_st = inputs + (quantized.to(inputs.dtype) - inputs).detach()
        return quantized_st, vq_loss, encoding_indices


class FSO_VQVAE(nn.Module):
    def __init__(self, num_embeddings=16384, embedding_dim=4, commitment_cost=1.0):
        super().__init__()
        self.encoder = Encoder(in_channels=3, latent_dim=embedding_dim)
        self.pre_quant_conv = nn.Conv2d(embedding_dim, embedding_dim, 1)
        self.vq = VectorQuantizer(num_embeddings, embedding_dim, commitment_cost)
        self.post_quant_conv = nn.Conv2d(embedding_dim, embedding_dim, 1)
        self.decoder = Decoder(latent_dim=embedding_dim)

    def encode(self, x):
        if x.ndim != 4 or x.shape[1] != 3:
            raise ValueError("Expected RGB input with shape [batch, 3, height, width]")
        if x.shape[-2] % 8 or x.shape[-1] % 8:
            raise ValueError("Input height and width must be divisible by 8")
        continuous = self.pre_quant_conv(self.encoder(x))
        quantized, vq_loss, indices = self.vq(continuous)
        return quantized, vq_loss, indices

    def decode_quantized(self, quantized):
        decoder_input = self.post_quant_conv(quantized)
        return self.decoder(decoder_input, quantized_condition=quantized)

    def decode_indices(self, indices, spatial_shape):
        return self.decode_quantized(self.vq.lookup(indices, spatial_shape))

    def forward(self, x):
        quantized, vq_loss, indices = self.encode(x)
        return self.decode_quantized(quantized), vq_loss, indices


# The architecture is intentionally not instantiated here; it is large (~hundreds of M parameters).

In [ ]:
from dataclasses import dataclass
import math

import torch
import torch.nn.functional as F


BITS_PER_INDEX = 14
QAM_MAPPING_VERSION = "gray_qpsk_rect8qam_16qam_unit_energy_v1"
INTEGER_DTYPES = {torch.uint8, torch.int8, torch.int16, torch.int32, torch.int64}


def integers_to_bits(values, width):
    values = values.long()
    shifts = torch.arange(width - 1, -1, -1, device=values.device)
    return ((values.unsqueeze(-1) >> shifts) & 1).to(torch.uint8)


def bits_to_integers(bits):
    bits = bits.long()
    width = bits.shape[-1]
    weights = (2 ** torch.arange(width - 1, -1, -1, device=bits.device)).long()
    return (bits * weights).sum(dim=-1)


def indices_to_bits(indices, bits_per_index=BITS_PER_INDEX):
    if indices.dtype not in INTEGER_DTYPES:
        raise TypeError("indices must use an integer dtype")
    if indices.numel() and (
        (indices < 0).any().item() or (indices >= 2**bits_per_index).any().item()
    ):
        raise ValueError("Index is outside the fixed-width bit representation")
    return integers_to_bits(indices, bits_per_index)


def bits_to_indices(bits, bits_per_index=BITS_PER_INDEX):
    if bits.shape[-1] != bits_per_index:
        raise ValueError(f"Last dimension must be {bits_per_index} bits")
    if bits.dtype not in INTEGER_DTYPES and bits.dtype != torch.bool:
        raise TypeError("bits must use an integer or bool dtype")
    if bits.numel() and ((bits != 0) & (bits != 1)).any().item():
        raise ValueError("bits must contain only 0 and 1")
    return bits_to_integers(bits)


def qam_constellation(modulation, device):
    modulation = modulation.upper()
    supported = {"QPSK": 2, "8QAM": 3, "16QAM": 4}
    if modulation not in supported:
        raise ValueError(f"Unsupported modulation: {modulation}")
    bits_per_symbol = supported[modulation]
    labels = integers_to_bits(
        torch.arange(2**bits_per_symbol, device=device), bits_per_symbol
    ).float()

    if modulation == "QPSK":
        i = 1.0 - 2.0 * labels[:, 0]
        q = 1.0 - 2.0 * labels[:, 1]
        normalization = math.sqrt(2.0)
    elif modulation == "8QAM":
        # Rectangular Gray-labelled 8-QAM. The paper does not specify its 8-QAM geometry.
        gray_levels = torch.tensor([-3.0, -1.0, 3.0, 1.0], device=device)
        i_code = (2 * labels[:, 0] + labels[:, 1]).long()
        i = gray_levels[i_code]
        q = 2.0 * labels[:, 2] - 1.0
        normalization = math.sqrt(6.0)
    else:
        gray_levels = torch.tensor([-3.0, -1.0, 3.0, 1.0], device=device)
        i_code = (2 * labels[:, 0] + labels[:, 1]).long()
        q_code = (2 * labels[:, 2] + labels[:, 3]).long()
        i = gray_levels[i_code]
        q = gray_levels[q_code]
        normalization = math.sqrt(10.0)

    points = torch.complex(i, q) / normalization
    return points.to(torch.complex64), labels.to(torch.uint8), bits_per_symbol


def qam_modulate(bits, modulation):
    points, _, bits_per_symbol = qam_constellation(modulation, bits.device)
    if bits.shape[-1] != bits_per_symbol:
        raise ValueError(f"{modulation} expects {bits_per_symbol} bits per symbol")
    symbol_ids = bits_to_integers(bits).long()
    return points[symbol_ids]


def qam_demodulate(symbols, modulation):
    points, labels, _ = qam_constellation(modulation, symbols.device)
    # Cartesian-product QAM: limiting observations outside the outer levels
    # preserves exact nearest-point decisions and avoids large-float cancellation.
    i_limit = 2 * points.real.abs().max()
    q_limit = 2 * points.imag.abs().max()
    stable = torch.complex(symbols.real.clamp(-i_limit, i_limit),
                           symbols.imag.clamp(-q_limit, q_limit))
    distances = (stable.reshape(-1, 1) - points.reshape(1, -1)).abs().square()
    nearest = distances.argmin(dim=1)
    return labels[nearest].view(*symbols.shape, labels.shape[-1])


@dataclass(frozen=True)
class FSOChannelOutput:
    """Receiver observation before hard QAM decision.

    Return `is_equalized=True` after coherent receiver DSP, or provide the
    complex channel gain so this link can perform one-tap equalization.
    """

    symbols: torch.Tensor
    channel_gain: object = None
    is_equalized: bool = False
    diagnostics: object = None


class IdentityFSOChannel:
    """No-impairment receiver hook used to test the digital path."""

    def __call__(self, symbols, **channel_kwargs):
        return FSOChannelOutput(symbols=symbols, is_equalized=True)


def equalize_fso_output(channel_output, transmitted):
    if isinstance(channel_output, FSOChannelOutput):
        received = channel_output.symbols
        channel_gain = channel_output.channel_gain
        is_equalized = channel_output.is_equalized
    elif isinstance(channel_output, tuple) and len(channel_output) == 2:
        received, channel_gain = channel_output
        is_equalized = False
    else:
        raise TypeError(
            "FSO receiver must return FSOChannelOutput or (symbols, channel_gain)"
        )

    if not torch.is_tensor(received):
        raise TypeError("FSO receiver must return a torch.Tensor observation")
    if received.device != transmitted.device:
        raise ValueError(
            f"FSO receiver changed device: {transmitted.device} -> {received.device}"
        )
    if not torch.is_complex(received):
        raise TypeError("FSO receiver output must be complex baseband symbols")
    if received.shape != transmitted.shape:
        raise ValueError(
            f"FSO receiver changed symbol shape: {transmitted.shape} -> {received.shape}"
        )

    if not is_equalized:
        if not torch.is_tensor(channel_gain):
            raise TypeError("Unequalized FSO output must include a tensor channel_gain")
        if channel_gain.device != received.device:
            raise ValueError("channel_gain and received symbols must share a device")
        try:
            channel_gain = torch.broadcast_to(channel_gain, received.shape)
        except RuntimeError as error:
            raise ValueError("channel_gain is not broadcastable to the symbol shape") from error
        if (channel_gain.abs() == 0).any().item():
            raise ValueError("channel_gain contains an exact zero")
        if not torch.isfinite(channel_gain).all().item():
            raise ValueError("channel_gain contains NaN or Inf")
        denominator = channel_gain.abs().square().clamp_min(
            torch.finfo(received.real.dtype).tiny
        )
        received = received * channel_gain.conj() / denominator

    if not torch.isfinite(received).all().item():
        raise ValueError("FSO receiver output contains NaN or Inf")
    return received


@dataclass
class QAMIndexLink:
    """14-bit digital link ending in equalized complex [B, S] symbols."""

    fso_channel: object
    modulation: str = "QPSK"
    bits_per_index: int = BITS_PER_INDEX
    receiver_config: object = None

    def reproducibility_config(self):
        if self.receiver_config is not None and not isinstance(
            self.receiver_config, dict
        ):
            raise TypeError("receiver_config must be a serializable dict or None")
        return {
            "qam_mapping_version": QAM_MAPPING_VERSION,
            "modulation": self.modulation.upper(),
            "bits_per_index": self.bits_per_index,
            "receiver_config": self.receiver_config,
        }

    @torch.no_grad()
    def __call__(self, indices, return_statistics=False, **channel_kwargs):
        if indices.ndim < 2:
            raise ValueError("indices must retain a batch dimension")
        original_shape = indices.shape
        batch = original_shape[0]
        valid_bits_per_sample = indices[0].numel() * self.bits_per_index
        bitstream = indices_to_bits(indices, self.bits_per_index).reshape(batch, -1)
        _, _, bits_per_symbol = qam_constellation(self.modulation, indices.device)
        padding = (-valid_bits_per_sample) % bits_per_symbol
        if padding:
            bitstream = F.pad(bitstream, (0, padding))

        grouped_bits = bitstream.view(batch, -1, bits_per_symbol)
        transmitted = qam_modulate(grouped_bits, self.modulation)
        channel_output = self.fso_channel(transmitted, **channel_kwargs)
        received = equalize_fso_output(channel_output, transmitted)
        recovered_grouped = qam_demodulate(received, self.modulation)
        recovered = recovered_grouped.reshape(batch, -1)
        valid_mask = torch.arange(
            recovered.shape[1], device=indices.device
        ).view(1, -1) < valid_bits_per_sample
        transmitted_bit_error_mask = recovered != bitstream
        data_bit_error_mask = transmitted_bit_error_mask & valid_mask
        # SER is defined over every transmitted QAM symbol, including the final
        # padded symbol. Payload BER/TER continue to exclude padding bits.
        symbol_error_mask = transmitted_bit_error_mask.view(
            batch, -1, bits_per_symbol
        ).any(dim=-1)
        if padding:
            recovered = recovered[:, :-padding]
        recovered = recovered.view(*original_shape, self.bits_per_index)
        recovered_indices = bits_to_indices(
            recovered, self.bits_per_index
        ).view(original_shape)
        if not return_statistics:
            return recovered_indices

        bit_errors = int(data_bit_error_mask.sum().item())
        valid_bits = batch * valid_bits_per_sample
        transmitted_bit_errors = int(transmitted_bit_error_mask.sum().item())
        transmitted_bits = bitstream.numel()
        symbol_errors = int(symbol_error_mask.sum().item())
        valid_symbols = symbol_error_mask.numel()
        index_errors = int((recovered_indices != indices).sum().item())
        statistics = {
            "modulation": self.modulation.upper(),
            "padding_bits_per_sample": padding,
            "bit_errors": bit_errors,
            "valid_bits": valid_bits,
            "ber": bit_errors / max(valid_bits, 1),
            "transmitted_bit_errors": transmitted_bit_errors,
            "transmitted_bits": transmitted_bits,
            "transmitted_ber": transmitted_bit_errors / max(transmitted_bits, 1),
            "symbol_errors": symbol_errors,
            "valid_symbols": valid_symbols,
            "ser": symbol_errors / max(valid_symbols, 1),
            "index_errors": index_errors,
            "indices": indices.numel(),
            "ter": index_errors / max(indices.numel(), 1),
        }
        if isinstance(channel_output, FSOChannelOutput) and channel_output.diagnostics:
            statistics["channel"] = channel_output.diagnostics
        return recovered_indices, statistics


def qam_identity_roundtrip_test():
    test_indices = torch.tensor([
        [0, 1, 2, 8191, 8192, 16383, 77],
        [77, 16383, 8192, 8191, 2, 1, 0],
    ]).view(2, 1, 7)
    for modulation in ("QPSK", "8QAM", "16QAM"):
        link = QAMIndexLink(IdentityFSOChannel(), modulation=modulation)
        recovered = link(test_indices)
        assert torch.equal(recovered, test_indices), f"{modulation} round trip failed"
    print("Identity-channel index/QAM round-trip passed: QPSK, 8QAM, 16QAM")


qam_identity_roundtrip_test()

In [ ]:
"""Paper-parameter symbol-rate FSO channel with explicit heterodyne shot floor.

The vacuum/shot term is a physically motivated missing receiver detail, not an
explicit Chen et al. parameter or an LPIPS-curve calibration. One complex
heterodyne mode/symbol has E|n|^2=h*nu/eta, so the optical-field (sqrt W)
convention gives N_complex=(q_e/R)*R_s W; I and Q each have half that variance.
Reference: A. Martinez, Physical Review A 77,032116 (2008),
doi:10.1103/PhysRevA.77.032116, https://ir.cwi.nl/pub/13524/13524D.pdf .
"""

from dataclasses import asdict, dataclass, replace
import math
import warnings

import numpy as np
import torch


PAPER_FSO_CHANNEL_VERSION = "chen2025_eq3_18_heterodyne_shot_v6"
WEATHER_ATTENUATION_DB_PER_KM = {
    "clear": 0.147,
    "dust": 1.043,
    "fog": 33.979,
    "rain": 1.403,
    "snow": 2.145,
}

# The paper specifies FOV=0.1 but does not print a unit, and it does not give
# the optical filter width or the proprietary VPI receiver/DSP implementation.
# These assumptions are explicit and stored with the final model metadata.
PAPER_UNDISCLOSED_RECEIVER_ASSUMPTIONS = {
    "fov_interpretation": "0.1 rad",
    "optical_filter_bandwidth": "1 nm",
    "background_noise_conversion": (
        "Eq.-(4) optical background power is used as unit-energy complex "
        "matched-filter noise variance; proprietary coherent receiver is undisclosed"
    ),
    "ase_model": (
        "Assume post-transmitter-EDFA launch power (probe point undisclosed); standard two-polarization "
        "EDFA ASE is propagated through the optical path"
    ),
    "ase_noise_bandwidth": "28 GHz symbol-rate equivalent",
    "diversity_ase_correlation": "one common transmitter-ASE waveform across Rx branches",
    "photodiode_responsivity": (
        "Table-I R=0.85 A/W determines eta=R*h*nu/q_e; "
        "one-complex-mode/symbol heterodyne shot floor q_e*Rs/R is input-referred; "
        "strong LO and negligible receiver thermal/RIN/dark current are assumptions"
    ),
    "receiver_shot_noise": (
        "complex heterodyne mode variance h*nu/eta per symbol; "
        "N_complex=q_e*Rs/R W, with half per real quadrature; "
        "physically motivated undisclosed implementation detail, not curve-fitted"
    ),
    "receiver_shot_noise_reference": "https://doi.org/10.1103/PhysRevA.77.032116",
    "pulse_shaping": "symbol-rate matched-filter equivalent; RRC rolloff undisclosed",
    "carrier_recovery": (
        "raw channel returns true complex gain for an explicitly ideal/oracle baseline; "
        "the separate SignalDSP wrapper uses only block magnitude and estimates phase/frequency"
    ),
    "rytov_quadrature": "xi=u^2 ground-clustered trapezoid with convergence self-test",
    "malaga_phase_difference": "pi/2 chosen phase assumption; paper undisclosed",
    "malaga_beta_fit": (
        "paper omits beta and phase; nearest Eq.-(15) natural beta is used with "
        "an explicit feasibility/residual diagnostic (strict policy is available)"
    ),
    "fading_coherence": "one independent block fade per image and receiver",
    "8qam_geometry": "rectangular Gray 8-QAM; paper only states 8-QAM",
}


def paper_generator_device_matches(generator, tensor_device):
    generator_device = generator.device
    if generator_device.type != tensor_device.type:
        return False
    if tensor_device.type != "cuda":
        return generator_device == tensor_device
    generator_index = (
        torch.cuda.current_device()
        if generator_device.index is None
        else generator_device.index
    )
    tensor_index = (
        torch.cuda.current_device() if tensor_device.index is None else tensor_device.index
    )
    return generator_index == tensor_index


@dataclass(frozen=True)
class PaperFSOSystemConfig:
    # Paper Table I / Eqs. (4)-(18).
    wavelength_m: float = 1550e-9
    satellite_altitude_m: float = 620e3
    receiver_altitude_m: float = 0.0
    earth_radius_m: float = 6371e3
    earth_gravitational_parameter: float = 3.986004418e14
    light_speed_m_s: float = 299792458.0
    planck_constant_j_s: float = 6.62607015e-34
    electron_charge_c: float = 1.602176634e-19
    beam_radius_m: float = 0.05
    receiver_aperture_m: float = 0.318
    rms_wind_speed_m_s: float = 21.0
    ground_cn2_m_neg_two_thirds: float = 1.7e-14
    atmospheric_range_m: float = 20e3
    fog_range_m: float = 8e3
    field_of_view_rad: float = 0.1
    optical_filter_bandwidth_um: float = 1e-3  # 1 nm; not disclosed.
    symbol_rate_baud: float = 28e9
    amplifier_gain_db: float = 30.0
    amplifier_noise_figure_db: float = 4.0
    lo_linewidth_hz: float = 1000.0
    photodiode_responsivity_a_w: float = 0.85
    doppler_residual_ratio: float = 0.1
    malaga_omega: float = 0.5
    malaga_b0: float = 0.25
    malaga_phase_difference_rad: float = math.pi / 2
    malaga_beta: object = None
    max_derived_beta: int = 1000
    beta_fit_policy: str = "nearest_warn"
    beta_fit_relative_tolerance: float = 1e-3
    integration_points: int = 4097
    integration_grid_power: float = 2.0
    num_receivers: int = 1
    combining: str = "MRC"
    include_turbulence: bool = True
    include_background_noise: bool = True
    include_ase_noise: bool = True
    include_shot_noise: bool = True
    heterodyne_shot_noise_convention: float = 1.0
    include_doppler: bool = True
    include_phase_noise: bool = True

    # Convenience/reference condition; this combination is not a paper-wide default.
    launch_power_dbm: float = 33.0
    zenith_angle_deg: float = 30.0
    pointing_error_urad: float = 20.0
    scattered_power_ratio: float = 0.95
    background_light_radiance_w_m2_sr_um: float = 1e-4
    weather: str = "clear"

    def __post_init__(self):
        if self.integration_points < 257 or self.integration_points % 2 == 0:
            raise ValueError("integration_points must be an odd integer >= 257")
        if self.integration_grid_power < 1.0:
            raise ValueError("integration_grid_power must be at least 1")
        if self.photodiode_responsivity_a_w <= 0:
            raise ValueError("Photodiode responsivity must be positive")
        quantum_efficiency = (
            self.photodiode_responsivity_a_w * self.planck_constant_j_s
            * self.light_speed_m_s / self.wavelength_m / self.electron_charge_c
        )
        if not 0 < quantum_efficiency <= 1:
            raise ValueError("Responsivity implies unphysical photodiode quantum efficiency")
        if self.heterodyne_shot_noise_convention <= 0:
            raise ValueError("Heterodyne complex-noise convention factor must be positive")
        if self.num_receivers < 1:
            raise ValueError("num_receivers must be positive")
        if self.combining.upper() not in {"MRC", "EGC"}:
            raise ValueError("combining must be MRC or EGC")
        if not math.isclose(self.malaga_omega + 2 * self.malaga_b0, 1.0):
            raise ValueError("The paper normalizes Malaga power as Omega + 2*b0 = 1")
        if self.beta_fit_policy not in {"nearest_warn", "strict"}:
            raise ValueError("beta_fit_policy must be nearest_warn or strict")
        if self.beta_fit_relative_tolerance < 0:
            raise ValueError("beta_fit_relative_tolerance must be nonnegative")


def _trapezoid(values, grid):
    if hasattr(np, "trapezoid"):
        return np.trapezoid(values, grid)
    return np.trapz(values, grid)


def hufnagel_valley_cn2(altitude_m, config):
    h = np.asarray(altitude_m, dtype=np.float64)
    v = config.rms_wind_speed_m_s
    a = config.ground_cn2_m_neg_two_thirds
    return (
        0.00594 * (v / 27.0) ** 2 * (1e-5 * h) ** 10 * np.exp(-h / 1000.0)
        + 2.7e-16 * np.exp(-h / 1500.0)
        + a * np.exp(-h / 100.0)
    )


def satellite_slant_range_m(zenith_angle_deg, config):
    phi = math.radians(float(zenith_angle_deg))
    if not 0.0 <= phi < math.pi / 2:
        raise ValueError("zenith_angle_deg must be in [0, 90)")
    ground_radius = config.earth_radius_m + config.receiver_altitude_m
    satellite_radius = config.earth_radius_m + config.satellite_altitude_m
    return (
        math.sqrt(
            satellite_radius**2
            - ground_radius**2 * math.sin(phi) ** 2
        )
        - ground_radius * math.cos(phi)
    )


def path_altitude_m(path_fraction, slant_range_m, zenith_angle_deg, config):
    xi = np.asarray(path_fraction, dtype=np.float64)
    phi = math.radians(float(zenith_angle_deg))
    ground_radius = config.earth_radius_m + config.receiver_altitude_m
    distance = xi * slant_range_m
    radius = np.sqrt(
        ground_radius**2
        + distance**2
        + 2.0 * ground_radius * distance * math.cos(phi)
    )
    return radius - config.earth_radius_m


def shadowed_rician_second_moment(beta, omega_prime, gamma):
    mean_y = omega_prime + gamma
    beta_independent = omega_prime**2 + 4 * omega_prime * gamma + 2 * gamma**2
    return (
        beta_independent + omega_prime**2 / float(beta)
    ) / mean_y**2


def fit_malaga_beta_eq15(target_second_moment, omega_prime, gamma, max_beta):
    """Nearest natural beta plus an honest Eq.-(15) feasibility diagnostic."""
    mean_y = omega_prime + gamma
    beta_independent = omega_prime**2 + 4 * omega_prime * gamma + 2 * gamma**2
    denominator = target_second_moment * mean_y**2 - beta_independent
    continuous_beta = (
        omega_prime**2 / denominator if denominator > 0 else float("inf")
    )
    continuum_feasible = 1.0 <= continuous_beta <= float(max_beta)
    if math.isfinite(continuous_beta):
        candidates = {
            1,
            int(max_beta),
            min(max(int(math.floor(continuous_beta)), 1), int(max_beta)),
            min(max(int(math.ceil(continuous_beta)), 1), int(max_beta)),
        }
    else:
        candidates = {int(max_beta)}
    beta = min(
        candidates,
        key=lambda value: abs(
            shadowed_rician_second_moment(value, omega_prime, gamma)
            - target_second_moment
        ),
    )
    fitted = shadowed_rician_second_moment(beta, omega_prime, gamma)
    relative_residual = abs(fitted - target_second_moment) / max(
        abs(target_second_moment), 1e-30
    )
    return {
        "beta": int(beta),
        "continuous_beta": continuous_beta,
        "continuum_feasible_within_search": continuum_feasible,
        "fitted_second_moment": fitted,
        "relative_residual": relative_residual,
        "minimum_second_moment": beta_independent / mean_y**2,
        "maximum_natural_second_moment": shadowed_rician_second_moment(
            1, omega_prime, gamma
        ),
    }


def compute_paper_link_budget(
    config,
    launch_power_dbm=None,
    zenith_angle_deg=None,
    pointing_error_urad=None,
    scattered_power_ratio=None,
    background_light_radiance_w_m2_sr_um=None,
    weather=None,
):
    """Implement paper Eqs. (4)-(18) and Table I as scalar diagnostics."""
    launch_power_dbm = (
        config.launch_power_dbm if launch_power_dbm is None else float(launch_power_dbm)
    )
    zenith_angle_deg = (
        config.zenith_angle_deg if zenith_angle_deg is None else float(zenith_angle_deg)
    )
    pointing_error_urad = (
        config.pointing_error_urad
        if pointing_error_urad is None
        else float(pointing_error_urad)
    )
    rho = (
        config.scattered_power_ratio
        if scattered_power_ratio is None
        else float(scattered_power_ratio)
    )
    radiance = (
        config.background_light_radiance_w_m2_sr_um
        if background_light_radiance_w_m2_sr_um is None
        else float(background_light_radiance_w_m2_sr_um)
    )
    weather = (config.weather if weather is None else weather).lower()
    if weather not in WEATHER_ATTENUATION_DB_PER_KM:
        raise ValueError(f"Unsupported weather: {weather}")
    if not 0.0 <= rho <= 1.0:
        raise ValueError("scattered_power_ratio must be in [0, 1]")
    if radiance < 0.0:
        raise ValueError("background light radiance cannot be negative")

    wavelength = config.wavelength_m
    wavenumber = 2.0 * math.pi / wavelength
    slant_range = satellite_slant_range_m(zenith_angle_deg, config)

    theta0 = 1.0  # Collimated input beam: F0 -> infinity.
    lambda0 = 2.0 * slant_range / (wavenumber * config.beam_radius_m**2)
    denominator = theta0**2 + lambda0**2
    theta1 = theta0 / denominator
    lambda1 = lambda0 / denominator
    diffraction_radius = config.beam_radius_m / math.sqrt(theta1**2 + lambda1**2)

    # The Hufnagel-Valley ground term decays on a ~100-m altitude scale. A
    # uniform 4097-point slant grid undersamples it, so cluster samples near xi=0.
    xi = np.linspace(
        0.0, 1.0, config.integration_points, dtype=np.float64
    ) ** config.integration_grid_power
    altitude = path_altitude_m(xi, slant_range, zenith_angle_deg, config)
    cn2 = hufnagel_valley_cn2(altitude, config)
    mu_2d = float(_trapezoid(cn2 * xi ** (5.0 / 3.0), xi))
    spreading_factor = (
        4.35
        * mu_2d
        * lambda1 ** (5.0 / 6.0)
        * wavenumber ** (7.0 / 6.0)
        * slant_range ** (11.0 / 6.0)
    )
    long_term_radius = diffraction_radius * math.sqrt(1.0 + spreading_factor)

    geometric_scintillation_db = -10.0 * math.log10(
        config.receiver_aperture_m**2 / (2.0 * long_term_radius**2)
    )
    pointing_offset_m = slant_range * pointing_error_urad * 1e-6
    pointing_db = (10.0 / math.log(10.0)) * (
        2.0 * pointing_offset_m**2 / long_term_radius**2
    )
    weather_range_m = config.fog_range_m if weather == "fog" else config.atmospheric_range_m
    atmospheric_db = WEATHER_ATTENUATION_DB_PER_KM[weather] * weather_range_m / 1000.0
    total_attenuation_db = geometric_scintillation_db + pointing_db + atmospheric_db

    complex_kernel = (
        xi ** (5.0 / 6.0)
        * (
            lambda1 * xi
            + 1j * (1.0 - (1.0 - theta1) * xi)
        ) ** (5.0 / 6.0)
        - lambda1 ** (5.0 / 6.0) * xi ** (5.0 / 3.0)
    )
    rytov_variance = float(
        8.7
        * wavenumber ** (7.0 / 6.0)
        * slant_range ** (11.0 / 6.0)
        * np.real(_trapezoid(cn2 * complex_kernel, xi))
    )
    rytov_variance = max(rytov_variance, 0.0)
    sigma_lnx2 = (
        0.49
        * rytov_variance
        / (
            1.0
            + (1.0 + theta1) * 0.56 * rytov_variance ** (6.0 / 5.0)
        ) ** (7.0 / 6.0)
    )
    sigma_lny2 = (
        0.51
        * rytov_variance
        / (1.0 + 0.69 * rytov_variance ** (6.0 / 5.0)) ** (5.0 / 6.0)
    )
    alpha = 1.0 / math.expm1(max(sigma_lnx2, 1e-12))

    omega = config.malaga_omega
    b0 = config.malaga_b0
    phase_difference = config.malaga_phase_difference_rad
    omega_prime = (
        omega
        + 2.0 * b0 * rho
        + 2.0 * math.sqrt(2.0 * b0 * omega * rho) * math.cos(phase_difference)
    )
    gamma = 2.0 * b0 * (1.0 - rho)
    target_y_second_moment = math.exp(sigma_lny2)
    if config.malaga_beta is None:
        beta_fit = fit_malaga_beta_eq15(
            target_y_second_moment,
            omega_prime,
            gamma,
            config.max_derived_beta,
        )
        beta = beta_fit["beta"]
        beta_fit_feasible = (
            beta_fit["continuum_feasible_within_search"]
            and beta_fit["relative_residual"]
            <= config.beta_fit_relative_tolerance
        )
        beta_source = "eq15_nearest_natural"
        if config.beta_fit_policy == "strict" and not beta_fit_feasible:
            raise ValueError(
                "Paper-undisclosed Malaga beta/phase cannot satisfy Eq. (15) "
                f"within tolerance: target={target_y_second_moment:.8g}, "
                f"nearest={beta_fit['fitted_second_moment']:.8g}, "
                f"relative residual={beta_fit['relative_residual']:.3%}"
            )
    else:
        beta = int(config.malaga_beta)
        if beta < 1:
            raise ValueError("malaga_beta must be a natural number")
        fitted = shadowed_rician_second_moment(beta, omega_prime, gamma)
        beta_fit = {
            "beta": beta,
            "continuous_beta": None,
            "continuum_feasible_within_search": None,
            "fitted_second_moment": fitted,
            "relative_residual": abs(fitted - target_y_second_moment)
            / max(abs(target_y_second_moment), 1e-30),
            "minimum_second_moment": None,
            "maximum_natural_second_moment": None,
        }
        beta_fit_feasible = None
        beta_source = "explicit_user_assumption"
    fitted_y_second_moment = beta_fit["fitted_second_moment"]

    earth_radius = config.earth_radius_m
    orbit_radius = earth_radius + config.satellite_altitude_m
    orbital_angular_velocity = math.sqrt(
        config.earth_gravitational_parameter / orbit_radius**3
    )
    phi = math.radians(zenith_angle_deg)
    phi_max = math.pi / 2.0
    eta = math.cos(
        math.acos(earth_radius / orbit_radius * math.cos(phi_max)) - phi_max
    )
    optical_frequency_hz = config.light_speed_m_s / wavelength
    doppler_denominator = math.sqrt(
        earth_radius**2
        + orbit_radius**2
        - 2.0 * earth_radius * orbit_radius * math.cos(phi) * eta
    )
    doppler_hz = -(
        optical_frequency_hz
        / config.light_speed_m_s
        * orbital_angular_velocity
        * earth_radius
        * orbit_radius
        * math.sin(phi)
        * eta
        / doppler_denominator
    )
    residual_doppler_hz = config.doppler_residual_ratio * doppler_hz

    background_noise_w = (
        radiance
        * math.pi**2
        * config.field_of_view_rad**2
        * config.receiver_aperture_m**2
        / 16.0
        * config.optical_filter_bandwidth_um
    )
    amplifier_gain_linear = 10.0 ** (config.amplifier_gain_db / 10.0)
    noise_factor = 10.0 ** (config.amplifier_noise_figure_db / 10.0)
    spontaneous_emission_factor = (
        noise_factor - 1.0 / amplifier_gain_linear
    ) / (2.0 * (1.0 - 1.0 / amplifier_gain_linear))
    ase_noise_w = (
        2.0
        * spontaneous_emission_factor
        * config.planck_constant_j_s
        * optical_frequency_hz
        * (amplifier_gain_linear - 1.0)
        * config.symbol_rate_baud
    )

    # One detected complex temporal mode per Nyquist symbol. Equivalent
    # input field noise is independent between receiver branches, not common
    # transmitter ASE. No unknown LO power or arbitrary thermal floor is fitted.
    quantum_efficiency = (
        config.photodiode_responsivity_a_w * config.planck_constant_j_s
        * optical_frequency_hz / config.electron_charge_c
    )
    receiver_shot_noise_w = (
        config.heterodyne_shot_noise_convention * config.electron_charge_c
        / config.photodiode_responsivity_a_w * config.symbol_rate_baud
    )

    return {
        "channel_version": PAPER_FSO_CHANNEL_VERSION,
        "launch_power_dbm": launch_power_dbm,
        "launch_power_w": 10.0 ** ((launch_power_dbm - 30.0) / 10.0),
        "zenith_angle_deg": zenith_angle_deg,
        "pointing_error_urad": pointing_error_urad,
        "scattered_power_ratio": rho,
        "background_light_radiance_w_m2_sr_um": radiance,
        "weather": weather,
        "slant_range_m": slant_range,
        "theta1": theta1,
        "lambda1": lambda1,
        "diffraction_radius_m": diffraction_radius,
        "long_term_radius_m": long_term_radius,
        "geometric_scintillation_db": geometric_scintillation_db,
        "pointing_db": pointing_db,
        "atmospheric_db": atmospheric_db,
        "total_attenuation_db": total_attenuation_db,
        "rytov_variance": rytov_variance,
        "sigma_lnx2": sigma_lnx2,
        "sigma_lny2": sigma_lny2,
        "malaga_alpha": alpha,
        "malaga_beta": beta,
        "malaga_beta_source": beta_source,
        "malaga_beta_fit_feasible": beta_fit_feasible,
        "malaga_beta_continuous": beta_fit["continuous_beta"],
        "malaga_beta_fit_relative_residual": beta_fit["relative_residual"],
        "malaga_beta_minimum_second_moment": beta_fit["minimum_second_moment"],
        "malaga_beta_maximum_natural_second_moment": beta_fit[
            "maximum_natural_second_moment"
        ],
        "malaga_omega_prime": omega_prime,
        "malaga_gamma": gamma,
        "malaga_mean_y": omega_prime + gamma,
        "target_y_second_moment": target_y_second_moment,
        "fitted_y_second_moment": fitted_y_second_moment,
        "doppler_hz": doppler_hz,
        "residual_doppler_hz": residual_doppler_hz,
        "background_noise_w": background_noise_w,
        "amplifier_gain_linear": amplifier_gain_linear,
        "ase_noise_w": ase_noise_w,
        "photodiode_quantum_efficiency": quantum_efficiency,
        "receiver_shot_noise_w": receiver_shot_noise_w,
        "complex_shot_noise_convention": config.heterodyne_shot_noise_convention,
    }


def _real_dtype(complex_dtype):
    if complex_dtype == torch.complex128:
        return torch.float64
    return torch.float32


def _complex_normal(shape, device, complex_dtype, generator=None):
    real_dtype = _real_dtype(complex_dtype)
    real = torch.randn(shape, device=device, dtype=real_dtype, generator=generator)
    imag = torch.randn(shape, device=device, dtype=real_dtype, generator=generator)
    return torch.complex(real, imag) / math.sqrt(2.0)


def _gamma_unit_mean(shape_parameter, shape, device, dtype, generator=None):
    concentration = torch.full(
        shape,
        float(shape_parameter),
        device=device,
        dtype=dtype,
    )
    # PyTorch distributions do not expose a generator argument. This native
    # primitive does and is available in the paper's PyTorch 2.0 runtime.
    return torch._standard_gamma(concentration, generator=generator) / float(
        shape_parameter
    )


class PaperMalagaFSOChannel:
    """Paper Eq. (3) channel with Eqs. (4)-(18) and ideal coherent DSP.

    This is a symbol-rate matched-filter equivalent. Block Malaga fading is
    sampled independently per image/receiver. Doppler and LO phase are applied
    before ideal CFR/CPR/equalization through the returned complex channel gain.
    """

    def __init__(self, config=None):
        self.config = config or PaperFSOSystemConfig()
        self._budget_cache = {}
        self._reproducibility_config_cache = None
        self._warned_infeasible_beta_fit = False

    def reproducibility_config(self):
        if self._reproducibility_config_cache is None:
            reference = compute_paper_link_budget(self.config)
            self._reproducibility_config_cache = {
                "channel_version": PAPER_FSO_CHANNEL_VERSION,
                "system": asdict(self.config),
                "undisclosed_assumptions": PAPER_UNDISCLOSED_RECEIVER_ASSUMPTIONS,
                "reference_beta_diagnostic": {
                    "beta": reference["malaga_beta"],
                    "source": reference["malaga_beta_source"],
                    "fit_feasible": reference["malaga_beta_fit_feasible"],
                    "target_second_moment": round(
                        reference["target_y_second_moment"], 12
                    ),
                    "fitted_second_moment": round(
                        reference["fitted_y_second_moment"], 12
                    ),
                    "relative_residual": round(
                        reference["malaga_beta_fit_relative_residual"], 12
                    ),
                },
            }
        return self._reproducibility_config_cache

    def _budget(self, **condition):
        key = tuple(sorted(condition.items()))
        if key not in self._budget_cache:
            self._budget_cache[key] = compute_paper_link_budget(
                self.config, **condition
            )
        budget = self._budget_cache[key]
        if (
            self.config.malaga_beta is None
            and self.config.beta_fit_policy == "nearest_warn"
            and not budget["malaga_beta_fit_feasible"]
            and not self._warned_infeasible_beta_fit
        ):
            warnings.warn(
                "The paper omits Malaga beta and phase, and Eq. (15) has no "
                "fit within the configured range/tolerance for this condition; "
                f"using beta={budget['malaga_beta']} with relative moment "
                f"residual={budget['malaga_beta_fit_relative_residual']:.3%}. "
                "Set an explicit malaga_beta or beta_fit_policy='strict'.",
                RuntimeWarning,
            )
            self._warned_infeasible_beta_fit = True
        return budget

    def sample_malaga_power(self, shape, budget, device, dtype, generator=None):
        if not self.config.include_turbulence:
            return torch.ones(shape, device=device, dtype=dtype)
        large_scale = _gamma_unit_mean(
            budget["malaga_alpha"], shape, device, dtype, generator
        )
        shadowing = _gamma_unit_mean(
            budget["malaga_beta"], shape, device, dtype, generator
        )
        rho = budget["scattered_power_ratio"]
        coherent_phasor = (
            math.sqrt(self.config.malaga_omega)
            + math.sqrt(2.0 * self.config.malaga_b0 * rho)
            * complex(
                math.cos(-self.config.malaga_phase_difference_rad),
                math.sin(-self.config.malaga_phase_difference_rad),
            )
        )
        diffuse = _complex_normal(
            shape,
            device,
            torch.complex128 if dtype == torch.float64 else torch.complex64,
            generator,
        ) * math.sqrt(budget["malaga_gamma"])
        small_field = shadowing.sqrt().to(diffuse.dtype) * coherent_phasor + diffuse
        small_power = small_field.abs().square().to(dtype)
        return large_scale * small_power / budget["malaga_mean_y"]

    @torch.no_grad()
    def __call__(
        self,
        symbols,
        launch_power_dbm=None,
        zenith_angle_deg=None,
        pointing_error_urad=None,
        scattered_power_ratio=None,
        background_light_radiance_w_m2_sr_um=None,
        weather=None,
        generator=None,
    ):
        if symbols.ndim != 2 or not torch.is_complex(symbols):
            raise ValueError("symbols must be complex baseband with shape [batch, symbols]")
        if generator is not None and not paper_generator_device_matches(
            generator, symbols.device
        ):
            raise ValueError("generator and symbols must use the same device")

        condition = {
            "launch_power_dbm": (
                self.config.launch_power_dbm
                if launch_power_dbm is None
                else float(launch_power_dbm)
            ),
            "zenith_angle_deg": (
                self.config.zenith_angle_deg
                if zenith_angle_deg is None
                else float(zenith_angle_deg)
            ),
            "pointing_error_urad": (
                self.config.pointing_error_urad
                if pointing_error_urad is None
                else float(pointing_error_urad)
            ),
            "scattered_power_ratio": (
                self.config.scattered_power_ratio
                if scattered_power_ratio is None
                else float(scattered_power_ratio)
            ),
            "background_light_radiance_w_m2_sr_um": (
                self.config.background_light_radiance_w_m2_sr_um
                if background_light_radiance_w_m2_sr_um is None
                else float(background_light_radiance_w_m2_sr_um)
            ),
            "weather": self.config.weather if weather is None else str(weather),
        }
        budget = self._budget(**condition)
        batch, symbol_count = symbols.shape
        receivers = self.config.num_receivers
        real_dtype = _real_dtype(symbols.dtype)
        fading_shape = (batch, receivers, 1)
        scintillation_power = self.sample_malaga_power(
            fading_shape,
            budget,
            symbols.device,
            real_dtype,
            generator,
        )

        attenuation_amplitude = 10.0 ** (-budget["total_attenuation_db"] / 20.0)
        # Assume the listed power is launched after the transmitter AMP; the paper omits its probe point.
        # Therefore its 30-dB gain must not be applied to the signal a second time.
        propagation_amplitude = attenuation_amplitude * scintillation_power.sqrt()

        time_index = torch.arange(
            symbol_count, device=symbols.device, dtype=real_dtype
        )
        phase = torch.zeros(
            (batch, receivers, symbol_count),
            device=symbols.device,
            dtype=real_dtype,
        )
        if self.config.include_doppler:
            phase = phase + (
                2.0
                * math.pi
                * budget["residual_doppler_hz"]
                / self.config.symbol_rate_baud
                * time_index
            ).view(1, 1, -1)
        if self.config.include_phase_noise and self.config.lo_linewidth_hz > 0:
            increment_std = math.sqrt(
                2.0
                * math.pi
                * self.config.lo_linewidth_hz
                / self.config.symbol_rate_baud
            )
            increments = torch.randn(
                phase.shape,
                device=symbols.device,
                dtype=real_dtype,
                generator=generator,
            ) * increment_std
            phase = phase + increments.cumsum(dim=-1)

        propagation_gain = propagation_amplitude.to(symbols.dtype) * torch.exp(
            1j * phase
        ).to(symbols.dtype)
        channel_gain = math.sqrt(budget["launch_power_w"]) * propagation_gain
        received = channel_gain * symbols[:, None, :]

        if self.config.include_background_noise and budget["background_noise_w"] > 0:
            received = received + _complex_normal(
                received.shape, symbols.device, symbols.dtype, generator
            ) * math.sqrt(budget["background_noise_w"])
        if self.config.include_shot_noise and budget["receiver_shot_noise_w"] > 0:
            received = received + _complex_normal(
                received.shape, symbols.device, symbols.dtype, generator
            ) * math.sqrt(budget["receiver_shot_noise_w"])
        if self.config.include_ase_noise and budget["ase_noise_w"] > 0:
            transmitter_ase = _complex_normal(
                (batch, 1, symbol_count),
                symbols.device,
                symbols.dtype,
                generator,
            ) * math.sqrt(budget["ase_noise_w"])
            received = received + propagation_gain * transmitter_ase

        # The transmitter-EDFA ASE is one common source waveform, so diversity
        # cannot average it away. Receiver background noise is branch-independent.
        equivalent_noise_variance = torch.zeros(
            (batch, 1), device=symbols.device, dtype=real_dtype
        )
        if self.config.include_ase_noise:
            equivalent_noise_variance = equivalent_noise_variance + (
                budget["ase_noise_w"] / budget["launch_power_w"]
            )
        receiver_independent_noise_w = (
            budget["background_noise_w"] if self.config.include_background_noise else 0.0
        ) + (
            budget["receiver_shot_noise_w"] if self.config.include_shot_noise else 0.0
        )
        if receiver_independent_noise_w > 0:
            if receivers == 1 or self.config.combining.upper() == "MRC":
                combined_signal_power = channel_gain[..., :1].abs().square().sum(dim=1)
                equivalent_noise_variance = equivalent_noise_variance + (
                    receiver_independent_noise_w
                    / combined_signal_power.clamp_min(torch.finfo(real_dtype).tiny)
                )
            else:
                combined_signal_amplitude = channel_gain[..., :1].abs().sum(dim=1)
                equivalent_noise_variance = equivalent_noise_variance + (
                    receivers
                    * receiver_independent_noise_w
                    / combined_signal_amplitude.square().clamp_min(
                        torch.finfo(real_dtype).tiny
                    )
                )
        if bool((equivalent_noise_variance > 0).any()):
            instantaneous_snr = equivalent_noise_variance.clamp_min(
                torch.finfo(real_dtype).tiny
            ).reciprocal()
            snr_linear_sum = float(instantaneous_snr.sum().item())
            snr_sample_count = int(instantaneous_snr.numel())
            mean_snr_linear = snr_linear_sum / max(snr_sample_count, 1)
            mean_snr_db = 10.0 * math.log10(max(mean_snr_linear, 1e-30))
        else:
            snr_linear_sum = float("inf")
            snr_sample_count = batch
            mean_snr_db = float("inf")
        diagnostics = {
            "channel_version": PAPER_FSO_CHANNEL_VERSION,
            "mean_snr_db": mean_snr_db,
            "snr_linear_sum": snr_linear_sum,
            "snr_sample_count": snr_sample_count,
            "num_receivers": receivers,
            "combining": self.config.combining.upper(),
            "budget": budget,
        }

        if receivers == 1:
            return FSOChannelOutput(
                symbols=received[:, 0, :],
                channel_gain=channel_gain[:, 0, :],
                is_equalized=False,
                diagnostics=diagnostics,
            )

        if self.config.combining.upper() == "MRC":
            denominator = (
                channel_gain.abs().square().sum(dim=1)
            ).clamp_min(torch.finfo(real_dtype).tiny)
            combined = (channel_gain.conj() * received).sum(dim=1) / denominator
        else:
            magnitudes = channel_gain.abs().clamp_min(1e-30)
            combined = (
                channel_gain.conj() / magnitudes * received
            ).sum(dim=1) / magnitudes.sum(dim=1).clamp_min(1e-30)
        return FSOChannelOutput(
            symbols=combined,
            is_equalized=True,
            diagnostics=diagnostics,
        )


@dataclass
class PaperFSOIndexChannel:
    receiver: object
    modulations: tuple = ("QPSK", "8QAM", "16QAM")
    bits_per_index: int = BITS_PER_INDEX

    @property
    def receiver_config(self):
        return self.receiver.reproducibility_config()

    def reproducibility_config(self):
        return {
            "qam_mapping_version": QAM_MAPPING_VERSION,
            "modulations": tuple(name.upper() for name in self.modulations),
            "bits_per_index": self.bits_per_index,
            "receiver": self.receiver_config,
        }

    def __call__(
        self,
        indices,
        modulation="QPSK",
        return_statistics=False,
        **channel_kwargs,
    ):
        modulation = modulation.upper()
        if modulation not in tuple(name.upper() for name in self.modulations):
            raise ValueError(f"Unsupported paper modulation: {modulation}")
        link = QAMIndexLink(
            self.receiver,
            modulation=modulation,
            bits_per_index=self.bits_per_index,
            receiver_config=self.receiver_config,
        )
        return link(
            indices,
            return_statistics=return_statistics,
            **channel_kwargs,
        )


def paper_fig4_conditions():
    return [
        {
            "label": f"{modulation}, Pt={power} dBm, zenith={zenith} deg",
            "channel_kwargs": {
                "modulation": modulation,
                "launch_power_dbm": float(power),
                "zenith_angle_deg": float(zenith),
                "pointing_error_urad": 20.0,
                "scattered_power_ratio": 0.95,
                "background_light_radiance_w_m2_sr_um": 1e-4,
                "weather": "clear",
            },
        }
        for power in (30, 33, 36)
        for modulation in ("QPSK", "8QAM", "16QAM")
        for zenith in range(0, 81, 10)
    ]


def paper_fig6_pointing_conditions():
    return [
        {
            "label": (
                f"{modulation}, Pt=36 dBm, zenith={zenith} deg, "
                f"pointing={pointing} urad"
            ),
            "channel_kwargs": {
                "modulation": modulation,
                "launch_power_dbm": 36.0,
                "zenith_angle_deg": float(zenith),
                "pointing_error_urad": float(pointing),
                "scattered_power_ratio": 0.95,
                "background_light_radiance_w_m2_sr_um": 1e-4,
                "weather": "clear",
            },
        }
        for modulation in ("QPSK", "8QAM", "16QAM")
        for zenith in (0, 30, 60)
        for pointing in range(18, 23)
    ]


def paper_fig6_background_conditions():
    return [
        {
            "label": (
                f"{modulation}, Pt={power} dBm, "
                f"BGL={radiance:g} W/m2/sr/um"
            ),
            "channel_kwargs": {
                "modulation": modulation,
                "launch_power_dbm": float(power),
                "zenith_angle_deg": 0.0,
                "pointing_error_urad": 19.0,
                "scattered_power_ratio": 0.95,
                "background_light_radiance_w_m2_sr_um": float(radiance),
                "weather": "clear",
            },
        }
        for modulation in ("QPSK", "8QAM", "16QAM")
        for radiance in (1e-4, 1e-2, 1.0)
        for power in range(28, 37)
    ]


def paper_fig7_weather_conditions():
    return [
        {
            "label": f"QPSK, Pt={power} dBm, weather={weather}",
            "channel_kwargs": {
                "modulation": "QPSK",
                "launch_power_dbm": float(power),
                "zenith_angle_deg": 0.0,
                "pointing_error_urad": 15.0,
                "scattered_power_ratio": 0.95,
                "background_light_radiance_w_m2_sr_um": 1e-4,
                "weather": weather,
            },
        }
        for weather in ("clear", "dust", "rain", "snow", "fog")
        for power in range(28, 37)
    ]


def paper_fig8_turbulence_conditions():
    """Return the common FSO-SC rho/Pt sweep, not the BPG codec pipelines."""
    return [
        {
            "label": f"16QAM, Pt={power} dBm, rho={rho}",
            "channel_kwargs": {
                "modulation": "16QAM",
                "launch_power_dbm": float(power),
                "zenith_angle_deg": 0.0,
                "pointing_error_urad": 19.0,
                "scattered_power_ratio": float(rho),
                "background_light_radiance_w_m2_sr_um": 1e-4,
                "weather": "clear",
            },
        }
        for rho in (0.95, 0.75, 0.596)
        for power in range(28, 37)
    ]


def paper_fig8_diversity_index_channels(base_config=None):
    """Expose the paper's 2/4-Rx EGC/MRC comparisons.

    The paper does not disclose inter-branch turbulence correlation, so the
    receiver implementation records its independent-block-fade assumption.
    """
    base_config = PaperFSOSystemConfig() if base_config is None else base_config
    channels = {
        "1Rx": PaperFSOIndexChannel(
            PaperMalagaFSOChannel(replace(base_config, num_receivers=1))
        )
    }
    for receivers in (2, 4):
        for combining in ("EGC", "MRC"):
            label = f"{receivers}Rx-{combining}"
            receiver_config = replace(
                base_config,
                num_receivers=receivers,
                combining=combining,
            )
            channels[label] = PaperFSOIndexChannel(
                PaperMalagaFSOChannel(receiver_config)
            )
    return channels


def paper_fso_equation_self_test():
    config = PaperFSOSystemConfig()
    reference = compute_paper_link_budget(
        config,
        launch_power_dbm=33.0,
        zenith_angle_deg=0.0,
        pointing_error_urad=20.0,
        weather="clear",
    )
    assert abs(reference["slant_range_m"] - 620e3) < 1e-6
    assert abs(reference["geometric_scintillation_db"] - 28.69) < 0.05
    assert abs(reference["pointing_db"] - 35.68) < 0.05
    assert abs(reference["atmospheric_db"] - 2.94) < 1e-9
    # The paper omits beta and the coherent phase. With the documented pi/2
    # phase, its Eq. (15) has no feasible beta here; this must stay explicit.
    assert reference["malaga_beta_fit_feasible"] is False
    assert reference["malaga_beta_fit_relative_residual"] > 0.0
    try:
        compute_paper_link_budget(
            PaperFSOSystemConfig(beta_fit_policy="strict"),
            zenith_angle_deg=0.0,
            scattered_power_ratio=0.95,
        )
    except ValueError:
        pass
    else:
        raise AssertionError("Strict beta fitting must reject the underdetermined fit")
    expected_ranges_km = {
        0: 620.000,
        20: 655.958,
        30: 705.628,
        60: 1107.733,
        80: 1977.252,
    }
    for zenith, expected_km in expected_ranges_km.items():
        actual_km = satellite_slant_range_m(zenith, config) / 1000.0
        assert abs(actual_km - expected_km) < 0.001
    expected_weather_db = {
        "clear": 2.94,
        "dust": 20.86,
        "rain": 28.06,
        "snow": 42.90,
        "fog": 271.832,
    }
    for weather, expected_db in expected_weather_db.items():
        weather_budget = compute_paper_link_budget(config, weather=weather)
        assert abs(weather_budget["atmospheric_db"] - expected_db) < 0.001
    peak = compute_paper_link_budget(config, zenith_angle_deg=20.0)
    high = compute_paper_link_budget(config, zenith_angle_deg=80.0)
    assert abs(peak["doppler_hz"]) > abs(high["doppler_hz"])

    convergence_reference = compute_paper_link_budget(
        PaperFSOSystemConfig(integration_points=32769),
        zenith_angle_deg=0.0,
        pointing_error_urad=20.0,
    )
    assert abs(
        reference["rytov_variance"] / convergence_reference["rytov_variance"] - 1.0
    ) < 5e-3
    assert abs(
        reference["malaga_alpha"] / convergence_reference["malaga_alpha"] - 1.0
    ) < 5e-3

    receiver = PaperMalagaFSOChannel(config)
    generator_a = torch.Generator().manual_seed(123)
    generator_b = torch.Generator().manual_seed(123)
    sample_a = receiver.sample_malaga_power(
        (4096, 1, 1), reference, torch.device("cpu"), torch.float32, generator_a
    )
    sample_b = receiver.sample_malaga_power(
        (4096, 1, 1), reference, torch.device("cpu"), torch.float32, generator_b
    )
    assert torch.equal(sample_a, sample_b)
    assert bool((sample_a >= 0).all()) and bool(torch.isfinite(sample_a).all())
    assert 0.9 < float(sample_a.mean()) < 1.1
    print(
        "Paper FSO Eq. (3)-(18) self-test passed: "
        f"asg={reference['geometric_scintillation_db']:.2f} dB, "
        f"ape={reference['pointing_db']:.2f} dB, "
        f"alpha={reference['malaga_alpha']:.2f}, beta={reference['malaga_beta']} "
        f"(Eq15 residual={reference['malaga_beta_fit_relative_residual']:.2%})"
    )


paper_fso_equation_self_test()


In [ ]:
"""Signal-estimated symbol-rate coherent carrier recovery for the FSO notebook.

Embed after ``qam_index_link`` and ``paper_fso_channel_v1``. This is a
plausible receiver implementation, NOT the undisclosed VPI DSP used by Chen
et al. It introduces no pilot symbols and never uses transmitted symbol values
or the simulated channel phase to recover the carrier. The default magnitude
mode retains explicitly assumed block-amplitude CSI; ``blind_rms`` is also
available, but its uniform-symbol assumption is unsuitable for strongly biased
semantic bitstreams.

Reference for feed-forward blind phase search: Pfau, Hoffmann, and Noe,
JLT 27(8), 989-999 (2009), doi:10.1109/JLT.2008.2010511. Fourth-power spectral
CFR is a conventional modulation-removal approximation, with its finite-frame
failure modes retained. Exact frame/timing/IQ calibration and an initial phase
branch near zero remain assumptions. No RRC/ADC waveform is synthesized here.
"""

from dataclasses import asdict, dataclass
import math

import torch
import torch.nn.functional as F


SIGNAL_DSP_VERSION = "fourth_power_fft_newton_bps_v1"
SIGNAL_DSP_REFERENCE = {
    "blind_phase_search": "https://doi.org/10.1109/JLT.2008.2010511",
    "scope": "symbol-rate, single receiver; no waveform/timing/IQ calibration model",
    "cfr": "fourth-power FFT peak plus continuous periodogram Newton refinement",
    "cpr": "windowed minimum-distance blind phase search and symmetry unwrapping",
    "phase_ambiguity": (
        "initial carrier phase lies in the branch nearest zero, consistent with "
        "the simulator phase origin; no transmitted data is used to select a branch"
    ),
    "no_pilots": True,
    "amplitude_default": "known block magnitude only, not the complex channel gain",
    "failure_policy": "keep estimates/errors; do not drop low-SNR frames",
}


@dataclass(frozen=True)
class SignalDSPConfig:
    amplitude_mode: str = "known_block_magnitude"
    cfo_fft_oversampling: int = 4
    cfo_newton_steps: int = 4
    bps_test_phases: int = 64
    bps_window_symbols: int = 65
    cfo_enabled: bool = True
    cpr_enabled: bool = True

    def __post_init__(self):
        if self.amplitude_mode not in {"known_block_magnitude", "blind_rms"}:
            raise ValueError("Unsupported amplitude_mode")
        if self.cfo_fft_oversampling < 1 or self.cfo_newton_steps < 0:
            raise ValueError("Invalid CFR FFT/refinement configuration")
        if self.bps_test_phases < 8:
            raise ValueError("Use at least eight BPS trial phases")
        if self.bps_window_symbols < 1 or self.bps_window_symbols % 2 != 1:
            raise ValueError("BPS window must be a positive odd integer")


def _qam_axis_levels(modulation, device, dtype):
    modulation = modulation.upper()
    if modulation == "QPSK":
        i = q = torch.tensor([-1.0, 1.0], device=device, dtype=dtype) / math.sqrt(2)
        period = math.pi / 2
    elif modulation == "16QAM":
        i = q = torch.tensor([-3.0, -1.0, 1.0, 3.0], device=device, dtype=dtype) / math.sqrt(10)
        period = math.pi / 2
    elif modulation == "8QAM":
        # Rectangular geometry in qam_constellation, NOT square-QAM symmetry.
        i = torch.tensor([-3.0, -1.0, 1.0, 3.0], device=device, dtype=dtype) / math.sqrt(6)
        q = torch.tensor([-1.0, 1.0], device=device, dtype=dtype) / math.sqrt(6)
        period = math.pi
    else:
        raise ValueError(f"Unsupported modulation: {modulation}")
    return i, q, period


def _estimate_cfo_fourth_power(observed, symbol_rate_baud, config):
    batch, count = observed.shape
    # The fourth harmonic repeats at Fs/4, so the unambiguous CFO interval is
    # [-Fs/8, Fs/8). Paper residual offsets are within that acquisition range.
    nfft = 1 << max(1, (count * config.cfo_fft_oversampling - 1).bit_length())
    # Severe fog can make gain-normalized receiver noise enormous. Remove a
    # positive block scale before the fourth power; it does not change the
    # frequency peak and prevents overflow in frames whose data is unrecoverable.
    peak_scale = observed.abs().amax(dim=-1, keepdim=True).clamp_min(
        torch.finfo(observed.real.dtype).tiny
    )
    cfr_samples = observed / peak_scale
    fourth = cfr_samples.square().square()
    spectrum = torch.fft.fft(fourth, n=nfft, dim=-1)
    power = spectrum.abs().square()
    peak = power.argmax(dim=-1)
    signed_peak = torch.where(peak > nfft // 2, peak - nfft, peak)
    real_dtype = observed.real.dtype
    omega = signed_peak.to(real_dtype) * (2.0 * math.pi / nfft)
    index = torch.arange(count, device=observed.device, dtype=real_dtype)
    centered = index - (count - 1) / 2.0
    max_step = 2.0 * math.pi / nfft
    for _ in range(config.cfo_newton_steps):
        derotated = fourth * torch.exp(-1j * omega[:, None] * centered[None, :])
        value = derotated.sum(dim=-1)
        derivative = (-1j * centered[None, :] * derotated).sum(dim=-1)
        curvature = (-centered.square()[None, :] * derotated).sum(dim=-1)
        slope = 2 * (derivative * value.conj()).real
        second = 2 * ((curvature * value.conj()).real + derivative.abs().square())
        stable = second < -torch.finfo(real_dtype).eps
        denominator = torch.where(stable, second, torch.ones_like(second))
        step = torch.where(stable, slope / denominator, torch.zeros_like(slope))
        omega = omega - step.clamp(min=-max_step, max=max_step)
    omega = torch.remainder(omega + math.pi, 2 * math.pi) - math.pi
    estimated_hz = omega * symbol_rate_baud / (8 * math.pi)
    phase = omega[:, None] * index[None, :] / 4.0
    compensated = observed * torch.exp(-1j * phase)
    peak_ratio = power.max(dim=-1).values / power.mean(dim=-1).clamp_min(
        torch.finfo(real_dtype).tiny
    )
    return compensated, estimated_hz, peak_ratio


def _unwrap_symmetry_phase(phase, period):
    differences = torch.diff(phase, dim=-1)
    wrapped = torch.remainder(differences + period / 2, period) - period / 2
    corrections = wrapped - differences
    cumulative = torch.cat(
        [torch.zeros_like(phase[:, :1]), corrections.cumsum(dim=-1)], dim=-1
    )
    return phase + cumulative, (corrections.abs() > period / 2).sum(dim=-1)


def _blind_phase_search(observed, modulation, config):
    i_levels, q_levels, period = _qam_axis_levels(
        modulation, observed.device, observed.real.dtype
    )
    trial_phases = (
        torch.arange(config.bps_test_phases, device=observed.device, dtype=observed.real.dtype)
        / config.bps_test_phases - 0.5
    ) * period
    rotated = observed[:, None, :] * torch.exp(-1j * trial_phases[None, :, None])
    i_distance = (rotated.real[..., None] - i_levels).square().amin(dim=-1)
    q_distance = (rotated.imag[..., None] - q_levels).square().amin(dim=-1)
    costs = F.avg_pool1d(
        i_distance + q_distance,
        kernel_size=config.bps_window_symbols,
        stride=1,
        padding=config.bps_window_symbols // 2,
        count_include_pad=False,
    )
    selected = trial_phases[costs.argmin(dim=1)]
    phase, unwrap_events = _unwrap_symmetry_phase(selected, period)
    # Global rotational ambiguity is not solved using the source symbols. The
    # initial branch near zero is an explicit simulator convention, not a pilot.
    phase = phase - torch.round(phase[:, :1] / period) * period
    corrected = observed * torch.exp(-1j * phase)
    return corrected, phase, unwrap_events


@torch.no_grad()
def recover_qam_carrier(
    received,
    modulation,
    symbol_rate_baud,
    config=None,
    known_block_magnitude=None,
):
    """Estimate CFR/CPR from received samples, without source data/true phase.

    ``known_block_magnitude`` may contain positive block-amplitude CSI only.
    It must not contain the true phase trajectory or the transmitted symbols.
    """
    config = SignalDSPConfig() if config is None else config
    if received.ndim != 2 or not torch.is_complex(received) or received.shape[-1] < 8:
        raise ValueError("Expected complex [batch, symbols>=8] receiver samples")
    if symbol_rate_baud <= 0 or not torch.isfinite(received).all():
        raise ValueError("Invalid receiver samples or symbol rate")
    if config.amplitude_mode == "known_block_magnitude":
        if known_block_magnitude is None or torch.is_complex(known_block_magnitude):
            raise ValueError("Positive REAL block-amplitude CSI is required")
        if not torch.isfinite(known_block_magnitude).all() or (known_block_magnitude <= 0).any():
            raise ValueError("Invalid block-amplitude CSI")
        amplitude = torch.broadcast_to(known_block_magnitude, (received.shape[0], 1))
    else:
        amplitude = received.abs().square().mean(dim=-1, keepdim=True).sqrt()
        amplitude = amplitude.clamp_min(torch.finfo(received.real.dtype).tiny)
    observed = received / amplitude
    zeros = torch.zeros(received.shape[0], device=received.device, dtype=received.real.dtype)
    if config.cfo_enabled:
        observed, cfo_hz, peak_ratio = _estimate_cfo_fourth_power(
            observed, symbol_rate_baud, config
        )
    else:
        cfo_hz, peak_ratio = zeros, zeros
    if config.cpr_enabled:
        observed, phase, unwrap_events = _blind_phase_search(observed, modulation, config)
    else:
        phase = torch.zeros_like(received.real)
        unwrap_events = zeros.to(torch.int64)
    diagnostics = {
        "receiver_version": SIGNAL_DSP_VERSION,
        "amplitude_mode": config.amplitude_mode,
        "estimated_cfo_hz": cfo_hz.detach().cpu().tolist(),
        "cfr_peak_to_mean_power": peak_ratio.detach().cpu().tolist(),
        "cfr_unambiguous_interval_hz": [-symbol_rate_baud / 8, symbol_rate_baud / 8],
        "estimated_phase_start_rad": phase[:, 0].detach().cpu().tolist(),
        "estimated_phase_end_rad": phase[:, -1].detach().cpu().tolist(),
        "phase_symmetry_unwrap_events": unwrap_events.detach().cpu().tolist(),
        "failure_policy": "estimates/errors retained, no frame rejection",
    }
    return observed, diagnostics


@dataclass
class SignalDSPFSOReceiver:
    """Replace phase-oracle equalization for one physical receiver branch."""

    physical_receiver: object
    modulation: str = "QPSK"
    dsp_config: object = None

    def __post_init__(self):
        if self.dsp_config is None:
            self.dsp_config = SignalDSPConfig()
        if self.physical_receiver.config.num_receivers != 1:
            raise ValueError("Signal DSP wrapper supports single Rx only; no oracle MRC")

    def reproducibility_config(self):
        return {
            "physical_receiver": self.physical_receiver.reproducibility_config(),
            "receiver_version": SIGNAL_DSP_VERSION,
            "receiver_config": asdict(self.dsp_config),
            "receiver_assumptions": SIGNAL_DSP_REFERENCE,
            "modulation": self.modulation.upper(),
        }

    @torch.no_grad()
    def __call__(self, symbols, **channel_kwargs):
        observation = self.physical_receiver(symbols, **channel_kwargs)
        if observation.is_equalized:
            raise ValueError("Expected un-equalized physical receiver observation")
        magnitude = None
        if self.dsp_config.amplitude_mode == "known_block_magnitude":
            # Deliberately read only real block magnitude, NOT its phase. Do
            # not pass complex channel_gain to recover_qam_carrier.
            magnitude = observation.channel_gain.abs().mean(dim=-1, keepdim=True)
        recovered, receiver_diagnostics = recover_qam_carrier(
            observation.symbols,
            self.modulation,
            self.physical_receiver.config.symbol_rate_baud,
            self.dsp_config,
            known_block_magnitude=magnitude,
        )
        diagnostics = dict(observation.diagnostics or {})
        diagnostics["signal_dsp"] = receiver_diagnostics
        return FSOChannelOutput(symbols=recovered, is_equalized=True, diagnostics=diagnostics)


@dataclass
class SignalDSPPaperIndexChannel:
    """Drop-in single-Rx alternative to PaperFSOIndexChannel in this notebook."""

    receiver: object
    dsp_config: object = None
    modulations: tuple = ("QPSK", "8QAM", "16QAM")
    bits_per_index: int = 14

    def __post_init__(self):
        if self.dsp_config is None:
            self.dsp_config = SignalDSPConfig()
        if self.receiver.config.num_receivers != 1:
            raise ValueError("Estimated carrier receiver is currently single-Rx only")

    @property
    def receiver_config(self):
        return {
            "physical_receiver": self.receiver.reproducibility_config(),
            "receiver_version": SIGNAL_DSP_VERSION,
            "receiver_config": asdict(self.dsp_config),
            "receiver_assumptions": SIGNAL_DSP_REFERENCE,
        }

    def reproducibility_config(self):
        return {
            "qam_mapping_version": QAM_MAPPING_VERSION,
            "modulations": tuple(name.upper() for name in self.modulations),
            "bits_per_index": self.bits_per_index,
            "receiver": self.receiver_config,
        }

    def __call__(self, indices, modulation="QPSK", return_statistics=False, **channel_kwargs):
        modulation = modulation.upper()
        if modulation not in tuple(name.upper() for name in self.modulations):
            raise ValueError(f"Unsupported modulation: {modulation}")
        receiver = SignalDSPFSOReceiver(self.receiver, modulation, self.dsp_config)
        link = QAMIndexLink(
            receiver,
            modulation=modulation,
            bits_per_index=self.bits_per_index,
            receiver_config=receiver.reproducibility_config(),
        )
        return link(indices, return_statistics=return_statistics, **channel_kwargs)


def signal_dsp_offline_self_test(device="cpu"):
    """Synthetic local checks only; does not download weights or run training."""
    device = torch.device(device)
    generator = torch.Generator(device=device).manual_seed(71819)
    count = 4096
    sample_rate = 28e9
    time = torch.arange(count, device=device) / sample_rate
    records = {}
    for modulation in ("QPSK", "8QAM", "16QAM"):
        points, _, _ = qam_constellation(modulation, device)
        ids = torch.randint(len(points), (2, count), generator=generator, device=device)
        source = points[ids]
        true_cfo = torch.tensor([700e6, -320e6], device=device)
        phase = 0.03 + 2 * math.pi * true_cfo[:, None] * time[None, :]
        amplitude = torch.tensor([[0.002], [0.0005]], device=device)
        noise = torch.complex(
            torch.randn(source.shape, generator=generator, device=device),
            torch.randn(source.shape, generator=generator, device=device),
        ) * (0.005 / math.sqrt(2))
        received = amplitude * (source * torch.exp(1j * phase) + noise)
        recovered, stats = recover_qam_carrier(
            received, modulation, sample_rate, known_block_magnitude=amplitude
        )
        recovered_ids = (recovered[..., None] - points).abs().square().argmin(dim=-1)
        errors = int((recovered_ids != ids).sum())
        estimated_cfo = torch.tensor(stats["estimated_cfo_hz"], device=device)
        assert errors == 0, (modulation, errors, stats)
        # Finite-frame fourth-moment self-noise is nonzero for QAM. Require
        # error below one 4th-harmonic FFT bin; BPS must still give exact data.
        assert (estimated_cfo - true_cfo).abs().max() < sample_rate / (4 * count), stats
        records[modulation] = {"symbol_errors": errors, "estimated_cfo_hz": stats["estimated_cfo_hz"]}

    # Nonzero phase trajectory must be inferred and unwrapped, not divided out
    # with a hidden simulator phase. Deliberately turn CFR off for this check.
    points, _, _ = qam_constellation("QPSK", device)
    ids = torch.randint(len(points), (1, count), generator=generator, device=device)
    source = points[ids]
    phase = torch.linspace(0, 3, count, device=device)[None, :]
    recovered, stats = recover_qam_carrier(
        source * torch.exp(1j * phase), "QPSK", sample_rate,
        SignalDSPConfig(cfo_enabled=False), torch.ones((1, 1), device=device),
    )
    recovered_ids = (recovered[..., None] - points).abs().square().argmin(dim=-1)
    assert torch.equal(ids, recovered_ids), stats
    assert stats["phase_symmetry_unwrap_events"][0] >= 1

    # A global pi/2 ambiguity is intentionally NOT resolved using source data.
    received = source * torch.exp(torch.tensor(1j * math.pi / 2, device=device))
    recovered, _ = recover_qam_carrier(
        received, "QPSK", sample_rate,
        known_block_magnitude=torch.ones((1, 1), device=device),
    )
    recovered_ids = (recovered[..., None] - points).abs().square().argmin(dim=-1)
    assert not torch.equal(ids, recovered_ids), "Undisclosed source-aided phase ambiguity resolution"
    print("Signal-estimated CFR/CPR offline self-test passed:", records)
    return records


In [ ]:
"""Received-sample-only rectangular-8QAM carrier acquisition repair.

Append this source after ``fso_sc_signal_dsp.py`` in a notebook.  Standalone
imports are also supported.  QPSK and 16QAM dispatch to the unmodified v1
receiver, and the existing SignalDSPConfig dataclass is not changed.

The fourth-moment periodogram is only a coarse frequency estimator: a
nonuniform finite payload can bias its peak even without receiver noise.
Rectangular 8QAM also has *data-dependent* phase aliases on constant-amplitude
runs; these are not the constellation's pi rotational symmetry.  Independent
BPS windows can therefore jump between aliases on zero-padded coded data.

The repair refines frequency AND constant phase by the AWGN constellation
likelihood, then uses windowed decision-directed phase maximum likelihood
inside the acquired basin.  For fixed decisions that phase estimate is
arg(sum(y * conj(decision))).  No source bits, channel phase, codec state,
parity checks, or image-quality metric enters this receiver.  The initial
phase-branch-near-zero assumption and known real block magnitude are retained
from v1; fundamental blind phase ambiguity is not claimed to be solved.

Related work: Yan et al., JLT 37, 5856-5866 (2019) document nonuniform-QAM
failure of conventional fourth-power frequency estimation.  Meiyappan et al.,
JLT 31, 2055-2069 (2013) study decision-aided ML phase/frequency recovery.
These motivate the estimator family, not a claim that this simple local
likelihood search reproduces either paper's particular algorithm.
"""

import math

import torch
import torch.nn.functional as F


if "recover_qam_carrier" not in globals():
    pass  # already embedded above
_rect8_legacy_recover_qam_carrier = globals().get(
    "_rect8_legacy_recover_qam_carrier", recover_qam_carrier
)


RECT8_DSP_VERSION = "rect8_constellation_ml_cfr_dd_ml_cpr_v2"
RECT8_DSP_REFERENCE = {
    "affected_modulation": "8QAM",
    "scope": "rectangular 8QAM only; QPSK and 16QAM use unchanged v1",
    "coarse_cfr": "v1 fourth-power FFT and Newton periodogram refinement",
    "fine_cfr": "received-sample constellation-distance likelihood; frequency and constant phase jointly fitted",
    "cpr": "two windowed decision-directed ML phase updates, arg(sum(y*conj(decision)))",
    "acquisition_search": "one unoversampled fourth-harmonic CFO bin either side of the coarse estimate",
    "acquisition_samples": "at most 2048 uniformly spaced received samples; no data-dependent sampling",
    "fine_frequency_trials": 17,
    "fine_newton_steps": 5,
    "phase_trials_and_window": "unchanged SignalDSPConfig phase count and window length",
    "phase_ambiguity": "initial phase branch nearest zero; unidentifiable likelihood ties use that convention",
    "no_pilots": True,
    "uses_transmitted_bits_or_symbols": False,
    "uses_true_channel_phase": False,
    "uses_decoder_or_codec_feedback": False,
    "amplitude_default": "known positive real block magnitude only, as in v1",
    "failure_policy": "estimates and errors retained; no frame rejection",
    "legacy_bps_reference": "https://doi.org/10.1109/JLT.2008.2010511",
    "nonuniform_qam_cfr_related_work": "https://opg.optica.org/jlt/abstract.cfm?uri=jlt-37-23-5856",
    "decision_aided_ml_related_work": "https://opg.optica.org/jlt/abstract.cfm?uri=jlt-31-13-2055",
    "related_work_scope": "estimator-family motivation only; neither paper's exact algorithm is reproduced",
}


def rect8_repair_metadata():
    """JSON-safe provenance for the notebook's additional receiver stage."""
    return {"receiver_version": RECT8_DSP_VERSION, **RECT8_DSP_REFERENCE}


def _rect8_nearest_point(value):
    """Exact nearest point for I={-3,-1,1,3}/sqrt(6), Q={-1,1}/sqrt(6)."""
    root = math.sqrt(6.0)
    i = (2 * ((value.real * root + 3) / 2).round().clamp(0, 3) - 3) / root
    q = (2 * (value.imag >= 0).to(value.real.dtype) - 1) / root
    return torch.complex(i, q)


def _rect8_joint_likelihood_acquisition(observed, symbol_rate_baud, config, fit_frequency):
    """Local frequency acquisition with continuous constant-phase refinement.

    A fourth-harmonic Fourier bin corresponds to Fs/(4*N) carrier Hz, or
    pi/2 radians of end-to-end carrier phase.  This fixes the search width
    from the acquisition resolution, not from any image/BER experiment.
    Equally likely phase solutions are resolved by the declared initial
    near-zero phase convention, with a floating-point roundoff tolerance.
    """
    batch, count = observed.shape
    dtype, device = observed.real.dtype, observed.device
    sample_indices = torch.linspace(0, count - 1, min(count, 2048), device=device).round().long()
    sample_time = sample_indices.to(dtype) / (count - 1) - 0.5
    initial_phases = (torch.arange(config.bps_test_phases, device=device, dtype=dtype)
                      / config.bps_test_phases - 0.5) * math.pi
    slopes = (torch.linspace(-math.pi / 2, math.pi / 2, 17, device=device, dtype=dtype)
              if fit_frequency else torch.zeros(1, device=device, dtype=dtype))
    fitted_hz, fitted_intercept, fitted_cost = [], [], []
    for row in observed:
        sample = row[sample_indices]
        scale = sample.abs().amax().clamp_min(1.0)
        phase_candidates, slope_candidates = [], []
        # Bound temporary memory independently of frame/batch length.
        for phase_chunk in initial_phases.split(16):
            angle = phase_chunk[:, None, None] + slopes[None, :, None] * sample_time
            rotated = sample * torch.exp(-1j * angle)
            costs = ((rotated - _rect8_nearest_point(rotated)) / scale).abs().square().mean(-1)
            phase_candidates.append(phase_chunk)
            slope_candidates.append(slopes[costs.argmin(-1)])
        phase = torch.cat(phase_candidates)
        slope = torch.cat(slope_candidates)
        # Newton optimization for the local piecewise-smooth likelihood.
        # Keeping all phase seeds also retains data-dependent ambiguous basins.
        for _ in range(5):
            rotated = sample * torch.exp(-1j * (phase[:, None] + slope[:, None] * sample_time))
            decisions = _rect8_nearest_point(rotated)
            cross = (rotated / scale) * decisions.conj()
            h00 = cross.real.mean(-1)
            g0 = cross.imag.mean(-1)
            if fit_frequency:
                h01 = (cross.real * sample_time).mean(-1)
                h11 = (cross.real * sample_time.square()).mean(-1)
                g1 = (cross.imag * sample_time).mean(-1)
                determinant = h00 * h11 - h01.square()
                stable = (h00 > 0) & (determinant > torch.finfo(dtype).eps * (h00 * h11).abs())
                denominator = torch.where(stable, determinant, torch.ones_like(determinant))
                phase_step = torch.where(stable, (h11 * g0 - h01 * g1) / denominator, 0.0)
                slope_step = torch.where(stable, (h00 * g1 - h01 * g0) / denominator, 0.0)
            else:
                stable = h00 > torch.finfo(dtype).tiny
                phase_step = torch.where(stable, g0 / h00.clamp_min(torch.finfo(dtype).tiny), 0.0)
                slope_step = torch.zeros_like(slope)
            phase = phase + phase_step.clamp(-math.pi / 8, math.pi / 8)
            slope = (slope + slope_step.clamp(-math.pi / 4, math.pi / 4)).clamp(-math.pi / 2, math.pi / 2)
        rotated = sample * torch.exp(-1j * (phase[:, None] + slope[:, None] * sample_time))
        costs = ((rotated - _rect8_nearest_point(rotated)) / scale).abs().square().mean(-1)
        intercept = torch.remainder(phase - slope / 2 + math.pi / 2, math.pi) - math.pi / 2
        tolerance = 32 * torch.finfo(dtype).eps
        eligible = costs <= costs.amin() + tolerance
        selection = torch.where(eligible, intercept.abs(), float("inf")).argmin()
        fitted_hz.append(slope[selection] * symbol_rate_baud / (2 * math.pi * (count - 1)))
        fitted_intercept.append(intercept[selection])
        fitted_cost.append(costs[selection])
    frequency = torch.stack(fitted_hz)
    intercept = torch.stack(fitted_intercept)
    index = torch.arange(count, device=device, dtype=dtype)
    phase = intercept[:, None] + frequency[:, None] * (2 * math.pi / symbol_rate_baud) * index
    return observed * torch.exp(-1j * phase), frequency, intercept, torch.stack(fitted_cost)


def _rect8_decision_directed_phase(observed, config):
    """Windowed constant-phase ML, initialized inside the acquired basin."""
    corrected = observed
    scale = observed.abs().amax(-1, keepdim=True).clamp_min(1.0)
    phase = torch.zeros_like(observed.real)
    unwrap_events = torch.zeros(observed.shape[0], device=observed.device, dtype=torch.int64)
    for _ in range(2):
        decisions = _rect8_nearest_point(corrected)
        cross = (observed / scale) * decisions.conj()
        kwargs = dict(kernel_size=config.bps_window_symbols, stride=1,
                      padding=config.bps_window_symbols // 2, count_include_pad=False)
        average = torch.complex(F.avg_pool1d(cross.real[:, None], **kwargs)[:, 0],
                                F.avg_pool1d(cross.imag[:, None], **kwargs)[:, 0])
        phase, unwrap_events = _unwrap_symmetry_phase(average.angle(), 2 * math.pi)
        corrected = observed * torch.exp(-1j * phase)
    return corrected, phase, unwrap_events


@torch.no_grad()
def recover_qam_carrier(received, modulation, symbol_rate_baud, config=None, known_block_magnitude=None):
    """V1-compatible API; only rectangular-8QAM takes the repaired path."""
    if modulation.upper() != "8QAM":
        return _rect8_legacy_recover_qam_carrier(
            received, modulation, symbol_rate_baud, config, known_block_magnitude
        )
    config = SignalDSPConfig() if config is None else config
    if received.ndim != 2 or not torch.is_complex(received) or received.shape[-1] < 8:
        raise ValueError("Expected complex [batch, symbols>=8] receiver samples")
    if symbol_rate_baud <= 0 or not torch.isfinite(received).all():
        raise ValueError("Invalid receiver samples or symbol rate")
    if config.amplitude_mode == "known_block_magnitude":
        if known_block_magnitude is None or torch.is_complex(known_block_magnitude):
            raise ValueError("Positive REAL block-amplitude CSI is required")
        if not torch.isfinite(known_block_magnitude).all() or (known_block_magnitude <= 0).any():
            raise ValueError("Invalid block-amplitude CSI")
        amplitude = torch.broadcast_to(known_block_magnitude, (received.shape[0], 1))
    else:
        amplitude = received.abs().square().mean(-1, keepdim=True).sqrt()
        amplitude = amplitude.clamp_min(torch.finfo(received.real.dtype).tiny)
    observed = received / amplitude
    zeros = torch.zeros(received.shape[0], device=received.device, dtype=received.real.dtype)
    if config.cfo_enabled:
        observed, coarse_hz, peak_ratio = _estimate_cfo_fourth_power(observed, symbol_rate_baud, config)
    else:
        coarse_hz, peak_ratio = zeros, zeros
    if config.cfo_enabled or config.cpr_enabled:
        acquired, fine_hz, initial_phase, cost = _rect8_joint_likelihood_acquisition(
            observed, symbol_rate_baud, config, config.cfo_enabled
        )
        if config.cpr_enabled:
            observed = acquired
        else:
            index = torch.arange(received.shape[-1], device=received.device, dtype=received.real.dtype)
            observed = observed * torch.exp(-1j * fine_hz[:, None] * (2 * math.pi / symbol_rate_baud) * index)
    else:
        fine_hz, initial_phase, cost = zeros, zeros, zeros
    if config.cpr_enabled:
        observed, phase, unwrap_events = _rect8_decision_directed_phase(observed, config)
        phase = phase + initial_phase[:, None]
    else:
        phase = torch.zeros_like(received.real)
        unwrap_events = zeros.to(torch.int64)
    diagnostics = {
        "receiver_version": RECT8_DSP_VERSION,
        "amplitude_mode": config.amplitude_mode,
        "estimated_cfo_hz": (coarse_hz + fine_hz).detach().cpu().tolist(),
        "coarse_cfo_hz": coarse_hz.detach().cpu().tolist(),
        "fine_cfo_correction_hz": fine_hz.detach().cpu().tolist(),
        "acquisition_normalized_distance": cost.detach().cpu().tolist(),
        "cfr_peak_to_mean_power": peak_ratio.detach().cpu().tolist(),
        "cfr_unambiguous_interval_hz": [-symbol_rate_baud / 8, symbol_rate_baud / 8],
        "estimated_phase_start_rad": phase[:, 0].detach().cpu().tolist(),
        "estimated_phase_end_rad": phase[:, -1].detach().cpu().tolist(),
        "phase_symmetry_unwrap_events": unwrap_events.detach().cpu().tolist(),
        "failure_policy": "estimates/errors retained, no frame rejection",
    }
    return observed, diagnostics


def rect8_repair_self_test(device="cpu"):
    """Small synthetic preflight; no model, image, codec, or external data."""
    device = torch.device(device)
    generator = torch.Generator(device=device).manual_seed(20261028)
    count, sample_rate = 7168, 28e9
    ids = torch.randint(8, (1, count), generator=generator, device=device)
    ids[:, 5000:] = 0
    i = torch.tensor([-3, -3, -1, -1, 3, 3, 1, 1], device=device) / math.sqrt(6)
    q = torch.tensor([-1, 1, -1, 1, -1, 1, -1, 1], device=device) / math.sqrt(6)
    points = torch.complex(i, q)
    source = points[ids]
    index = torch.arange(count, device=device, dtype=source.real.dtype)
    records = []
    for cfo in (0.0, 700e6):
        received = source * torch.exp(1j * (0.03 + 2 * math.pi * cfo / sample_rate * index))
        recovered, diagnostics = recover_qam_carrier(
            received, "8QAM", sample_rate, known_block_magnitude=torch.ones((1, 1), device=device)
        )
        errors = int(((recovered[..., None] - points).abs().argmin(-1) != ids).sum())
        if errors:
            raise AssertionError(("rect8 preflight failed", cfo, errors, diagnostics))
        records.append({"cfo_hz": cfo, "symbol_errors": errors,
                        "estimated_cfo_hz": diagnostics["estimated_cfo_hz"],
                        "acquisition_normalized_distance": diagnostics["acquisition_normalized_distance"]})
    return {"receiver_version": RECT8_DSP_VERSION, "synthetic_only": True,
            "symbols_per_condition": count, "zero_tail_symbols": 2168, "conditions": records}


In [ ]:
"""Explicit noise interpretations; embeddable after the legacy channel namespace.

No training/model changes and no default profile. These are sensitivity profiles,
not settings selected to match the published LPIPS curve or a full VPI replica.
"""

import math


_PROFILE_FLAGS = {
    "paper_eq3_4_background_only": (True, False, False),
    "coherent_receiver_extension_v6": (True, True, True),
    "background_plus_ase_ablation": (True, True, False),
}
PROFILE_NAMES = tuple(_PROFILE_FLAGS)
_NOISE_FLAGS = (
    "include_background_noise", "include_ase_noise", "include_shot_noise"
)


def _profile_flags(name):
    if name not in _PROFILE_FLAGS:
        raise ValueError(f"Unknown explicit channel profile {name!r}; choose {PROFILE_NAMES}")
    return dict(zip(_NOISE_FLAGS, _PROFILE_FLAGS[name]))


def make_channel_profile(name, config_class, **overrides):
    """Construct the caller's config class; noise flags cannot contradict its label."""
    flags = _profile_flags(name)
    protected = set(overrides).intersection(_NOISE_FLAGS)
    if protected:
        raise ValueError(f"Profile noise flags cannot be overridden: {sorted(protected)}")
    return config_class(**flags, **overrides)


def profile_metadata(name):
    """Return fresh JSON-safe provenance, including what this profile cannot claim."""
    flags = _profile_flags(name)
    basis = {
        "paper_eq3_4_background_only": (
            "Chen et al. Eq. (3) defines n(t) as background-light noise; Eq. (4) "
            "gives its collected optical power. No receiver shot or ASE term is "
            "printed in those equations."
        ),
        "coherent_receiver_extension_v6": (
            "Unchanged v6 interpretation: background plus propagated transmitter "
            "EDFA ASE and input-referred single-polarization heterodyne shot noise."
        ),
        "background_plus_ase_ablation": (
            "Diagnostic ablation only: retain v6 background and transmitter ASE "
            "but omit its receiver shot term. Not an independently published model."
        ),
    }
    return {
        "profile": name,
        "noise_flags": flags,
        "basis": basis[name],
        "diagnostic_only": name == "background_plus_ase_ablation",
        "full_vpi_reproduction": False,
        "physical_superiority_claim": False,
        "curve_fitted": False,
        "limitations": [
            "Eq. (4) optical background power to complex matched-filter variance "
            "is an implementation approximation, not a disclosed receiver transfer.",
            "FOV unit, optical filter width, Malaga phase/beta, fading coherence "
            "and proprietary coherent receiver/DSP details remain assumptions.",
            "Background-only omission of quantum shot noise is an equation-level "
            "idealization, not a physically complete coherent receiver.",
            "ASE bandwidth/post-EDFA launch reference and shot bandwidth/polarization "
            "are inferred; neither establishes the undisclosed VPI configuration.",
        ],
        "provenance": {
            "paper": "https://ieeexplore.ieee.org/document/10969992/",
            "published_channel": "Chen et al., Sec. II-B, Eqs. (3)-(4); Table I",
            "v6_shot_reference": "https://doi.org/10.1103/PhysRevA.77.032116",
            "selection_rule": "Choose explicitly before examining evaluation outcomes.",
        },
    }


def _positive_power_dbm(power_w):
    return 10.0 * math.log10(power_w) + 30.0 if 0.0 < power_w < math.inf else None


def summarize_channel_budget(config, condition, budget_function):
    """Scalar no-fade, single-branch Es/N0; excludes fading and DSP errors.

    Powers are in W; the active noise terms denote E|n|^2 at the receiver.
    ASE is propagated through the same power attenuation as the signal. Raw
    formula outputs remain visible even for disabled noise terms. No Torch is
    imported; pass the existing compute_paper_link_budget function explicitly.
    """
    flags = {key: bool(getattr(config, key)) for key in _NOISE_FLAGS}
    matches = [name for name in PROFILE_NAMES if _profile_flags(name) == flags]
    if len(matches) != 1:
        raise ValueError("Config noise flags do not match an explicit channel profile")
    budget = budget_function(config, **dict(condition))
    attenuation = float(budget["total_attenuation_db"])
    launch = float(budget["launch_power_w"])
    if not math.isfinite(attenuation) or not math.isfinite(launch) or launch < 0.0:
        raise ValueError("Attenuation and nonnegative launch power must be finite")
    transmission = 10.0 ** (-attenuation / 10.0)
    signal = launch * transmission
    computed = {
        "background": float(budget["background_noise_w"]),
        "transmitter_ase": float(budget["ase_noise_w"]),
        "receiver_shot": float(budget["receiver_shot_noise_w"]),
    }
    if any(not math.isfinite(value) or value < 0.0 for value in computed.values()):
        raise ValueError("Computed noise powers must be finite and nonnegative")
    active = {
        "background": computed["background"] if flags[_NOISE_FLAGS[0]] else 0.0,
        "transmitter_ase": computed["transmitter_ase"] * transmission
        if flags[_NOISE_FLAGS[1]] else 0.0,
        "receiver_shot": computed["receiver_shot"] if flags[_NOISE_FLAGS[2]] else 0.0,
    }
    active["total"] = sum(active.values())
    snr = signal / active["total"] if active["total"] > 0.0 else None
    snr = snr if snr is None or math.isfinite(snr) else None
    snr_db = 10.0 * math.log10(snr) if snr is not None and snr > 0.0 else None
    return {
        "profile": matches[0],
        "condition": dict(condition),
        "noise_flags": flags,
        "total_attenuation_db": attenuation,
        "weather_attenuation_db": float(budget["atmospheric_db"]),
        "signal_power_no_fading_w": signal,
        "signal_power_no_fading_dbm": _positive_power_dbm(signal),
        "computed_noise_variance_w": computed,
        "computed_noise_power_dbm": {
            key: _positive_power_dbm(value) for key, value in computed.items()
        },
        "active_received_noise_variance_w": active,
        "active_received_noise_power_dbm": {
            key: _positive_power_dbm(value) for key, value in active.items()
        },
        "no_fading_esn0_linear": snr,
        "no_fading_esn0_db": snr_db,
        "snr_status": "finite" if snr_db is not None else "zero_signal_or_noise",
        "snr_scope": "single branch; unit-energy symbols; no fading or DSP errors",
    }


def qpsk_awgn_ber_from_esn0_db(esn0_db):
    """Gray QPSK coherent AWGN BER: Q(sqrt(Es/N0)); not a fading/DSP prediction."""
    esn0_db = float(esn0_db)
    if not math.isfinite(esn0_db):
        raise ValueError("Es/N0 must be finite dB")
    if esn0_db > 300.0:
        return 0.0
    return 0.5 * math.erfc(math.sqrt(10.0 ** (esn0_db / 10.0) / 2.0))


In [ ]:
"""Experimental QPSK block-likelihood acquisition, not author VPI DSP.

V1/V2 are preserved.  This candidate replaces only QPSK when called explicitly.
It uses received samples and the existing real block-amplitude CSI, never source
bits, true phase/CFO, decoder feedback or image scores.  A global residual-CFO
bound is derived from the configured orbit and wavelength, NOT the actual angle
or simulated CFO.  No frame is dropped.  The zero-frequency grid candidate is
always included, alongside received-spectrum candidates, and is never forced.

The short frames at 28 GBaud / 1 kHz linewidth motivate one affine phase per
frame instead of unwrapping independent 65-symbol noisy BPS windows.  This is
an explicit slow-phase assumption and can fail for fast phase wander.  Blind
QPSK's pi/2 ambiguity is retained; nearest-zero initial branch is an assumption.

References: Viterbi & Viterbi, IEEE Trans. IT 29(4), 543-551 (1983),
doi:10.1109/TIT.1983.1056713 (nonlinear M-PSK phase estimation); OptiCommPy
fourthPowerFOE documentation for M-th power spectral acquisition.  The exact
finite candidate search below is an independent implementation, not either
publication's released algorithm and not a confirmed FSO-SC author setting.
"""
from dataclasses import asdict, dataclass
import math

import torch
import torch.nn.functional as F

if "_rect8_legacy_recover_qam_carrier" not in globals():
    from fso_sc_signal_dsp_v2 import recover_qam_carrier as _v3_previous_recover
    from fso_sc_signal_dsp import SignalDSPConfig
else:
    _v3_previous_recover = recover_qam_carrier


QPSK_BLOCK_DSP_VERSION = "qpsk_bounded_spectral_marginal_ml_block_v3_candidate"


@dataclass(frozen=True)
class QPSKBlockMLConfig:
    wavelength_m: float = 1550e-9
    orbital_radius_m: float = 6371e3 + 620e3
    gravitational_parameter: float = 3.986004418e14
    residual_doppler_ratio: float = .1
    spectral_peaks_per_statistic: int = 8
    phase_trials: int = 16
    newton_steps: int = 6
    maximum_fit_samples: int = 8192

    def __post_init__(self):
        if not all(math.isfinite(v) and v > 0 for v in (self.wavelength_m,
                self.orbital_radius_m,self.gravitational_parameter,self.residual_doppler_ratio)):
            raise ValueError("Positive finite orbital/CFO parameters required")
        if any(type(value) is not int for value in (self.spectral_peaks_per_statistic,
                self.phase_trials,self.newton_steps,self.maximum_fit_samples)):
            raise ValueError("Integer QPSK search parameters required")
        if (self.spectral_peaks_per_statistic < 1 or self.phase_trials < 8
                or self.newton_steps < 1 or self.maximum_fit_samples < 128):
            raise ValueError("Invalid QPSK block-ML search configuration")

    @property
    def maximum_residual_cfo_hz(self):
        return self.residual_doppler_ratio * math.sqrt(
            self.gravitational_parameter/self.orbital_radius_m)/self.wavelength_m


def qpsk_block_ml_metadata(config=None):
    config = QPSKBlockMLConfig() if config is None else config
    return {"receiver_version":QPSK_BLOCK_DSP_VERSION,"experimental_candidate":True,
        "scope":"QPSK only; 8QAM/16QAM unchanged v2/v1", "config":asdict(config),
        "maximum_residual_cfo_hz":config.maximum_residual_cfo_hz,
        "frequency_prior":"global circular-orbit speed / wavelength times residual ratio; no actual CFO/angle",
        "phase_model":"one affine phase across a short frame; no local BPS unwrapping",
        "likelihood":"uniform-QPSK symbol-marginal AWGN likelihood; no transmitted decisions",
        "noise_variance":"normalized frame energy minus unit QPSK signal power, floor 1e-3",
        "acquisition_gate":"retain nominal zero residual CFO unless N*loglikelihood gain exceeds .5*log(N)+log(number of resolvable global CFO bins); conservative independent implementation choice",
        "nominal_cfo_prior":"zero after nominal carrier/precompensation, never actual simulated CFO; nonzero weak signals may fail acquisition",
        "phase_ambiguity":"initial branch nearest zero, not blind ambiguity resolution",
        "uses_true_channel_phase":False,"uses_true_cfo":False,
        "uses_transmitted_bits_or_symbols":False,"uses_codec_or_lpips_feedback":False,
        "no_pilots":True,"failure_policy":"all estimated frames and errors retained",
        "references":["https://doi.org/10.1109/TIT.1983.1056713",
        "https://opticommpy.readthedocs.io/en/stable/generated/optic.dsp.carrierRecovery.fourthPowerFOE.html"]}


def _qpsk_candidates(row, rate, config):
    count=row.numel(); nfft=1 << (4*count-1).bit_length()
    frequencies=torch.fft.fftfreq(nfft,1/rate,device=row.device)/4
    eligible=frequencies.abs() <= config.maximum_residual_cfo_hz
    indices=eligible.nonzero().flatten()
    candidates=[torch.zeros(1,device=row.device,dtype=row.real.dtype)]
    # Raw fourth moment and bounded-amplitude fourth moment.  The latter limits
    # the influence of rare AWGN amplitude outliers without selecting symbols.
    energy=row.abs().square().mean().clamp_min(torch.finfo(row.real.dtype).tiny)
    for value in (row, row/(row.abs().square()+energy).sqrt()):
        scale=value.abs().amax().clamp_min(torch.finfo(row.real.dtype).tiny)
        fourth=(value/scale).square().square()
        power=torch.fft.fft(fourth,nfft).abs().square()
        k=min(config.spectral_peaks_per_statistic,indices.numel())
        selected=indices[power[indices].topk(k).indices]
        candidates.append(frequencies[selected])
    return torch.unique(torch.cat(candidates)), rate/(4*count)


def _qpsk_fit(row, rate, config, fit_cfo=True):
    count=row.numel();dtype=row.real.dtype;device=row.device
    frequencies,bin_width=_qpsk_candidates(row,rate,config) if fit_cfo else (
        torch.zeros(1,device=device,dtype=dtype),rate/(4*count))
    ids=torch.linspace(0,count-1,min(count,config.maximum_fit_samples),device=device).round().long()
    time=ids.to(dtype)/(count-1)-.5
    sample=row[ids]
    noise=(row.abs().square().mean()-1).clamp_min(1e-3)
    c=math.sqrt(2)/noise
    phase_grid=(torch.arange(config.phase_trials,device=device,dtype=dtype)
                /config.phase_trials-.5)*(math.pi/2)
    base_frequency=frequencies.repeat_interleave(config.phase_trials)
    phase=phase_grid.repeat(frequencies.numel())
    slope=torch.zeros_like(phase)
    base_angle=base_frequency[:,None]*(2*math.pi/rate)*ids.to(dtype)
    for _ in range(config.newton_steps):
        rotated=sample*torch.exp(-1j*(base_angle+phase[:,None]+slope[:,None]*time))
        x,y=rotated.real,rotated.imag
        tx,ty=torch.tanh(c*x),torch.tanh(c*y)
        g=c*(y*tx-x*ty)
        h=c.square()*(y.square()*(1-tx.square())+x.square()*(1-ty.square()))-c*(x*tx+y*ty)
        h00=h.mean(-1);h01=(h*time).mean(-1);h11=(h*time.square()).mean(-1)
        g0=g.mean(-1);g1=(g*time).mean(-1)
        determinant=h00*h11-h01.square()
        stable=(h00 < 0)&(h11 < 0)&(determinant > 1e-8*(h00*h11).abs())
        denominator=torch.where(stable,determinant,torch.ones_like(determinant))
        phase_step=(torch.where(stable,(h11*g0-h01*g1)/denominator,0) if fit_cfo else
                    torch.where(h00<0,g0/h00.clamp_max(-torch.finfo(dtype).tiny),0))
        slope_step=torch.where(stable,(h00*g1-h01*g0)/denominator,0) if fit_cfo else torch.zeros_like(slope)
        phase=phase-phase_step.clamp(-math.pi/8,math.pi/8)
        slope=(slope-slope_step.clamp(-math.pi/4,math.pi/4)).clamp(-math.pi/2,math.pi/2)
    rotated=sample*torch.exp(-1j*(base_angle+phase[:,None]+slope[:,None]*time))
    logcosh=lambda x:F.softplus(2*x)-x-math.log(2)
    scores=(logcosh(c*rotated.real)+logcosh(c*rotated.imag)).mean(-1)
    frequency=base_frequency+slope*rate/(2*math.pi*(count-1))
    allowed=frequency.abs() <= config.maximum_residual_cfo_hz
    scores=torch.where(allowed,scores,-float("inf"))
    selected=scores.argmax()
    initial=phase[selected]-slope[selected]/2
    initial=torch.remainder(initial+math.pi/4,math.pi/2)-math.pi/4
    freq=frequency[selected]
    index=torch.arange(count,device=device,dtype=dtype)
    trajectory=initial+(2*math.pi/rate)*freq*index
    return row*torch.exp(-1j*trajectory),freq,initial,noise,scores[selected]


@torch.no_grad()
def recover_qam_carrier_v3(received,modulation,symbol_rate_baud,config=None,
                           known_block_magnitude=None,*,block_config=None):
    if modulation.upper() != "QPSK":
        return _v3_previous_recover(received,modulation,symbol_rate_baud,config,known_block_magnitude)
    config=SignalDSPConfig() if config is None else config
    block_config=QPSKBlockMLConfig() if block_config is None else block_config
    if received.ndim != 2 or not torch.is_complex(received) or received.shape[-1]<8:
        raise ValueError("Expected complex [batch,symbols>=8] receiver samples")
    if not math.isfinite(symbol_rate_baud) or symbol_rate_baud<=0 or not torch.isfinite(received).all():
        raise ValueError("Invalid receiver samples/rate")
    if block_config.maximum_residual_cfo_hz >= symbol_rate_baud/8:
        raise ValueError("Orbital CFO prior exceeds the fourth-power acquisition range")
    if config.amplitude_mode != "known_block_magnitude":
        raise ValueError("QPSK marginal likelihood requires declared real block-amplitude CSI")
    if known_block_magnitude is None or torch.is_complex(known_block_magnitude):
        raise ValueError("Positive real amplitude CSI required")
    if not torch.isfinite(known_block_magnitude).all() or (known_block_magnitude<=0).any():
        raise ValueError("Invalid amplitude CSI")
    amplitude=torch.broadcast_to(known_block_magnitude,(received.shape[0],1))
    observed=received/amplitude
    if not config.cfo_enabled and not config.cpr_enabled:
        return observed,{"receiver_version":QPSK_BLOCK_DSP_VERSION,"both_stages_disabled":True}
    rows=[];cfo=[];phase=[];noise=[];score=[];gates=[];gains=[];thresholds=[]
    for row in observed:
        recovered,f,p,n,s=_qpsk_fit(row,symbol_rate_baud,block_config,config.cfo_enabled)
        nominal_used=False; gain=0.;threshold=0.
        if config.cfo_enabled:
            nominal,nf,np,nn,ns=_qpsk_fit(row,symbol_rate_baud,block_config,False)
            fit_count=min(row.numel(),block_config.maximum_fit_samples)
            resolvable=max(1.,2*block_config.maximum_residual_cfo_hz/(symbol_rate_baud/(4*row.numel())))
            threshold=.5*math.log(fit_count)+math.log(resolvable)
            gain=float((s-ns)*fit_count)
            if gain<=threshold:
                recovered,f,p,n,s=nominal,nf,np,nn,ns
                nominal_used=True
        if not config.cpr_enabled:
            recovered=recovered*torch.exp(1j*p)
        rows.append(recovered);cfo.append(float(f));phase.append(float(p));noise.append(float(n));score.append(float(s))
        gates.append(nominal_used);gains.append(gain);thresholds.append(threshold)
    return torch.stack(rows),{"receiver_version":QPSK_BLOCK_DSP_VERSION,
        "estimated_cfo_hz":cfo,"estimated_phase_start_rad":phase,
        "estimated_normalized_noise_variance":noise,"marginal_log_likelihood":score,
        "nominal_cfo_prior_used":gates,"frequency_loglikelihood_gain":gains,
        "frequency_evidence_threshold":thresholds,
        "maximum_residual_cfo_hz":block_config.maximum_residual_cfo_hz,
        "phase_symmetry_unwrap_events":[0]*received.shape[0],
        "experimental_candidate":True,"failure_policy":"estimates/errors retained, no frame rejection"}


In [ ]:
"""Fixed, disclosed channel interpretations for the v12 rebuild.

The primary profile reproduces the public-equation idealization, not the unknown
VPI receiver. The coherent extension remains a separate stress interpretation.
No channel parameter or noise multiplier is fitted to image quality.
"""
import hashlib
import json
import math

import torch
import torch.nn.functional as F


REBUILD_CHANNEL_VERSION_V12 = "public_equations_and_coherent_stress_v12"
REBUILD_PROFILES_V12 = ("paper_eq3_4_background_only", "coherent_receiver_extension_v6")
_FIELDS_V12 = {"modulation", "launch_power_dbm", "zenith_angle_deg", "pointing_error_urad",
    "weather", "scattered_power_ratio", "background_light_radiance_w_m2_sr_um"}


def rebuild_training_conditions_v12():
    """A fixed training grid, never selected by development/holdout LPIPS."""
    rows = [None]
    for power in (30., 33., 36.):
        for zenith in (0., 30., 60.):
            for modulation in ("QPSK", "8QAM", "16QAM"):
                rows.append({"modulation": modulation, "launch_power_dbm": power,
                    "zenith_angle_deg": zenith, "pointing_error_urad": 20., "weather": "clear",
                    "scattered_power_ratio": .95, "background_light_radiance_w_m2_sr_um": 1e-4})
    for power in (28., 32., 36.):
        rows.append({"modulation": "QPSK", "launch_power_dbm": power,
            "zenith_angle_deg": 0., "pointing_error_urad": 15., "weather": "snow",
            "scattered_power_ratio": .95, "background_light_radiance_w_m2_sr_um": 1e-4})
    return rows


def _rebuild_sha_v12(tensor):
    tensor = tensor.detach().cpu().contiguous()
    digest = hashlib.sha256((str(tensor.dtype)+str(tuple(tensor.shape))).encode())
    digest.update(memoryview(tensor.numpy()).cast("B"))
    return digest.hexdigest()


def _rebuild_condition_v12(condition):
    if condition is None:return None
    if not isinstance(condition,dict) or set(condition)!=_FIELDS_V12:
        raise ValueError("Supply an explicit modulation and the six fixed channel arguments; no noise tuning")
    result = dict(condition)
    if result["modulation"] not in ("QPSK","8QAM","16QAM"):
        raise ValueError("Unsupported paper modulation")
    if result["weather"] not in ("clear","snow","fog","rain","dust"):
        raise ValueError("Unsupported published weather")
    for key in _FIELDS_V12-{"modulation","weather"}:
        if isinstance(result[key],bool) or not isinstance(result[key],(float,int)) or not math.isfinite(result[key]):
            raise ValueError("Nonfinite/non-numeric channel condition")
        result[key]=float(result[key])
    if (not 28<=result["launch_power_dbm"]<=36 or not 0<=result["zenith_angle_deg"]<=80
            or result["pointing_error_urad"]<0 or not 0<=result["scattered_power_ratio"]<=1
            or result["background_light_radiance_w_m2_sr_um"]<0):
        raise ValueError("Channel condition outside the declared paper range")
    return result


def _rebuild_observe_v12(physical,symbols,generator,kwargs):
    """Scope deterministic mode to this synchronous physical call only."""
    enabled=torch.are_deterministic_algorithms_enabled()
    warn_only=torch.is_deterministic_algorithms_warn_only_enabled()
    if symbols.device.type!="cuda":
        return physical(symbols,generator=generator,**kwargs),"cpu_native"
    initial=generator.get_state().clone()
    try:
        torch.use_deterministic_algorithms(True,warn_only=False)
        try:
            return physical(symbols,generator=generator,**kwargs),"cuda_scoped_deterministic"
        except RuntimeError as error:
            if "deterministic" not in str(error).lower() or (enabled and not warn_only):raise
            # Older Torch lacks deterministic CUDA cumsum. Do not consume a
            # second realization or claim byte-identical paired observations.
            generator.set_state(initial)
            torch.use_deterministic_algorithms(enabled,warn_only=warn_only)
            return physical(symbols,generator=generator,**kwargs),"cuda_unsupported_fallback_not_pair_verified"
    finally:torch.use_deterministic_algorithms(enabled,warn_only=warn_only)


def _rebuild_awgn_sanity_v12(namespace,device):
    rows=[];samples=65536
    for offset,snr_db in enumerate((0.,6.,12.)):
        generator=torch.Generator(device=device).manual_seed(20261113+offset)
        bits=torch.randint(0,2,(1,samples,2),device=device,generator=generator,dtype=torch.uint8)
        symbols=namespace["qam_modulate"](bits,"QPSK")
        noise=torch.complex(torch.randn(symbols.shape,device=device,generator=generator),
            torch.randn(symbols.shape,device=device,generator=generator))/math.sqrt(2)
        decoded=namespace["qam_demodulate"](symbols+noise*10**(-snr_db/20),"QPSK")
        errors=int((decoded!=bits).sum());expected=.5*math.erfc(math.sqrt(10**(snr_db/10)/2))
        measured=errors/(2*samples)
        tolerance=max(6*math.sqrt(expected*(1-expected)/(2*samples)),5/(2*samples))
        rows.append({"esn0_db":snr_db,"bit_errors":errors,"valid_bits":2*samples,
            "measured_ber":measured,"theoretical_ber":expected,"absolute_tolerance":tolerance,
            "passed":abs(measured-expected)<=tolerance})
    if not all(row["passed"] for row in rows):raise RuntimeError("QPSK AWGN theory sanity check failed")
    return rows


class RebuildIndexLinkV12:
    def __init__(self,namespace,profile="paper_eq3_4_background_only",base_seed=20261103):
        if profile not in REBUILD_PROFILES_V12:raise ValueError("Only the two preregistered interpretations are allowed")
        if type(base_seed) is not int or not 0<=base_seed<2**63:raise ValueError("Invalid channel base seed")
        self.ns=namespace;self.profile=profile;self.base_seed=base_seed
        self.config=namespace["make_channel_profile"](profile,namespace["PaperFSOSystemConfig"])
        self.physical=namespace["PaperMalagaFSOChannel"](self.config)
        expected=(True,False,False) if profile==REBUILD_PROFILES_V12[0] else (True,True,True)
        if (tuple(getattr(self.config,name) for name in ("include_background_noise","include_ase_noise","include_shot_noise"))!=expected
                or self.config.num_receivers!=1 or not all((self.config.include_turbulence,
                    self.config.include_doppler,self.config.include_phase_noise))):raise ValueError("Profile identity differs")

    def metadata(self):
        return {"version":REBUILD_CHANNEL_VERSION_V12,"profile":self.profile,"base_seed":self.base_seed,
            "frozen_physical_version":self.ns["PAPER_FSO_CHANNEL_VERSION"],
            "receiver":"unchanged v3 QPSK / prior 8QAM-16QAM DSP",
            "power_convention":"nominal launch power for unit-uniform-constellation energy; actual per-frame power is recorded, not secretly normalized",
            "receiver_csi":"existing ideal real block magnitude only; no true complex phase, CFO or transmitted symbols",
            "same_external_randomness_claimed":False,"curve_fitted":False,"full_vpi_reproduction":False}

    @torch.no_grad()
    def transmit(self,indices,conditions,seeds,*,device):
        if (not isinstance(indices,torch.Tensor) or indices.dtype!=torch.int64 or indices.ndim!=3
                or tuple(indices.shape[1:])!=(32,32) or len(indices)<1
                or bool(((indices<0)|(indices>=16384)).any())):
            raise ValueError("Expected current-model int64 [B,32,32] indices in [0,16384)")
        if len(conditions)!=len(indices) or len(seeds)!=len(indices):raise ValueError("Incomplete per-image channel plan")
        plans=[_rebuild_condition_v12(c) for c in conditions]
        if any(type(s) is not int or not 0<=s<2**63 for s in seeds):raise ValueError("Invalid per-image seed")
        device=torch.device(device);indices=indices.to(device);outputs=[];records=[]
        for i,(condition,seed) in enumerate(zip(plans,seeds)):
            source=indices[i:i+1]
            record={"profile":self.profile,"condition":condition,"seed":seed,"bit_errors":0,"valid_bits":14336,
                "index_errors":0,"indices":1024,"symbol_errors":0,"valid_symbols":0,"padding_bits":0,
                "physical_calls":0,"channel_gain_sha256":None,"channel_rng_state_sha256":None,
                "transmitted_symbols_mean_power":None,"nominal_launch_power_w":None,"actual_launch_power_w":None,
                "receiver_diagnostics":{},"deterministic_physical":"clean_identity","same_external_randomness_verified":False}
            record["condition_label"]="clean" if condition is None else (
                f"{condition['weather']}_{condition['modulation']}_P{condition['launch_power_dbm']:g}_z{condition['zenith_angle_deg']:g}_p{condition['pointing_error_urad']:g}")
            if condition is None:
                output=source.clone()
            else:
                modulation=condition["modulation"]
                _,_,bps=self.ns["qam_constellation"](modulation,device)
                bits=self.ns["indices_to_bits"](source).reshape(1,-1)
                padding=(-bits.shape[-1])%bps
                grouped=F.pad(bits,(0,padding)).reshape(1,-1,bps)
                symbols=self.ns["qam_modulate"](grouped,modulation)
                generator=torch.Generator(device=device).manual_seed(seed)
                observed,mode=_rebuild_observe_v12(self.physical,symbols,generator,
                    {k:v for k,v in condition.items() if k!="modulation"})
                if (observed.is_equalized or observed.channel_gain is None
                        or observed.symbols.shape!=symbols.shape or observed.channel_gain.shape!=symbols.shape
                        or not bool(torch.isfinite(observed.symbols).all()) or not bool(torch.isfinite(observed.channel_gain).all())):
                    raise FloatingPointError("Invalid raw physical observation; no frame discarded")
                magnitude=observed.channel_gain.abs().mean(-1,keepdim=True)
                if not bool((magnitude>0).all()):raise FloatingPointError("Nonpositive block amplitude")
                recovered,diagnostics=self.ns["recover_qam_carrier_v3"](observed.symbols,modulation,
                    self.config.symbol_rate_baud,known_block_magnitude=magnitude)
                if recovered.shape!=symbols.shape or not bool(torch.isfinite(recovered).all()):
                    raise FloatingPointError("Invalid receiver output; no frame discarded")
                hard_groups=self.ns["qam_demodulate"](recovered,modulation)
                if hard_groups.shape!=grouped.shape:raise ValueError("Receiver bit grouping changed")
                hard=hard_groups.reshape(1,-1)[:,:bits.shape[-1]]
                output=self.ns["bits_to_indices"](hard.reshape(1,32,32,14)).long()
                energy=float(symbols.abs().square().mean());launch=10**((condition["launch_power_dbm"]-30)/10)
                record.update(bit_errors=int((hard!=bits).sum()),index_errors=int((output!=source).sum()),
                    symbol_errors=int((hard_groups!=grouped).any(-1).sum()),valid_symbols=grouped.shape[1],padding_bits=padding,
                    physical_calls=1,channel_gain_sha256=_rebuild_sha_v12(observed.channel_gain),
                    channel_rng_state_sha256=_rebuild_sha_v12(generator.get_state()),deterministic_physical=mode,
                    transmitted_symbols_mean_power=energy,nominal_launch_power_w=launch,actual_launch_power_w=launch*energy,
                    receiver_diagnostics=diagnostics)
            record.update(payload_ber=record["bit_errors"]/record["valid_bits"],
                index_error_rate=record["index_errors"]/record["indices"],
                symbol_error_rate=record["symbol_errors"]/record["valid_symbols"] if record["valid_symbols"] else None)
            json.dumps(record,allow_nan=False)
            outputs.append(output);records.append(record)
        return torch.cat(outputs),records


def preflight_rebuild_channel_v12(namespace,device="cpu"):
    """Eight synthetic physical frames; no weights, image metrics or tuning."""
    device=torch.device(device);generator=torch.Generator(device=device).manual_seed(20261103)
    indices=torch.randint(0,16384,(1,32,32),device=device,generator=generator)
    rows=[];warnings=[];awgn=_rebuild_awgn_sanity_v12(namespace,device)
    for profile in REBUILD_PROFILES_V12:
        link=RebuildIndexLinkV12(namespace,profile)
        for modulation in ("QPSK","16QAM"):
            for weather in ("clear","snow"):
                condition={"modulation":modulation,"launch_power_dbm":36.,"zenith_angle_deg":0.,
                    "pointing_error_urad":15.,"weather":weather,"scattered_power_ratio":.95,
                    "background_light_radiance_w_m2_sr_um":1e-4}
                _,records=link.transmit(indices,[condition],[20261103+len(rows)],device=device)
                budget=namespace["compute_paper_link_budget"](link.config,
                    **{k:v for k,v in condition.items() if k!="modulation"})
                transmission=10**(-budget["total_attenuation_db"]/10)
                noise=(budget["background_noise_w"]
                    +(budget["ase_noise_w"]*transmission if link.config.include_ase_noise else 0.)
                    +(budget["receiver_shot_noise_w"] if link.config.include_shot_noise else 0.))
                signal=budget["launch_power_w"]*transmission
                records[0]["no_fading_scalar_budget"]={"attenuation_db":budget["total_attenuation_db"],
                    "signal_w":signal,"complex_noise_variance_w":noise,"esn0_db":10*math.log10(signal/noise),
                    "scope":"unit-uniform-constellation nominal energy; excludes random fading and DSP error"}
                rows.extend(records)
                if weather=="snow" and records[0]["payload_ber"]<1e-3:
                    warnings.append(profile+"/"+modulation+": near-error-free snow does not establish the paper's BPG failure or full curve")
                if records[0]["payload_ber"]>.1:
                    warnings.append(profile+"/"+modulation+": severe raw errors; do not promise semantic structure recovery")
    return {"version":REBUILD_CHANNEL_VERSION_V12,"synthetic_only":True,"frames":len(rows),"records":rows,
        "qpsk_awgn_theory_checks":awgn,"all_awgn_theory_checks_passed":True,
        "training_updates":0,"model_weights_loaded":False,"lpips_used_for_channel_selection":False,
        "same_external_randomness_across_profiles_verified":False,"warnings":warnings,
        "interpretation":"Two fixed interpretations, not a profile search or proof of authors' receiver settings"}


In [ ]:
import hashlib,io,json,os
from pathlib import Path
import numpy as np
from PIL import Image
import torch

def rebuild_file_sha256_v12(path):
    digest=hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda:handle.read(8*1024**2),b""):digest.update(block)
    return digest.hexdigest()

def rebuild_save_json_v12(path,value):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    if path.is_symlink():raise ValueError("Refuse linked output")
    pending=path.with_name(path.name+".partial")
    if pending.is_symlink():raise ValueError("Refuse linked pending output")
    with pending.open("w",encoding="utf-8") as handle:
        json.dump(value,handle,ensure_ascii=False,indent=2,allow_nan=False)
        handle.flush();os.fsync(handle.fileno())
    pending.replace(path)

def _rebuild_image_v12(content):
    with Image.open(io.BytesIO(content)) as opened:
        image=opened.convert("RGB");w,h=image.size;side=min(w,h)
        image=image.crop(((w-side)/2,(h-side)/2,(w+side)/2,(h+side)/2))
        image=image.resize((256,256),resample=Image.Resampling.BICUBIC,reducing_gap=1)
        array=np.asarray(image,dtype=np.float32)/127.5-1.
    return torch.from_numpy(np.ascontiguousarray(array.transpose(2,0,1)))


In [ ]:
import hashlib,json
from pathlib import Path,PurePosixPath
import torch

def _pilot_json_sha_v12(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=False,
        allow_nan=False, separators=(",", ":")).encode()).hexdigest()

def _pilot_sha_valid_v12(value):
    return isinstance(value, str) and len(value) == 64 and all(x in "0123456789abcdef" for x in value)

def _pilot_file_sha_v12(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(8 * 1024**2), b""):
            digest.update(block)
    return digest.hexdigest()

def _pilot_state_sha_v12(state):
    """Same model-state digest as the frozen v9/v12 exporter."""
    if not isinstance(state, dict) or not state:
        raise ValueError("Missing Stage1/model tensor state")
    digest = hashlib.sha256()
    for name, value in sorted(state.items()):
        if not isinstance(name, str) or not isinstance(value, torch.Tensor):
            raise ValueError("Model state must contain named tensors only")
        array = value.detach().cpu().contiguous()
        if not bool(torch.isfinite(array).all()):
            raise FloatingPointError("Nonfinite model tensor")
        digest.update(name.encode())
        digest.update(str(array.dtype).encode())
        digest.update(str(tuple(array.shape)).encode())
        digest.update(memoryview(array.numpy()).cast("B"))
    return digest.hexdigest()

def _pilot_child_v12(root, relative):
    root = Path(root).resolve()
    if not isinstance(relative, str):
        raise ValueError("Pilot-relative output must be a string")
    name = PurePosixPath(relative)
    if (name.is_absolute() or ".." in name.parts
            or "\\" in relative or not name.parts):
        raise ValueError("Unsafe pilot-relative output")
    target = root.joinpath(*name.parts)
    for ancestor in (target, *target.parents):
        if ancestor == root:
            break
        if ancestor.is_symlink():
            raise ValueError("Linked pilot output is not permitted")
    if root == Path(root.anchor) or root not in target.resolve().parents:
        raise ValueError("Pilot output escaped its scoped directory")
    return target

def _pilot_save_json_v12(ns, root, relative, value):
    path = _pilot_child_v12(root, relative)
    ns["rebuild_save_json_v12"](path, value)
    return path

def _pilot_write_identity_v12(ns, root, relative, value):
    path = _pilot_child_v12(root, relative)
    if path.exists():
        if json.loads(path.read_text(encoding="utf-8")) != value:
            raise RuntimeError("Existing pilot identity differs: " + relative)
    else:
        _pilot_save_json_v12(ns, root, relative, value)

def _pilot_inventory_v12(root, paths):
    root = Path(root).resolve()
    rows = []
    for path in paths:
        relative = Path(path).resolve().relative_to(root).as_posix()
        candidate = _pilot_child_v12(root, relative)
        if not candidate.is_file():
            raise RuntimeError("Missing pilot result: " + relative)
        rows.append({"file": relative, "bytes": candidate.stat().st_size,
            "sha256": _pilot_file_sha_v12(candidate)})
    return sorted(rows, key=lambda row: row["file"])

def _pilot_verify_inventory_v12(root, rows):
    if not isinstance(rows, list) or not rows:
        return False
    if len({row.get("file") for row in rows if isinstance(row, dict)}) != len(rows):
        return False
    for row in rows:
        if (not isinstance(row, dict) or type(row.get("bytes")) is not int
                or row["bytes"] < 1 or not _pilot_sha_valid_v12(row.get("sha256"))):
            return False
        path = _pilot_child_v12(root, row.get("file"))
        if (not path.is_file() or path.stat().st_size != row["bytes"]
                or _pilot_file_sha_v12(path) != row["sha256"]):
            return False
    return True


In [ ]:
import json,os,shutil
from pathlib import Path
import torch

def _extension_runtime_v22(device):
    return {"torch": str(torch.__version__), "cuda": torch.version.cuda, "device": str(device),
        "gpu": torch.cuda.get_device_name(device) if torch.device(device).type == "cuda" else None}

def _extension_load_checkpoint_file_v22(source, entry, local):
    """One verified mounted-file transfer; subsequent reads use local staging."""
    source, local = Path(source), Path(local).resolve()
    local.mkdir(parents=True, exist_ok=True)
    name = "stage2_resume_"+entry["sha256"]+".pth"
    target = _pilot_child_v12(local, name)
    if target.exists() and (not target.is_file() or target.stat().st_nlink > 1):
        raise ValueError("Local resume staging must not be linked")
    if (not target.is_file() or target.stat().st_size != entry["bytes"]
            or _pilot_file_sha_v12(target) != entry["sha256"]):
        if source.is_symlink() or not source.is_file() or source.stat().st_size != entry["bytes"]:
            raise RuntimeError("Reviewed checkpoint is missing or incomplete")
        pending = _pilot_child_v12(local, name+".partial")
        with source.open("rb") as reader, pending.open("wb") as writer:
            shutil.copyfileobj(reader, writer, 8*1024**2);writer.flush();os.fsync(writer.fileno())
        if pending.stat().st_size != entry["bytes"] or _pilot_file_sha_v12(pending) != entry["sha256"]:
            raise RuntimeError("Checkpoint transport SHA256 differs; no training started")
        pending.replace(target)
    return torch.load(target, map_location="cpu", weights_only=True)

def _extension_checkpoint_reference_valid_v22(branch, reference):
    """Accept either retained generation, not only the latest pointer entry."""
    try:
        pointer = _pilot_child_v12(branch, reference["pointer_file"])
        identity_path = _pilot_child_v12(branch, reference["store_identity_file"])
        saved = json.loads(pointer.read_text(encoding="utf-8"))
        identity = json.loads(identity_path.read_text(encoding="utf-8"))
        generation = reference["generation"]
        checkpoint = _pilot_child_v12(branch, reference["checkpoint_file"])
        return (identity.get("configuration_sha256") == reference["configuration_sha256"]
            and _pilot_json_sha_v12(identity.get("configuration")) == reference["configuration_sha256"]
            and saved.get("configuration_sha256") == reference["configuration_sha256"]
            and generation in saved.get("generations", [])
            and checkpoint.name == generation["file"] and checkpoint.is_file()
            and not checkpoint.is_symlink() and checkpoint.stat().st_size == generation["bytes"])
    except (OSError, KeyError, TypeError, ValueError):return False


In [ ]:
"""Real libbpg CLI transport, with no subprocess/network work at import.

Provenance: https://bellard.org/bpg/ ; official libbpg-0.9.8.tar.gz SHA256
c0788e23bdf1a7d36cb4424ccb2fae4c7789ac94949563c4ad0e2569d3bf0095.
The 256x256 input/output are RGB uint8. Compression explicitly uses x265,
8-bit YCbCr 4:2:0, full range, no alpha/metadata, and lossy q=0..51.

Example (the PNG input must already be RGB8 256x256)::

    bpgenc -e x265 -b 8 -c ycbcr -f 420 -m 4 -q 28 -o image.bpg image.png
    bpgdec -b 8 -o recovered.png image.bpg

Linux source build requires a 64-bit target, make, gcc/g++, cmake, yasm,
libpng-dev, libjpeg-dev and zlib1g-dev. Build only the two utilities with
``make -j2 USE_BPGVIEW= CMAKE_OPTS=-DENABLE_LIBNUMA=OFF bpgenc bpgdec``;
this avoids the optional SDL viewer. The official source bundles its x265
fork: replacing that fork with a current system libx265 is not equivalent.
The verified build pins CMake3.28.3:3.31+ also breaks its old ASM_YASM linker
module, while CMake4 removes its CMP0025/CMP0054 OLD behavior. The installer
configures every variant explicitly so failed configure caches can recover.
Yasm cannot simply be disabled on x86: that x265 source rejects doing so.
The official README supports Windows cross-compilation from Linux, not a
native MSVC build; official portable Windows binaries are also provided.
Building/installing/downloads are intentionally the caller's responsibility.
"""

from dataclasses import dataclass
import hashlib
import math
import operator
import os
from pathlib import Path
import shutil
import subprocess
import tempfile
import warnings

import numpy as np
from PIL import Image, UnidentifiedImageError


BPG_CODEC_VERSION = "libbpg_cli_rgb8_v1"
BPG_MAGIC = b"BPG\xfb"
LIBBPG_SOURCE_URL = "https://bellard.org/bpg/libbpg-0.9.8.tar.gz"
LIBBPG_SOURCE_SHA256 = "c0788e23bdf1a7d36cb4424ccb2fae4c7789ac94949563c4ad0e2569d3bf0095"


class BPGError(RuntimeError):
    """Fatal codec setup/encoding problem, not an ordinary damaged frame."""


class BPGExecutableNotFound(BPGError):
    pass


class BPGEncodeError(BPGError):
    pass


class BPGRateBudgetError(BPGError):
    """No q in the full lossy range fits; never silently increases the budget."""

    def __init__(self, max_bytes, tested_q_sizes):
        self.max_bytes = max_bytes
        self.tested_q_sizes = tuple(sorted(tested_q_sizes))
        self.minimum_size_q, self.minimum_size = min(self.tested_q_sizes, key=lambda item: (item[1], item[0]))
        super().__init__(f"No lossy BPG q=0..51 fits {max_bytes} bytes; minimum measured "
                         f"size is {self.minimum_size} bytes at q={self.minimum_size_q}")


@dataclass(frozen=True)
class EncodedBPG:
    data: bytes
    q: int
    max_bytes: int
    image_hw: tuple
    tested_q_sizes: tuple
    search_mode: str
    optimality: str
    encoder_settings: dict
    cache_hits: int = 0

    @property
    def num_bytes(self):
        return len(self.data)

    @property
    def bpp(self):
        return 8 * len(self.data) / (self.image_hw[0] * self.image_hw[1])

    def metadata(self):
        return {"codec": BPG_CODEC_VERSION, "q": self.q, "actual_bytes": self.num_bytes,
                "actual_bpp": self.bpp, "budget_bytes": self.max_bytes,
                "bpg_header_included": True, "image_hw": list(self.image_hw),
                "search_mode": self.search_mode, "optimality": self.optimality,
                "tested_q_sizes": {str(q): size for q, size in self.tested_q_sizes},
                "tested_q_count": len(self.tested_q_sizes), "cache_hits": self.cache_hits,
                "encoder_settings": dict(self.encoder_settings),
                "source_url": LIBBPG_SOURCE_URL, "source_sha256": LIBBPG_SOURCE_SHA256}


@dataclass(frozen=True)
class DecodeResult:
    status: str
    image: object = None
    error: str = ""
    returncode: object = None

    @property
    def ok(self):
        return self.status == "ok"


def _integer(value, name, minimum, maximum=None):
    if isinstance(value, (bool, np.bool_)):
        raise ValueError(f"{name} must be an integer")
    try:
        value = operator.index(value)
    except TypeError as exc:
        raise ValueError(f"{name} must be an integer") from exc
    if value < minimum or (maximum is not None and value > maximum):
        raise ValueError(f"{name} is outside its permitted range")
    return value


def bytes_to_bits(data):
    """Every byte, including the BPG header, in MSB-first transmission order."""
    if not isinstance(data, (bytes, bytearray, memoryview)):
        raise TypeError("data must be bytes-like")
    return np.unpackbits(np.frombuffer(bytes(data), dtype=np.uint8), bitorder="big")


def bits_to_bytes(bits):
    """Reverse bytes_to_bits; rejects truncation, nonbinary values and padding."""
    bits = np.asarray(bits)
    if bits.ndim != 1 or bits.size % 8:
        raise ValueError("bits must be one-dimensional and byte-aligned")
    if not np.isin(bits, [0, 1]).all():
        raise ValueError("bits must contain only 0 or 1")
    return np.packbits(bits.astype(np.uint8), bitorder="big").tobytes()


def _read_ue7(data, offset):
    value = 0
    for i in range(5):
        if offset >= len(data):
            raise ValueError("Truncated BPG header")
        byte = data[offset]
        offset += 1
        if i == 0 and byte == 0x80:
            raise ValueError("Noncanonical BPG header integer")
        value = (value << 7) | (byte & 127)
        if value > 0xffffffff:
            raise ValueError("Oversized BPG header integer")
        if not byte & 128:
            return value, offset
    raise ValueError("Oversized BPG header integer")


def _validate_header(data, expected_hw):
    """Bound dimensions before handing an untrusted bitstream to native code."""
    if len(data) < 9 or not data.startswith(BPG_MAGIC):
        raise ValueError("Missing or truncated BPG header")
    if (data[4] & 15) != 0:
        raise ValueError("Expected an 8-bit BPG stream")
    if data[4] >> 5 > 5 or data[5] >> 4 > 4:
        raise ValueError("Invalid BPG pixel format/color space")
    if data[5] & 1:
        raise ValueError("Animated BPG is not supported by the still-image baseline")
    width, offset = _read_ue7(data, 6)
    height, offset = _read_ue7(data, offset)
    if (height, width) != expected_hw:
        raise ValueError(f"BPG dimensions {(height, width)} do not match {expected_hw}")
    payload_size, offset = _read_ue7(data, offset)
    if data[5] & 8:
        extension_size, offset = _read_ue7(data, offset)
        offset += extension_size
    if offset >= len(data) or (payload_size and payload_size > len(data) - offset):
        raise ValueError("Truncated BPG payload/extension")


def _command_prefix(command):
    if isinstance(command, (str, os.PathLike)):
        result = [os.fspath(command)]
    else:
        result = [os.fspath(item) for item in command]
    if not result or not result[0]:
        raise ValueError("Executable command cannot be empty")
    executable = shutil.which(result[0])
    if executable is None:
        raise BPGExecutableNotFound(f"BPG executable not found: {result[0]}. "
                                    f"Install official libbpg: {LIBBPG_SOURCE_URL}")
    result[0] = str(Path(executable).resolve())
    return result


def _run(command, directory, timeout):
    # Files prevent an invalid/native decoder's diagnostics from filling RAM.
    with tempfile.TemporaryFile() as stdout, tempfile.TemporaryFile() as stderr:
        try:
            process = subprocess.run(command, cwd=directory, stdin=subprocess.DEVNULL,
                                     stdout=stdout, stderr=stderr, timeout=timeout, check=False)
        except FileNotFoundError as exc:
            raise BPGExecutableNotFound(f"BPG executable disappeared: {command[0]}") from exc
        except OSError as exc:
            raise BPGError(f"Cannot launch BPG executable {command[0]}: {exc}") from exc
        stderr.seek(0, os.SEEK_END)
        length = stderr.tell()
        stderr.seek(max(0, length - 4096))
        detail = stderr.read(4096).decode("utf-8", errors="replace").strip()
        return process.returncode, detail


class BPGCodec:
    """Bounded native codec wrapper; a one-image cache avoids duplicate q work.

    Missing/unlaunchable executables are fatal. Damaged received streams return
    DecodeResult failures; no image or score is synthesized by the codec. The
    evaluator must declare its failure convention and retain all frames. The
    paper's explicit LPIPS=1 failure marker and a separately measured gray
    reconstruction metric are permitted, provided they are clearly separated.
    Success requires exit status zero AND a valid expected-size RGB8 image;
    a nonzero exit is failure even if the process left a partial PNG behind.

    binary_checked picks the best *tested* feasible q without claiming global
    optimality for nonmonotonic sizes. exhaustive proves the smallest feasible
    q. If q51 does not fit, binary_checked falls back to exhaustive rather than
    falsely reporting that no q can fit.
    """

    def __init__(self, encoder="bpgenc", decoder="bpgdec", timeout=30.0,
                 expected_hw=(256, 256), speed=4):
        self.encoder = _command_prefix(encoder)
        self.decoder = _command_prefix(decoder)
        self.timeout = float(timeout)
        if not math.isfinite(self.timeout) or self.timeout <= 0:
            raise ValueError("timeout must be finite and positive")
        if len(expected_hw) != 2:
            raise ValueError("expected_hw must be (height, width)")
        self.expected_hw = tuple(_integer(v, "image dimension", 1, 16384) for v in expected_hw)
        self.speed = _integer(speed, "speed", 1, 9)
        self._cached_image = None
        self._encoded = {}
        self._cache_hits = 0

    @property
    def encoder_settings(self):
        return {"encoder": "x265", "bit_depth": 8, "color_space": "ycbcr",
                "chroma_format": "420", "speed": self.speed, "lossless": False,
                "input_mode": "RGB", "output_dtype": "uint8", "metadata": False}

    def _prepare_image(self, image):
        if not isinstance(image, Image.Image) or image.mode != "RGB":
            raise ValueError("image must be a PIL RGB image")
        if image.size != self.expected_hw[::-1]:
            raise ValueError(f"Input dimensions must be {self.expected_hw[::-1]}")
        key = hashlib.sha256(image.tobytes()).digest(), self.speed
        if key != self._cached_image:
            self._cached_image, self._encoded = key, {}
        return image

    def encode(self, image, q):
        """Return genuine complete .bpg bytes at one lossy quantizer value."""
        image = self._prepare_image(image)
        q = _integer(q, "q", 0, 51)
        if q in self._encoded:
            self._cache_hits += 1
            return self._encoded[q]
        with tempfile.TemporaryDirectory(prefix="fso_bpg_encode_") as directory:
            source, output = Path(directory) / "source.png", Path(directory) / "encoded.bpg"
            # A fresh RGB image deliberately strips ICC/EXIF and alpha metadata.
            Image.frombytes("RGB", image.size, image.tobytes()).save(source, format="PNG")
            command = self.encoder + ["-e", "x265", "-b", "8", "-c", "ycbcr", "-f", "420",
                                      "-m", str(self.speed), "-q", str(q), "-o", str(output), str(source)]
            try:
                returncode, detail = _run(command, directory, self.timeout)
            except subprocess.TimeoutExpired as exc:
                raise BPGEncodeError(f"bpgenc timed out at q={q}") from exc
            if returncode != 0:
                raise BPGEncodeError(f"bpgenc exited {returncode} at q={q}: {detail}")
            if not output.is_file():
                raise BPGEncodeError(f"bpgenc produced no output at q={q}")
            data = output.read_bytes()
            try:
                _validate_header(data, self.expected_hw)
            except ValueError as exc:
                raise BPGEncodeError(f"Invalid bpgenc output at q={q}: {exc}") from exc
        self._encoded[q] = data
        return data

    def encode_to_budget(self, image, max_bytes, search_mode="binary_checked"):
        """Budget includes every BPG byte, but not caller-owned transport framing."""
        self._prepare_image(image)
        max_bytes = _integer(max_bytes, "max_bytes", 1)
        if search_mode not in ("binary_checked", "exhaustive"):
            raise ValueError("search_mode must be binary_checked or exhaustive")
        tested, cache_hits_before = {}, self._cache_hits

        def measure(q):
            if q not in tested:
                tested[q] = self.encode(image, q)
            return len(tested[q]) <= max_bytes

        mode = search_mode
        if search_mode == "exhaustive":
            for q in range(52):
                if measure(q):
                    break
        elif not measure(0):
            if not measure(51):
                mode = "exhaustive_fallback"
                for q in range(1, 51):
                    if measure(q):
                        break
            else:
                lo, hi = 0, 51
                while hi - lo > 1:
                    mid = (lo + hi) // 2
                    if measure(mid):
                        hi = mid
                    else:
                        lo = mid
                best = min(q for q, data in tested.items() if len(data) <= max_bytes)
                for q in range(max(0, best - 2), min(51, best + 2) + 1):
                    measure(q)
        # Previously measured q values for this exact image are evidence too:
        # never ignore a better known feasible stream when reusing the cache.
        tested.update(self._encoded)
        feasible = [q for q, data in tested.items() if len(data) <= max_bytes]
        sizes = tuple(sorted((q, len(data)) for q, data in tested.items()))
        if not feasible:
            # The q51 failure route has measured every q before reaching here.
            raise BPGRateBudgetError(max_bytes, sizes)
        q = min(feasible)
        proven = mode != "binary_checked" or q == 0
        optimality = ("lowest feasible q over 0..51 (all lower q measured)" if proven else
                      "lowest tested feasible q; not exhaustive/global under nonmonotonic size")
        return EncodedBPG(tested[q], q, max_bytes, self.expected_hw, sizes, mode, optimality,
                          self.encoder_settings, self._cache_hits - cache_hits_before)

    def decode(self, data):
        """Decode one hard received stream; no fallback is synthesized here."""
        if not isinstance(data, (bytes, bytearray, memoryview)):
            raise TypeError("data must be bytes-like")
        data = bytes(data)
        try:
            _validate_header(data, self.expected_hw)
        except ValueError as exc:
            return DecodeResult("invalid_stream", error=str(exc))
        with tempfile.TemporaryDirectory(prefix="fso_bpg_decode_") as directory:
            source, output = Path(directory) / "received.bpg", Path(directory) / "decoded.png"
            source.write_bytes(data)
            try:
                returncode, detail = _run(self.decoder + ["-b", "8", "-o", str(output), str(source)],
                                          directory, self.timeout)
            except subprocess.TimeoutExpired:
                return DecodeResult("timeout", error=f"bpgdec exceeded {self.timeout:g}s")
            if returncode != 0:
                return DecodeResult("decoder_exit", error=detail or "bpgdec failed", returncode=returncode)
            if not output.is_file():
                return DecodeResult("invalid_output", error="bpgdec produced no output")
            # A valid RGB8 PNG is comfortably smaller; reject runaway output.
            maximum_png_bytes = self.expected_hw[0] * self.expected_hw[1] * 8 + 65536
            if output.stat().st_size > maximum_png_bytes:
                return DecodeResult("invalid_output", error="bpgdec output exceeds sanity bound")
            try:
                with warnings.catch_warnings():
                    warnings.simplefilter("error", Image.DecompressionBombWarning)
                    with Image.open(output) as decoded:
                        if decoded.format != "PNG" or decoded.size != self.expected_hw[::-1]:
                            return DecodeResult("invalid_output", error="Wrong output format/dimensions")
                        if decoded.mode not in ("RGB", "RGBA", "L", "LA", "P"):
                            return DecodeResult("invalid_output", error="Expected an 8-bit PNG output")
                        decoded.load()
                        pixels = np.asarray(decoded.convert("RGB"), dtype=np.uint8).copy()
            except (OSError, ValueError, UnidentifiedImageError, Image.DecompressionBombError,
                    Image.DecompressionBombWarning) as exc:
                return DecodeResult("invalid_output", error=f"Unreadable decoder image: {exc}")
            if pixels.shape != self.expected_hw + (3,) or pixels.dtype != np.uint8:
                return DecodeResult("invalid_output", error="Invalid RGB uint8 reconstruction")
            return DecodeResult("ok", pixels, returncode=0)


In [ ]:
"""Rate-only, dataset-mean matching of genuine native BPG streams.

The official libbpg 0.9.8 CLI parses ``-q`` with ``atoi`` and its x265 glue
uses integer constant-QP mode; fractional QP is not an exposed rate control.
Source: https://bellard.org/bpg/libbpg-0.9.8.tar.gz (bpgenc.c, x265_glue.c).

For each image we retain an under-target native stream and the first tested
lower-QP stream above the target. A deterministic subset-sum then promotes
some images using *only encoded byte counts*. No reconstruction, metric,
SNR, decoder outcome, padding, metadata stuffing, or source-image change is
used. Freeze the selected streams before evaluating any channel condition.

This matches the DATASET MEAN, not each image's rate. Individual streams may
exceed the per-image reference. Report their actual minimum/maximum/rates.
Subset-sum is exact over the retained candidates, not over all possible
codec settings. The default native QP search remains explicitly non-global.
The legacy strict-budget codec is deliberately unchanged.
"""

from dataclasses import dataclass
import hashlib
import math
import operator
from types import MappingProxyType



RATE_CONTROL_VERSION = "bpg_native_integer_qp_dataset_mean_v1"


def _positive_integer(value, name):
    if isinstance(value, bool):
        raise ValueError(f"{name} must be a positive integer")
    try:
        value = operator.index(value)
    except TypeError as exc:
        raise ValueError(f"{name} must be a positive integer") from exc
    if value < 1:
        raise ValueError(f"{name} must be a positive integer")
    return value


@dataclass(frozen=True)
class NativeBPGCandidate:
    data: bytes
    q: int
    image_hw: tuple

    def __post_init__(self):
        if not isinstance(self.data, bytes) or not self.data.startswith(BPG_MAGIC):
            raise ValueError("candidate must contain complete native BPG bytes")
        if isinstance(self.q, bool) or not isinstance(self.q, int) or not 0 <= self.q <= 51:
            raise ValueError("candidate q must be an integer from 0 to 51")
        if len(self.image_hw) != 2 or any(_positive_integer(v, "image dimension") != v for v in self.image_hw):
            raise ValueError("candidate image_hw must contain two positive integers")

    @property
    def num_bytes(self):
        return len(self.data)

    @property
    def bpp(self):
        return 8 * self.num_bytes / math.prod(self.image_hw)

    @property
    def sha256(self):
        return hashlib.sha256(self.data).hexdigest()

    def metadata(self):
        return {"q": self.q, "actual_bytes": self.num_bytes, "actual_bpp": self.bpp,
                "image_hw": list(self.image_hw), "sha256": self.sha256,
                "bpg_header_included": True, "padding_bytes": 0}


@dataclass(frozen=True)
class BPGRateBracket:
    image_name: str
    target_bytes: int
    low: NativeBPGCandidate
    high: object
    tested_q_sizes: tuple
    search_mode: str
    optimality: str
    encoder_settings: dict
    cache_hits: int = 0

    def __post_init__(self):
        if not isinstance(self.image_name, str) or not self.image_name:
            raise ValueError("image_name must be a nonempty string")
        _positive_integer(self.target_bytes, "target_bytes")
        if self.low.num_bytes > self.target_bytes:
            raise ValueError("low candidate exceeds the reference target")
        if self.high is not None:
            if self.high.num_bytes <= self.target_bytes or self.high.q >= self.low.q:
                raise ValueError("high candidate must exceed target at a lower integer QP")
            if self.high.image_hw != self.low.image_hw:
                raise ValueError("candidate image dimensions differ")

    def metadata(self):
        return {"image_name": self.image_name, "target_bytes": self.target_bytes,
                "low": self.low.metadata(),
                "high": self.high.metadata() if self.high is not None else None,
                "tested_q_sizes": {str(q): size for q, size in self.tested_q_sizes},
                "tested_q_count": len(self.tested_q_sizes), "search_mode": self.search_mode,
                "optimality": self.optimality, "encoder_settings": dict(self.encoder_settings),
                "cache_hits": self.cache_hits}


def encode_rate_bracket(codec, image, image_name, target_bytes=1792,
                        search_mode="binary_checked"):
    """Measure an under/over integer-QP pair without decoding or scoring.

    ``low`` means fewer bytes, not lower QP. The higher-rate candidate is
    the first actual over-target stream encountered below low.q. Already
    tested same-image results are reused by the unmodified native codec.
    ``exhaustive`` proves the lowest feasible QP; the default does not.
    """
    target_bytes = _positive_integer(target_bytes, "target_bytes")
    encoded = codec.encode_to_budget(image, target_bytes, search_mode=search_mode)
    low = NativeBPGCandidate(encoded.data, encoded.q, tuple(encoded.image_hw))
    sizes = dict(encoded.tested_q_sizes)
    high = None
    cache_before = getattr(codec, "_cache_hits", 0)
    for q in range(low.q - 1, -1, -1):
        data = codec.encode(image, q)
        sizes[q] = len(data)
        candidate = NativeBPGCandidate(data, q, tuple(encoded.image_hw))
        if candidate.num_bytes > target_bytes:
            high = candidate
            break
        low = candidate
    return BPGRateBracket(image_name, target_bytes, low, high, tuple(sorted(sizes.items())),
                          encoded.search_mode, encoded.optimality,
                          dict(encoded.encoder_settings), encoded.cache_hits +
                          getattr(codec, "_cache_hits", 0) - cache_before)


@dataclass(frozen=True)
class SelectedBPG:
    candidate: NativeBPGCandidate
    bracket: BPGRateBracket
    selected_level: str

    @property
    def data(self):
        return self.candidate.data

    @property
    def q(self):
        return self.candidate.q

    @property
    def num_bytes(self):
        return self.candidate.num_bytes

    @property
    def bpp(self):
        return self.candidate.bpp

    def metadata(self):
        result = self.candidate.metadata()
        result.update(codec=BPG_CODEC_VERSION, rate_control=RATE_CONTROL_VERSION,
                      rate_matching_scope="dataset_mean_not_per_image",
                      per_image_reference_bytes=self.bracket.target_bytes,
                      per_image_hard_cap_bytes=None, selected_level=self.selected_level,
                      tested_q_sizes={str(q): size for q, size in self.bracket.tested_q_sizes},
                      tested_q_count=len(self.bracket.tested_q_sizes),
                      search_mode=self.bracket.search_mode, optimality=self.bracket.optimality,
                      cache_hits=self.bracket.cache_hits,
                      encoder_settings=dict(self.bracket.encoder_settings),
                      bracket=self.bracket.metadata())
        return result


@dataclass(frozen=True)
class MatchedBPGSources:
    entries: object
    target_bytes_per_image: int
    seed: int
    subset_sum_states: int

    def metadata(self):
        entries = list(self.entries.values())
        sizes = [entry.num_bytes for entry in entries]
        target_total = len(entries) * self.target_bytes_per_image
        total = sum(sizes)
        total_pixels = sum(math.prod(entry.candidate.image_hw) for entry in entries)
        manifest = "\n".join(f"{name}\t{entry.q}\t{entry.candidate.sha256}"
                             for name, entry in sorted(self.entries.items()))
        return {"rate_control": RATE_CONTROL_VERSION, "selection_inputs": ["image_name", "encoded_byte_count"],
                "quality_metric_used_for_selection": False, "snr_used_for_selection": False,
                "rate_matching_scope": "dataset_mean_not_per_image", "num_images": len(entries),
                "target_bytes_per_image": self.target_bytes_per_image, "target_total_bytes": target_total,
                "actual_total_bytes": total, "aggregate_shortfall_bytes": target_total - total,
                "mean_actual_bytes": total / len(entries), "actual_mean_bpp": total * 8 / total_pixels,
                "target_mean_bpp": target_total * 8 / total_pixels,
                "min_actual_bytes": min(sizes), "max_actual_bytes": max(sizes),
                "min_actual_bpp": min(entry.bpp for entry in entries),
                "max_actual_bpp": max(entry.bpp for entry in entries),
                "per_image_hard_cap_bytes": None,
                "images_above_per_image_reference": sum(size > self.target_bytes_per_image for size in sizes),
                "selected_high_count": sum(entry.selected_level == "high" for entry in entries),
                "selected_low_count": sum(entry.selected_level == "low" for entry in entries),
                "padding_bytes": 0, "seed": self.seed, "subset_sum_states": self.subset_sum_states,
                "aggregate_optimality": "largest achievable total <= target over retained native candidate pairs",
                "qp_search_global_optimality_claimed": False,
                "source_manifest_sha256": hashlib.sha256(manifest.encode()).hexdigest(),
                "reuse_policy": "freeze this exact mapping across both BPG branches and every channel condition"}


def select_rate_matched_sources(brackets, target_bytes_per_image=1792, seed=20261028):
    """Exact bounded subset-sum on measured size increments, never quality.

    Ties use seeded image-name hashes, independently of caller iteration
    order. The aggregate never exceeds N*target, although individual images
    can exceed target. An unattainable target is reported as a real shortfall.
    Runtime/memory scale with aggregate shortfall, not with image pixels.
    """
    target = _positive_integer(target_bytes_per_image, "target_bytes_per_image")
    if isinstance(seed, bool):
        raise ValueError("seed must be an integer")
    try:
        seed = operator.index(seed)
    except TypeError as exc:
        raise ValueError("seed must be an integer") from exc
    brackets = list(brackets.values() if hasattr(brackets, "values") else brackets)
    if not brackets:
        raise ValueError("at least one rate bracket is required")
    names = [bracket.image_name for bracket in brackets]
    if len(set(names)) != len(names):
        raise ValueError("image names must be unique")
    if any(bracket.target_bytes != target for bracket in brackets):
        raise ValueError("all brackets must use the requested target")
    ordered = sorted(brackets, key=lambda item: (hashlib.sha256(
        f"{seed}\0{item.image_name}".encode()).digest(), item.image_name))
    deficit = len(ordered) * target - sum(bracket.low.num_bytes for bracket in ordered)
    mask = (1 << (deficit + 1)) - 1
    reachable, history = 1, [1]
    increments = []
    for bracket in ordered:
        increment = bracket.high.num_bytes - bracket.low.num_bytes if bracket.high is not None else 0
        increments.append(increment)
        # A huge out-of-range increment must not allocate a huge bigint.
        if 0 < increment <= deficit:
            reachable = (reachable | reachable << increment) & mask
        history.append(reachable)
    remaining = reachable.bit_length() - 1
    selected = {}
    for index in range(len(ordered) - 1, -1, -1):
        bracket = ordered[index]
        use_high = not ((history[index] >> remaining) & 1)
        candidate = bracket.high if use_high else bracket.low
        if use_high:
            remaining -= increments[index]
        selected[bracket.image_name] = SelectedBPG(candidate, bracket, "high" if use_high else "low")
    if remaining != 0:
        raise AssertionError("subset-sum reconstruction failed")
    selected = MappingProxyType(dict(sorted(selected.items())))
    return MatchedBPGSources(selected, target, seed, reachable.bit_count())


def prepare_rate_matched_sources(codec, images, target_bytes_per_image=1792,
                                 seed=20261028, search_mode="binary_checked", progress=None):
    """Encode then freeze one mean-matched source mapping.

    ``images`` is a mapping or iterable of (name, PIL RGB image). ``progress``
    receives (completed_count, total_count, bracket) after each native image;
    it is a status callback and must not mutate the source-selection policy.
    For parallel/pre-cached preparation call encode_rate_bracket separately
    and pass the complete fixed set to select_rate_matched_sources.
    """
    items = list(images.items() if hasattr(images, "items") else images)
    if len({name for name, _ in items}) != len(items):
        raise ValueError("image names must be unique")
    brackets = []
    for name, image in items:
        bracket = encode_rate_bracket(codec, image, name, target_bytes_per_image, search_mode)
        brackets.append(bracket)
        if progress is not None:
            progress(len(brackets), len(items), bracket)
    return select_rate_matched_sources(brackets, target_bytes_per_image, seed)


In [ ]:
"""Actual rate-3/4 (2048,1536) systematic repeat-accumulate LDPC.

The FSO-SC paper specifies rate 3/4 and block length 2048 (Sec. IV,
printed p.9173), but does not publish its parity-check matrix or decoder.
This module therefore supplies a reproducible *replacement matrix*, not
the undisclosed paper code.  Its systematic repeat-accumulate construction
uses H=[A|D], where A has column weight three and D is lower bidiagonal.
This is the sparse systematic RA/IRA family described in H. Jin's thesis,
Chapter 7: https://thesis.caltech.edu/3191/1/HJinthesis.pdf .  The particular
balanced, seeded, four-cycle-free A below is an implementation assumption;
no published performance claim or optimized degree distribution is implied.

Decoder inputs are genuine soft log(P(bit=0)/P(bit=1)) likelihood ratios.
Parity convergence is not a CRC and does not establish correct recovery.
Only PyTorch plus the Python standard library are required; all iterative
encoding/decoding operations stay on the module's CPU or CUDA device.
"""

from __future__ import annotations

import hashlib
import math
import random
import struct

import torch
from torch import nn


def _construct_rows(seed: int) -> list[list[int]]:
    """Build A with balanced check degrees, excluding all length-4 cycles.

An information column cannot reuse a pair of checks.  Adjacent check
pairs are already occupied by a column of the bidiagonal D, so they are
excluded too.  Local Python RNG keeps global training RNGs untouched.
"""
    m, k = 512, 1536
    rng = random.Random(seed)
    rows: list[list[int]] = [[] for _ in range(m)]
    degrees = [0] * m
    used_pairs = {(i, i + 1) for i in range(m - 1)}
    for variable in range(k):
        selected: list[int] = []
        for _ in range(3):
            candidates = [
                check for check in range(m)
                if check not in selected
                and all(tuple(sorted((check, old))) not in used_pairs for old in selected)
            ]
            if not candidates:
                raise RuntimeError("LDPC construction exhausted four-cycle-free candidates")
            minimum = min(degrees[check] for check in candidates)
            chosen = rng.choice([check for check in candidates if degrees[check] == minimum])
            selected.append(chosen)
            degrees[chosen] += 1
            rows[chosen].append(variable)
        for a in range(3):
            for b in range(a + 1, 3):
                used_pairs.add(tuple(sorted((selected[a], selected[b]))))
    for check in range(m):
        rows[check].append(k + check)
        if check:
            rows[check].append(k + check - 1)
        rows[check].sort()
    return rows


class SystematicRALDPC(nn.Module):
    """Fixed dimensions; batched systematic encoder and soft min-sum decoder.

    ``decode_llr`` returns ``(information_bits, stats)``. All statistics are
    tensors on the input device; iterations are zero for initially valid
    codewords. A successful syndrome does not rule out undetected errors.
    ``seed``, normalization, iteration limit and LLR clipping must be fixed
    before any image-quality sweep, not selected to fit reported curves.
    """

    n = 2048
    k = 1536
    m = 512
    rate = 0.75

    def __init__(
        self,
        *,
        max_iterations: int = 50,
        normalization: float = 0.8,
        seed: int = 20261028,
        llr_clip: float = 30.0,
    ) -> None:
        super().__init__()
        if isinstance(max_iterations, bool) or not isinstance(max_iterations, int) or max_iterations < 1:
            raise ValueError("max_iterations must be a positive integer")
        if not math.isfinite(float(normalization)) or not 0.0 < normalization <= 1.0:
            raise ValueError("normalization must be finite and in (0,1]")
        if not math.isfinite(float(llr_clip)) or llr_clip <= 0:
            raise ValueError("llr_clip must be finite and positive")
        if isinstance(seed, bool) or not isinstance(seed, int):
            raise ValueError("seed must be an integer")
        self.max_iterations = max_iterations
        self.normalization = float(normalization)
        self.seed = seed
        self.llr_clip = float(llr_clip)
        rows = _construct_rows(seed)
        self.edge_count = sum(map(len, rows))
        # Canonical byte layout: row-major (check, variable) uint16 little endian.
        canonical = b"".join(struct.pack("<HH", i, j) for i, row in enumerate(rows) for j in row)
        self.matrix_sha256 = hashlib.sha256(canonical).hexdigest()
        self._row_degree_range = (min(map(len, rows)), max(map(len, rows)))
        width = max(map(len, rows))
        neighbors = torch.zeros((self.m, width), dtype=torch.long)
        valid = torch.zeros((self.m, width), dtype=torch.bool)
        information_mask = torch.zeros((self.m, width), dtype=torch.bool)
        for check, row in enumerate(rows):
            neighbors[check, :len(row)] = torch.tensor(row, dtype=torch.long)
            valid[check, :len(row)] = True
            information_mask[check, :len(row)] = torch.tensor([v < self.k for v in row])
        self.register_buffer("check_variables", neighbors)
        self.register_buffer("check_mask", valid)
        self.register_buffer("information_mask", information_mask)

    def metadata(self) -> dict:
        return {
            "name": "systematic_ra_2048_1536_seeded_v1",
            "n": self.n, "k": self.k, "m": self.m, "rate": self.rate,
            "seed": self.seed, "matrix_sha256": self.matrix_sha256,
            "matrix_hash_encoding": "row-major (check,variable), uint16 little endian pairs",
            "edge_count": self.edge_count,
            "information_column_weight": 3,
            "check_degree_min": self._row_degree_range[0],
            "check_degree_max": self._row_degree_range[1],
            "four_cycles": 0,
            "decoder": "flooding normalized min-sum with syndrome early stopping",
            "normalization": self.normalization,
            "max_iterations": self.max_iterations,
            "llr_clip": self.llr_clip,
            "llr_convention": "log(P0/P1); positive means bit zero",
            "paper_dimensions_matched": True,
            "paper_matrix_reproduced": False,
            "matrix_assumption": "H=[seeded balanced column-weight-3 A | lower-bidiagonal D]",
            "construction_reference": "H. Jin thesis, Chapter 7, https://thesis.caltech.edu/3191/1/HJinthesis.pdf",
            "convergence_is_crc": False,
        }

    def _check_input(self, value: torch.Tensor, width: int, *, bits: bool) -> None:
        if not isinstance(value, torch.Tensor) or value.ndim != 2 or value.shape[1] != width:
            raise ValueError(f"expected a tensor with shape [batch,{width}]")
        if value.device != self.check_variables.device:
            raise ValueError("input and LDPC module must share a device; use ldpc.to(device)")
        if value.is_complex():
            raise ValueError("expected real bits or real-valued LLRs")
        if bits:
            if not bool(((value == 0) | (value == 1)).all()):
                raise ValueError("bits must contain only zeros and ones")
        elif not value.is_floating_point() or not bool(torch.isfinite(value).all()):
            raise ValueError("LLRs must be finite floating-point values")

    @torch.no_grad()
    def encode_bits(self, bits: torch.Tensor) -> torch.Tensor:
        self._check_input(bits, self.k, bits=True)
        # Padding and parity neighbor slots are masked, not included in Au.
        information_indices = self.check_variables.clamp_max(self.k - 1)
        local = bits[:, information_indices].to(torch.int64)
        au = (local * self.information_mask).sum(dim=-1).remainder(2)
        parity = au.cumsum(dim=1).remainder(2).to(torch.uint8)
        return torch.cat((bits.to(torch.uint8), parity), dim=1)

    @torch.no_grad()
    def syndrome(self, codeword: torch.Tensor) -> torch.Tensor:
        self._check_input(codeword, self.n, bits=True)
        return self._syndrome_unchecked(codeword)

    def _syndrome_unchecked(self, codeword: torch.Tensor) -> torch.Tensor:
        return (codeword[:, self.check_variables].to(torch.int64) * self.check_mask).sum(-1).remainder(2).to(torch.uint8)

    @torch.no_grad()
    def decode_llr(self, llr: torch.Tensor) -> tuple[torch.Tensor, dict[str, torch.Tensor]]:
        self._check_input(llr, self.n, bits=False)
        channel = llr.to(torch.float32).clamp(-self.llr_clip, self.llr_clip)
        batch = channel.shape[0]
        posterior = channel.clone()
        hard = (posterior < 0).to(torch.uint8)
        initial_weight = self._syndrome_unchecked(hard).sum(dim=1)
        syndrome_weight = initial_weight.clone()
        converged = initial_weight == 0
        iterations = torch.zeros(batch, dtype=torch.int64, device=channel.device)
        r = torch.zeros((batch,) + tuple(self.check_variables.shape), device=channel.device)
        q = channel[:, self.check_variables]
        flat_indices = self.check_variables.reshape(1, -1).expand(batch, -1)
        slots = torch.arange(self.check_variables.shape[1], device=channel.device)
        for iteration in range(1, self.max_iterations + 1):
            if bool(converged.all()):
                break
            active = ~converged
            signs = torch.where(q < 0, -1.0, 1.0)
            signs = torch.where(self.check_mask, signs, 1.0)
            abs_q = torch.where(self.check_mask, q.abs(), float("inf"))
            minima, locations = abs_q.topk(2, dim=-1, largest=False, sorted=True)
            excluded_minimum = torch.where(slots == locations[..., :1], minima[..., 1:2], minima[..., :1])
            new_r = self.normalization * signs.prod(-1, keepdim=True) * signs * excluded_minimum
            new_r = torch.where(self.check_mask, new_r, 0.0)
            # Freeze converged blocks. They must not drift while others iterate.
            r = torch.where(active[:, None, None], new_r, r)
            candidate = channel.clone().scatter_add_(1, flat_indices, r.reshape(batch, -1))
            posterior = torch.where(active[:, None], candidate, posterior)
            hard = (posterior < 0).to(torch.uint8)
            syndrome_weight = self._syndrome_unchecked(hard).sum(dim=1)
            iterations[active] = iteration
            converged = converged | (syndrome_weight == 0)
            q = (posterior[:, self.check_variables] - r).clamp(-self.llr_clip, self.llr_clip)
        return hard[:, :self.k], {
            "converged": converged,
            "iterations": iterations,
            "syndrome_weight": syndrome_weight,
            "initial_syndrome_weight": initial_weight,
        }


In [ ]:
"""Pinned official Bellard libbpg installer for Colab/Linux or local Win64 tests.

No API credentials, import-time work or third-party binary republishing.
Checksums identify the retrieved official distribution, not publisher signatures.
"""
import hashlib
import json
import os
from pathlib import Path
import platform
import re
import shutil
import subprocess
import tarfile
import urllib.request
import zipfile


BPG_RELEASE = "0.9.8"
BPG_SOURCE_URL = "https://bellard.org/bpg/libbpg-0.9.8.tar.gz"
# Independently matched to FreeBSD graphics/libbpg/distinfo (2029021 bytes).
BPG_SOURCE_SHA256 = "c0788e23bdf1a7d36cb4424ccb2fae4c7789ac94949563c4ad0e2569d3bf0095"
BPG_SOURCE_CHECKSUM_REFERENCE = "https://raw.githubusercontent.com/freebsd/freebsd-ports/main/graphics/libbpg/distinfo"
BPG_WIN64_URL = "https://bellard.org/bpg/bpg-0.9.8-win64.zip"
BPG_WIN64_SHA256 = "44034757c7dbf44498bfdb4c28b8ed3fbc498b32e7d18de4b98a6995371d457c"
BPG_CMAKE_VERSION = "3.28.3"
BPG_BUILD_RECIPE = "libbpg098-cmake3283-v2"


def _validated_cmake_version(output):
    found = re.search(r"cmake version (\d+\.\d+\.\d+)",output)
    if not found or found[1] != BPG_CMAKE_VERSION:
        raise RuntimeError("This verified libbpg build requires cmake==3.28.3; "
                           "rerun the notebook installation cell before codec setup. "
                           "CMake3.31+ changes the legacy YASM linker modules.")
    return found[1]


def _get_verified_archive(url,digest,path):
    if re.fullmatch(r"[0-9a-f]{64}",digest) is None:
        raise ValueError("Expected a pinned lowercase SHA256 digest")
    if path.exists() and hashlib.sha256(path.read_bytes()).hexdigest() == digest:
        return
    partial = path.with_suffix(path.suffix+".partial")
    with urllib.request.urlopen(url,timeout=60) as response, partial.open("wb") as handle:
        shutil.copyfileobj(response,handle)
    observed = hashlib.sha256(partial.read_bytes()).hexdigest()
    if observed != digest:
        raise RuntimeError("BPG archive SHA256 mismatch; do not execute it. "
                           f"URL={url}; bytes={partial.stat().st_size}; "
                           f"expected={digest}; observed={observed}")
    partial.replace(path)


def _safe_member(root,name):
    target = (root/name).resolve()
    if not target.is_relative_to(root.resolve()):
        raise ValueError("Unsafe codec archive path")
    return target


def install_official_bpg(cache_dir):
    """Return (encoder,decoder,provenance); Linux build requires apt dependencies.

    On Colab install build-essential/cmake/yasm/libpng-dev/libjpeg-dev/zlib1g-dev
    first. Uses official bundled x265; GUI and optional JCTVC are disabled.
    Never installs system packages or changes PATH on Windows.
    """
    root = Path(cache_dir).resolve()
    root.mkdir(parents=True,exist_ok=True)
    if platform.system() == "Windows":
        url,digest = BPG_WIN64_URL,BPG_WIN64_SHA256
        archive = root/"bpg-0.9.8-win64.zip"
        target = root/"win64"
        _get_verified_archive(url,digest,archive)
        encoder,decoder = target/"bpgenc.exe",target/"bpgdec.exe"
        if not encoder.is_file() or not decoder.is_file():
            with zipfile.ZipFile(archive) as source:
                for item in source.infolist():
                    path = _safe_member(target,item.filename)
                    if item.is_dir():
                        path.mkdir(parents=True,exist_ok=True)
                    else:
                        path.parent.mkdir(parents=True,exist_ok=True)
                        with source.open(item) as src,path.open("wb") as dest:
                            shutil.copyfileobj(src,dest)
        build_command = "official win64 portable distribution; no build"
    elif platform.system() == "Linux":
        url,digest = BPG_SOURCE_URL,BPG_SOURCE_SHA256
        archive = root/"libbpg-0.9.8.tar.gz"
        _get_verified_archive(url,digest,archive)
        # Never reuse the failed 3.31 build tree from earlier notebook versions.
        extract_root = root/BPG_BUILD_RECIPE
        target = extract_root/"libbpg-0.9.8"
        encoder,decoder = target/"bpgenc",target/"bpgdec"
        manifest_path = extract_root/"completed_build.json"
        expected = {"recipe":BPG_BUILD_RECIPE,"archive_sha256":digest,"cmake_version":BPG_CMAKE_VERSION}
        valid_cached_build = False
        if manifest_path.is_file() and encoder.is_file() and decoder.is_file():
            try:
                manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
                valid_cached_build = (all(manifest.get(k)==v for k,v in expected.items())
                    and manifest.get("encoder_sha256")==hashlib.sha256(encoder.read_bytes()).hexdigest()
                    and manifest.get("decoder_sha256")==hashlib.sha256(decoder.read_bytes()).hexdigest())
            except (OSError,ValueError):
                pass
        if not valid_cached_build:
            with tarfile.open(archive,"r:gz") as source:
                for item in source.getmembers():
                    path = _safe_member(extract_root,item.name)
                    if not path.is_relative_to(target):
                        raise ValueError("Unexpected top-level BPG source directory")
                    if item.isdir():
                        path.mkdir(parents=True,exist_ok=True)
                    elif item.isfile():
                        path.parent.mkdir(parents=True,exist_ok=True)
                        with source.extractfile(item) as src,path.open("wb") as dest:
                            shutil.copyfileobj(src,dest)
                        path.chmod(item.mode & 0o777)
                    else:
                        raise ValueError("Unexpected link/device in source archive")
            # Keep upstream source unchanged and use the tested CMake version.
            environment = dict(os.environ)
            try:
                import cmake
                environment["PATH"] = str(cmake.CMAKE_BIN_DIR)+os.pathsep+environment.get("PATH","")
            except ImportError:
                pass
            version = subprocess.run(["cmake","--version"],env=environment,check=True,
                                     capture_output=True,text=True,timeout=15).stdout
            cmake_version = _validated_cmake_version(version)
            command = ["make","-j2","bpgenc","bpgdec","USE_BPGVIEW=","USE_JCTVC=",
                       "CMAKE_OPTS=-DENABLE_LIBNUMA=OFF"]
            log = root/"libbpg_build.log"
            print("Building official BPG0.9.8 once; compiler log:",log,flush=True)
            try:
                with log.open("wb") as handle:
                    # Official Makefile configures only if x265.out does not
                    # exist. Configure explicitly so a failed earlier configure
                    # cannot trap the next attempt in an incomplete directory.
                    common = ["-DENABLE_LIBNUMA=OFF","-DENABLE_SHARED=OFF","-DENABLE_CLI=OFF"]
                    variants = (
                        ("12bit",["-DHIGH_BIT_DEPTH=ON","-DEXPORT_C_API=OFF","-DMAIN12=ON"]),
                        ("10bit",["-DHIGH_BIT_DEPTH=ON","-DEXPORT_C_API=OFF","-DMAIN10=ON"]),
                        ("8bit",["-DLINKED_10BIT=ON","-DLINKED_12BIT=ON"]))
                    for name,flags in variants:
                        build = target/"x265.out"/name
                        build.mkdir(parents=True,exist_ok=True)
                        subprocess.run(["cmake","-S",str(target/"x265/source"),"-B",str(build),*common,*flags],
                                       env=environment,check=True,timeout=120,stdout=handle,stderr=subprocess.STDOUT)
                    subprocess.run(command,cwd=target,env=environment,check=True,timeout=900,
                                   stdout=handle,stderr=subprocess.STDOUT)
            except (subprocess.CalledProcessError,subprocess.TimeoutExpired) as error:
                full_log = log.read_text(encoding="utf-8",errors="replace")
                errors = [line for line in full_log.splitlines() if any(token in line.lower()
                          for token in ("fatal error:"," error:","cmake error","undefined reference","***"))]
                detail = "Error lines:\n"+"\n".join(errors[-20:])+"\nCompiler tail:\n"+full_log[-4000:]
                raise RuntimeError("Official BPG source build failed. "+detail) from error
            if not encoder.is_file() or not decoder.is_file():
                raise RuntimeError("Build exited successfully without both BPG executables")
            manifest = {**expected,"encoder_sha256":hashlib.sha256(encoder.read_bytes()).hexdigest(),
                        "decoder_sha256":hashlib.sha256(decoder.read_bytes()).hexdigest()}
            manifest_path.write_text(json.dumps(manifest,indent=2),encoding="utf-8")
        build_command = "CMake3.28.3 explicit8/10/12bit configure; make -j2 bpgenc bpgdec USE_BPGVIEW= USE_JCTVC= CMAKE_OPTS=-DENABLE_LIBNUMA=OFF"
    else:
        raise RuntimeError("This notebook supports Colab Linux; local smoke testing also supports Win64")
    return str(encoder),str(decoder),{
        "release":BPG_RELEASE,"official_distribution_url":url,"archive_sha256":digest,
        "publisher_signature_verified":False,"platform":platform.platform(),
        "build_command":build_command,
        "build_recipe":BPG_BUILD_RECIPE if platform.system()=="Linux" else "official-win64-098",
        "encoder_binary_sha256":hashlib.sha256(encoder.read_bytes()).hexdigest(),
        "decoder_binary_sha256":hashlib.sha256(decoder.read_bytes()).hexdigest(),
        "redistribution":"Do not commit codecs/DLLs to GitHub. Upstream encoder GPLv2, decoder mixed BSD/LGPL2.1; see bundled licenses."}


In [ ]:
import torch

def soft_qam_llr(symbols,modulation,noise_variance,namespace,clip=30.):
    """Exact log-MAP bit LLR, log(P0/P1), E|complex noise|²=N0.

    The equalized noise variance uses disclosed real-amplitude/noise-PSD CSI,
    not the simulator's actual noise sample or exact phase trajectory. Residual
    phase-estimation errors are deliberately NOT oracle-corrected here.
    """
    if not torch.is_complex(symbols) or symbols.ndim != 2:
        raise ValueError("Expected complex [batch,symbols]")
    points,labels,_ = namespace["qam_constellation"](modulation,symbols.device)
    n0 = torch.as_tensor(noise_variance,device=symbols.device,dtype=torch.float64)
    if not bool(torch.isfinite(n0).all()) or bool((n0 < 0).any()):
        raise ValueError("Noise variance must be finite and nonnegative")
    if n0.ndim == 0:
        n0 = n0.reshape(1,1)
    n0 = torch.broadcast_to(n0,symbols.shape).clamp_min(1e-15)
    # Float64 avoids overflow/cancellation in the severe-fog diagnostic. No
    # precision change is made to the semantic model or the carrier recovery.
    distances = (symbols.to(torch.complex128)[...,None]-points.to(torch.complex128)).abs().square()
    scores = -distances/n0[...,None]
    llr = torch.stack([torch.logsumexp(scores[...,labels[:,bit]==0],dim=-1)
                       -torch.logsumexp(scores[...,labels[:,bit]==1],dim=-1)
                       for bit in range(labels.shape[1])],dim=-1)
    if not bool(torch.isfinite(llr).all()):
        raise FloatingPointError("Nonfinite soft QAM LLR; do not drop the frame")
    return llr.clamp(-clip,clip).to(torch.float32)


In [ ]:
"""Predeclared image-disjoint confirmation without reading training ZIPs.

Only GEE_test.zip is copied and decoded. The selected 200 images exclude
the archived development200, old holdout200, and wire-label confirmation100.
Raw train-file overlap is checked against the pinned checkpoint's 11000-entry
training inventory. RGB checks here cover the test pool, not unseen train RGB
or geographical/near-duplicate independence.
"""
import hashlib
import json
import os
from pathlib import Path, PurePosixPath
import shutil
import stat
import zipfile

import torch



COMPACT_FINAL_DATA_VERSION_V26 = "compact_final_data_predeclared_v26_1"
_COMPACT_DATA_LOCK_VERSION_V26 = "fixed_compact_final_data_v26_1"
_COMPACT_DATA_SELECTION_VERSION_V26 = "exclude_known500_sha256_rank_v26_1"
_COMPACT_DATA_PREPROCESSING_V26 = "center-square/PIL-bicubic256/reducing_gap1/RGB[-1,1]"
_COMPACT_EXCLUSION_ROLES_V26 = ("prior_development_200", "prior_holdout_200", "wire_confirmation_100")


def _compact_data_json_sha_v26(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=False,
        allow_nan=False, separators=(",", ":")).encode()).hexdigest()


def _compact_data_valid_sha_v26(value):
    return isinstance(value, str) and len(value) == 64 and all(x in "0123456789abcdef" for x in value)


def _compact_data_hash_list_v26(values, name, count=None):
    if (not isinstance(values, list) or (count is not None and len(values) != count)
            or not values or len(set(values)) != len(values)
            or any(not _compact_data_valid_sha_v26(value) for value in values)):
        raise ValueError("Invalid distinct file SHA256 list: " + name)
    return set(values)


def _compact_data_rank_v26(digest, seed):
    return hashlib.sha256(("compact-final-v26\0" + str(seed) + "\0" + digest).encode()).hexdigest()


def validate_compact_final_data_lock_v26(lock):
    """Check membership from identities only; never resample after seeing pixels."""
    if (not isinstance(lock, dict) or lock.get("version") != _COMPACT_DATA_LOCK_VERSION_V26
            or lock.get("selection_version") != _COMPACT_DATA_SELECTION_VERSION_V26
            or lock.get("selection_uses_image_content_or_quality_scores") is not False
            or lock.get("test_zip_file") != "GEE_test.zip"
            or lock.get("preprocessing") != _COMPACT_DATA_PREPROCESSING_V26
            or lock.get("source_zips_modified") is not False
            or lock.get("geographic_or_near_duplicate_independence_established") is not False
            or lock.get("entirely_new_external_dataset_claimed") is not False
            or lock.get("training_RGB_duplicate_check_this_run") is not False):
        raise ValueError("Use the predeclared confirmation data lock and disclosed scope")
    unsigned = {key: value for key, value in lock.items() if key != "data_lock_sha256"}
    if _compact_data_json_sha_v26(unsigned) != lock.get("data_lock_sha256"):
        raise ValueError("The confirmation data-lock SHA256 differs")
    for key in ("selection_seed", "expected_total_images", "count", "eligible_images", "excluded_unique_images", "source_training_images"):
        if type(lock.get(key)) is not int or lock[key] < (0 if key == "selection_seed" else 1):
            raise ValueError("Invalid fixed data count/seed: " + key)
    for key in ("test_zip_sha256", "source_training_inventory_sha256", "source_data_manifest_sha256"):
        if not _compact_data_valid_sha_v26(lock.get(key)):
            raise ValueError("Pin the reviewed source corpus digest: " + key)
    all_hashes = _compact_data_hash_list_v26(lock.get("all_file_sha256"), "test pool", lock["expected_total_images"])
    if lock["all_file_sha256"] != sorted(all_hashes):
        raise ValueError("Retain the sorted complete test inventory")
    roles = lock.get("excluded_roles")
    if not isinstance(roles, dict) or set(roles) != set(_COMPACT_EXCLUSION_ROLES_V26):
        raise ValueError("Retain all three known previous-evaluation roles")
    excluded = set()
    for role in _COMPACT_EXCLUSION_ROLES_V26:
        hashes = _compact_data_hash_list_v26(roles[role], role)
        if not hashes.issubset(all_hashes) or excluded & hashes:
            raise ValueError("Previous-evaluation memberships are invalid or overlap")
        excluded.update(hashes)
    eligible = all_hashes - excluded
    selected = _compact_data_hash_list_v26(lock.get("selected_file_sha256"), "selected confirmation", lock["count"])
    if (len(excluded) != lock["excluded_unique_images"] or len(eligible) != lock["eligible_images"]
            or lock["count"] > len(eligible) or selected & excluded):
        raise ValueError("Confirmation eligibility or exclusion counts differ")
    expected = sorted(eligible, key=lambda value: _compact_data_rank_v26(value, lock["selection_seed"]))[:lock["count"]]
    if lock["selected_file_sha256"] != expected:
        raise ValueError("Confirmation membership/order must be frozen before pixels and metrics")
    return {"all": all_hashes, "excluded": excluded, "selected": selected, "eligible": eligible}


def _compact_source_training_inventory_v26(configuration, lock, memberships):
    data = configuration.get("data") if isinstance(configuration, dict) else None
    if not isinstance(data, dict) or _compact_data_json_sha_v26(data) != lock["source_data_manifest_sha256"]:
        raise RuntimeError("The fixed primary model's complete data manifest differs")
    count = lock["source_training_images"]
    train_hashes = data.get("training_file_sha256")
    train = _compact_data_hash_list_v26(train_hashes, "source training", count)
    inventory_sha = hashlib.sha256("\n".join(train_hashes).encode()).hexdigest()
    if (configuration.get("training_images") != count or data.get("training_images") != count
            or data.get("training_inventory_sha256") != inventory_sha
            or inventory_sha != lock["source_training_inventory_sha256"]
            or data.get("zip_sha256", {}).get("gee_test") != lock["test_zip_sha256"]
            or data.get("preprocessing") != _COMPACT_DATA_PREPROCESSING_V26):
        raise RuntimeError("The fixed model training inventory/preprocessing identity differs")
    if train & memberships["all"]:
        raise RuntimeError("Exact training-file content overlaps the test corpus")
    return inventory_sha


def _compact_test_zip_copy_v26(drive_root, local_root, expected_sha):
    drive_arg, local_arg = Path(drive_root), Path(local_root)
    if drive_arg.is_symlink() or local_arg.is_symlink():
        raise ValueError("Use unlinked scoped source/cache directories")
    drive, local = drive_arg.resolve(), local_arg.resolve()
    if (drive == Path(drive.anchor) or local == Path(local.anchor)
            or drive == local or drive in local.parents or local in drive.parents):
        raise ValueError("Keep local test staging separate from Drive")
    source = drive / "GEE_test.zip"
    if source.is_symlink() or not source.is_file():
        raise RuntimeError("Restore the existing GEE_test.zip; no dataset replacement fallback")
    if rebuild_file_sha256_v12(source) != expected_sha:
        raise RuntimeError("GEE_test.zip differs from the frozen test corpus")
    local.mkdir(parents=True, exist_ok=True)
    target = local / ("GEE_test_" + expected_sha + ".zip")
    for path in (target, target.with_name(target.name + ".partial")):
        if path.exists() and (path.is_symlink() or not path.is_file() or path.stat().st_nlink > 1):
            raise ValueError("Unsafe own test staging file")
    copied = False
    if not target.is_file() or rebuild_file_sha256_v12(target) != expected_sha:
        pending = target.with_name(target.name + ".partial")
        with source.open("rb") as reader, pending.open("wb") as writer:
            shutil.copyfileobj(reader, writer, 8*1024**2); writer.flush(); os.fsync(writer.fileno())
        if rebuild_file_sha256_v12(pending) != expected_sha:
            raise RuntimeError("Local test ZIP copy SHA256 differs")
        pending.replace(target); copied = True
    return target, {"source_zip_verified": True, "local_zip_verified": True,
        "local_copy_performed_this_call": copied, "zip_sha256": expected_sha}


def _compact_rgb_sha_v26(image):
    if (not isinstance(image, torch.Tensor) or image.dtype != torch.float32
            or tuple(image.shape) != (3, 256, 256) or not bool(torch.isfinite(image).all())):
        raise ValueError("Keep the frozen finite FP32 RGB256 preprocessing")
    raster = ((image.permute(1,2,0)+1)*127.5).round().to(torch.uint8).contiguous().numpy()
    return hashlib.sha256(raster.tobytes()).hexdigest()


def prepare_compact_final_data_v26(drive_root, local_root, lock, source_configuration):
    """Return fixed RGB200, ordered file hashes and an audit-only manifest.

    Decode all test members to detect selected RGB duplicates against the entire
    test pool. Keep only the selected tensor images. Training ZIPs, models and
    quality metrics are never loaded here.
    """
    memberships = validate_compact_final_data_lock_v26(lock)
    training_sha = _compact_source_training_inventory_v26(source_configuration, lock, memberships)
    path, copy_audit = _compact_test_zip_copy_v26(drive_root, local_root, lock["test_zip_sha256"])
    rows, selected_images, rgb_members = [], {}, {}
    with zipfile.ZipFile(path) as archive:
        members = []
        for item in archive.infolist():
            source_names = (item.filename, item.orig_filename)
            if (any(PurePosixPath(name).is_absolute() or ".." in PurePosixPath(name).parts
                    or "\\" in name or any(":" in part for part in PurePosixPath(name).parts)
                    for name in source_names) or item.file_size > 128*1024**2
                    or stat.S_ISLNK(item.external_attr >> 16)):
                raise ValueError("Unsafe test ZIP member; no extraction performed")
            name = PurePosixPath(item.filename)
            if item.is_dir() or name.name.startswith(".") or "__MACOSX" in name.parts:continue
            if name.suffix.lower() in {".png", ".jpg", ".jpeg"}:members.append(item)
        if (len(members) != lock["expected_total_images"]
                or len({item.filename for item in members}) != len(members)):
            raise RuntimeError("The frozen test ZIP image count/names differ")
        for i, item in enumerate(sorted(members, key=lambda value: value.filename)):
            content = archive.read(item); digest = hashlib.sha256(content).hexdigest()
            image = _rebuild_image_v12(content); rgb_sha = _compact_rgb_sha_v26(image)
            rows.append({"file_sha256": digest, "bytes": len(content), "rgb256_sha256": rgb_sha})
            rgb_members.setdefault(rgb_sha, []).append(digest)
            if digest in memberships["selected"]:
                if digest in selected_images:raise RuntimeError("Duplicate selected file content in test ZIP")
                selected_images[digest] = image
            if (i+1) % 250 == 0 or i+1 == len(members):
                print(f"Frozen test-only RGB audit: {i+1}/{len(members)}", flush=True)
    observed = [row["file_sha256"] for row in rows]
    if len(set(observed)) != len(observed) or set(observed) != memberships["all"]:
        raise RuntimeError("The complete test-file SHA256 inventory differs")
    if set(selected_images) != memberships["selected"]:
        raise RuntimeError("A predeclared confirmation image is missing")
    collisions = [digests for digests in rgb_members.values()
                  if len(digests) > 1 and memberships["selected"].intersection(digests)]
    if collisions:
        raise RuntimeError("A selected image duplicates preprocessed RGB in the test pool; stop without resampling")
    selected_hashes = list(lock["selected_file_sha256"])
    images = torch.stack([selected_images[digest] for digest in selected_hashes])
    by_hash = {row["file_sha256"]: row for row in rows}
    canonical_inventory = sorted(rows, key=lambda row: row["file_sha256"])
    manifest = {"version": COMPACT_FINAL_DATA_VERSION_V26, "data_lock_sha256": lock["data_lock_sha256"],
        "images": len(images), "test_images": len(rows), "test_zip_sha256": lock["test_zip_sha256"],
        "selected_file_sha256": selected_hashes,
        "selected_rgb256_sha256": [by_hash[digest]["rgb256_sha256"] for digest in selected_hashes],
        "all_test_file_inventory_sha256": hashlib.sha256("\n".join(sorted(observed)).encode()).hexdigest(),
        "all_test_rgb_inventory_sha256": _compact_data_json_sha_v26(canonical_inventory),
        "source_training_inventory_sha256": training_sha,
        "preprocessing": _COMPACT_DATA_PREPROCESSING_V26,
        "known_prior_evaluation_excluded": len(memberships["excluded"]),
        "prior_role_counts": {key: len(value) for key, value in lock["excluded_roles"].items()},
        "selected_prior_file_overlap": 0, "selected_train_file_overlap": 0,
        "all_test_train_file_overlap": 0, "selected_test_rgb_duplicate_count": 0,
        "nonselected_test_rgb_duplicate_groups": sum(len(values)>1 for values in rgb_members.values()),
        "membership_used_pixels_or_quality_scores": False, "selection_role": lock["selection_role"],
        "training_RGB_duplicate_check_this_run": False,
        "geographic_or_near_duplicate_independence_established": False,
        "entirely_new_external_dataset_claimed": False, "source_zips_modified": False,
        "training_zips_loaded": False, "quality_metrics_called": False, "earth_engine_calls": 0,
        # Transfer work is per-call operational state, not experiment identity.
        # A verified local-cache hit must reproduce exactly the same manifest.
        "copy_audit": {key: value for key, value in copy_audit.items()
                       if key != "local_copy_performed_this_call"}}
    return images, selected_hashes, manifest


In [ ]:
"""Frozen-model compact evaluation with the current common physical receiver.

No optimizer, fitting, model selection, CRC oracle or failure-frame omission.
BPG means the actual native byte stream; LDPC consumes soft QAM LLRs.  The
existing v12 real block-magnitude and noise-PSD CSI assumptions are retained.
"""
import hashlib
import json
import math

import numpy as np
import torch



COMPACT_FINAL_CORE_VERSION_V26 = "frozen_wide1375_compact_final_core_v26_1"
_COMPACT_METHODS_V26 = ("fso_sc", "bpg", "bpg_ldpc")
_COMPACT_PROFILE_V26 = "coherent_receiver_extension_v6"
_COMPACT_COUNTERS_V26 = ("bit_errors", "valid_bits", "pre_fec_bit_errors",
    "pre_fec_valid_bits", "index_errors", "indices", "symbol_errors", "valid_symbols",
    "physical_calls", "source_bits", "coded_bits", "transmitted_bits", "channel_symbols",
    "qam_padding_bits", "information_padding_bits", "ldpc_blocks", "ldpc_nonconverged_blocks")


def compact_conditions_v26():
    """Six predeclared measured points, not an interpolated full-paper sweep."""
    result = []
    for group, weather, modulation, powers, zenith, pointing in (
            ("clear16_zenith30", "clear", "16QAM", (30,33,36),30,20),
            ("snow_qpsk_zenith0", "snow", "QPSK", (32,34,36),0,15)):
        for power in powers:
            kwargs = {"modulation":modulation,"launch_power_dbm":float(power),
                "zenith_angle_deg":float(zenith),"pointing_error_urad":float(pointing),
                "weather":weather,"scattered_power_ratio":.95,
                "background_light_radiance_w_m2_sr_um":1e-4}
            digest = hashlib.sha256(json.dumps(kwargs,sort_keys=True,separators=(",",":"),
                                              allow_nan=False).encode()).hexdigest()[:16]
            result.append({"id":digest,"label":f"{weather}_{modulation}_P{power}",
                           "group":group,"channel_kwargs":kwargs})
    return result


def _compact_condition_v26(condition):
    if condition is None:return None
    if condition not in compact_conditions_v26():
        raise ValueError("Keep the six frozen compact-final conditions; no post-result channel edits")
    return condition["channel_kwargs"]


def compact_seed_v26(condition, image_sha256, base_seed=20261107):
    if (not isinstance(image_sha256,str) or len(image_sha256)!=64
            or any(c not in "0123456789abcdef" for c in image_sha256)
            or type(base_seed) is not int or not 0<=base_seed<2**63):
        raise ValueError("Use one explicit image SHA256 and nonnegative integer channel seed")
    _compact_condition_v26(condition)
    label = "clean" if condition is None else condition["id"]
    return int.from_bytes(hashlib.sha256(
        f"compact-final-v26:{base_seed}:{label}:{image_sha256}".encode()).digest()[:8],"big")%(2**63-1)


def _compact_link_v26(link):
    if link is None or link.metadata().get("profile") != _COMPACT_PROFILE_V26:
        raise ValueError("Keep the frozen coherent receiver extension for every evaluated method")
    if not callable(link.ns.get("recover_qam_carrier_v3")):
        raise ValueError("The current v3 receiver must serve both semantic and byte-stream methods")


@torch.inference_mode()
def transport_bytes_v26(payload, link, condition, seed, device="cpu", ldpc=None):
    """One real BPG transport using exactly the link's current physical/DSP code.

    Equal seeds do not assert identical waveform noise across unequal payload
    lengths.  Neither the source bytes nor true phase/CFO enter the receiver.
    """
    _compact_link_v26(link);kwargs = _compact_condition_v26(condition)
    if kwargs is None:raise ValueError("Use direct codec reconstruction for clean BPG")
    if not isinstance(payload,bytes) or not payload:raise ValueError("Transmit complete native BPG bytes")
    if type(seed) is not int or not 0<=seed<2**63:raise ValueError("Invalid private channel seed")
    device=torch.device(device);ns=link.ns;physical=link.physical
    bits=torch.from_numpy(np.unpackbits(np.frombuffer(payload,dtype=np.uint8)).copy()).to(device)
    source_count=bits.numel();blocks=math.ceil(source_count/ldpc.k) if ldpc is not None else 0
    information_padding=blocks*ldpc.k-source_count if ldpc is not None else 0
    if ldpc is None:transmitted=bits.reshape(1,-1)
    else:
        padded=torch.cat((bits,torch.zeros(information_padding,dtype=torch.uint8,device=device)))
        transmitted=ldpc.encode_bits(padded.reshape(blocks,ldpc.k)).reshape(1,-1)
    modulation=kwargs["modulation"];_,_,bps=ns["qam_constellation"](modulation,device)
    coded_count=transmitted.numel();padding=(-coded_count)%bps
    grouped=torch.cat((transmitted,torch.zeros((1,padding),dtype=torch.uint8,device=device)),dim=-1).reshape(1,-1,bps)
    symbols=ns["qam_modulate"](grouped,modulation)
    generator=torch.Generator(device=device).manual_seed(seed)
    observation,mode=_rebuild_observe_v12(physical,symbols,generator,
        {k:v for k,v in kwargs.items() if k!="modulation"})
    if (observation.is_equalized or observation.channel_gain is None
            or observation.symbols.shape!=symbols.shape
            or observation.channel_gain.shape!=symbols.shape
            or not bool(torch.isfinite(observation.symbols).all())
            or not bool(torch.isfinite(observation.channel_gain).all())):
        raise FloatingPointError("Invalid raw physical observation; no frame may be omitted")
    magnitude=observation.channel_gain.abs().mean(-1,keepdim=True)
    if not bool((magnitude>0).all()):raise FloatingPointError("Nonpositive block magnitude")
    recovered,diagnostics=ns["recover_qam_carrier_v3"](observation.symbols,modulation,
        link.config.symbol_rate_baud,known_block_magnitude=magnitude)
    if recovered.shape!=symbols.shape or not bool(torch.isfinite(recovered).all()):
        raise FloatingPointError("Nonfinite/wrong-shaped recovered byte-stream samples")
    hard_groups=ns["qam_demodulate"](recovered,modulation)
    if hard_groups.shape!=grouped.shape:raise ValueError("Receiver bit grouping changed")
    hard=hard_groups.reshape(1,-1)[:,:coded_count]
    budget=observation.diagnostics["budget"]
    receiver_noise=budget["background_noise_w"] if link.config.include_background_noise else 0.
    receiver_noise+=budget["receiver_shot_noise_w"] if link.config.include_shot_noise else 0.
    n0=receiver_noise/magnitude.to(torch.float64).square().clamp_min(1e-300)
    if link.config.include_ase_noise:n0+=budget["ase_noise_w"]/budget["launch_power_w"]
    statistics=None
    if ldpc is None:decoded_bits=hard.reshape(-1)[:source_count]
    else:
        soft=soft_qam_llr(recovered,modulation,n0,ns,clip=ldpc.llr_clip)
        information,statistics=ldpc.decode_llr(soft.reshape(-1)[:coded_count].reshape(blocks,ldpc.n))
        decoded_bits=information.reshape(-1)[:source_count]
    energy=float(symbols.abs().square().mean());launch=10**((kwargs["launch_power_dbm"]-30)/10)
    result={"received_bytes":np.packbits(decoded_bits.cpu().numpy()).tobytes(),"seed":seed,
        "bit_errors":int((decoded_bits!=bits).sum()),"valid_bits":source_count,
        "pre_fec_bit_errors":int((hard!=transmitted).sum()),"pre_fec_valid_bits":coded_count,
        "index_errors":0,"indices":0,"symbol_errors":int((hard_groups!=grouped).any(-1).sum()),
        "valid_symbols":grouped.shape[1],"physical_calls":1,
        "source_bytes":len(payload),"source_bits":source_count,"coded_bits":coded_count,
        "transmitted_bits":coded_count+padding,"channel_symbols":symbols.numel(),
        "qam_padding_bits":padding,"padding_bits":padding,
        "information_padding_bits":information_padding,"ldpc_blocks":blocks,
        "ldpc_nonconverged_blocks":0,"ldpc_iterations_mean":0.,
        "noise_variance_equalized":float(n0.mean()),"receiver_diagnostics":diagnostics,
        "channel_gain_sha256":_rebuild_sha_v12(observation.channel_gain),
        "channel_rng_state_sha256":_rebuild_sha_v12(generator.get_state()),
        "transmitted_symbols_mean_power":energy,"nominal_launch_power_w":launch,
        "actual_launch_power_w":launch*energy,"deterministic_physical":mode,
        "receiver":"same link.ns recover_qam_carrier_v3 as semantic transmission",
        "same_waveform_noise_across_methods_claimed":False,
        "frame_integrity_policy":"Known file length; actual native decode/header validation; no truth BER or parity-based erasure"}
    if statistics is not None:
        result["ldpc_nonconverged_blocks"]=int((~statistics["converged"]).sum())
        result["ldpc_iterations_mean"]=float(statistics["iterations"].float().mean())
    return result


def _compact_image_v26(image):
    if (not isinstance(image,torch.Tensor) or image.dtype!=torch.float32
            or tuple(image.shape)!=(1,3,256,256) or not bool(torch.isfinite(image).all())
            or bool((image.abs()>1.000001).any())):
        raise ValueError("Expected finite preprocessed FP32 RGB[1,3,256,256] in[-1,1]")


def _compact_clean_transport_v26(payload,seed):
    return {"seed":seed,"received_bytes":payload,"source_bytes":len(payload),
        **{k:0 for k in _COMPACT_COUNTERS_V26},"source_bits":len(payload)*8,
        "valid_bits":len(payload)*8,"padding_bits":0,"ldpc_iterations_mean":0.,
        "receiver_diagnostics":{},"nominal_launch_power_w":None,"actual_launch_power_w":None,
        "transmitted_symbols_mean_power":None,"channel_gain_sha256":None,
        "channel_rng_state_sha256":None,"deterministic_physical":"clean_identity"}


@torch.inference_mode()
def evaluate_compact_image_v26(model,image,image_sha256,metrics,link,condition,payload,
        codec,ldpc,method,*,device="cpu",base_seed=20261107,cached_indices=None):
    """One retained frame, separately usable by the runner's chunk journal."""
    if method not in _COMPACT_METHODS_V26:raise ValueError("Unknown frozen evaluation method")
    _compact_image_v26(image);_compact_link_v26(link);kwargs=_compact_condition_v26(condition)
    if set(metrics)!={"alex","vgg"}:raise ValueError("Report the fixed primary Alex and secondary VGG metrics")
    seed=compact_seed_v26(condition,image_sha256,base_seed);device=torch.device(device)
    target=image.to(device);decode_failed=False;decode_status="semantic_reconstruction";decode_error=None
    if method=="fso_sc":
        if model is None:raise ValueError("The fixed semantic model is required")
        previous=model.training;model.eval()
        try:
            if cached_indices is None:_,_,indices=model.encode(target)
            else:indices=cached_indices.to(device)
            if (indices.dtype!=torch.int64 or tuple(indices.shape)!=(1,32,32)
                    or bool(((indices<0)|(indices>=16384)).any())):
                raise ValueError("Semantic cache must be current model int64[1,32,32] indices")
            received,rows=link.transmit(indices,[kwargs],[seed],device=device)
            prediction=model.decode_indices(received,received.shape)
            result=dict(rows[0]);padding=result.get("padding_bits",0)
            physical=result["physical_calls"]==1
            result.update(source_bytes=1792,source_bits=14336,coded_bits=14336 if physical else 0,
                transmitted_bits=14336+padding if physical else 0,
                channel_symbols=result["valid_symbols"] if physical else 0,qam_padding_bits=padding,
                information_padding_bits=0,ldpc_blocks=0,ldpc_nonconverged_blocks=0,
                ldpc_iterations_mean=0.,pre_fec_bit_errors=result["bit_errors"] if physical else 0,
                pre_fec_valid_bits=14336 if physical else 0,
                same_waveform_noise_across_methods_claimed=False)
        finally:model.train(previous)
    else:
        if codec is None or not isinstance(payload,bytes) or not payload:
            raise ValueError("Native BPG codec and selected complete byte stream are required")
        if method=="bpg_ldpc" and (ldpc is None or kwargs is None):
            raise ValueError("BPG+LDPC requires the fixed code and physical condition; clean duplicate is not evaluated")
        result=(_compact_clean_transport_v26(payload,seed) if kwargs is None else
            transport_bytes_v26(payload,link,condition,seed,device,ldpc if method=="bpg_ldpc" else None))
        decoded=codec.decode(result.pop("received_bytes"))
        decode_failed=not decoded.ok;decode_status=decoded.status
        decode_error=decoded.error if decode_failed else None
        if decode_failed:raster=np.full((256,256,3),128,dtype=np.uint8)
        else:
            raster=np.asarray(decoded.image,dtype=np.uint8)
            if raster.shape!=(256,256,3):raise ValueError("Successful BPG decode returned the wrong RGB shape")
        prediction=torch.from_numpy(raster.copy()).permute(2,0,1).unsqueeze(0).to(device,dtype=torch.float32)/127.5-1.
    if prediction.shape!=target.shape or not bool(torch.isfinite(prediction).all()):
        raise FloatingPointError("Invalid reconstructed image; no frame may be discarded")
    prediction=prediction.clamp(-1,1);scores={}
    for name,metric in metrics.items():
        score=metric(prediction,target).reshape(-1)
        if score.numel()!=1 or not bool(torch.isfinite(score).all()):
            raise FloatingPointError("Nonfinite LPIPS; no frame may be discarded")
        scores[name]=float(score[0])
    record={**result,"image_sha256":image_sha256,"image_name":image_sha256,"repeat":0,
        "method":method,"profile":_COMPACT_PROFILE_V26,"condition":condition,
        "lpips_alex":1. if decode_failed else scores["alex"],
        "lpips_vgg":1. if decode_failed else scores["vgg"],
        "rendered_lpips_alex":scores["alex"],"rendered_lpips_vgg":scores["vgg"],
        "lpips":1. if decode_failed else scores["alex"],"rendered_lpips":scores["alex"],
        "mse":float((prediction-target).square().mean()),"decode_failed":decode_failed,
        "decode_status":decode_status,"decode_error":decode_error,
        "failure_marker_policy":"Native BPG decode/header failure -> marker1 in each LPIPS; actual gray LPIPS separately measured; no mathematical LPIPS upper bound"}
    record["payload_ber"]=record["bit_errors"]/record["valid_bits"]
    record["index_error_rate"]=record["index_errors"]/record["indices"] if record["indices"] else None
    record["symbol_error_rate"]=record["symbol_errors"]/record["valid_symbols"] if record["valid_symbols"] else None
    record["source_bpp"]=record["source_bits"]/65536
    record["transmitted_bpp"]=record["transmitted_bits"]/65536
    json.dumps(record,allow_nan=False)
    return {"record":record,"prediction":prediction.detach().cpu().clone()}


def summarize_compact_records_v26(records,method,condition,*,base_seed=20261107,bootstrap_samples=2000):
    """Image-only bootstrap; does not cover channel/model/setting uncertainty."""
    _compact_condition_v26(condition)
    if (method not in _COMPACT_METHODS_V26 or not isinstance(records,list) or not records
            or type(bootstrap_samples) is not int or bootstrap_samples<1
            or len({r["image_sha256"] for r in records})!=len(records)
            or any(r["method"]!=method or r["condition"]!=condition for r in records)):
        raise ValueError("Case must retain each declared image exactly once under one frozen method/condition")
    n=len(records);rng=np.random.default_rng(base_seed)
    summary={}
    for key in ("lpips_alex","lpips_vgg","rendered_lpips_alex","rendered_lpips_vgg","mse"):
        values=np.asarray([r[key] for r in records],dtype=np.float64)
        if not bool(np.isfinite(values).all()):raise FloatingPointError("Nonfinite case scores")
        summary[key]=float(values.mean())
        summary[key+"_ci95"]=np.quantile(values[rng.integers(0,n,(bootstrap_samples,n))].mean(1),[.025,.975]).tolist()
    counts={key:sum(r[key] for r in records) for key in _COMPACT_COUNTERS_V26}
    summary.update(mean=summary["lpips_alex"],lpips_mean=summary["lpips_alex"],ci95=summary["lpips_alex_ci95"],
        rendered_lpips_mean=summary["rendered_lpips_alex"],mse_mean=summary["mse"],
        counts=counts,independent_images=n,unique_images=n,image_channel_realizations=n,
        bootstrap_samples=bootstrap_samples,ci_unit="image only; one channel realization per image",
        payload_ber=counts["bit_errors"]/counts["valid_bits"],
        pre_fec_ber=counts["pre_fec_bit_errors"]/counts["pre_fec_valid_bits"] if counts["pre_fec_valid_bits"] else None,
        index_error_rate=counts["index_errors"]/counts["indices"] if counts["indices"] else None,
        symbol_error_rate=counts["symbol_errors"]/counts["valid_symbols"] if counts["valid_symbols"] else None,
        decode_failures=sum(r["decode_failed"] for r in records),
        decode_failure_rate=sum(r["decode_failed"] for r in records)/n,
        source_bpp_mean=float(np.mean([r["source_bpp"] for r in records])),
        transmitted_bpp_mean=float(np.mean([r["transmitted_bpp"] for r in records])),
        channel_symbols_mean=counts["channel_symbols"]/n,
        primary_metric="Alex LPIPS v0.1 with native-decode failure marker1",
        secondary_metric="VGG LPIPS v0.1 with separately reported rendered fallback scores",
        rate_comparison="matched mean source rate, NOT equal total channel-use budget",
        same_waveform_noise_across_methods_claimed=False)
    json.dumps(summary,allow_nan=False)
    return summary


@torch.inference_mode()
def evaluate_compact_case_v26(model,images,hashes,metrics,link,condition,streams,codec,ldpc,method,
        *,device="cpu",base_seed=20261107,example_count=3,cached_indices=None,bootstrap_samples=2000,progress=None):
    """Convenience wrapper; the runner may instead journal per-image records."""
    if (len(images)!=len(hashes) or not hashes or len(set(hashes))!=len(hashes)
            or type(example_count) is not int or example_count<0):
        raise ValueError("Invalid compact case image identities")
    if method!="fso_sc" and (not isinstance(streams,dict) or set(streams)!=set(hashes)):
        raise ValueError("Every byte-stream case must cover exactly its selected images")
    records=[];examples={}
    for i,digest in enumerate(hashes):
        cached=None
        if cached_indices is not None:
            cached=cached_indices[digest] if isinstance(cached_indices,dict) else cached_indices[i:i+1]
        packet=evaluate_compact_image_v26(model,images[i:i+1],digest,metrics,link,condition,
            None if method=="fso_sc" else streams[digest],codec,ldpc,method,
            device=device,base_seed=base_seed,cached_indices=cached)
        records.append(packet["record"])
        if i<example_count:examples[digest]=packet["prediction"]
        if progress is not None:progress(i+1,len(hashes))
    return {"report":{"version":COMPACT_FINAL_CORE_VERSION_V26,"method":method,"condition":condition,
        "images":len(hashes),"records":records,"summary":summarize_compact_records_v26(records,method,condition,
            base_seed=base_seed,bootstrap_samples=bootstrap_samples)},"examples":examples}


In [ ]:
"""Frozen compact confirmation, genuine BPG streams, and 25-image repair units.

No optimizer, training, model selection, source modification or Drive API.
Only the pinned wide1375 model is eligible. Native BPG candidates and selection
are frozen before any metric is created, and reused for every channel branch.
"""
import copy
import gc
import hashlib
import io
import json
import math
from pathlib import Path
import time
import zipfile

import numpy as np
from PIL import Image, ImageDraw
import torch



COMPACT_FINAL_RUNNER_VERSION_V26 = "fixed_wide1375_compact_confirmation_v26_1"
COMPACT_FINAL_CHUNK_SIZE_V26 = 25
COMPACT_FINAL_RATE_BYTES_V26 = 1792
COMPACT_FINAL_RATE_SEED_V26 = 20261107
COMPACT_FINAL_CHANNEL_SEED_V26 = 20261107
_COMPACT_METHODS_V26 = ("fso_sc", "bpg", "bpg_ldpc")


def _compact_source_v26(ns):
    lock = ns["COMPACT_FINAL_INPUT_LOCK_V26"]
    source = Path(ns["SOURCE_V25_DIR"])
    if (source.is_symlink() or (lock.get("source_branch") is not None and source.name != lock["source_branch"])
            or source.parent.name != lock["source_root_name"]
            or source.resolve() == Path(source.resolve().anchor)):
        raise ValueError("Use only the pinned wide1375 result folder")
    path = _pilot_child_v12(source, lock["source_summary_file"])
    if (not path.is_file() or path.stat().st_size != lock["source_summary_bytes"]
            or _pilot_file_sha_v12(path) != lock["source_summary_file_sha256"]):
        raise RuntimeError("Reviewed wide1375 summary is missing or changed")
    summary = json.loads(path.read_text(encoding="utf-8"))
    cp = lock["source_checkpoint"]
    common = json.loads(_pilot_child_v12(source, cp["configuration_file"]).read_text(encoding="utf-8"))
    configuration = common["configuration"]
    endpoint = summary["endpoint"]
    reference = endpoint["checkpoint"]
    if (summary.get("status") != "one_total_epoch_completed"
            or summary["binding"]["code_sha256"] != lock["source_code_sha256"]
            or (lock.get("source_summary_json_sha256") is not None and _pilot_json_sha_v12(summary) != lock["source_summary_json_sha256"])
            or (lock.get("source_common_configuration_sha256") is not None
                and _pilot_json_sha_v12(summary["common_configuration"]) != lock["source_common_configuration_sha256"])
            or source.name != "extension_"+_pilot_json_sha_v12(summary["common_configuration"])[:16]
            or common != summary["common_configuration"]
            or ns["source_configuration"] != configuration
            or _pilot_json_sha_v12(configuration) != cp["configuration_sha256"]
            or endpoint["model_state_sha256"] != cp["model_state_sha256"]
            or (lock.get("source_progress_json_sha256") is not None
                and _pilot_json_sha_v12(endpoint["progress"]) != lock["source_progress_json_sha256"])
            or reference["configuration_sha256"] != cp["configuration_sha256"]
            or reference["checkpoint_file"] != cp["file"]
            or any(reference["generation"][k] != cp[k] for k in ("sha256", "bytes", "updates"))
            or not _extension_checkpoint_reference_valid_v22(source, reference)):
        raise RuntimeError("Pinned primary model/configuration/progress differs; do not select another model on confirmation data")
    progress = endpoint["progress"]
    if (cp["updates"] != 1375 or progress.get("updates") != 1375
            or progress.get("epoch") != 1 or progress.get("cursor") != 0
            or progress.get("status") != "range_extension_budget_reached"
            or progress.get("arm") != "wide" or progress.get("counts") != [10, 102, 307, 512]
            or summary["best_reference"]["case"] != "wide1375"):
        raise RuntimeError("Continue evaluation only with the preselected wide1375 primary")
    if ns["link"].metadata() != configuration["channel"]:
        raise RuntimeError("The frozen physical channel or current receiver changed")
    hashes = ns["evaluation_hashes"]
    images = ns["evaluation_images"]
    manifest = ns["evaluation_manifest"]
    if (not isinstance(hashes, list) or len(set(hashes)) != len(hashes)
            or len(hashes) != lock["evaluation_count"] or len(images) != len(hashes)
            or manifest.get("selected_file_sha256") != hashes
            or (lock.get("evaluation_manifest_json_sha256") is not None
                and _pilot_json_sha_v12(manifest) != lock["evaluation_manifest_json_sha256"])
            or manifest.get("data_lock_sha256") != ns["COMPACT_FINAL_DATA_LOCK_V26"].get("data_lock_sha256")
            or tuple(images.shape) != (len(hashes), 3, 256, 256)
            or images.dtype != torch.float32 or not bool(torch.isfinite(images).all())
            or bool(((images < -1) | (images > 1)).any())):
        raise RuntimeError("Frozen confirmation membership, preprocessing or image tensors differ")
    return source.resolve(), summary, configuration


def _compact_atomic_bytes_v26(root, relative, content):
    path = _pilot_child_v12(root, relative)
    path.parent.mkdir(parents=True, exist_ok=True)
    pending = path.with_name(path.name+".partial")
    if pending.is_symlink():raise ValueError("Linked compact-result pending file")
    with pending.open("wb") as handle:
        handle.write(content);handle.flush()
    pending.replace(path)
    return path


def _compact_rgb_v26(value):
    if tuple(value.shape) != (1, 3, 256, 256):raise ValueError("Expected one RGB256 tensor")
    return ((value.detach().cpu()[0].permute(1, 2, 0).clamp(-1, 1)+1)*127.5).round().to(torch.uint8).numpy()


def _compact_rate_bracket_packet_v26(bracket):
    return {"image_name": bracket.image_name, "target_bytes": bracket.target_bytes,
        "low": bracket.low.metadata(), "high": bracket.high.metadata() if bracket.high else None,
        "tested_q_sizes": [list(row) for row in bracket.tested_q_sizes],
        "search_mode": bracket.search_mode, "optimality": bracket.optimality,
        "encoder_settings": dict(bracket.encoder_settings), "cache_hits": bracket.cache_hits}


def _compact_rate_candidates_v26(ns, branch, codec, build, hashes, images):
    """Eight small native-candidate ZIPs normally; no source RGB copied to Drive."""
    n = len(hashes);all_brackets = [];artifacts = []
    for first in range(0, n, COMPACT_FINAL_CHUNK_SIZE_V26):
        last = min(first+COMPACT_FINAL_CHUNK_SIZE_V26, n)
        relative = f"rate_candidates/block_{first:04d}_{last:04d}.zip"
        marker_relative = relative+".identity.json"
        archive = _pilot_child_v12(branch, relative)
        marker = _pilot_child_v12(branch, marker_relative)
        identity = {"version": COMPACT_FINAL_RUNNER_VERSION_V26, "rate_control": RATE_CONTROL_VERSION,
            "image_sha256": hashes[first:last], "target_bytes": COMPACT_FINAL_RATE_BYTES_V26,
            "codec_source_sha256": ns["COMPACT_FINAL_INPUT_LOCK_V26"]["bpg"]["source_sha256"],
            "encoder_settings": dict(codec.encoder_settings), "search_mode": "binary_checked"}
        cached = json.loads(marker.read_text(encoding="utf-8")) if marker.is_file() else None
        if cached is not None and cached.get("identity") != identity:
            raise RuntimeError("Native candidate cache belongs to different codec/images/settings")
        if not (cached is not None and _pilot_verify_inventory_v12(branch, cached.get("artifacts"))):
            brackets = []
            for index in range(first, last):
                raster = Image.fromarray(_compact_rgb_v26(images[index:index+1]))
                brackets.append(encode_rate_bracket(codec, raster, hashes[index],
                    COMPACT_FINAL_RATE_BYTES_V26, search_mode="binary_checked"))
                print(f"BPG native source {index+1}/{n}", flush=True)
            buffer = io.BytesIO()
            with zipfile.ZipFile(buffer, "w", compression=zipfile.ZIP_DEFLATED) as z:
                metadata = []
                for bracket in brackets:
                    metadata.append(_compact_rate_bracket_packet_v26(bracket))
                    z.writestr(bracket.image_name+"/low.bpg", bracket.low.data)
                    if bracket.high is not None:z.writestr(bracket.image_name+"/high.bpg", bracket.high.data)
                z.writestr("metadata.json", json.dumps(metadata, sort_keys=True, ensure_ascii=False, allow_nan=False))
            _compact_atomic_bytes_v26(branch, relative, buffer.getvalue())
            rows = _pilot_inventory_v12(branch, [archive])
            _pilot_save_json_v12(ns, branch, marker_relative, {"identity": identity, "artifacts": rows})
        with zipfile.ZipFile(archive) as z:
            if z.testzip() is not None:raise RuntimeError("Native BPG candidate ZIP CRC differs")
            metadata = json.loads(z.read("metadata.json"))
            if [row["image_name"] for row in metadata] != hashes[first:last]:
                raise RuntimeError("Native BPG candidate image order differs")
            for row in metadata:
                def candidate(label):
                    value = row[label]
                    if value is None:return None
                    raw = z.read(row["image_name"]+"/"+label+".bpg")
                    if len(raw) != value["actual_bytes"] or hashlib.sha256(raw).hexdigest() != value["sha256"]:
                        raise RuntimeError("Native BPG candidate content differs")
                    return NativeBPGCandidate(raw, value["q"], tuple(value["image_hw"]))
                all_brackets.append(BPGRateBracket(row["image_name"], row["target_bytes"], candidate("low"),
                    candidate("high"), tuple(tuple(x) for x in row["tested_q_sizes"]), row["search_mode"],
                    row["optimality"], row["encoder_settings"], row["cache_hits"]))
        artifacts += _pilot_inventory_v12(branch, [archive, marker])
    return all_brackets, artifacts


def _compact_streams_v26(ns, branch, codec, build, hashes, images, binding):
    archive = _pilot_child_v12(branch, "sources/selected_native_bpg.zip")
    marker = _pilot_child_v12(branch, "sources/selected_native_bpg.identity.json")
    identity = {"binding_sha256": _pilot_json_sha_v12(binding),
        "codec_source_sha256": ns["COMPACT_FINAL_INPUT_LOCK_V26"]["bpg"]["source_sha256"],
        "image_sha256": hashes, "target_bytes": COMPACT_FINAL_RATE_BYTES_V26,
        "rate_seed": COMPACT_FINAL_RATE_SEED_V26, "rate_control": RATE_CONTROL_VERSION}
    cached = json.loads(marker.read_text(encoding="utf-8")) if marker.is_file() else None
    if cached is not None and cached.get("identity") != identity:
        raise RuntimeError("Frozen native BPG mapping belongs to different protocol")
    candidate_artifacts = []
    if not (cached is not None and _pilot_verify_inventory_v12(branch, cached.get("artifacts"))):
        if cached is not None:
            # A frozen mapping must not be replaced by a new native encoder build.
            print("Repairing frozen source archive from identical retained native candidates", flush=True)
        brackets, candidate_artifacts = _compact_rate_candidates_v26(ns, branch, codec, build, hashes, images)
        matched = select_rate_matched_sources(brackets, target_bytes_per_image=COMPACT_FINAL_RATE_BYTES_V26,
            seed=COMPACT_FINAL_RATE_SEED_V26)
        metadata = {"identity": identity, "rate_matching": matched.metadata(), "original_codec_build": build,
            "entries": {digest: matched.entries[digest].metadata() for digest in hashes},
            "quality_or_channel_outcome_used_for_rate_selection": False}
        if cached is not None:
            frozen = cached["frozen_metadata"]
            if any(metadata["entries"][digest][key] != frozen["entries"][digest][key]
                   for digest in hashes for key in ("q", "actual_bytes", "sha256")):
                raise RuntimeError("Re-encoded native sources differ from the frozen pre-metric byte mapping")
            metadata = frozen
        buffer = io.BytesIO()
        with zipfile.ZipFile(buffer, "w", compression=zipfile.ZIP_DEFLATED) as z:
            z.writestr("metadata.json", json.dumps(metadata, sort_keys=True, ensure_ascii=False, allow_nan=False))
            for digest in hashes:z.writestr(digest+".bpg", matched.entries[digest].data)
        if cached is not None and cached.get("mapping_sha256") != _pilot_json_sha_v12(metadata):
            raise RuntimeError("Recomputed native source selection differs from the frozen pre-metric mapping")
        _compact_atomic_bytes_v26(branch, "sources/selected_native_bpg.zip", buffer.getvalue())
        paths = [archive, _pilot_save_json_v12(ns, branch, "sources/rate_matching.json", metadata)]
        rows = _pilot_inventory_v12(branch, paths)
        _pilot_save_json_v12(ns, branch, "sources/selected_native_bpg.identity.json",
            {"identity": identity, "mapping_sha256": _pilot_json_sha_v12(metadata),
             "frozen_metadata": metadata, "artifacts": rows})
    with zipfile.ZipFile(archive) as z:
        if z.testzip() is not None:raise RuntimeError("Frozen native BPG source CRC differs")
        metadata = json.loads(z.read("metadata.json"))
        if metadata["identity"] != identity:raise RuntimeError("Frozen native source protocol differs")
        streams = {digest: z.read(digest+".bpg") for digest in hashes}
        if set(z.namelist()) != {"metadata.json"} | {digest+".bpg" for digest in hashes}:
            raise RuntimeError("Unexpected/missing frozen native BPG entries")
        for digest, raw in streams.items():
            row = metadata["entries"][digest]
            if len(raw) != row["actual_bytes"] or hashlib.sha256(raw).hexdigest() != row["sha256"]:
                raise RuntimeError("Frozen native BPG content differs")
    cached = json.loads(marker.read_text(encoding="utf-8"))
    if _pilot_json_sha_v12(metadata) != cached["mapping_sha256"]:
        raise RuntimeError("Frozen native BPG rate manifest differs")
    target = ns["COMPACT_FINAL_INPUT_LOCK_V26"]["source_bpp_target"]
    tolerance = ns["COMPACT_FINAL_INPUT_LOCK_V26"]["rate_tolerance_bpp"]
    if abs(metadata["rate_matching"]["actual_mean_bpp"]-target) > tolerance:
        raise RuntimeError("Native BPG mean source bitrate misses its predeclared tolerance; do not tune using metrics")
    return streams, metadata, candidate_artifacts+cached["artifacts"]+_pilot_inventory_v12(branch, [marker])


def _compact_case_v26(ns, branch, binding, model, metrics, link, codec, ldpc, streams, hashes, images,
                     indices, method, condition, stream_sha):
    label = "clean" if condition is None else condition["id"]
    root = f"cases/{label}/{method}"
    identity = {"binding_sha256": _pilot_json_sha_v12(binding), "method": method, "condition": condition,
        "image_sha256": hashes, "frozen_bpg_mapping_sha256": stream_sha,
        "chunk_size": COMPACT_FINAL_CHUNK_SIZE_V26, "examples": hashes[:3]}
    marker = _pilot_child_v12(branch, root+"/completed.json")
    cached = json.loads(marker.read_text(encoding="utf-8")) if marker.is_file() else None
    if cached is not None and cached.get("identity") != identity:raise RuntimeError("Condition cache identity changed")
    if cached is not None and _pilot_verify_inventory_v12(branch, cached.get("artifacts")):
        report = json.loads(_pilot_child_v12(branch, root+"/metrics.json").read_text(encoding="utf-8"))
        if [r["image_sha256"] for r in report["records"]] != hashes:raise RuntimeError("Cached case membership differs")
        return report, cached["artifacts"]+_pilot_inventory_v12(branch, [marker])
    records = [];artifacts = []
    evaluator = ns.get("evaluate_compact_image_v26", evaluate_compact_image_v26)
    for first in range(0, len(hashes), COMPACT_FINAL_CHUNK_SIZE_V26):
        last = min(first+COMPACT_FINAL_CHUNK_SIZE_V26, len(hashes))
        block = root+f"/blocks/{first:04d}_{last:04d}"
        block_marker = _pilot_child_v12(branch, block+".identity.json")
        block_identity = {"case_identity_sha256": _pilot_json_sha_v12(identity), "first": first, "last": last}
        saved = json.loads(block_marker.read_text(encoding="utf-8")) if block_marker.is_file() else None
        if saved is not None and saved.get("identity") != block_identity:raise RuntimeError("Case block identity differs")
        if saved is not None and _pilot_verify_inventory_v12(branch, saved.get("artifacts")):
            packet = json.loads(_pilot_child_v12(branch, block+".json").read_text(encoding="utf-8"))
            paths = saved["artifacts"]+_pilot_inventory_v12(branch, [block_marker])
            if [r["image_sha256"] for r in packet["records"]] != hashes[first:last]:raise RuntimeError("Case block order differs")
        else:
            rows = [];files = []
            for index in range(first, last):
                digest = hashes[index]
                one = evaluator(model, images[index:index+1], digest, metrics, link, condition,
                    None if method == "fso_sc" else streams[digest], codec, ldpc, method,
                    device=ns["device"], base_seed=COMPACT_FINAL_CHANNEL_SEED_V26,
                    cached_indices=indices[index:index+1] if method == "fso_sc" else None)
                row = one["record"]
                if row.get("image_sha256") != digest:raise RuntimeError("Image evaluator changed its identity")
                rows.append(row)
                if index < 3:
                    panel = Image.new("RGB", (512, 286), "white")
                    panel.paste(Image.fromarray(_compact_rgb_v26(images[index:index+1])), (0, 30))
                    panel.paste(Image.fromarray(_compact_rgb_v26(one["prediction"])), (256, 30))
                    draw = ImageDraw.Draw(panel);draw.text((4, 8), "Original RGB", fill="black")
                    draw.text((260, 8), method+" / "+label, fill="black")
                    buffer = io.BytesIO();panel.save(buffer, format="PNG")
                    files.append(_compact_atomic_bytes_v26(branch, root+"/example_"+digest+".png", buffer.getvalue()))
                del one
            files.append(_pilot_save_json_v12(ns, branch, block+".json", {"records": rows}))
            paths = _pilot_inventory_v12(branch, files)
            _pilot_save_json_v12(ns, branch, block+".identity.json", {"identity": block_identity, "artifacts": paths})
            paths += _pilot_inventory_v12(branch, [block_marker]);packet = {"records": rows}
            print(f"Compact {label}/{method}: committed {last}/{len(hashes)}", flush=True)
        records += packet["records"];artifacts += paths
    report = {"version": COMPACT_FINAL_RUNNER_VERSION_V26, "method": method, "condition": condition,
        "images": len(hashes), "records": records,
        "summary": summarize_compact_records_v26(records, method, condition,
            base_seed=COMPACT_FINAL_CHANNEL_SEED_V26, bootstrap_samples=2000)}
    path = _pilot_save_json_v12(ns, branch, root+"/metrics.json", report)
    artifacts += _pilot_inventory_v12(branch, [path])
    _pilot_save_json_v12(ns, branch, root+"/completed.json", {"identity": identity, "artifacts": artifacts})
    return report, artifacts+_pilot_inventory_v12(branch, [marker])


def run_compact_final_v26(ns, *, code_sha256):
    required = ("COMPACT_FINAL_INPUT_LOCK_V26", "COMPACT_FINAL_DATA_LOCK_V26", "SOURCE_V25_DIR", "PILOT_ROOT", "LOCAL_CACHE",
        "source_configuration", "evaluation_images", "evaluation_hashes", "evaluation_manifest",
        "link", "device", "FSO_VQVAE", "rebuild_save_json_v12",
        "create_compact_metrics_v26", "create_compact_codec_v26", "SystematicRALDPC")
    if any(k not in ns for k in required):raise RuntimeError("Prepare the frozen compact confirmation first")
    if not isinstance(code_sha256, str) or len(code_sha256) != 64:raise ValueError("Pin complete evaluation code")
    # Transport work is operational evidence, not corpus membership. The data
    # helper reports a fresh local copy on first execution and a cache hit on
    # restart; that flag must never change the frozen run identity.
    caller_ns = ns
    ns = dict(ns)
    manifest = copy.deepcopy(ns["evaluation_manifest"])
    if isinstance(manifest.get("copy_audit"), dict):
        manifest["copy_audit"].pop("local_copy_performed_this_call", None)
    ns["evaluation_manifest"] = manifest
    source, source_summary, source_configuration = _compact_source_v26(ns)
    lock = ns["COMPACT_FINAL_INPUT_LOCK_V26"];cp = lock["source_checkpoint"]
    root, local = Path(ns["PILOT_ROOT"]), Path(ns["LOCAL_CACHE"])
    if (root.name != "stage2_compact_final_v26" or root.is_symlink() or local.is_symlink()
            or any(p.resolve() == Path(p.resolve().anchor) for p in (root, local))
            or any(a.resolve() == b.resolve() or a.resolve() in b.resolve().parents or b.resolve() in a.resolve().parents
                for a, b in ((source, root), (source, local), (root, local)))):
        raise ValueError("Use separate scoped source, result and local-cache directories")
    hashes, images = ns["evaluation_hashes"], ns["evaluation_images"]
    conditions = compact_conditions_v26()
    if len(conditions) != 6:raise RuntimeError("Keep the predeclared six compact physical conditions")
    binding = {"version": COMPACT_FINAL_RUNNER_VERSION_V26, "code_sha256": code_sha256,
        "input_lock_sha256": _pilot_json_sha_v12(lock),
        "source_summary_json_sha256": _pilot_json_sha_v12(source_summary),
        "model_checkpoint_sha256": cp["sha256"], "model_state_sha256": cp["model_state_sha256"],
        "configuration_sha256": cp["configuration_sha256"], "primary_model_updates": 1375,
        "selection_manifest_sha256": _pilot_json_sha_v12(ns["evaluation_manifest"]),
        "condition_grid": conditions, "channel": ns["link"].metadata(),
        "metrics": {"primary": "alex v0.1", "secondary": "vgg v0.1"},
        "images": len(hashes), "methods": list(_COMPACT_METHODS_V26),
        "physical_image_method_evaluations": len(conditions)*len(hashes)*3,
        "clean_image_method_evaluations": len(hashes)*2,
        "rate_control": RATE_CONTROL_VERSION, "target_source_bytes_per_image": COMPACT_FINAL_RATE_BYTES_V26,
        "native_qp_search": "binary_checked", "rate_selection_seed": COMPACT_FINAL_RATE_SEED_V26,
        "channel_seed": COMPACT_FINAL_CHANNEL_SEED_V26,
        "ldpc": {"version": "seeded_systematic_RA_replacement_not_author_matrix", "n": 2048, "k": 1536,
            "max_iterations": 50, "normalization": .8, "seed": 20261028, "llr_clip": 30},
        "no_training_or_model_selection": True, "no_exact_cross_method_waveform_noise_pairing_claim": True}
    root.mkdir(parents=True, exist_ok=True);local.mkdir(parents=True, exist_ok=True)
    branch = _pilot_child_v12(root, "frozen_"+_pilot_json_sha_v12(binding)[:16]);branch.mkdir(exist_ok=True)
    cache = _pilot_child_v12(local, branch.name);cache.mkdir(parents=True, exist_ok=True)
    ns["COMPACT_FINAL_OUTPUT_DIR"] = branch
    caller_ns["COMPACT_FINAL_OUTPUT_DIR"] = branch
    _pilot_write_identity_v12(ns, branch, "protocol.json", binding)
    _pilot_write_identity_v12(ns, branch, "evaluation_manifest.json", ns["evaluation_manifest"])
    runtime = _extension_runtime_v22(ns["device"])
    history_path = _pilot_child_v12(branch, "execution_history.json")
    executions = json.loads(history_path.read_text(encoding="utf-8")) if history_path.is_file() else []
    if not executions or executions[-1]["runtime"] != runtime:
        executions.append({"runtime": runtime, "matches_source_runtime": runtime == lock["source_runtime"]})
        _pilot_save_json_v12(ns, branch, "execution_history.json", executions)
    summary_path = _pilot_child_v12(branch, "compact_final_summary.json")
    if summary_path.is_file():
        completed = json.loads(summary_path.read_text(encoding="utf-8"))
        if completed.get("binding") != binding:raise RuntimeError("Completed confirmation protocol differs")
        if _pilot_verify_inventory_v12(branch, completed.get("result_artifacts")):
            print("Verified complete compact confirmation; no model/codec/metrics/evaluation repeated", flush=True)
            print(summary_path, flush=True);return completed
    started = time.monotonic()
    codec, build = ns["create_compact_codec_v26"](_pilot_child_v12(cache, "codec"))
    build_path = _pilot_save_json_v12(ns, branch, "codec_build.json", build)
    # Selection is committed BEFORE metric creation, model loading or channel evaluation.
    streams, rate_manifest, artifacts = _compact_streams_v26(ns, branch, codec, build, hashes, images, binding)
    mapping_sha = _pilot_json_sha_v12(rate_manifest)
    metrics = ns["create_compact_metrics_v26"](ns["device"])
    if set(metrics) != {"alex", "vgg"}:raise RuntimeError("Report both predeclared metric backbones")
    payload = _extension_load_checkpoint_file_v22(_pilot_child_v12(source, cp["file"]), cp,
        _pilot_child_v12(cache, "primary_model"))
    if (payload.get("configuration") != source_configuration
            or payload.get("configuration_sha256") != cp["configuration_sha256"]
            or payload.get("progress") != source_summary["endpoint"]["progress"]
            or _pilot_state_sha_v12(payload["state_dict"]) != cp["model_state_sha256"]):
        raise RuntimeError("Actual primary checkpoint model/configuration/progress differs")
    model = ns["FSO_VQVAE"]().to(ns["device"]).eval().requires_grad_(False)
    model.load_state_dict(payload["state_dict"], strict=True);del payload;gc.collect()
    indices = []
    with torch.inference_mode():
        for image in images.split(1):
            _, _, index = model.encode(image.to(ns["device"]))
            if index.dtype != torch.int64 or tuple(index.shape) != (1, 32, 32):
                raise RuntimeError("Primary model latent shape differs")
            indices.append(index.detach().cpu())
    indices = torch.cat(indices)
    ldpc = ns["SystematicRALDPC"](max_iterations=50, normalization=.8, seed=20261028, llr_clip=30).to(ns["device"])
    results = {}
    work = [("fso_sc", None), ("bpg", None)]+[(method, condition) for condition in conditions for method in _COMPACT_METHODS_V26]
    for method, condition in work:
        label = "clean" if condition is None else condition["id"]
        report, files = _compact_case_v26(ns, branch, binding, model, metrics, ns["link"], codec, ldpc,
            streams, hashes, images, indices, method, condition, mapping_sha)
        results[label+"/"+method] = report;artifacts += files
    if _pilot_state_sha_v12(model.state_dict()) != cp["model_state_sha256"]:
        raise RuntimeError("Evaluation changed the primary model tensors")
    artifacts += _pilot_inventory_v12(branch, [build_path,
        _pilot_child_v12(branch, "protocol.json"), _pilot_child_v12(branch, "evaluation_manifest.json")])
    # Deduplicate candidate artifacts, which may be absent on a source-cache hit.
    artifacts = list({r["file"]: r for r in artifacts}.values())
    if any(r["file"].endswith(".pth") for r in artifacts):raise RuntimeError("Do not copy model/Adam checkpoints to result output")
    archive = _pilot_child_v12(branch, "compact_final_examples.zip")
    buffer = io.BytesIO()
    with zipfile.ZipFile(buffer, "w", compression=zipfile.ZIP_DEFLATED) as z:
        for row in sorted(artifacts, key=lambda r: r["file"]):
            if row["file"].endswith((".png", "/metrics.json")) or row["file"] in {"protocol.json", "evaluation_manifest.json"}:
                z.write(_pilot_child_v12(branch, row["file"]), arcname=row["file"])
    _compact_atomic_bytes_v26(branch, archive.name, buffer.getvalue())
    with zipfile.ZipFile(archive) as z:
        if z.testzip() is not None:raise RuntimeError("Compact final example archive CRC differs")
    artifacts += _pilot_inventory_v12(branch, [archive])
    summary = {"version": COMPACT_FINAL_RUNNER_VERSION_V26, "status": "compact_confirmation_completed",
        "binding": binding, "evaluation_manifest": copy.deepcopy(ns["evaluation_manifest"]),
        "source_checkpoint": {"source_directory": str(source), **copy.deepcopy(cp)},
        "source_model_preselected_before_confirmation": True, "primary_model_updates": 1375,
        "source_checkpoint_or_configuration_modified": False, "training_updates": 0,
        "optimizer_created": False, "checkpoint_weights_copied_to_result_directory": False,
        "baseline_development_scores_reused": False, "codec_build": build,
        "rate_matching": rate_manifest["rate_matching"], "rate_mapping_sha256": mapping_sha,
        "results": results, "condition_method_rows": len(results),
        "physical_image_method_evaluations": len(conditions)*len(hashes)*3,
        "clean_image_method_evaluations": len(hashes)*2,
        "cross_method_seed_shared": True, "cross_method_exact_waveform_noise_pairing_claimed": False,
        "execution_history": executions, "cross_runtime_bitwise_exactness_claimed": False,
        "result_artifacts": sorted(artifacts, key=lambda r: r["file"]), "examples_archive": archive.name,
        "elapsed_seconds": time.monotonic()-started,
        "paper_numerical_reproduction_claimed": False, "automatic_model_reselection_or_channel_fitting": False,
        "limitations": ["Fixed primary model and compact grid, not a full paper-figure or 3dB-gain replication",
            "Same corpus and undisclosed author images/LPIPS backbone/receiver settings limit direct comparison",
            "Image-disjoint from recorded prior use; not geographically independent or a new dataset",
            "One channel realization per image and condition; image bootstrap does not cover all channel/model uncertainty",
            "Native BPG average source bytes are matched; LDPC adds coded bits and padding, so channel-use budgets differ",
            "Seeded RA matrix and DSP/noise assumptions are independent implementations, not the authors' original settings",
            "BPG decode failure marker1 and separately measured gray fallback LPIPS must remain distinct",
            "Condition work is committed in25-image blocks; interruption can repeat at most25 uncommitted image evaluations"]}
    _pilot_save_json_v12(ns, branch, summary_path.name, summary)
    del model, ldpc, metrics, codec, indices;gc.collect()
    if torch.cuda.is_available():torch.cuda.empty_cache()
    print("Fixed primary compact confirmation complete; no automatic retraining or reselection", flush=True)
    print("Send:", summary_path, archive, sep="\n", flush=True)
    return summary


In [ ]:
def create_compact_metrics_v26(device):
    import lpips
    return {name:lpips.LPIPS(net=name,version="0.1").to(device).eval().requires_grad_(False)
            for name in ("alex","vgg")}

def create_compact_codec_v26(local):
    encoder,decoder,provenance=install_official_bpg(local)
    codec=BPGCodec(encoder,decoder,expected_hw=(256,256),timeout=20,speed=4)
    # Real native codec smoke before full dataset compression or model loading.
    smoke=Image.new("RGB",(256,256),(24,112,208))
    stream=codec.encode(smoke,30);decoded=codec.decode(stream)
    if (not decoded.ok or np.asarray(decoded.image).shape!=(256,256,3)
            or np.asarray(decoded.image).dtype!=np.uint8):
        raise RuntimeError("Official native BPG clean round-trip failed; stop rather than count infrastructure failure as bad-channel output")
    return codec,provenance


In [ ]:
"""Printed Fig4-8 sample grids, with shared physical cases evaluated only once."""
import hashlib
import json


def _sweep_hash_v27(value):
    return hashlib.sha256(json.dumps(value,sort_keys=True,separators=(',',':'),allow_nan=False).encode()).hexdigest()


def paper_sweep_protocol_v27():
    cases={};panels=[]
    def condition(**changes):
        value=dict(modulation='QPSK',launch_power_dbm=36.,zenith_angle_deg=0.,pointing_error_urad=20.,
            scattered_power_ratio=.95,background_light_radiance_w_m2_sr_um=1e-4,weather='clear')
        value.update(changes)
        return {'id':_sweep_hash_v27(value)[:16],'label':str(value),'group':'paper_figures_v27','channel_kwargs':value}
    def add(panel,label,points,method='fso_sc',receivers=1,combining='MRC'):
        ids=[]
        for x,c in points:
            entry={'method':method,'condition':c,'receivers':receivers,'combining':combining}
            key=_sweep_hash_v27(entry)[:20]
            cases[key]={'id':key,**entry};ids.append({'x':x,'case_id':key})
        panel['series'].append({'label':label,'method':method,'receivers':receivers,'combining':combining,'points':ids})
    for letter,power in zip('abc',(30.,33.,36.)):
        panel={'figure':'fig4','panel':letter,'title':f'P={power:g} dBm','x_axis':'zenith_angle_deg','series':[]}
        for mod in ('QPSK','8QAM','16QAM'):
            points=[(z,condition(modulation=mod,launch_power_dbm=power,zenith_angle_deg=float(z))) for z in range(0,81,10)]
            for method in ('fso_sc','bpg','bpg_ldpc'):add(panel,mod+' / '+method,points,method)
        panels.append(panel)
    for letter,mod in zip('abc',('QPSK','8QAM','16QAM')):
        panel={'figure':'fig6','panel':letter,'title':mod+' / pointing jitter','x_axis':'pointing_error_urad','series':[]}
        for z in (0.,30.,60.):
            points=[(p,condition(modulation=mod,zenith_angle_deg=z,pointing_error_urad=float(p))) for p in range(18,23)]
            for method in ('fso_sc','bpg_ldpc'):add(panel,f'z={z:g} / '+method,points,method)
        panels.append(panel)
    for letter,mod in zip('def',('QPSK','8QAM','16QAM')):
        panel={'figure':'fig6','panel':letter,'title':mod+' / background radiance','x_axis':'launch_power_dbm','series':[]}
        for radiance in (1e-4,1e-2,1.):
            points=[(p,condition(modulation=mod,launch_power_dbm=float(p),pointing_error_urad=19.,
                background_light_radiance_w_m2_sr_um=radiance)) for p in range(28,37)]
            for method in ('fso_sc','bpg_ldpc'):add(panel,f'BGL={radiance:g} / '+method,points,method)
        panels.append(panel)
    panel={'figure':'fig7','panel':'a','title':'Weather / QPSK / z=0 / pointing15urad','x_axis':'launch_power_dbm','series':[]}
    for weather in ('dust','rain','snow','fog'):
        points=[(p,condition(launch_power_dbm=float(p),pointing_error_urad=15.,weather=weather)) for p in range(28,37)]
        for method in ('fso_sc','bpg_ldpc'):add(panel,weather+' / '+method,points,method)
    panels.append(panel)
    for letter,rho,fec in (('a',.95,False),('b',.95,True),('c',.75,True),('d',.596,True)):
        panel={'figure':'fig8','panel':letter,'title':f'rho={rho:g} / LDPC={fec}','x_axis':'launch_power_dbm','series':[]}
        points=[(p,condition(modulation='16QAM',launch_power_dbm=float(p),pointing_error_urad=19.,scattered_power_ratio=rho)) for p in range(28,37)]
        add(panel,'FSO-SC 1Rx',points)
        method='bpg_ldpc' if fec else 'bpg'
        add(panel,method+' 1Rx',points,method)
        for count,combining in ((2,'EGC'),(4,'EGC'),(2,'MRC'),(4,'MRC')):
            add(panel,f'{method} {count}Rx-{combining}',points,method,count,combining)
        panels.append(panel)
    fig5=[]
    for power in (33.,36.):
        for mod in ('8QAM','16QAM'):
            for method in ('fso_sc','bpg_ldpc'):
                entry={'method':method,'condition':condition(modulation=mod,launch_power_dbm=power,zenith_angle_deg=30.),'receivers':1,'combining':'MRC'}
                key=_sweep_hash_v27(entry)[:20]
                if key not in cases:raise AssertionError('Fig5 must reuse Fig4')
                fig5.append({'power':power,'modulation':mod,'method':method,'case_id':key})
    # Prioritize complete power curves, then all remaining paper panels. No score-dependent ordering.
    priorities=[]
    for panel in sorted(panels,key=lambda p:({'fig7':0,'fig4':2,'fig6':3,'fig8':4}[p['figure']],p['panel'])):
        for series in panel['series']:
            priorities.extend(p['case_id'] for p in series['points'])
    order=list(dict.fromkeys(priorities))
    result={'version':'paper_fig4_to8_full_grid_v27_1','cases':[cases[k] for k in order],'panels':panels,'fig5_examples':fig5,
        'raw_paper_curve_points':sum(len(s['points']) for p in panels for s in p['series']),
        'unique_condition_method_cases':len(cases),'images':200,'channel_realizations_per_image':1,
        'physical_image_method_evaluations':len(cases)*200,'clean_image_method_evaluations':400,
        'power_grid_dbm':list(range(28,37)),
        'fig8d_ambiguity':'LDPC enabled according to plotted legend and discussion; caption says withoutLDPC',
        'diversity_assumption':'Independent branch fading/noise, shared transmitter ASE; ideal relative branch phase and magnitude CSI, common carrier still estimated by same DSP',
        'hardware_fig9_to12_reproduced':False,'model_preselected':True,'channel_fitted':False,
        'monotonic_quality_enforced':False,'image_count_from_paper':False}
    result['protocol_sha256']=_sweep_hash_v27(result)
    return result


def paper_sweep_seed_v27(condition,image_sha256,base_seed=20261107):
    """Common random numbers across power, not common method-dependent payloads."""
    family=None if condition is None else {k:v for k,v in condition['channel_kwargs'].items() if k!='launch_power_dbm'}
    return int.from_bytes(hashlib.sha256(json.dumps([base_seed,family,image_sha256],sort_keys=True,separators=(',',':')).encode()).digest()[:8],'big')%(2**63-1)


In [ ]:
"""Frozen-model compact evaluation with the current common physical receiver.

No optimizer, fitting, model selection, CRC oracle or failure-frame omission.
BPG means the actual native byte stream; LDPC consumes soft QAM LLRs.  The
existing v12 real block-magnitude and noise-PSD CSI assumptions are retained.
"""
import hashlib
import json
import math

import numpy as np
import torch

from dataclasses import replace

_SWEEP_ALLOWED_CONDITIONS_V27={json.dumps(c['condition'],sort_keys=True) for c in paper_sweep_protocol_v27()['cases']}


COMPACT_FINAL_CORE_VERSION_V27 = "paper_fig4_to8_evaluation_core_v27_1"
_COMPACT_METHODS_V27 = ("fso_sc", "bpg", "bpg_ldpc")
_COMPACT_PROFILE_V27 = "coherent_receiver_extension_v6"
_COMPACT_COUNTERS_V27 = ("bit_errors", "valid_bits", "pre_fec_bit_errors",
    "pre_fec_valid_bits", "index_errors", "indices", "symbol_errors", "valid_symbols",
    "physical_calls", "source_bits", "coded_bits", "transmitted_bits", "channel_symbols",
    "qam_padding_bits", "information_padding_bits", "ldpc_blocks", "ldpc_nonconverged_blocks")


def compact_conditions_v27():
    return list({c['condition']['id']:c['condition'] for c in paper_sweep_protocol_v27()['cases']}.values())


def _compact_condition_v27(condition):
    if condition is None:return None
    if json.dumps(condition,sort_keys=True) not in _SWEEP_ALLOWED_CONDITIONS_V27:
        raise ValueError("Keep the frozen full-paper condition grid; no post-result channel edits")
    return condition["channel_kwargs"]


def compact_seed_v27(condition, image_sha256, base_seed=20261107):
    if (not isinstance(image_sha256,str) or len(image_sha256)!=64
            or any(c not in "0123456789abcdef" for c in image_sha256)
            or type(base_seed) is not int or not 0<=base_seed<2**63):
        raise ValueError("Use one explicit image SHA256 and nonnegative integer channel seed")
    _compact_condition_v27(condition)
    return paper_sweep_seed_v27(condition,image_sha256,base_seed)


def _compact_link_v27(link):
    if link is None or link.metadata().get("profile") != _COMPACT_PROFILE_V27:
        raise ValueError("Keep the frozen coherent receiver extension for every evaluated method")
    if not callable(link.ns.get("recover_qam_carrier_v3")):
        raise ValueError("The current v3 receiver must serve both semantic and byte-stream methods")


@torch.inference_mode()
def transport_bytes_v27(payload, link, condition, seed, device="cpu", ldpc=None):
    """One real BPG transport using exactly the link's current physical/DSP code.

    Equal seeds do not assert identical waveform noise across unequal payload
    lengths.  Neither the source bytes nor true phase/CFO enter the receiver.
    """
    _compact_link_v27(link);kwargs = _compact_condition_v27(condition)
    if kwargs is None:raise ValueError("Use direct codec reconstruction for clean BPG")
    if not isinstance(payload,bytes) or not payload:raise ValueError("Transmit complete native BPG bytes")
    if type(seed) is not int or not 0<=seed<2**63:raise ValueError("Invalid private channel seed")
    device=torch.device(device);ns=link.ns;physical=link.physical
    bits=torch.from_numpy(np.unpackbits(np.frombuffer(payload,dtype=np.uint8)).copy()).to(device)
    source_count=bits.numel();blocks=math.ceil(source_count/ldpc.k) if ldpc is not None else 0
    information_padding=blocks*ldpc.k-source_count if ldpc is not None else 0
    if ldpc is None:transmitted=bits.reshape(1,-1)
    else:
        padded=torch.cat((bits,torch.zeros(information_padding,dtype=torch.uint8,device=device)))
        transmitted=ldpc.encode_bits(padded.reshape(blocks,ldpc.k)).reshape(1,-1)
    modulation=kwargs["modulation"];_,_,bps=ns["qam_constellation"](modulation,device)
    coded_count=transmitted.numel();padding=(-coded_count)%bps
    grouped=torch.cat((transmitted,torch.zeros((1,padding),dtype=torch.uint8,device=device)),dim=-1).reshape(1,-1,bps)
    symbols=ns["qam_modulate"](grouped,modulation)
    generator=torch.Generator(device=device).manual_seed(seed)
    observation,mode=_rebuild_observe_v12(physical,symbols,generator,
        {k:v for k,v in kwargs.items() if k!="modulation"})
    if (observation.is_equalized or observation.channel_gain is None
            or observation.symbols.shape!=symbols.shape
            or observation.channel_gain.shape!=symbols.shape
            or not bool(torch.isfinite(observation.symbols).all())
            or not bool(torch.isfinite(observation.channel_gain).all())):
        raise FloatingPointError("Invalid raw physical observation; no frame may be omitted")
    magnitude=observation.channel_gain.abs().mean(-1,keepdim=True)
    if not bool((magnitude>0).all()):raise FloatingPointError("Nonpositive block magnitude")
    recovered,diagnostics=ns["recover_qam_carrier_v3"](observation.symbols,modulation,
        link.config.symbol_rate_baud,known_block_magnitude=magnitude)
    if recovered.shape!=symbols.shape or not bool(torch.isfinite(recovered).all()):
        raise FloatingPointError("Nonfinite/wrong-shaped recovered byte-stream samples")
    hard_groups=ns["qam_demodulate"](recovered,modulation)
    if hard_groups.shape!=grouped.shape:raise ValueError("Receiver bit grouping changed")
    hard=hard_groups.reshape(1,-1)[:,:coded_count]
    budget=observation.diagnostics["budget"]
    receiver_noise=budget["background_noise_w"] if link.config.include_background_noise else 0.
    receiver_noise+=budget["receiver_shot_noise_w"] if link.config.include_shot_noise else 0.
    n0=receiver_noise/magnitude.to(torch.float64).square().clamp_min(1e-300)
    if link.config.include_ase_noise:n0+=budget["ase_noise_w"]/budget["launch_power_w"]
    statistics=None
    if ldpc is None:decoded_bits=hard.reshape(-1)[:source_count]
    else:
        soft=soft_qam_llr(recovered,modulation,n0,ns,clip=ldpc.llr_clip)
        information,statistics=ldpc.decode_llr(soft.reshape(-1)[:coded_count].reshape(blocks,ldpc.n))
        decoded_bits=information.reshape(-1)[:source_count]
    energy=float(symbols.abs().square().mean());launch=10**((kwargs["launch_power_dbm"]-30)/10)
    result={"received_bytes":np.packbits(decoded_bits.cpu().numpy()).tobytes(),"seed":seed,
        "bit_errors":int((decoded_bits!=bits).sum()),"valid_bits":source_count,
        "pre_fec_bit_errors":int((hard!=transmitted).sum()),"pre_fec_valid_bits":coded_count,
        "index_errors":0,"indices":0,"symbol_errors":int((hard_groups!=grouped).any(-1).sum()),
        "valid_symbols":grouped.shape[1],"physical_calls":1,
        "source_bytes":len(payload),"source_bits":source_count,"coded_bits":coded_count,
        "transmitted_bits":coded_count+padding,"channel_symbols":symbols.numel(),
        "qam_padding_bits":padding,"padding_bits":padding,
        "information_padding_bits":information_padding,"ldpc_blocks":blocks,
        "ldpc_nonconverged_blocks":0,"ldpc_iterations_mean":0.,
        "noise_variance_equalized":float(n0.mean()),"receiver_diagnostics":diagnostics,
        "channel_gain_sha256":_rebuild_sha_v12(observation.channel_gain),
        "channel_rng_state_sha256":_rebuild_sha_v12(generator.get_state()),
        "transmitted_symbols_mean_power":energy,"nominal_launch_power_w":launch,
        "actual_launch_power_w":launch*energy,"deterministic_physical":mode,
        "receiver":"same link.ns recover_qam_carrier_v3 as semantic transmission",
        "same_waveform_noise_across_methods_claimed":False,
        "frame_integrity_policy":"Known file length; actual native decode/header validation; no truth BER or parity-based erasure"}
    if statistics is not None:
        result["ldpc_nonconverged_blocks"]=int((~statistics["converged"]).sum())
        result["ldpc_iterations_mean"]=float(statistics["iterations"].float().mean())
    return result


def _compact_image_v27(image):
    if (not isinstance(image,torch.Tensor) or image.dtype!=torch.float32
            or tuple(image.shape)!=(1,3,256,256) or not bool(torch.isfinite(image).all())
            or bool((image.abs()>1.000001).any())):
        raise ValueError("Expected finite preprocessed FP32 RGB[1,3,256,256] in[-1,1]")


def _compact_clean_transport_v27(payload,seed):
    return {"seed":seed,"received_bytes":payload,"source_bytes":len(payload),
        **{k:0 for k in _COMPACT_COUNTERS_V27},"source_bits":len(payload)*8,
        "valid_bits":len(payload)*8,"padding_bits":0,"ldpc_iterations_mean":0.,
        "receiver_diagnostics":{},"nominal_launch_power_w":None,"actual_launch_power_w":None,
        "transmitted_symbols_mean_power":None,"channel_gain_sha256":None,
        "channel_rng_state_sha256":None,"deterministic_physical":"clean_identity"}


@torch.inference_mode()
def _evaluate_sweep_image_base_v27(model,image,image_sha256,metrics,link,condition,payload,
        codec,ldpc,method,*,device="cpu",base_seed=20261107,cached_indices=None):
    """One retained frame, separately usable by the runner's chunk journal."""
    if method not in _COMPACT_METHODS_V27:raise ValueError("Unknown frozen evaluation method")
    _compact_image_v27(image);_compact_link_v27(link);kwargs=_compact_condition_v27(condition)
    if set(metrics)!={"alex","vgg"}:raise ValueError("Report the fixed primary Alex and secondary VGG metrics")
    seed=compact_seed_v27(condition,image_sha256,base_seed);device=torch.device(device)
    target=image.to(device);decode_failed=False;decode_status="semantic_reconstruction";decode_error=None
    if method=="fso_sc":
        if model is None:raise ValueError("The fixed semantic model is required")
        previous=model.training;model.eval()
        try:
            if cached_indices is None:_,_,indices=model.encode(target)
            else:indices=cached_indices.to(device)
            if (indices.dtype!=torch.int64 or tuple(indices.shape)!=(1,32,32)
                    or bool(((indices<0)|(indices>=16384)).any())):
                raise ValueError("Semantic cache must be current model int64[1,32,32] indices")
            received,rows=link.transmit(indices,[kwargs],[seed],device=device)
            prediction=model.decode_indices(received,received.shape)
            result=dict(rows[0]);padding=result.get("padding_bits",0)
            physical=result["physical_calls"]==1
            result.update(source_bytes=1792,source_bits=14336,coded_bits=14336 if physical else 0,
                transmitted_bits=14336+padding if physical else 0,
                channel_symbols=result["valid_symbols"] if physical else 0,qam_padding_bits=padding,
                information_padding_bits=0,ldpc_blocks=0,ldpc_nonconverged_blocks=0,
                ldpc_iterations_mean=0.,pre_fec_bit_errors=result["bit_errors"] if physical else 0,
                pre_fec_valid_bits=14336 if physical else 0,
                same_waveform_noise_across_methods_claimed=False)
        finally:model.train(previous)
    else:
        if codec is None or not isinstance(payload,bytes) or not payload:
            raise ValueError("Native BPG codec and selected complete byte stream are required")
        if method=="bpg_ldpc" and (ldpc is None or kwargs is None):
            raise ValueError("BPG+LDPC requires the fixed code and physical condition; clean duplicate is not evaluated")
        result=(_compact_clean_transport_v27(payload,seed) if kwargs is None else
            transport_bytes_v27(payload,link,condition,seed,device,ldpc if method=="bpg_ldpc" else None))
        decoded=codec.decode(result.pop("received_bytes"))
        decode_failed=not decoded.ok;decode_status=decoded.status
        decode_error=decoded.error if decode_failed else None
        if decode_failed:raster=np.full((256,256,3),128,dtype=np.uint8)
        else:
            raster=np.asarray(decoded.image,dtype=np.uint8)
            if raster.shape!=(256,256,3):raise ValueError("Successful BPG decode returned the wrong RGB shape")
        prediction=torch.from_numpy(raster.copy()).permute(2,0,1).unsqueeze(0).to(device,dtype=torch.float32)/127.5-1.
    if prediction.shape!=target.shape or not bool(torch.isfinite(prediction).all()):
        raise FloatingPointError("Invalid reconstructed image; no frame may be discarded")
    prediction=prediction.clamp(-1,1);scores={}
    for name,metric in metrics.items():
        score=metric(prediction,target).reshape(-1)
        if score.numel()!=1 or not bool(torch.isfinite(score).all()):
            raise FloatingPointError("Nonfinite LPIPS; no frame may be discarded")
        scores[name]=float(score[0])
    record={**result,"image_sha256":image_sha256,"image_name":image_sha256,"repeat":0,
        "method":method,"profile":_COMPACT_PROFILE_V27,"condition":condition,
        "lpips_alex":1. if decode_failed else scores["alex"],
        "lpips_vgg":1. if decode_failed else scores["vgg"],
        "rendered_lpips_alex":scores["alex"],"rendered_lpips_vgg":scores["vgg"],
        "lpips":1. if decode_failed else scores["alex"],"rendered_lpips":scores["alex"],
        "mse":float((prediction-target).square().mean()),"decode_failed":decode_failed,
        "decode_status":decode_status,"decode_error":decode_error,
        "failure_marker_policy":"Native BPG decode/header failure -> marker1 in each LPIPS; actual gray LPIPS separately measured; no mathematical LPIPS upper bound"}
    record["payload_ber"]=record["bit_errors"]/record["valid_bits"]
    record["index_error_rate"]=record["index_errors"]/record["indices"] if record["indices"] else None
    record["symbol_error_rate"]=record["symbol_errors"]/record["valid_symbols"] if record["valid_symbols"] else None
    record["source_bpp"]=record["source_bits"]/65536
    record["transmitted_bpp"]=record["transmitted_bits"]/65536
    json.dumps(record,allow_nan=False)
    return {"record":record,"prediction":prediction.detach().cpu().clone()}


def summarize_compact_records_v27(records,method,condition,*,base_seed=20261107,bootstrap_samples=2000):
    """Image-only bootstrap; does not cover channel/model/setting uncertainty."""
    _compact_condition_v27(condition)
    if (method not in _COMPACT_METHODS_V27 or not isinstance(records,list) or not records
            or type(bootstrap_samples) is not int or bootstrap_samples<1
            or len({r["image_sha256"] for r in records})!=len(records)
            or any(r["method"]!=method or r["condition"]!=condition for r in records)):
        raise ValueError("Case must retain each declared image exactly once under one frozen method/condition")
    n=len(records);rng=np.random.default_rng(base_seed)
    summary={}
    for key in ("lpips_alex","lpips_vgg","rendered_lpips_alex","rendered_lpips_vgg","mse"):
        values=np.asarray([r[key] for r in records],dtype=np.float64)
        if not bool(np.isfinite(values).all()):raise FloatingPointError("Nonfinite case scores")
        summary[key]=float(values.mean())
        summary[key+"_ci95"]=np.quantile(values[rng.integers(0,n,(bootstrap_samples,n))].mean(1),[.025,.975]).tolist()
    counts={key:sum(r[key] for r in records) for key in _COMPACT_COUNTERS_V27}
    summary.update(mean=summary["lpips_alex"],lpips_mean=summary["lpips_alex"],ci95=summary["lpips_alex_ci95"],
        rendered_lpips_mean=summary["rendered_lpips_alex"],mse_mean=summary["mse"],
        counts=counts,independent_images=n,unique_images=n,image_channel_realizations=n,
        bootstrap_samples=bootstrap_samples,ci_unit="image only; one channel realization per image",
        payload_ber=counts["bit_errors"]/counts["valid_bits"],
        pre_fec_ber=counts["pre_fec_bit_errors"]/counts["pre_fec_valid_bits"] if counts["pre_fec_valid_bits"] else None,
        index_error_rate=counts["index_errors"]/counts["indices"] if counts["indices"] else None,
        symbol_error_rate=counts["symbol_errors"]/counts["valid_symbols"] if counts["valid_symbols"] else None,
        decode_failures=sum(r["decode_failed"] for r in records),
        decode_failure_rate=sum(r["decode_failed"] for r in records)/n,
        source_bpp_mean=float(np.mean([r["source_bpp"] for r in records])),
        transmitted_bpp_mean=float(np.mean([r["transmitted_bpp"] for r in records])),
        channel_symbols_mean=counts["channel_symbols"]/n,
        primary_metric="Alex LPIPS v0.1 with native-decode failure marker1",
        secondary_metric="VGG LPIPS v0.1 with separately reported rendered fallback scores",
        rate_comparison="matched mean source rate, NOT equal total channel-use budget",
        same_waveform_noise_across_methods_claimed=False)
    json.dumps(summary,allow_nan=False)
    return summary


@torch.inference_mode()
def evaluate_compact_case_v27(model,images,hashes,metrics,link,condition,streams,codec,ldpc,method,
        *,device="cpu",base_seed=20261107,example_count=3,cached_indices=None,bootstrap_samples=2000,progress=None):
    """Convenience wrapper; the runner may instead journal per-image records."""
    if (len(images)!=len(hashes) or not hashes or len(set(hashes))!=len(hashes)
            or type(example_count) is not int or example_count<0):
        raise ValueError("Invalid compact case image identities")
    if method!="fso_sc" and (not isinstance(streams,dict) or set(streams)!=set(hashes)):
        raise ValueError("Every byte-stream case must cover exactly its selected images")
    records=[];examples={}
    for i,digest in enumerate(hashes):
        cached=None
        if cached_indices is not None:
            cached=cached_indices[digest] if isinstance(cached_indices,dict) else cached_indices[i:i+1]
        packet=_evaluate_sweep_image_base_v27(model,images[i:i+1],digest,metrics,link,condition,
            None if method=="fso_sc" else streams[digest],codec,ldpc,method,
            device=device,base_seed=base_seed,cached_indices=cached)
        records.append(packet["record"])
        if i<example_count:examples[digest]=packet["prediction"]
        if progress is not None:progress(i+1,len(hashes))
    return {"report":{"version":COMPACT_FINAL_CORE_VERSION_V27,"method":method,"condition":condition,
        "images":len(hashes),"records":records,"summary":summarize_compact_records_v27(records,method,condition,
            base_seed=base_seed,bootstrap_samples=bootstrap_samples)},"examples":examples}


class RelativeBranchCombinerV27:
    """Ideal relative branch CSI; common phase/CFO stays for the same DSP.

    Independent atmospheric fades/Rx noise and common transmitter ASE. Relative
    branch estimation and spatial correlations are undisclosed implementation
    assumptions, not an experimental-receiver reproduction.
    """
    def __init__(self,namespace,config):
        self.ns=namespace;self.config=config;self.base=namespace['PaperMalagaFSOChannel'](config)

    @torch.no_grad()
    def __call__(self,symbols,generator=None,**condition):
        c=self.config;ns=self.ns
        budget=self.base._budget(**condition);batch,count=symbols.shape;rx=c.num_receivers
        real=symbols.real.dtype;device=symbols.device
        fade=self.base.sample_malaga_power((batch,rx,1),budget,device,real,generator)
        amplitude=10**(-budget['total_attenuation_db']/20)*fade.sqrt()
        index=torch.arange(count,device=device,dtype=real)
        phase=torch.zeros((batch,rx,count),device=device,dtype=real)
        if c.include_doppler:phase+=((2*math.pi*budget['residual_doppler_hz']/c.symbol_rate_baud)*index).view(1,1,-1)
        if c.include_phase_noise and c.lo_linewidth_hz>0:
            phase+=(torch.randn(phase.shape,device=device,dtype=real,generator=generator)*
                math.sqrt(2*math.pi*c.lo_linewidth_hz/c.symbol_rate_baud)).cumsum(-1)
        propagation=amplitude.to(symbols.dtype)*torch.exp(1j*phase).to(symbols.dtype)
        gain=math.sqrt(budget['launch_power_w'])*propagation
        observed=gain*symbols[:,None,:]
        def noise(shape):
            return torch.complex(torch.randn(shape,device=device,dtype=real,generator=generator),
                torch.randn(shape,device=device,dtype=real,generator=generator))/math.sqrt(2)
        receiver_noise=0.
        for flag,key in ((c.include_background_noise,'background_noise_w'),(c.include_shot_noise,'receiver_shot_noise_w')):
            if flag and budget[key]>0:
                observed+=noise(observed.shape)*math.sqrt(budget[key]);receiver_noise+=budget[key]
        if c.include_ase_noise and budget['ase_noise_w']>0:
            observed+=propagation*noise((batch,1,count))*math.sqrt(budget['ase_noise_w'])
        magnitudes=gain.abs().clamp_min(torch.finfo(real).tiny)
        unit=gain/magnitudes;relative=unit[:,0:1,:]*unit.conj()
        if c.combining.upper()=='MRC':
            norm=magnitudes.square().sum(1).sqrt();weights=relative*magnitudes/norm[:,None,:]
        else:
            norm=magnitudes.sum(1)/math.sqrt(rx);weights=relative/math.sqrt(rx)
        combined=(weights*observed).sum(1);equivalent_gain=unit[:,0,:]*norm
        n0=receiver_noise/norm[:,:1].double().square()
        if c.include_ase_noise:n0+=budget['ase_noise_w']/budget['launch_power_w']
        snr=10*torch.log10(n0.reciprocal().clamp_min(1e-300))
        diagnostics={'budget':budget,'num_receivers':rx,'combining':c.combining.upper(),
            'mean_snr_db':float(snr.mean()),'relative_branch_CSI_is_ideal':True,
            'common_carrier_phase_or_CFO_removed_by_combiner':False,
            'independent_branch_fading_and_rx_noise':True,'common_transmitter_ASE':True}
        return ns['FSOChannelOutput'](symbols=combined,channel_gain=equivalent_gain,is_equalized=False,diagnostics=diagnostics)


class TracedPhysicalV27:
    def __init__(self,physical):self.physical=physical;self.last=None
    def __call__(self,*args,**kwargs):
        packet=self.physical(*args,**kwargs);self.last=packet.diagnostics
        return packet


class PaperSweepLinkV27:
    def __init__(self,base,receivers=1,combining='MRC'):
        if receivers not in (1,2,4) or combining not in ('MRC','EGC'):raise ValueError('Unknown paper diversity branch')
        self.ns=base.ns;self.profile=base.profile;self.base_seed=base.base_seed
        self.config=replace(base.config,num_receivers=receivers,combining=combining)
        self.delegate=self.ns['RebuildIndexLinkV12'](self.ns,base.profile,base_seed=base.base_seed)
        physical=self.delegate.physical if receivers==1 else RelativeBranchCombinerV27(self.ns,self.config)
        self.physical=TracedPhysicalV27(physical);self.delegate.physical=self.physical
        self.receivers=receivers;self.combining=combining;self.base=base
    def metadata(self):return self.base.metadata()
    def transmit(self,*args,**kwargs):
        if self.receivers!=1:raise ValueError('FSO-SC remains the single-receiver primary')
        return self.delegate.transmit(*args,**kwargs)


@torch.inference_mode()
def evaluate_paper_sweep_image_v27(model,image,digest,metrics,link,case,payload,codec,ldpc,*,device='cpu',cached_indices=None,clean_prediction=None):
    condition=case['condition'];method=case['method'];link.physical.last=None
    packet=_evaluate_sweep_image_base_v27(model,image,digest,metrics,link,condition,payload,codec,ldpc,method,
        device=device,cached_indices=cached_indices)
    row=packet['record'];row['case_id']=case['id'];row['receivers']=case.get('receivers',1);row['combining']=case.get('combining','MRC')
    row['ideal_relative_branch_CSI']=row['receivers']>1;row['common_carrier_phase_CFO_oracle']=False
    if condition is not None:
        diag=link.physical.last
        if not isinstance(diag,dict) or not math.isfinite(diag['mean_snr_db']):raise FloatingPointError('Missing finite physical SNR')
        row['unit_symbol_esn0_db']=diag['mean_snr_db']
        row['actual_symbol_esn0_db']=diag['mean_snr_db']+10*math.log10(row['transmitted_symbols_mean_power'])
        row['attenuation_db']=diag['budget']['total_attenuation_db'];row['equalized_noise_variance']=10**(-diag['mean_snr_db']/10)
        row['noise_budget_w']={k:diag['budget'][k] for k in ('background_noise_w','ase_noise_w','receiver_shot_noise_w')}
    else:
        for key in ('unit_symbol_esn0_db','actual_symbol_esn0_db','attenuation_db','equalized_noise_variance','noise_budget_w'):row[key]=None
    if clean_prediction is not None:row['clean_relative_pixel_mse']=float((packet['prediction']-clean_prediction.cpu()).square().mean())
    json.dumps(row,allow_nan=False)
    return packet




In [ ]:
"""Frozen full Fig4-8 evaluation; no training, channel fitting or Drive API."""
import copy
import gc
import io
import json
import math
from pathlib import Path
import time
import zipfile

import numpy as np
from PIL import Image,ImageDraw
import torch


PAPER_SWEEP_VERSION_V27='frozen_paper_fig4_to8_all_points_v27_1'
PAPER_SWEEP_CHUNK_V27=25


def _sweep_case_v27(ns,branch,binding,case,model,metrics,link,codec,ldpc,streams,indices,clean_predictions,examples):
    hashes=ns['evaluation_hashes'];images=ns['evaluation_images'];root='cases/'+case['id']
    identity={'binding_sha256':_pilot_json_sha_v12(binding),'case':case,'chunk_size':PAPER_SWEEP_CHUNK_V27,'example':bool(examples)}
    marker=_pilot_child_v12(branch,root+'/completed.json')
    saved=json.loads(marker.read_text(encoding='utf-8')) if marker.is_file() else None
    if saved is not None and saved.get('identity')!=identity:raise RuntimeError('Case cache belongs to another protocol')
    if saved is not None and _pilot_verify_inventory_v12(branch,saved.get('artifacts')):
        return json.loads(_pilot_child_v12(branch,root+'/metrics.json').read_text(encoding='utf-8')),saved['artifacts']+_pilot_inventory_v12(branch,[marker]),True
    records=[];example_paths=[];started=time.monotonic()
    for first in range(0,len(hashes),PAPER_SWEEP_CHUNK_V27):
        last=min(first+PAPER_SWEEP_CHUNK_V27,len(hashes));relative=root+f'/blocks/{first:04d}_{last:04d}'
        bm=_pilot_child_v12(branch,relative+'.identity.json')
        bid={'case_identity_sha256':_pilot_json_sha_v12(identity),'first':first,'last':last}
        old=json.loads(bm.read_text(encoding='utf-8')) if bm.is_file() else None
        if old is not None and old.get('identity')!=bid:raise RuntimeError('Chunk identity changed')
        if old is not None and _pilot_verify_inventory_v12(branch,old.get('artifacts')):
            packet=json.loads(_pilot_child_v12(branch,relative+'.json').read_text(encoding='utf-8'))
            example_paths.extend(_pilot_child_v12(branch,r['file']) for r in old['artifacts'] if r['file'].endswith('.png'))
        else:
            rows=[];paths=[]
            for i in range(first,last):
                evaluate=ns.get('evaluate_paper_sweep_image_v27',evaluate_paper_sweep_image_v27)
                one=evaluate(model,images[i:i+1],hashes[i],metrics,link,case,
                    None if case['method']=='fso_sc' else streams[hashes[i]],codec,ldpc,
                    device=ns['device'],cached_indices=indices[i:i+1] if case['method']=='fso_sc' else None,
                    clean_prediction=clean_predictions[i:i+1] if case['method']=='fso_sc' else None)
                rows.append(one['record'])
                if examples and i==0:
                    path=_compact_atomic_bytes_v26(branch,root+'/example.png',_sweep_panel_v27(images[i:i+1],one['prediction'],case))
                    paths.append(path);example_paths.append(path)
                del one
            packet={'records':rows}
            paths.append(_pilot_save_json_v12(ns,branch,relative+'.json',packet))
            _pilot_save_json_v12(ns,branch,relative+'.identity.json',{'identity':bid,'artifacts':_pilot_inventory_v12(branch,paths)})
            print(f"Sweep {case['id']}/{case['method']}: {last}/{len(hashes)} committed",flush=True)
        if [r['image_sha256'] for r in packet['records']]!=hashes[first:last]:raise RuntimeError('Chunk membership/order changed')
        records.extend(packet['records'])
    summary=summarize_compact_records_v27(records,case['method'],case['condition'],bootstrap_samples=2000)
    for key in ('unit_symbol_esn0_db','actual_symbol_esn0_db','attenuation_db','equalized_noise_variance','actual_launch_power_w','clean_relative_pixel_mse'):
        values=[r[key] for r in records if r.get(key) is not None]
        summary[key+'_mean']=float(np.mean(values)) if values else None
    if case['condition'] is not None and case['method']=='fso_sc':
        summary['no_index_errors_fraction']=sum(r['index_errors']==0 for r in records)/len(records)
    report={'version':PAPER_SWEEP_VERSION_V27,'case':case,'images':len(hashes),'records':records,'summary':summary,
        'computed_seconds':time.monotonic()-started}
    path=_pilot_save_json_v12(ns,branch,root+'/metrics.json',report)
    files=_pilot_inventory_v12(branch,[path]+list(dict.fromkeys(example_paths)))
    _pilot_save_json_v12(ns,branch,root+'/completed.json',{'identity':identity,'artifacts':files})
    return report,files+_pilot_inventory_v12(branch,[marker]),False


def _sweep_panel_v27(target,prediction,case):
    a=Image.fromarray(_compact_rgb_v26(target));b=Image.fromarray(_compact_rgb_v26(prediction))
    panel=Image.new('RGB',(768,286),'white');panel.paste(a,(0,30));panel.paste(b,(256,30))
    # The same fixed central128px crop, not best-case region selection.
    panel.paste(b.crop((64,64,192,192)).resize((256,256),Image.Resampling.NEAREST),(512,30))
    d=ImageDraw.Draw(panel);d.text((4,8),'Original',fill='black');d.text((260,8),case['method']+' '+case['id'],fill='black')
    d.text((516,8),'Fixed center crop',fill='black');buffer=io.BytesIO();panel.save(buffer,format='PNG');return buffer.getvalue()


def run_paper_sweep_v27(ns,*,code_sha256):
    caller=ns;ns=dict(ns)
    ns['evaluation_manifest']=copy.deepcopy(ns['evaluation_manifest'])
    ns['evaluation_manifest'].get('copy_audit',{}).pop('local_copy_performed_this_call',None)
    source,source_report,config=_compact_source_v26(ns)
    protocol=paper_sweep_protocol_v27()
    root=Path(ns['PILOT_ROOT']);local=Path(ns['LOCAL_CACHE'])
    if root.name!='stage2_paper_sweep_v27' or root.is_symlink() or local.is_symlink():raise ValueError('Use separate v27 evaluation output')
    for a,b in ((root,source),(local,source),(root,local)):
        if a.resolve()==b.resolve() or a.resolve() in b.resolve().parents or b.resolve() in a.resolve().parents:raise ValueError('Source/result/local paths must be disjoint')
    cp=ns['COMPACT_FINAL_INPUT_LOCK_V26']['source_checkpoint'];hashes=ns['evaluation_hashes'];images=ns['evaluation_images']
    binding={'version':PAPER_SWEEP_VERSION_V27,'code_sha256':code_sha256,'protocol':protocol,
        'source_checkpoint_sha256':cp['sha256'],'model_state_sha256':cp['model_state_sha256'],
        'selection_manifest_sha256':_pilot_json_sha_v12(ns['evaluation_manifest']),'channel':ns['link'].metadata(),
        'images':len(hashes),'metrics':{'primary':'alex v0.1','secondary':'vgg v0.1'},
        'rate_target_bpp':.21875,'no_training':True,'channel_fitted':False,
        'seed_policy':'same image/condition-except-P seed across power; payloadlength and Rx count can change waveform RNG consumption',
        'runtime_not_in_identity':True,'monotonic_metrics_enforced':False}
    root.mkdir(parents=True,exist_ok=True);local.mkdir(parents=True,exist_ok=True)
    branch=_pilot_child_v12(root,'frozen_'+_pilot_json_sha_v12(binding)[:16]);branch.mkdir(exist_ok=True)
    cache=_pilot_child_v12(local,branch.name);cache.mkdir(exist_ok=True)
    caller['PAPER_SWEEP_OUTPUT_DIR']=branch
    _pilot_write_identity_v12(ns,branch,'protocol.json',binding)
    _pilot_write_identity_v12(ns,branch,'evaluation_manifest.json',ns['evaluation_manifest'])
    summary_path=_pilot_child_v12(branch,'paper_sweep_summary.json')
    if summary_path.is_file():
        complete=json.loads(summary_path.read_text(encoding='utf-8'))
        if complete.get('binding')!=binding:raise RuntimeError('Completed sweep protocol differs')
        if _pilot_verify_inventory_v12(branch,complete.get('result_artifacts')):
            print('Verified complete full sweep; no model, codec or metrics loaded again',flush=True);return complete
    started=time.monotonic();runtime=_extension_runtime_v22(ns['device'])
    _pilot_save_json_v12(ns,branch,'last_execution.json',runtime)
    codec,build=ns['create_compact_codec_v26'](_pilot_child_v12(cache,'codec'))
    streams,rate,source_artifacts=_compact_streams_v26(ns,branch,codec,build,hashes,images,binding)
    metrics=ns['create_compact_metrics_v26'](ns['device'])
    payload=_extension_load_checkpoint_file_v22(_pilot_child_v12(source,cp['file']),cp,_pilot_child_v12(cache,'primary_model'))
    if (payload.get('configuration')!=config or payload.get('configuration_sha256')!=cp['configuration_sha256']
            or payload.get('progress')!=source_report['endpoint']['progress'] or _pilot_state_sha_v12(payload['state_dict'])!=cp['model_state_sha256']):raise RuntimeError('Actual source checkpoint identity differs')
    model=ns['FSO_VQVAE']().to(ns['device']).eval().requires_grad_(False)
    model.load_state_dict(payload['state_dict'],strict=True);del payload;gc.collect()
    indices=[];clean=[]
    with torch.inference_mode():
        for image in images.split(1):
            _,_,idx=model.encode(image.to(ns['device']));indices.append(idx.cpu())
            clean.append(model.decode_indices(idx,idx.shape).clamp(-1,1).cpu())
    indices=torch.cat(indices);clean=torch.cat(clean)
    ldpc=ns['SystematicRALDPC'](max_iterations=50,normalization=.8,seed=20261028,llr_clip=30).to(ns['device'])
    link_factory=ns.get('PaperSweepLinkV27',PaperSweepLinkV27)
    links={(n,c):link_factory(ns['link'],n,c) for n,c in ((1,'MRC'),(2,'EGC'),(4,'EGC'),(2,'MRC'),(4,'MRC'))}
    clean_cases=[{'id':'clean_'+m,'method':m,'condition':None,'receivers':1,'combining':'MRC'} for m in ('fso_sc','bpg')]
    work=clean_cases+protocol['cases'];examples={r['case_id'] for r in protocol['fig5_examples']}
    results={};artifacts=[];fresh_seconds=0.;fresh_evals=0;reused=0
    for i,case in enumerate(work):
        report,files,hit=_sweep_case_v27(ns,branch,binding,case,model,metrics,links[(case['receivers'],case['combining'])],codec,ldpc,streams,indices,clean,
            case['id'] in examples or case['condition'] is None)
        results[case['id']]={'case':case,'summary':report['summary'],'report_file':'cases/'+case['id']+'/metrics.json'}
        artifacts.extend(files);reused+=int(hit)
        if not hit:fresh_seconds+=report['computed_seconds'];fresh_evals+=len(hashes)
        eta=(len(work)-i-1)*len(hashes)*fresh_seconds/fresh_evals if fresh_evals else None
        progress={'completed_cases':i+1,'total_cases':len(work),'cache_reused_cases':reused,
            'images_per_case':len(hashes),'image_method_evaluations_completed':(i+1)*len(hashes),
            'estimated_remaining_hours':eta/3600 if eta is not None else None,
            'elapsed_this_execution_hours':(time.monotonic()-started)/3600,'last_case':case}
        _pilot_save_json_v12(ns,branch,'paper_sweep_progress.json',progress)
        print(f"Full sweep {i+1}/{len(work)} cases | reused {reused} | remaining~{eta/3600:.2f}h" if eta is not None else f'Cached {i+1}/{len(work)}',flush=True)
        # Small live curves file: no enormous records JSON is repeatedly copied.
        _pilot_save_json_v12(ns,branch,'paper_sweep_live.json',{'binding':binding,'results':results,'status':'in_progress'})
    if _pilot_state_sha_v12(model.state_dict())!=cp['model_state_sha256']:raise RuntimeError('Inference changed source model')
    archive=_pilot_child_v12(branch,'paper_sweep_examples.zip');buffer=io.BytesIO()
    with zipfile.ZipFile(buffer,'w',compression=zipfile.ZIP_DEFLATED) as z:
        for row in artifacts:
            if row['file'].endswith('.png'):z.write(_pilot_child_v12(branch,row['file']),row['file'])
        z.writestr('protocol.json',json.dumps(binding,sort_keys=True,allow_nan=False))
    _compact_atomic_bytes_v26(branch,archive.name,buffer.getvalue())
    artifacts+=source_artifacts+_pilot_inventory_v12(branch,[archive,_pilot_child_v12(branch,'protocol.json'),_pilot_child_v12(branch,'evaluation_manifest.json')])
    artifacts=list({r['file']:r for r in artifacts}.values())
    summary={'version':PAPER_SWEEP_VERSION_V27,'status':'paper_sweep_completed','binding':binding,'results':results,
        'source_checkpoint':cp,'evaluation_manifest':ns['evaluation_manifest'],'rate_matching':rate['rate_matching'],'codec_build':build,
        'case_count':len(work),'physical_image_method_evaluations':len(protocol['cases'])*len(hashes),
        'clean_image_method_evaluations':2*len(hashes),'training_updates':0,'optimizer_created':False,
        'source_model_unchanged':True,'hardware_figures_reproduced':False,'paper_numerical_reproduction_claimed':False,
        'channel_fitted':False,'monotonic_LPIPS_enforced':False,'examples_archive':archive.name,
        'result_artifacts':sorted(artifacts,key=lambda r:r['file']),'elapsed_this_execution_seconds':time.monotonic()-started,
        'runtime':runtime,'limitations':['Author data/backbone/VPI receiver and LDPC matrix undisclosed',
            'Same test corpus,200 image-disjoint members; count is our choice; one fade per image/condition',
            'Ideal relative branch CSI for diversity; common phase/CFO still estimated; branch correlation undisclosed',
            'Matched mean source rate, not equal coded channel-use budget',
            'Fig8d follows LDPC-enabled legend/discussion despite contradictory caption',
            'Hardware Fig9-12 and trained noSN/noNT controls are not reproduced']}
    _pilot_save_json_v12(ns,branch,summary_path.name,summary)
    del model,metrics,ldpc,codec,indices,clean;gc.collect()
    if torch.cuda.is_available():torch.cuda.empty_cache()
    print('Full sweep completed. Send:',summary_path,archive,sep='\n',flush=True)
    return summary


In [ ]:
import hashlib, json, zipfile
from pathlib import Path
FIG6D_V27_PARENT_BYTES = 3791018
FIG6D_V27_PARENT_SHA = "da91da3c71f34117087523e46a0bd5c58476325543a79e83b2869920d21acfc5"


def _fig6d_require_v30(check, message):
    if not check:
        raise ValueError(message)

def fig6d_parent_v30(drive_root, checkpoint):
    root = Path(drive_root)/'stage2_paper_sweep_v27'
    _fig6d_require_v30(root.is_dir() and not root.is_symlink(), 'Restore original stage2_paper_sweep_v27 folder')
    matches = []
    for folder in root.iterdir():
        if folder.is_symlink() or not folder.is_dir() or not folder.name.startswith('frozen_'):
            continue
        candidate = _pilot_child_v12(folder,'paper_sweep_summary.json')
        if (candidate.is_file() and candidate.stat().st_size == FIG6D_V27_PARENT_BYTES
                and _pilot_file_sha_v12(candidate) == FIG6D_V27_PARENT_SHA):
            matches.append((folder,candidate))
    _fig6d_require_v30(len(matches)==1, 'Need exactly one original v27 result directory matching the pinned summary')
    folder, path = matches[0]
    parent = json.loads(path.read_text(encoding='utf-8'))
    _fig6d_require_v30(parent['status']=='paper_sweep_completed'
        and parent['binding']['source_checkpoint_sha256']==checkpoint['sha256']
        and parent['binding']['model_state_sha256']==checkpoint['model_state_sha256']
        and parent['binding']['channel']['profile']=='coherent_receiver_extension_v6',
        'Parent checkpoint, physical profile or completed status differs')
    return folder, parent

def _fig6d_source_bpg_v30(folder, pin, hashes, parent_hashes):
    _fig6d_require_v30(_pilot_verify_inventory_v12(folder,[pin]), 'Original v27 BPG archive missing or modified')
    path = _pilot_child_v12(folder,pin['file'])
    with zipfile.ZipFile(path) as archive:
        expected_names = {'metadata.json'} | {f'{h}.bpg' for h in parent_hashes}
        _fig6d_require_v30(set(archive.namelist())==expected_names and len(archive.namelist())==len(expected_names)
                           and archive.testzip() is None,'Unexpected files or CRC failure in pinned v27 source archive')
        metadata = json.loads(archive.read('metadata.json'))
        streams = {h:archive.read(h+'.bpg') for h in hashes}
    for h, raw in streams.items():
        row = metadata['entries'][h]
        _fig6d_require_v30(len(raw)==row['actual_bytes'] and hashlib.sha256(raw).hexdigest()==row['sha256'],
                          'Original native BPG stream hash differs')
    return streams, {h:metadata['entries'][h] for h in hashes}


In [ ]:
import hashlib, json
SOURCE_FACTOR_VERSION_V28 = "frozen_source_codec_rate_chroma_controls_v28_1"


def _source_json_sha_v28(value):
    return hashlib.sha256(json.dumps(value,sort_keys=True,separators=(',',':'),allow_nan=False).encode()).hexdigest()

def _source_hashes_v28(hashes):
    if (not isinstance(hashes,list) or not hashes or len(set(hashes))!=len(hashes)
            or any(not isinstance(h,str) or len(h)!=64 or any(c not in '0123456789abcdef' for c in h) for h in hashes)):
        raise ValueError('Use ordered distinct lowercase file SHA256 identities')

def select_source_diagnostic_members_v28(all_hashes,count=32,seed=20261108):
    """Result-blind diagnostic subset of an already-used image pool, NOT holdout."""
    _source_hashes_v28(all_hashes)
    if type(count) is not int or not 1<=count<=len(all_hashes) or type(seed) is not int or seed<0:
        raise ValueError('Invalid explicit diagnostic subset size or seed')
    ranked=sorted(all_hashes,key=lambda digest:hashlib.sha256(f'factor-source-v28:{seed}:{digest}'.encode()).digest())
    selected=ranked[:count];positions={h:i for i,h in enumerate(all_hashes)}
    manifest={'version':SOURCE_FACTOR_VERSION_V28,'selection':'SHA256 rank without pixels or quality metrics',
        'selected_file_sha256':selected,'source_membership_sha256':_source_json_sha_v28(all_hashes),
        'source_pool_count':len(all_hashes),'count':count,'seed':seed,
        'already_used_v26_v27_pool':True,'untouched_holdout_claimed':False,'selection_used_quality':False}
    manifest['manifest_sha256']=_source_json_sha_v28(manifest)
    return {'selected_hashes':selected,'positions':[positions[h] for h in selected],'manifest':manifest}


In [ ]:
def create_fig6d_codec_v30(local, first_original_stream):
    encoder,decoder,build = install_official_bpg(local)
    codec = BPGCodec(encoder,decoder,expected_hw=(256,256),timeout=30,speed=4)
    # Decode exact v27 bytes, no QP search, no encoder round-trip changing source.
    smoke = codec.decode(first_original_stream)
    if not smoke.ok or np.asarray(smoke.image).shape != (256,256,3):
        raise RuntimeError("The original clean v27 BPG stream did not decode; codec setup failure, not a channel result")
    return codec,build


In [ ]:
FIG8B_PAPER_REFERENCE_V35 = {'conditions': {'modulation': '16QAM', 'zenith_deg': 0, 'pointing_urad': 19, 'scattered_power_ratio': 0.95, 'bpg_ldpc': True}, 'series': {'bpg_ldpc_2rx_egc': [{'power_dbm': 28, 'lpips': 0.765}, {'power_dbm': 29, 'lpips': 0.616}, {'power_dbm': 30, 'lpips': 0.175}, {'power_dbm': 31, 'lpips': 0.175}, {'power_dbm': 32, 'lpips': 0.175}, {'power_dbm': 33, 'lpips': 0.175}, {'power_dbm': 34, 'lpips': 0.175}, {'power_dbm': 35, 'lpips': 0.175}, {'power_dbm': 36, 'lpips': 0.175}], 'bpg_ldpc_4rx_egc': [{'power_dbm': 28, 'lpips': 0.713}, {'power_dbm': 29, 'lpips': 0.175}, {'power_dbm': 30, 'lpips': 0.175}, {'power_dbm': 31, 'lpips': 0.175}, {'power_dbm': 32, 'lpips': 0.175}, {'power_dbm': 33, 'lpips': 0.175}, {'power_dbm': 34, 'lpips': 0.175}, {'power_dbm': 35, 'lpips': 0.175}, {'power_dbm': 36, 'lpips': 0.175}], 'bpg_ldpc_2rx_mrc': [{'power_dbm': 28, 'lpips': 0.673}, {'power_dbm': 29, 'lpips': 0.175}, {'power_dbm': 30, 'lpips': 0.175}, {'power_dbm': 31, 'lpips': 0.175}, {'power_dbm': 32, 'lpips': 0.175}, {'power_dbm': 33, 'lpips': 0.175}, {'power_dbm': 34, 'lpips': 0.175}, {'power_dbm': 35, 'lpips': 0.175}, {'power_dbm': 36, 'lpips': 0.175}], 'bpg_ldpc_4rx_mrc': [{'power_dbm': 28, 'lpips': 0.335}, {'power_dbm': 29, 'lpips': 0.175}, {'power_dbm': 30, 'lpips': 0.175}, {'power_dbm': 31, 'lpips': 0.175}, {'power_dbm': 32, 'lpips': 0.175}, {'power_dbm': 33, 'lpips': 0.175}, {'power_dbm': 34, 'lpips': 0.175}, {'power_dbm': 35, 'lpips': 0.175}, {'power_dbm': 36, 'lpips': 0.175}], 'fso_sc_1rx': [{'power_dbm': 28, 'lpips': 0.362}, {'power_dbm': 29, 'lpips': 0.263}, {'power_dbm': 30, 'lpips': 0.197}, {'power_dbm': 31, 'lpips': 0.134}, {'power_dbm': 32, 'lpips': 0.125}, {'power_dbm': 33, 'lpips': 0.103}, {'power_dbm': 34, 'lpips': 0.095}, {'power_dbm': 35, 'lpips': 0.094}, {'power_dbm': 36, 'lpips': 0.089}], 'bpg_ldpc_1rx': [{'power_dbm': 28, 'lpips': 0.772}, {'power_dbm': 29, 'lpips': 0.713}, {'power_dbm': 30, 'lpips': 0.335}, {'power_dbm': 31, 'lpips': 0.175}, {'power_dbm': 32, 'lpips': 0.175}, {'power_dbm': 33, 'lpips': 0.175}, {'power_dbm': 34, 'lpips': 0.175}, {'power_dbm': 35, 'lpips': 0.175}, {'power_dbm': 36, 'lpips': 0.175}]}}


In [ ]:
"""Predeclared Fig. 8(b) extension; only launch power changes from v27."""

from contextlib import contextmanager
from copy import deepcopy
import json



FIG8B_POWERS_V35 = (28, 26, 24, 22, 20, 18)
FIG8B_BRANCHES_V35 = (("fso_sc", 1), ("bpg_ldpc", 1), ("bpg_ldpc", 2), ("bpg_ldpc", 4))
FIG8B_FIXED_V35 = {
    "modulation": "16QAM", "weather": "clear", "zenith_angle_deg": 0.,
    "pointing_error_urad": 19., "scattered_power_ratio": .95,
    "background_light_radiance_w_m2_sr_um": 1e-4,
}


def _fig8b_case_v35(anchor, power):
    if power == 28:
        return deepcopy(anchor)
    # Keep the exact v27 case/condition ID recipes and seed convention.
    kwargs = {**anchor["condition"]["channel_kwargs"], "launch_power_dbm": float(power)}
    condition = {**anchor["condition"], "id": _sweep_hash_v27(kwargs)[:16],
                 "label": str(kwargs), "channel_kwargs": kwargs}
    entry = {k: anchor[k] for k in ("method", "receivers", "combining")}
    entry["condition"] = condition
    return {"id": _sweep_hash_v27(entry)[:20], **entry}


def fig8b_protocol_v35(parent):
    """Lock the overlap to original Fig. 8(b), not a manually guessed geometry."""
    original = parent["binding"]["protocol"]
    if original != paper_sweep_protocol_v27():
        raise ValueError("Pinned v27 paper grid differs from the inherited v27 implementation")
    panel, = [p for p in original["panels"] if (p["figure"], p["panel"]) == ("fig8", "b")]
    anchors = []
    for method, receivers in FIG8B_BRANCHES_V35:
        series, = [s for s in panel["series"] if (s["method"], s["receivers"], s["combining"]) ==
                   (method, receivers, "MRC")]
        point, = [p for p in series["points"] if p["x"] == 28]
        case = parent["results"][point["case_id"]]["case"]
        want = {**FIG8B_FIXED_V35, "launch_power_dbm": 28.}
        if (case["id"] != point["case_id"] or case["condition"]["channel_kwargs"] != want
                or (case["method"], case["receivers"], case["combining"]) != (method, receivers, "MRC")):
            raise ValueError("Unexpected 28 dBm original condition/receiver identity")
        anchors.append(case)
    if any(case["condition"] != anchors[0]["condition"] for case in anchors):
        raise ValueError("The original four 28 dBm cases must share one physical condition")
    cases = [_fig8b_case_v35(anchor, power) for power in FIG8B_POWERS_V35 for anchor in anchors]
    if len({c["id"] for c in cases}) != 24:
        raise ValueError("Invalid Fig. 8(b) extension: repeated case IDs")
    return {"version": "fig8b_lowpower_protocol_v35_1", "paper_figure": "8(b)", "images": 32,
            "powers_dbm": list(sorted(FIG8B_POWERS_V35)), "paper_power_dbm": list(range(28, 37)),
            "fixed_channel": dict(FIG8B_FIXED_V35), "cases": cases,
            "clean_cases": [{"id": "clean_" + m, "method": m, "condition": None, "receivers": 1,
                             "combining": "MRC"} for m in ("fso_sc", "bpg")],
            "new_below_paper_range_image_evaluations": 640, "overlap_image_evaluations": 128,
            "clean_image_evaluations": 64, "channel_realizations_per_image": 1,
            "same_model_and_native_bpg_as_v27": True, "channel_or_curve_fitting": False,
            "paired_unit": "image; v27 seeds across power, not identical branch observations across receiver counts",
            "diversity_assumption": "v27 identical per-branch aperture, ideal relative CSI, independent fades/Rx noise; common carrier recovery retained"}


@contextmanager
def register_fig8b_grid_v35(protocol, channel_namespace):
    """Temporarily admit exactly the new powers through the unchanged v27 evaluator."""
    allowed = _SWEEP_ALLOWED_CONDITIONS_V27
    added = set()
    seen = set()
    for case in protocol["cases"]:
        cond = case["condition"]
        kwargs = cond["channel_kwargs"]
        power = kwargs["launch_power_dbm"]
        scheme = (case["method"], case["receivers"])
        key = (power, scheme)
        if (power not in FIG8B_POWERS_V35 or scheme not in FIG8B_BRANCHES_V35
                or case["combining"] != "MRC" or key in seen
                or kwargs != {**FIG8B_FIXED_V35, "launch_power_dbm": float(power)}
                or cond["id"] != _sweep_hash_v27(kwargs)[:16]
                or case["id"] != _sweep_hash_v27({k: v for k, v in case.items() if k != "id"})[:20]):
            raise ValueError("Only the predeclared Fig. 8(b) power/Rx grid is allowed")
        seen.add(key)
        token = json.dumps(cond, sort_keys=True)
        if power == 28 and token not in allowed:
            raise ValueError("The overlap must be a valid original v27 condition")
        if token not in allowed:
            added.add(token)
    if len(seen) != 24:
        raise ValueError("Incomplete Fig. 8(b) extension grid")
    base_guard = channel_namespace["_rebuild_condition_v12"]
    lower_kwargs = [{**FIG8B_FIXED_V35, "launch_power_dbm": float(power)}
                    for power in FIG8B_POWERS_V35 if power < 28]

    def guard_for_lowpower(condition):
        if isinstance(condition, dict) and condition in lower_kwargs:
            # The frozen FSO-SC delegate validates 28–36 dBm. Validate the exact
            # shared 28 dBm anchor through that guard, then restore the actual
            # lower power before the unchanged transmitter/physical/DSP code.
            checked = base_guard({**condition, "launch_power_dbm": 28.})
            return {**checked, "launch_power_dbm": float(condition["launch_power_dbm"])}
        return base_guard(condition)

    allowed.update(added)
    channel_namespace["_rebuild_condition_v12"] = guard_for_lowpower
    try:
        yield
    finally:
        channel_namespace["_rebuild_condition_v12"] = base_guard
        allowed.difference_update(added)


In [ ]:
"""Report only measured Fig. 8(b) low-power results, with image-paired uncertainty."""

import csv
import json
import math
import re
from pathlib import Path

import numpy as np


FIG8B_V35_REPORT_VERSION = 'fig8b_low_power_image_paired_v35_1'
FIG8B_V35_POWERS = (18, 20, 22, 24, 26, 28)
FIG8B_V35_PAPER_POWERS = tuple(range(28, 37))
FIG8B_V35_IMAGES = 32
FIG8B_V35_BOOTSTRAP_SEED = 20261135
FIG8B_V35_CHANNEL = dict(modulation='16QAM', zenith_angle_deg=0., pointing_error_urad=19.,
    scattered_power_ratio=.95, background_light_radiance_w_m2_sr_um=1e-4, weather='clear')
FIG8B_V35_SERIES = {'fso_sc_1rx': ('fso_sc', 1), 'bpg_ldpc_1rx': ('bpg_ldpc', 1),
    'bpg_ldpc_2rx_mrc': ('bpg_ldpc', 2), 'bpg_ldpc_4rx_mrc': ('bpg_ldpc', 4)}
_FIG8B_V35_STYLE = {'fso_sc_1rx': ('FSO-SC 1 Rx', '#235781', 'o'),
    'bpg_ldpc_1rx': ('BPG+LDPC 1 Rx', '#943d33', 's'),
    'bpg_ldpc_2rx_mrc': ('BPG+LDPC 2 Rx', '#7b5722', '^'),
    'bpg_ldpc_4rx_mrc': ('BPG+LDPC 4 Rx', '#256d57', 'D')}
_FIG8B_V35_QUALITY = ('lpips_alex', 'lpips_vgg', 'rendered_lpips_alex', 'rendered_lpips_vgg', 'mse')
_FIG8B_V35_CONTRASTS = {'four_rx_minus_fso': 'fso_sc_1rx', 'four_rx_minus_one_rx': 'bpg_ldpc_1rx'}


def _fig8b_report_require_v35(ok, message):
    if not ok:
        raise ValueError(message)


def _fig8b_finite_v35(value):
    return type(value) in (float, int) and math.isfinite(value)


def _fig8b_paper_v35(reference):
    expected = dict(modulation='16QAM', zenith_deg=0, pointing_urad=19,
                    scattered_power_ratio=.95, bpg_ldpc=True)
    _fig8b_report_require_v35(isinstance(reference, dict) and reference.get('conditions') == expected
        and isinstance(reference.get('series'), dict), 'Missing the digitized Fig. 8(b) panel and geometry')
    keep = {}
    for series in FIG8B_V35_SERIES:
        points = reference['series'].get(series)
        _fig8b_report_require_v35(isinstance(points, list) and all(isinstance(p, dict) for p in points)
            and tuple(p.get('power_dbm') for p in points) == FIG8B_V35_PAPER_POWERS
            and all(_fig8b_finite_v35(p.get('lpips')) and p['lpips'] >= 0 for p in points),
            'Fig. 8(b) original markers must span only 28–36 dBm: '+series)
        keep[series] = [{'power_dbm': int(p['power_dbm']), 'lpips': float(p['lpips'])} for p in points]
    return {'figure': '8(b)', 'doi': '10.1109/TCOMM.2025.3562356', 'conditions': dict(expected),
            'series': keep, 'note': 'Approximate paper digitization, not author raw measurements.'}


def _fig8b_summary_metric_v35(values, resamples):
    values = np.asarray(values, dtype=np.float64)
    return {'mean': float(values.mean()),
            'ci95': np.quantile(values[resamples].mean(axis=1), (.025, .975)).tolist()}


def _fig8b_rate_v35(rows, numerator, denominator, resamples):
    errors = np.asarray([r[numerator] for r in rows], dtype=np.int64)
    totals = np.asarray([r[denominator] for r in rows], dtype=np.int64)
    return {'rate': float(errors.sum()/totals.sum()), 'errors': int(errors.sum()), 'bits': int(totals.sum()),
            'ci95': np.quantile(errors[resamples].sum(axis=1)/totals[resamples].sum(axis=1), (.025,.975)).tolist()}


def _fig8b_grid_v35(reports):
    _fig8b_report_require_v35(isinstance(reports, list) and len(reports) == 26,
                              'Expected 24 physical + 2 clean v27 reports')
    slots, clean, ids = {}, {}, set()
    for packet in reports:
        case = packet['case']; method = case['method']; receivers = case['receivers']; cid = case['id']
        _fig8b_report_require_v35(type(receivers) is int and case['combining'] == 'MRC'
            and isinstance(cid, str) and bool(cid) and cid not in ids, 'Invalid or duplicate case identity')
        ids.add(cid)
        condition = case['condition']
        physical = condition is not None
        if physical:
            kwargs = condition['channel_kwargs']; power = kwargs.get('launch_power_dbm')
            _fig8b_report_require_v35(_fig8b_finite_v35(power) and power in FIG8B_V35_POWERS
                and kwargs == dict(FIG8B_V35_CHANNEL, launch_power_dbm=float(power)),
                'Unexpected Fig. 8(b) channel or launch power')
            series = [label for label, key in FIG8B_V35_SERIES.items() if key == (method, receivers)]
            _fig8b_report_require_v35(len(series) == 1 and (int(power), series[0]) not in slots,
                                      'Unexpected or duplicated physical Fig. 8(b) method/Rx')
            output, key = slots, (int(power), series[0])
        else:
            _fig8b_report_require_v35(method in ('fso_sc', 'bpg') and receivers == 1 and method not in clean,
                                      'Expected one no-channel FSO-SC case and one no-channel BPG case')
            output, key = clean, method
        rows = packet['records']; ordered = []
        _fig8b_report_require_v35(packet['images'] == FIG8B_V35_IMAGES and isinstance(rows, list)
            and len(rows) == FIG8B_V35_IMAGES, 'Missing frames: all 32 images must remain in each report')
        for row in rows:
            digest = row.get('image_sha256')
            _fig8b_report_require_v35(isinstance(digest, str) and bool(re.fullmatch('[a-f0-9]{64}', digest))
                and row.get('case_id') == cid and row.get('condition') == condition and row.get('method') == method
                and row.get('receivers') == receivers and row.get('combining') == 'MRC', 'Frame does not match its case')
            _fig8b_report_require_v35(all(_fig8b_finite_v35(row.get(k)) and row[k] >= 0 for k in _FIG8B_V35_QUALITY),
                                      'Non-finite or negative quality score')
            failed = row.get('decode_failed')
            _fig8b_report_require_v35(type(failed) is bool and (not failed or (physical and method == 'bpg_ldpc')),
                                      'Decode failure is reserved for physical BPG+LDPC')
            for metric in ('lpips_alex', 'lpips_vgg'):
                _fig8b_report_require_v35(row[metric] == (1. if failed else row['rendered_'+metric]),
                                          'Failure-marker and actual-render LPIPS must stay separate')
            counter_pairs = [('bit_errors', 'valid_bits')]
            if physical:
                counter_pairs.append(('pre_fec_bit_errors', 'pre_fec_valid_bits'))
                _fig8b_report_require_v35(_fig8b_finite_v35(row.get('actual_symbol_esn0_db'))
                    and type(row.get('seed')) is int and 0 <= row['seed'] < 2**63,
                    'Physical frames require a finite actual Es/N0 and channel seed')
            for a,b in counter_pairs:
                _fig8b_report_require_v35(type(row.get(a)) is int and type(row.get(b)) is int
                    and row[b] > 0 and 0 <= row[a] <= row[b], 'Invalid bit-error or payload counters')
            ordered.append(digest)
        _fig8b_report_require_v35(len(set(ordered)) == len(ordered), 'An image appears twice in a case')
        output[key] = rows
    expected = {(p, k) for p in FIG8B_V35_POWERS for k in FIG8B_V35_SERIES}
    _fig8b_report_require_v35(set(slots) == expected and set(clean) == {'fso_sc', 'bpg'}, 'Incomplete report grid')
    hashes = [r['image_sha256'] for r in clean['fso_sc']]
    _fig8b_report_require_v35(all([r['image_sha256'] for r in rows] == hashes
        for rows in (*clean.values(), *slots.values())), 'All reports must contain the same ordered image identities')
    for series in FIG8B_V35_SERIES:
        # Rx or payload length changes can alter the random draws; do not require cross-series waveform identity.
        first = [(r['seed'], r['valid_bits'], r['pre_fec_valid_bits']) for r in slots[FIG8B_V35_POWERS[0],series]]
        _fig8b_report_require_v35(all([(r['seed'], r['valid_bits'], r['pre_fec_valid_bits'])
            for r in slots[power,series]] == first for power in FIG8B_V35_POWERS),
            'Seed or payload length changed within a series across power')
    return slots, clean


def summarize_fig8b_v35(reports, paper_reference, *, bootstrap_samples=2000):
    """Pair by image; all confidence intervals are pointwise, not a global crossover test."""
    _fig8b_report_require_v35(type(bootstrap_samples) is int and bootstrap_samples >= 2, 'Invalid image bootstrap count')
    reference = _fig8b_paper_v35(paper_reference)
    slots, clean = _fig8b_grid_v35(reports)
    n = FIG8B_V35_IMAGES
    resamples = np.random.default_rng(FIG8B_V35_BOOTSTRAP_SEED).integers(0,n,(bootstrap_samples,n))
    quality = lambda rows, key: np.asarray([r[key] for r in rows], dtype=np.float64)
    clean_refs = {method: {metric: _fig8b_summary_metric_v35(quality(rows, metric),resamples)
                          for metric in _FIG8B_V35_QUALITY} for method, rows in clean.items()}
    points, paired = [], []
    for power in FIG8B_V35_POWERS:
        for series, (method, receivers) in FIG8B_V35_SERIES.items():
            rows = slots[power, series]
            values = {metric: _fig8b_summary_metric_v35(quality(rows,metric),resamples) for metric in
                      (*_FIG8B_V35_QUALITY,'actual_symbol_esn0_db')}
            p = {'launch_power_dbm': power, 'series': series, 'method': method, 'receivers': receivers,
                 'combining': 'MRC', 'in_published_power_range': power == 28, 'images': n,
                 'paper_lpips_approx_same_power': reference['series'][series][0]['lpips'] if power==28 else None}
            for key, summary in values.items():
                p[key],p[key+'_ci95']=summary['mean'],summary['ci95']
            for key, a, b in (('pre_fec_ber','pre_fec_bit_errors','pre_fec_valid_bits'),
                              ('payload_ber','bit_errors','valid_bits')):
                ratio=_fig8b_rate_v35(rows,a,b,resamples)
                p[key],p[key+'_ci95']=ratio['rate'],ratio['ci95']
                p[a],p[b]=ratio['errors'],ratio['bits']
            p['payload_error_frames']=sum(r['bit_errors'] > 0 for r in rows)
            p['payload_error_frame_rate']=p['payload_error_frames']/n
            p['native_decode_failures']=(sum(r['decode_failed'] for r in rows) if method=='bpg_ldpc' else None)
            if p['native_decode_failures'] is not None:
                failure=_fig8b_summary_metric_v35(quality(rows,'decode_failed'),resamples)
                p['native_decode_failure_rate'],p['native_decode_failure_rate_ci95']=failure['mean'],failure['ci95']
            else:
                p['native_decode_failure_rate']=p['native_decode_failure_rate_ci95']=None
            points.append(p)
        four = slots[power, 'bpg_ldpc_4rx_mrc']
        row = {'launch_power_dbm': power, 'images': n}
        for contrast, other in _FIG8B_V35_CONTRASTS.items():
            a = slots[power, other]
            row[contrast] = {key: _fig8b_summary_metric_v35(quality(four,key)-quality(a,key),resamples)
                             for key in ('lpips_alex','rendered_lpips_alex')}
        one = slots[power,'bpg_ldpc_1rx']
        row['four_rx_minus_one_rx']['native_decode_failure_rate'] = _fig8b_summary_metric_v35(
            quality(four,'decode_failed')-quality(one,'decode_failed'),resamples)
        row['one_rx_failed_four_rx_decoded_images'] = sum(a['decode_failed'] and not b['decode_failed']
                                                        for a,b in zip(one,four))
        row['four_rx_failed_one_rx_decoded_images'] = sum(b['decode_failed'] and not a['decode_failed']
                                                        for a,b in zip(one,four))
        paired.append(row)
    descriptive = {}
    for contrast in _FIG8B_V35_CONTRASTS:
        descriptive[contrast] = {}
        for metric in ('lpips_alex','rendered_lpips_alex'):
            xy = [(r['launch_power_dbm'], r[contrast][metric]) for r in paired]
            descriptive[contrast][metric] = {
                'observed_four_rx_lower_at_dbm': [p for p, d in xy if d['mean'] < 0],
                'pointwise_ci_entirely_below_zero_at_dbm': [p for p, d in xy if d['ci95'][1] < 0],
                'observed_sign_change_brackets_dbm': [[a,b] for (a,x),(b,y) in zip(xy,xy[1:])
                                                     if x['mean']*y['mean'] < 0],
                'exact_tie_means_at_dbm': [p for p,d in xy if d['mean'] == 0]}
    result = {'version':FIG8B_V35_REPORT_VERSION, 'images':n, 'physical_cases':len(slots),
        'physical_image_method_evaluations':len(slots)*n,'clean_image_method_evaluations':len(clean)*n,
        'power_grid_dbm':list(FIG8B_V35_POWERS),'published_overlap_dbm':[28],
        'paper_reference':reference,'points':points,'paired_points':paired,'clean_references':clean_refs,
        'descriptive_contrasts':descriptive,'bootstrap_samples':bootstrap_samples,
        'bootstrap_seed':FIG8B_V35_BOOTSTRAP_SEED,'training_updates':0,'channel_or_curve_fitting':False,
        'paper_numerical_reproduction_claimed':False,
        'limitations':[
            'Below 28 dBm is a diagnostic extension, not published operating points. No interpolation or curve fit.',
            'Matched images and common seed policy, not identical payload waveforms or identical draws across Rx counts.',
            'Pointwise 95% image-bootstrap CIs; one realization per image/condition, not simultaneous bands or new holdout.',
            'v27 per-branch aperture and ideal relative branch CSI. This is not fixed-total-aperture or a hardware claim.',
            'Matched mean source bitrate, not matched coded transmission rate. A native BPG failure uses marker 1; rendered gray LPIPS is separate.'
        ]}
    json.dumps(result,allow_nan=False)
    return result


def _fig8b_plot_axis_v35(ax, *, paper=False):
    ax.set_xlabel('Launch power (dBm)')
    ax.set_xticks(list(FIG8B_V35_POWERS)+(list(FIG8B_V35_PAPER_POWERS[2::2]) if paper else []))
    ax.set_xlim(17.5,36.5 if paper else 28.5)
    ax.axvspan(17.5,28,alpha=.2,color='#dbe0e5',zorder=-2)
    ax.axvline(28,color='#82909b',lw=.8,ls=':')
    ax.grid(alpha=.22,linewidth=.6)
    ax.spines[['top','right']].set_visible(False)


def _fig8b_plot_series_v35(ax, rows, field, color, label, marker='o', ls='-', interval=True):
    rows = [r for r in rows if r[field] is not None]
    if not rows:
        return
    powers = [r['launch_power_dbm'] for r in rows]
    ax.plot(powers,[r[field] for r in rows],color=color,label=label,marker=marker,ms=4,lw=1.5,ls=ls)
    if interval:
        bounds=np.asarray([r[field+'_ci95'] for r in rows])
        ax.fill_between(powers,bounds[:,0],bounds[:,1],color=color,alpha=.11,lw=0)


def plot_fig8b_v35(result, output_dir):
    """Save two measured-point plots plus a flat CSV; return paths for the results ZIP."""
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt

    out=Path(output_dir);out.mkdir(parents=True,exist_ok=True)
    qfig,qgrid=plt.subplots(2,2,figsize=(12.4,8.1),layout='constrained')
    tfig,tgrid=plt.subplots(2,2,figsize=(12.4,8.1),layout='constrained')
    qaxes=list(qgrid.ravel()); taxes=list(tgrid.ravel())
    transport=('pre_fec_ber','payload_ber','native_decode_failure_rate','actual_symbol_esn0_db')
    for name,(label,color,marker) in _FIG8B_V35_STYLE.items():
        points=[r for r in result['points'] if r['series']==name]
        _fig8b_plot_series_v35(qaxes[0],points,'lpips_alex',color,label,marker)
        _fig8b_plot_series_v35(qaxes[1],points,'rendered_lpips_alex',color,label,marker)
        ref=result['paper_reference']['series'][name]
        qaxes[0].plot([r['power_dbm'] for r in ref],[r['lpips'] for r in ref],
                      color=color,ls='--',marker=marker,mfc='none',ms=4,lw=1.05,alpha=.72)
        for ax, field in zip(taxes,transport):
            _fig8b_plot_series_v35(ax,points,field,color,label,marker)
    qaxes[0].plot([],[],c='#5d6267',ls='--',marker='o',mfc='none',ms=4,label='Paper digitized (approx.)')
    qaxes[0].legend(loc='best',fontsize=8,ncols=2)
    for ax in qaxes[:2]:
        for method,c in (('fso_sc','#235781'),('bpg','#943d33')):
            ax.axhline(result['clean_references'][method]['lpips_alex']['mean'],ls=':',lw=.9,color=c,alpha=.6)
        ax.set_ylabel('Alex LPIPS (lower is better)')
    qaxes[0].set_title('Marked LPIPS; dotted horizontal lines: clean')
    qaxes[1].set_title('Rendered LPIPS including gray fallback')
    for ax,(contrast, other) in zip(qaxes[2:],_FIG8B_V35_CONTRASTS.items()):
        for metric,color,marker,ls,label in (('lpips_alex','#303c56','o','-','Failure marker 1'),
                ('rendered_lpips_alex','#778090','D','--','Rendered gray fallback')):
            pairs=[{'launch_power_dbm':r['launch_power_dbm'],metric:r[contrast][metric]['mean'],
                    metric+'_ci95':r[contrast][metric]['ci95']} for r in result['paired_points']]
            _fig8b_plot_series_v35(ax,pairs,metric,color,label,marker,ls=ls)
        ax.axhline(0,c='#717171',lw=.9,ls=':')
        ax.set_title('BPG+LDPC 4 Rx minus '+_FIG8B_V35_STYLE[other][0]);ax.set_ylabel('Paired Δ Alex LPIPS')
        ax.legend(loc='best',fontsize=8)
    qfig.suptitle('Fig. 8(b) low-power extension | v35: 16QAM / clear / MRC\n'
                 'Gray: below paper range · bands: pointwise image bootstrap · no axis shift or curve fit',fontsize=13)
    for i,ax in enumerate(qaxes):
        _fig8b_plot_axis_v35(ax,paper=(i==0))
    for ax, title in zip(taxes, ('Pre-FEC BER (FSO-SC is uncoded)', 'Post-FEC payload BER (FSO-SC is uncoded)',
                               'Native BPG decode failure rate', 'Mean actual symbol Es/N0 (dB)')):
        ax.set_title(title);_fig8b_plot_axis_v35(ax)
    for ax in taxes[:2]:
        ax.set_yscale('symlog',linthresh=1e-5)
        ax.set_ylim(bottom=0)
    taxes[2].set_ylim(-.025,1.025);taxes[0].legend(loc='best',fontsize=8)
    tfig.suptitle('Transport diagnostics | zero errors means observed zero, not an error-rate bound',fontsize=13)
    quality_path, transport_path, csv_path = [out/name for name in
        ('fig8b_low_power_quality.png','fig8b_low_power_transport.png','fig8b_low_power_points.csv')]
    qfig.savefig(quality_path,dpi=155);tfig.savefig(transport_path,dpi=155);plt.close(qfig);plt.close(tfig)
    fields=('launch_power_dbm','series','images','lpips_alex','rendered_lpips_alex','actual_symbol_esn0_db',
            'pre_fec_ber','payload_ber','payload_error_frames','native_decode_failures',
            'native_decode_failure_rate','paper_lpips_approx_same_power')
    with csv_path.open('w',encoding='utf-8',newline='') as f:
        writer=csv.DictWriter(f,fieldnames=fields,extrasaction='ignore')
        writer.writeheader();writer.writerows(result['points'])
    return [quality_path,transport_path,csv_path]


In [ ]:
"""Resume and package the narrow Fig. 8(b) extension with the unchanged v27 evaluator."""
import gc
import hashlib
import io
import json
from pathlib import Path
import platform
import re
import time
import zipfile

import numpy as np
import torch



FIG8B_RUN_VERSION_V35 = 'frozen_fig8b_low_launch_power_v35_1'
FIG8B_ROOT_NAME_V35 = 'stage2_fig8b_lowpower_v35'
FIG8B_ARCHIVE_V35 = 'fig8b_lowpower_v35_results.zip'
_FIG8B_NATIVE_DECODE_FAILURES_V35 = frozenset(('invalid_stream', 'decoder_exit', 'invalid_output'))


def _fig8b_run_require_v35(check, message):
    if not check:
        raise ValueError(message)


def _fig8b_case_power_v35(case):
    condition = case['condition']
    return None if condition is None else condition['channel_kwargs']['launch_power_dbm']


def _fig8b_paths_v35(ns, parent_folder):
    drive = Path(ns['DRIVE_DATA_DIR']).resolve()
    root, local, source = (Path(ns[k]) for k in ('PILOT_ROOT', 'LOCAL_CACHE', 'SOURCE_V25_ROOT'))
    _fig8b_run_require_v35(not any(p.is_symlink() for p in (root, local, source)),
                           'Linked v25/Colab result directories are not allowed')
    root, local, source, parent = (p.resolve() for p in (root, local, source, parent_folder))
    _fig8b_run_require_v35(root == drive / FIG8B_ROOT_NAME_V35 and drive not in local.parents,
                           'Keep the v35 output on its own Drive path and staging outside Drive')
    for a,b in ((root,local), (root,source), (root,parent), (local,source), (local,parent)):
        _fig8b_run_require_v35(a!=b and a not in b.parents and b not in a.parents,
                               'Source, results and local staging paths must be disjoint')
    return root, local


def _fig8b_cached_case_v35(branch, binding, case):
    """Read the same completion identity that the unmodified v27 journal writes."""
    root = 'cases/' + case['id']
    marker = _pilot_child_v12(branch,root+'/completed.json')
    if not marker.is_file():
        return None
    saved = json.loads(marker.read_text(encoding='utf-8'))
    identity = {'binding_sha256':_pilot_json_sha_v12(binding),'case':case,
                'chunk_size':PAPER_SWEEP_CHUNK_V27,'example':False}
    _fig8b_run_require_v35(saved.get('identity') == identity,'Saved v35 case belongs to another experiment')
    if not _pilot_verify_inventory_v12(branch,saved.get('artifacts')):
        return None
    _fig8b_run_require_v35([r['file'] for r in saved['artifacts']]==[root+'/metrics.json'],
                           'Completed v35 case inventory does not match its expected metrics')
    return json.loads(_pilot_child_v12(branch,root+'/metrics.json').read_text(encoding='utf-8'))


def _fig8b_checked_report_v35(report, case, hashes):
    _fig8b_run_require_v35(report['case']==case and report['images']==len(hashes)
                           and [r['image_sha256'] for r in report['records']]==hashes,
                           'Case identity or 32-image pairing differs')
    for row in report['records']:
        _fig8b_run_require_v35(row['case_id']==case['id'] and row['method']==case['method']
            and row['receivers']==case['receivers'] and row['combining']==case['combining']
            and row['condition']==case['condition'], 'A record belongs to a different v27 link case')
        _fig8b_check_frame_v35(row, case)
    return report


def _fig8b_check_frame_v35(row, case):
    if case['method']=='bpg':
        _fig8b_run_require_v35(not row['decode_failed'],
                               'Clean archived BPG failed; codec setup error, not an optical result')
    if row['decode_failed']:
        _fig8b_run_require_v35(case['method']=='bpg_ldpc' and row['decode_status'] in _FIG8B_NATIVE_DECODE_FAILURES_V35,
                               f"BPG infrastructure error ({row['decode_status']}); frame not committed")


def _fig8b_evaluate_frame_v35(model,image,digest,metrics,link,case,payload,codec,ldpc,**kwargs):
    packet=evaluate_paper_sweep_image_v27(model,image,digest,metrics,link,case,payload,codec,ldpc,**kwargs)
    _fig8b_check_frame_v35(packet['record'],case)
    return packet


def _fig8b_runtime_v35(device):
    # Same committed environment required only if a partially completed sweep still needs evaluations.
    return {**_extension_runtime_v22(device), 'python':platform.python_version(), 'numpy':str(np.__version__),
            'cudnn':torch.backends.cudnn.version(), 'matmul_precision':torch.get_float32_matmul_precision()}


def _fig8b_anchor_context_v35(parent, protocol):
    """Previously released 200-image means are not paired with the new 32-image measurements."""
    keys = ('lpips_alex','rendered_lpips_alex','pre_fec_ber','payload_ber','decode_failures',
            'decode_failure_rate','actual_symbol_esn0_db_mean','unique_images')
    anchors=[]
    for case in protocol['cases']:
        if _fig8b_case_power_v35(case)!=28:
            continue
        source = parent['results'][case['id']]
        _fig8b_run_require_v35(source['case']==case and source['summary']['unique_images']==200,
                               'Original 28 dBm anchor is not the completed v27 case')
        anchors.append({'case_id':case['id'],'method':case['method'],'receivers':case['receivers'],
                        **{key:source['summary'][key] for key in keys}})
    _fig8b_run_require_v35(len(anchors)==4,'The original panel must contribute four 28 dBm anchors')
    return anchors


def _fig8b_publish_v35(branch, result):
    summary = _compact_atomic_bytes_v26(branch,'fig8b_lowpower_v35_summary.json',
        (json.dumps(result,allow_nan=False,ensure_ascii=False,indent=2)+'\n').encode('utf-8'))
    plots = [Path(path) for path in plot_fig8b_v35(result,branch)]
    _fig8b_run_require_v35(plots and all(p.suffix in ('.png','.csv') for p in plots),
                           'Expected only analytical graph and CSV artifacts')
    files = _pilot_inventory_v12(branch,[summary,*plots])
    _fig8b_run_require_v35(len({item['file'] for item in files})==len(files),'Repeated artifact name')
    buffer = io.BytesIO()
    with zipfile.ZipFile(buffer,'w',compression=zipfile.ZIP_DEFLATED) as z:
        for item in files:
            z.writestr(item['file'],_pilot_child_v12(branch,item['file']).read_bytes())
        z.writestr('manifest.json',json.dumps({'version':FIG8B_RUN_VERSION_V35,'files':files},indent=2))
    with zipfile.ZipFile(buffer) as z:
        _fig8b_run_require_v35(z.testzip() is None and all(
            hashlib.sha256(z.read(r['file'])).hexdigest()==r['sha256'] for r in files),
            'Output ZIP integrity check failed')
    return _compact_atomic_bytes_v26(branch,FIG8B_ARCHIVE_V35,buffer.getvalue())


def run_fig8b_v35(ns, *, code_sha256, paper_reference):
    """Evaluate fixed power/Rx cases with no optimizer, retuning, or image export."""
    started=time.monotonic()
    _fig8b_run_require_v35(isinstance(code_sha256,str) and len(code_sha256)==64
                          and all(c in '0123456789abcdef' for c in code_sha256),'Expected embedded v35 source hash')
    cp=ns['COMPACT_FINAL_INPUT_LOCK_V26']['source_checkpoint']
    folder,parent=fig6d_parent_v30(ns['DRIVE_DATA_DIR'],cp)
    protocol=fig8b_protocol_v35(parent)
    _fig8b_run_require_v35(len(protocol['cases'])==24 and len(protocol['clean_cases'])==2,'Unexpected v35 scope')
    work=protocol['cases']+protocol['clean_cases']
    _fig8b_run_require_v35(len({c['id'] for c in work})==26,'Duplicate work item')
    anchor_context=_fig8b_anchor_context_v35(parent,protocol)
    parent_hashes=parent['evaluation_manifest']['selected_file_sha256']
    _fig8b_run_require_v35(len(parent_hashes)==200 and parent_hashes==ns['COMPACT_FINAL_DATA_LOCK_V26']['selected_file_sha256'],
                           'Archived v27 image membership differs from the input lock')
    subset=select_source_diagnostic_members_v28(parent_hashes,count=32)
    hashes=subset['selected_hashes']
    source_pin,=(r for r in parent['result_artifacts'] if r['file']=='sources/selected_native_bpg.zip')
    root,local=_fig8b_paths_v35(ns,folder)
    run_label=ns.get('FIG8B_RUN_LABEL_V35','run1')
    _fig8b_run_require_v35(isinstance(run_label,str) and re.fullmatch('[A-Za-z0-9_-]{1,32}',run_label),
                           'Use a short run label such as run1 or run2')
    binding={'version':FIG8B_RUN_VERSION_V35,'code_sha256':code_sha256,'protocol':protocol,'run_label':run_label,
             'paper_reference_sha256':_pilot_json_sha_v12(paper_reference),
             'source_checkpoint_sha256':cp['sha256'],'model_state_sha256':cp['model_state_sha256'],
             'parent_v27_sha256':FIG6D_V27_PARENT_SHA,'native_bpg':source_pin,'membership':subset['manifest'],
             'channel':parent['binding']['channel'],'training_updates':0}
    branch=_pilot_child_v12(root,'run_'+_pilot_json_sha_v12(binding)[:16]);branch.mkdir(parents=True,exist_ok=True)
    _pilot_write_identity_v12(ns,branch,'protocol.json',binding)
    cached=[_fig8b_cached_case_v35(branch,binding,case) for case in work]
    reports=[None if old is None else _fig8b_checked_report_v35(old,case,hashes) for old,case in zip(cached,work)]
    reused=sum(report is not None for report in reports)
    print(f'Fig. 8(b) low power | 6 powers × 4 schemes × 32 images + 2 clean cases | cached {reused}/26',flush=True)
    runtime_path=_pilot_child_v12(branch,'runtime.json')
    codec_path=_pilot_child_v12(branch,'codec.json')
    if reused!=len(work):
        runtime=_fig8b_runtime_v35(ns['device'])
        if runtime_path.exists():
            _fig8b_run_require_v35(json.loads(runtime_path.read_text(encoding='utf-8'))==runtime,
                'Incomplete run uses another GPU/runtime. Resume in the recorded environment, or set FIG8B_RUN_LABEL_V35 to run2 in the setup cell for a separate run.')
        # Verify the small original archive before loading the large checkpoint or installing the codec.
        streams,_=_fig6d_source_bpg_v30(folder,source_pin,hashes,parent_hashes)
        ns['prepare_fig8b_inputs_v35']()
        source,source_report,config=_compact_source_v26(ns)
        _fig8b_run_require_v35(ns['evaluation_hashes']==parent_hashes and ns['link'].metadata()==binding['channel'],
                               'Pinned v27 data or physical channel/DSP differs')
        images=ns['evaluation_images'][subset['positions']]
        work_ns=dict(ns,evaluation_images=images,evaluation_hashes=hashes,
                     evaluate_paper_sweep_image_v27=_fig8b_evaluate_frame_v35)
        codec,build=ns['create_fig6d_codec_v30'](_pilot_child_v12(local,'codec'),streams[hashes[0]])
        codec_id={key:build[key] for key in ('release','archive_sha256','decoder_binary_sha256')}
        _pilot_write_identity_v12(ns,branch,'codec.json',codec_id)
        _pilot_write_identity_v12(ns,branch,'runtime.json',runtime)
        metrics=ns['create_compact_metrics_v26'](ns['device'])
        print('Loading original v25 wide1375 inference weights; no optimizer or training',flush=True)
        payload=_extension_load_checkpoint_file_v22(_pilot_child_v12(source,cp['file']),cp,
                                                     _pilot_child_v12(local,'primary_model'))
        _fig8b_run_require_v35(payload.get('configuration')==config
            and payload.get('configuration_sha256')==cp['configuration_sha256']
            and payload.get('progress')==source_report['endpoint']['progress']
            and _pilot_state_sha_v12(payload['state_dict'])==cp['model_state_sha256'],'Source checkpoint does not match its lock')
        model=ns['FSO_VQVAE']().to(ns['device']).eval().requires_grad_(False)
        model.load_state_dict(payload['state_dict'],strict=True);del payload;gc.collect()
        indices=[];clean=[]
        with torch.inference_mode():
            for image in images.split(1):
                _,_,idx=model.encode(image.to(ns['device']));indices.append(idx.cpu())
                clean.append(model.decode_indices(idx,idx.shape).clamp(-1,1).cpu())
        indices=torch.cat(indices);clean=torch.cat(clean)
        ldpc=ns['SystematicRALDPC'](max_iterations=50,normalization=.8,seed=20261028,llr_clip=30).to(ns['device'])
        links={count:PaperSweepLinkV27(ns['link'],count,'MRC') for count in (1,2,4)}
        with register_fig8b_grid_v35(protocol, ns['link'].ns):
            for i,case in enumerate(work):
                if reports[i] is not None:
                    continue
                report,_,_=_sweep_case_v27(work_ns,branch,binding,case,model,metrics,links[case['receivers']],
                                           codec,ldpc,streams,indices,clean,False)
                reports[i]=_fig8b_checked_report_v35(report,case,hashes)
                done=sum(item is not None for item in reports)
                power=_fig8b_case_power_v35(case)
                label='clean' if power is None else f'{power:g} dBm'
                print(f'{done}/26 | {label} | {case["method"]} {case["receivers"]}Rx committed',flush=True)
        _fig8b_run_require_v35(_pilot_state_sha_v12(model.state_dict())==cp['model_state_sha256'],
                               'Inference unexpectedly changed the model state')
        del model,metrics,ldpc,codec,indices,clean,links;gc.collect()
        if torch.cuda.is_available():torch.cuda.empty_cache()
    else:
        print('All case hashes verified; no dataset, checkpoint, or codec reopened',flush=True)
    result=summarize_fig8b_v35(reports,paper_reference)
    result.update(binding=binding,status='completed',training_updates=0,optimizer_created=False,
        physical_image_method_evaluations=768,clean_image_method_evaluations=64,
        parent_v27_28dbm_200_image_context=anchor_context,
        parent_context_is_not_paired_to_32_image_run=True,
        raw_case_records=[{'case':r['case'],'records':r['records']} for r in reports],
        runtime=json.loads(runtime_path.read_text(encoding='utf-8')),
        bpg_codec=json.loads(codec_path.read_text(encoding='utf-8')),
        elapsed_this_execution_seconds=time.monotonic()-started)
    archive=_fig8b_publish_v35(branch,result)
    print('Send:',archive,flush=True)
    return result,archive


In [ ]:
def prepare_fig8b_inputs_v35():
    global pin, SOURCE_V25_DIR, source_summary, source_common, source_configuration
    global evaluation_images, evaluation_hashes, evaluation_manifest, link
    pin=COMPACT_FINAL_INPUT_LOCK_V26
    if SOURCE_V25_ROOT.is_symlink() or not SOURCE_V25_ROOT.is_dir():
        raise RuntimeError("Restore the original v25 result folder; no model fallback")
    candidates=[]
    for folder in SOURCE_V25_ROOT.iterdir():
        if folder.is_symlink() or not folder.is_dir() or not folder.name.startswith("extension_"):continue
        summary_path=_pilot_child_v12(folder,pin["source_summary_file"])
        if (summary_path.is_file() and summary_path.stat().st_size==pin["source_summary_bytes"]
                and _pilot_file_sha_v12(summary_path)==pin["source_summary_file_sha256"]):
            candidates.append(folder)
    if len(candidates)!=1:
        raise RuntimeError("Need exactly one original v25 folder matching the reviewed summary SHA; restore it and avoid duplicate copies")
    SOURCE_V25_DIR=candidates[0]
    source_summary=json.loads(_pilot_child_v12(SOURCE_V25_DIR,pin["source_summary_file"]).read_text(encoding="utf-8"))
    if SOURCE_V25_DIR.name!="extension_"+_pilot_json_sha_v12(source_summary["common_configuration"])[:16]:
        raise RuntimeError("The pinned v25 directory identity differs")
    source_common=json.loads(_pilot_child_v12(SOURCE_V25_DIR,pin["source_checkpoint"]["configuration_file"]).read_text(encoding="utf-8"))
    source_configuration=source_common["configuration"]
    if (source_common!=source_summary["common_configuration"]
            or _pilot_json_sha_v12(source_configuration)!=pin["source_checkpoint"]["configuration_sha256"]
            or source_configuration["semantic_model"]!=SEMANTIC_MODEL_CONFIG
            or _pilot_json_sha_v12(source_configuration["data"])!=pin["data_manifest_json_sha256"]
            or source_configuration["channel"]["profile"]!=pin["profile"]):
        raise RuntimeError("Keep the exact primary model, data preprocessing and channel")
    evaluation_images,evaluation_hashes,evaluation_manifest=prepare_compact_final_data_v26(
        DRIVE_DATA_DIR,LOCAL_CACHE/"test_zip",COMPACT_FINAL_DATA_LOCK_V26,source_configuration)
    link=RebuildIndexLinkV12(globals(),pin["profile"],base_seed=source_configuration["channel"]["base_seed"])
    if link.metadata()!=source_configuration["channel"]:
        raise RuntimeError("The frozen physical channel or receiver changed")
    if len(evaluation_images)!=200:raise RuntimeError("Frozen final membership count differs")
    print("Fixed primary:",SOURCE_V25_DIR,"| updates1375; no checkpoint reselection")


In [ ]:
FIG8B_CODE_SHA256_V35 = '960b5e7f20fd06d511002d9fb0f8ec52fec7444ada558cb0857322949cd3baef'
FIG8B_RESULT_V35, FIG8B_RESULT_ZIP_V35 = run_fig8b_v35(
    globals(), code_sha256=FIG8B_CODE_SHA256_V35, paper_reference=FIG8B_PAPER_REFERENCE_V35)
print('완료. 보내줄 파일:', FIG8B_RESULT_ZIP_V35)
if AUTO_DOWNLOAD_RESULTS_V35:
    from google.colab import files
    files.download(str(FIG8B_RESULT_ZIP_V35))
